# Kaggriculture: rank your agent against a known ladder

[Kaggriculture](https://www.kaggle.com/competitions/kaggriculture) scores you on
**head-to-head wins**, not on an absolute metric. That makes progress genuinely hard
to read. Your agent banked 40,000 coins — is that good? It depends entirely on who it
played. Beating the built-in `starter` baseline tells you almost nothing, and the
public leaderboard only updates after you have spent a submission.

So I built a fixed ladder to measure against. This notebook:

1. loads **ten documented reference agents** spanning a very wide skill range, plus the
   top-meta agent that beats all ten of them,
2. lets you plug in **your own agent** three different ways,
3. plays a seat-swapped round robin and ranks everyone with **Bradley-Terry** —
   the same method the competition uses for final standings,
4. tells you which rung you landed on,
5. and writes a **submittable `submission.tar.gz`**, so the agent you ranked is
   literally the artifact you submit.

The whole default run takes a couple of minutes. There is a knob at the bottom for a
much heavier evaluation when you want tighter error bars.

> **What you need:** the
> [Kaggriculture Reference Agents](https://www.kaggle.com/datasets/raykkretzschmar/kaggriculture-reference-agents)
> dataset attached (Add Input → Datasets), the output of
> [Findings from Zero to Top Meta](https://www.kaggle.com/code/raykkretzschmar/kaggriculture-findings-from-zero-to-top-meta)
> attached (Add Input → Notebook Output) for the rung above the ladder, and
> **Internet on** so the notebook can install a matching `kaggle-environments`.


---
## 1. Setup

One thing worth being fussy about: **pin the engine version**, and then *verify the pin
actually took*.

This is not a formality. The same game on the same seed can pay out very differently
across releases. Every number in the current reference dataset was measured on
**1.32.7**, and the check below confirms that by replaying a game straight out of
`head_to_head_games.csv` and requiring it to reproduce to the coin.

Worse, checking the version string is not enough. `importlib.metadata.version()` reads
the *newly written* package metadata, while `import kaggle_environments` can still
resolve to an older copy earlier on `sys.path` — so a pip install reports success, the
version check passes, and the engine you are actually running is the old one. I lost
several hours to exactly that, comparing measurements taken on three different engines
without realising it.

So the cell below installs the pin, then **replays a game from the dataset and asserts
the bank matches**. A behavioural fixture cannot be fooled by a stale import.


In [ ]:
import subprocess, sys
from importlib.metadata import PackageNotFoundError, version as pkg_version

ENGINE_VERSION = "1.32.7"   # the version these reference agents were measured on


def installed_engine():
    try:
        return pkg_version("kaggle-environments")
    except PackageNotFoundError:
        return None


def ensure_engine(want=ENGINE_VERSION):
    have = installed_engine()
    if have != want:
        print(f"installing kaggle-environments=={want} (found: {have})")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", f"kaggle-environments=={want}"],
            check=False,
        )
        for mod in [m for m in list(sys.modules) if m.startswith("kaggle_environments")]:
            del sys.modules[mod]
        have = installed_engine()
    return have


actual = ensure_engine()
print("kaggle-environments (reported):", actual)

from kaggle_environments import make
make("kaggriculture", configuration={"episodeSteps": 24})
print("kaggriculture environment loads OK")


In [ ]:
import importlib.util, itertools, json, math, os, shutil, sys, tarfile, time, zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

def find_dataset():
    """Locate the reference-agent dataset.

    Prefers the canonical mount, but falls back to scanning /kaggle/input for the
    manifest so a renamed or forked copy still works.
    """
    root = Path("/kaggle/input")
    # Kaggle has two input layouts depending on the image: the older flat
    # /kaggle/input/SLUG and the newer /kaggle/input/datasets/OWNER/SLUG.
    # Try both, then fall back to searching for the manifest.
    for candidate in (root / "kaggriculture-reference-agents", root / "datasets" / "raykkretzschmar/kaggriculture-reference-agents"):
        if (candidate / "agents_manifest.csv").exists():
            return candidate
    for hit in sorted(root.rglob("agents_manifest.csv")):
        return hit.parent
    raise SystemExit(
        "Reference-agent dataset not attached.\n"
        "Add Input -> Datasets -> search 'Kaggriculture Reference Agents', then re-run."
    )


DATASET_DIR = find_dataset()
print("dataset:", DATASET_DIR)
print("contents:", sorted(p.name for p in DATASET_DIR.iterdir()))


In [ ]:
# Behavioural engine check. The version string can lie -- a pip install writes new
# package metadata while `import kaggle_environments` may still resolve to an older copy
# earlier on sys.path. So replay a real game out of the dataset and require the banks to
# match. Expected values are read from the CSV, so this stays correct if the dataset is
# ever re-measured.
def _fixture_agent(path, name):
    spec = importlib.util.spec_from_file_location(f"fixture_{name}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.agent


def verify_engine_behaviour():
    """Replay one recorded pairing; return (ok, detail)."""
    games = pd.read_csv(DATASET_DIR / "head_to_head_games.csv")
    row = games.iloc[0]
    a = _fixture_agent(DATASET_DIR / f"{row.agent_a}.py", "a")
    b = _fixture_agent(DATASET_DIR / f"{row.agent_b}.py", "b")
    seat = int(row.seat_of_a)
    pair = [a, b] if seat == 0 else [b, a]
    env = make("kaggriculture",
               configuration={"episodeSteps": 720, "seed": int(row.seed)}, debug=False)
    env.run(pair)
    rewards = [step.reward for step in env.steps[-1]]
    got_a, got_b = round(rewards[seat]), round(rewards[1 - seat])
    ok = (got_a == int(row.bank_a)) and (got_b == int(row.bank_b))
    return ok, (f"{row.agent_a} vs {row.agent_b}, seed {row.seed}, seat {seat}: "
                f"expected {int(row.bank_a):,}/{int(row.bank_b):,}, got {got_a:,}/{got_b:,}")


_ok, _detail = verify_engine_behaviour()
print(("PASS  " if _ok else "FAIL  ") + _detail)
if not _ok:
    raise SystemExit(
        "The engine is not behaving like the one the dataset was measured on "
        f"({ENGINE_VERSION}), whatever version string it reports.\n"
        "Every bank, margin and expected_bank below would be measured in a different "
        "game than the one documented. Restart the session after installing "
        f"kaggle-environments=={ENGINE_VERSION}, or re-measure the ladder yourself."
    )


---
## 2. Meet the opponents

Ten agents in two bands, and they isolate different variables.

**Tiers 0–5 — authored.** Written from scratch, all sharing a **byte-identical action
scheduler**; the only difference between them is a `POLICY` dict at the top of each
file. That is deliberate: any gap in results comes from *economic decisions* alone, not
from one agent having better pathfinding than another. Diff two of these and the diff
is the lesson.

**Tiers 6–9 — the shared meta line.** These hold the opposite variable constant. All
four run the *same* production plan — the public meta line that shows up identically
across large groups of unrelated teams in public replays — and differ only in their
**market layer**: what to sell, in what order, and when to hold. Their head-to-head
ordering differs from their standalone bank ordering.

Together: tiers 0–5 teach you how to build a farm, tiers 6–9 show you that once
everyone builds the same farm, selling is the whole game. Expect a large jump between
the two bands — tier 5 banks ~46k, while the meta band banks ~149k–165k.


In [ ]:
manifest = pd.read_csv(DATASET_DIR / "agents_manifest.csv").sort_values("tier")
manifest[["tier", "agent_name", "headline", "expected_bank",
          "hands", "extra_quadrants", "crops", "animals"]].to_string(index=False)


In [ ]:
for _, row in manifest.iterrows():
    print(f"--- tier {row.tier}: {row.agent_name} " + "-" * (52 - len(str(row.agent_name))))
    print("  strategy:", row.strategy)
    print("  lesson  :", row.lesson)
    print()


---
## 3. Why the top tiers do what they do

Before ranking anything, look at this table. It is the single most useful thing I
worked out about this game, and it explains the whole top half of the ladder.

Every product has an independent **glut curve**. Sell into the market and the price
drops — but *how fast* varies enormously. The `units_until_price_floor` column is the
punchline: it is how many units you can sell before that product is worth $1.


In [ ]:
curves = pd.read_csv(DATASET_DIR / "price_curves.csv")
curves.sort_values("base_price", ascending=False).to_string(index=False)


In [ ]:
# Same thing as a picture: revenue you can actually extract per product.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))

sold = [50, 150, 400, 1000]
for _, r in curves.iterrows():
    prices = [r.price_at_50_sold, r.price_at_150_sold,
              r.price_at_400_sold, r.price_at_1000_sold]
    style = "-o" if r.glut_target >= 1.0 else "--s"
    ax1.plot(sold, prices, style, label=f"{r['product']} ({r.glut_shape})", alpha=.85)
ax1.set_xscale("log")
ax1.set_xlabel("net units sold into the market")
ax1.set_ylabel("price per unit ($)")
ax1.set_title("Price decay by product\n(solid = collapses under one field's output)")
ax1.legend(fontsize=7, ncol=2)
ax1.grid(alpha=.3)

# Cumulative revenue if you sold N units, at the price after N units.
for _, r in curves.iterrows():
    rev = [n * p for n, p in zip(sold, [r.price_at_50_sold, r.price_at_150_sold,
                                        r.price_at_400_sold, r.price_at_1000_sold])]
    ax2.plot(sold, rev, "-o", label=r["product"], alpha=.85)
ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_xlabel("net units sold")
ax2.set_ylabel("gross revenue at that price ($)")
ax2.set_title("Where the money actually is")
ax2.legend(fontsize=7, ncol=2)
ax2.grid(alpha=.3)

plt.tight_layout(); plt.show()


Read that chart and the ladder stops looking arbitrary:

- **MELON** grosses ~115 per tile per day, about five times wheat — but its glut curve
  is *quadratic* (`above_target` 3.60), so the market absorbs only ~150 melons before
  the price floors. That is why **Melon Mateo** meters his sales into 12-unit lots and
  holds a price floor, and why buying more land does *not* help him.
- **MILK** and **WOOL** floor almost as fast. **Rancher Rita** still wins with them,
  because livestock earns far more *per action* than crops once `CARE` is running.
- **WHEAT** and **EGG** are logarithmic (`above_target` 0.20) — nearly glut-proof.
  Wheat is why Rita can run a feed chain without wrecking her own margins.

The general lesson: **in this game, deciding what to sell matters more than deciding
what to grow.**


---
## 4. Load the reference agents

Nothing clever here — the reference agents are plain single-file Python modules exposing
`agent(obs)`. Submission archives receive an additional raw-loader check matching the
competition's unusual **last callable wins** rule.


In [ ]:
def load_agent(path, name=None):
    """Import a single-file Kaggriculture agent and return its `agent` callable."""
    path = Path(path)
    name = name or path.stem
    spec = importlib.util.spec_from_file_location(f"kagri_{name}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    if not hasattr(module, "agent"):
        raise AttributeError(f"{path} defines no `agent` function")
    return module.agent


def load_submission_agent(path):
    '''Load exactly as Kaggle's raw-Python runner does: last callable wins.'''
    from kaggle_environments.agent import get_last_callable
    path = Path(path)
    entrypoint = get_last_callable(path.read_text(), path=str(path))
    if getattr(entrypoint, "__name__", None) != "agent":
        raise ValueError(
            f"Kaggle would execute {entrypoint.__name__} instead of agent(). "
            "Make agent the last newly-bound callable (for example, finish with "
            "`kaggle_entrypoint = agent`)."
        )
    return entrypoint


reference = {}
for _, row in manifest.iterrows():
    reference[row.agent_slug] = load_agent(DATASET_DIR / row.file, row.agent_slug)

TIER_OF = dict(zip(manifest.agent_slug, manifest.tier))
NAME_OF = dict(zip(manifest.agent_slug, manifest.agent_name))
print(f"loaded {len(reference)} reference agents:",
      ", ".join(f"{s} (t{TIER_OF[s]})" for s in reference))


### One more opponent: the top of the public meta

The ladder above tops out at Closer Cleo. There is a rung above it that is not in the
dataset, because it is not mine to redistribute: the agent shipped by my other notebook,
[Kaggriculture: Findings from Zero to Top Meta](https://www.kaggle.com/code/raykkretzschmar/kaggriculture-findings-from-zero-to-top-meta),
which is built on a public replay tape and beats **all ten** reference agents 60–0–0 —
Cleo included, by about 13,500 coins.

It is attached here as a *notebook output* (Add Input → Notebook Output) rather than
copied, so what you rank against is byte-for-byte the artifact that notebook published.
If the input is missing the cell below just skips it and the rest of the notebook runs
unchanged.


In [ ]:
def find_top_meta():
    """Locate the top-meta agent from the attached notebook output, if present."""
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for candidate in (root / "kaggriculture-findings-from-zero-to-top-meta" / "main.py",
                      root / "kernels" / "raykkretzschmar/kaggriculture-findings-from-zero-to-top-meta" / "main.py"):
        if candidate.exists():
            return candidate
    # Fall back to any attached main.py that carries the tape the agent is built on.
    for hit in sorted(root.rglob("main.py")):
        try:
            head = hit.read_text(errors="ignore")[:4000]
        except OSError:
            continue
        if "_ACTIONS" in head and "b85decode" in head:
            return hit
    return None


TOP_META_PATH = find_top_meta()
TOP_META_SLUG = "top_meta_host"
if TOP_META_PATH is None:
    top_meta = None
    print("top-meta host not attached -- skipping it. Add Input -> Notebook Output ->")
    print("  raykkretzschmar/kaggriculture-findings-from-zero-to-top-meta")
else:
    top_meta = load_agent(TOP_META_PATH, TOP_META_SLUG)
    TIER_OF[TOP_META_SLUG] = 10
    NAME_OF[TOP_META_SLUG] = "Top Meta Host"
    print(f"top-meta host: {TOP_META_PATH} ({TOP_META_PATH.stat().st_size:,} bytes)")


---
## 5. Two worked examples: an idea I killed, and one that survived

Before you plug in your own agent, here is the harness doing the job it exists for —
on me. I had a new agent, I was fairly confident in it, and it is not in the dataset.
This is why.

### The hypothesis

Straight out of `price_curves.csv`: **EGG never floors.** Its glut curve is logarithmic
with `above_target` 0.20, so 4,700 eggs only move the price from 50 to about 35. MILK is
linear at 1.60 and floors after roughly **76** units; WOOL is quadratic and floors after
**59**. Rancher Rita (tier 5) sells milk and wool. Her ceiling therefore looked like a
*price* problem, not a production problem — so bolting a goose wing onto her working
wheat feed chain should add an uncapped revenue stream to a herd already paid for.

That is a clean, evidence-backed argument. It is also wrong.

### Attempt 1 — add geese to Rita

32 configurations: coop share, flock size, when the coops start, feed float, sell chunk.
**All 32 lost to Rita**, the best by −7,569. But that test moved three things at once —
more mouths on the feed chain, fewer tiles growing wheat, more structures — so it does
not tell you *which* one hurt.

### Attempt 2 — hold everything constant, vary only the mix

Same 16 animals, same 16 structures, same land, same feed load. Only the composition
changes:

| cows | sheep | geese | bank | vs Rita |
| ---: | ---: | ---: | ---: | ---: |
| 10 | 6 | 0 | 52,957 | — (Rita) |
| 12 | 4 | 0 | 54,512 | +1,555 |
| **16** | **0** | **0** | **57,407** | **+4,450** |
| 10 | 0 | 6 | 38,845 | −14,112 |
| 8 | 0 | 8 | 36,638 | −16,319 |
| 0 | 0 | 16 | 10,602 | −42,356 |

Every goose variant loses badly, and an all-goose farm is a catastrophe. Meanwhile
dropping the sheep and running 16 cows looked like a **+4,450** improvement.

### The part that matters

That +4,450 was measured on the same seeds I tuned on. Re-run on **held-out** seeds
(8000–8005, both seats), the all-cow agent **loses to Rita 3–9**, margin −3,627. It beats
every other tier 12–0 and loses to the one that counts.

So there is no new tier. The idea died, and it died specifically because I checked it on
seeds it had not seen. If you take one habit from this notebook, take that one: **tune on
one seed set, decide on another.** Six games on the seeds you tuned with will tell you
whatever you want to hear.

The cell below reproduces the flip on a 3-seed subset so it finishes in about a minute —
expect roughly 4–2 for the candidate on the tuned seeds and 1–5 against it on the
held-out ones. The 12-game run quoted above (3–9) is the same effect measured harder.


### Why eggs lose, and why the price curve misled me

`price_curves.csv` measures a **static** market. Real games are not static: town shops
consume product every four turns, all season, which continuously drains inventory and
holds the price up. What actually decides your realised price is **how many shops demand
your product**, not how steep its glut curve is.

| Product | Shops demanding it | Base price | Shop demand/day |
| :--- | ---: | ---: | ---: |
| WHEAT | 5 | 25 | 30 |
| STRAWBERRY | 4 | 120 | 24 |
| **MILK** | **3** | **160** | **18** |
| EGG | 2 | 50 | 12 |
| CARROT / TOMATO | 2 | 35 / 60 | 12 |
| WOOL | 1 | 200 | 12 |
| **MELON** | **0** | **250** | **0** |

Measured at the end of a 720-turn season, this is what that does:

| Farm | MILK inventory | MILK price | EGG inventory | EGG price |
| :--- | ---: | ---: | ---: | ---: |
| 16 cows | **−148** (scarce) | **266** | −302 | 68 |
| 16 geese | −464 | 347 | **+104** (glutted) | **42** |

Three shops drain milk faster than sixteen cows can supply it, so milk sells **above** its
$160 base for the entire season — the 76-unit "ceiling" never binds. Eggs, on two shops at
a $50 base, do glut and sell at 42. The uncapped product is worth less per action than the
capped one that nobody can keep in stock.

The same table explains the rest of the ladder. **Melon appears in no shop at all** —
only the town centre buys it, a couple of units a day — which is the real reason Melon
Mateo tops out around 44k no matter how much land he buys. And **wool has a single shop**,
which is why deleting the sheep helped at all.

So: `units_until_price_floor` is the wrong column to optimise. Multiply base price by shop
demand and you get much closer to what you can actually bank.


In [ ]:
# Build the candidate, so you can re-run the experiment yourself.
#
# Every shipped agent exposes `act(obs, policy)` -- the scheduler and the policy are
# separate -- so a new agent is a dict, not a new file. This is the cheapest way to
# test a production idea in this game.
import copy

spec = importlib.util.spec_from_file_location("rita_mod", DATASET_DIR / "rancher_rita.py")
rita_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rita_mod)

DROVER = copy.deepcopy(rita_mod.POLICY)          # Rita's plan...
DROVER["build"] = [{"kind": "PASTURE", "target": 16, "share": 0.5,
                    "from_day": 0, "until_day": 20}]
DROVER["animals"] = ["COW"]                       # ...with the sheep leg replaced
DROVER["animal_target"] = {"COW": 16}             #    by four more cows
DROVER["sell_order"] = ["MILK", "WHEAT"]

drover = lambda obs: rita_mod.act(obs, DROVER)
rita = lambda obs: rita_mod.act(obs, rita_mod.POLICY)


def quick_duel(a, b, seeds):
    """Minimal seat-swapped comparison. Self-contained so this section can be read
    and run on its own, before the full harness further down."""
    wins = losses = 0
    margins = []
    for seed in seeds:
        for seat in (0, 1):
            pair = [a, b] if seat == 0 else [b, a]
            env = make("kaggriculture",
                       configuration={"episodeSteps": 720, "seed": seed}, debug=False)
            env.run(pair)
            r = [s.reward for s in env.steps[-1]]
            x, y = r[seat], r[1 - seat]
            if x is None or y is None:
                continue
            margins.append(x - y)
            wins += x > y
            losses += x < y
    return wins, losses, sum(margins) / len(margins)


# Tuning seeds say the all-cow build is better. Held-out seeds disagree.
for label, seed_set in (("tuned-on  (7000-7002)", [7000, 7001, 7002]),
                        ("held-out  (8000-8002)", [8000, 8001, 8002])):
    w, l, margin = quick_duel(drover, rita, seed_set)
    verdict = "candidate wins" if w > l else "Rita wins"
    print(f"{label}:  {w}-{l}  margin {margin:+,.0f}   -> {verdict}")


---
## 5. Default submission artifact: V38 low-pressure opening

This notebook packages **`v38_low_pressure_opening_20260913`** byte-for-byte as its default
`submission.tar.gz` and `main.py`. The evaluation below loads that exact archive.
Run the notebook, then submit the output from the Kaggriculture competition page.

The strategy retains the public V38 multi-route chassis: shop-conditioned route
selection, terminal physical closure, storage guards, adaptive animal/crop branches,
and economic feed/fertilizer overlays. The experiment here changes one bounded
decision: turn zero now buys 5 wheat, buys 10 more, and sells up to 60, replacing
V38's buy-13, buy-30, sell-30 sequence. Attribution and upstream source notes are
preserved in the embedded `main.py`.

Promotion was based on wins rather than bank margin:

- public V38 scored **316-4** on a 32-opponent, five-seed paired-seat confirmation;
- the low-pressure opening beat original V38 and two opening finalists **120-0**
  head-to-head over 20 fresh seeds;
- on the final untouched 32-opponent, ten-seed, paired-seat holdout it scored
  **630-10 (98.4%)**, swept 28 of 32 opponents, and had zero runtime errors;
- its worst per-opponent result was **16-4 (80%)**.

These are deterministic local gates against public and replay-derived agents, not a
promised leaderboard rating. No competition submission is created merely by running
this notebook; it writes a submission-ready archive to the notebook output.

Selected source SHA-256: `8f03b16618586e1d9a55c38d5232ea61729b82cdc5abcf383cca73227d4875b5`.


In [ ]:
import base64, hashlib, io, tarfile, zlib

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
SUBMISSION_B85 = 'c-q{(XM5_%vM~DHzd|^02*!&X78#SX3C7rW0RjmSK>|r61ODx|gCfDpvuB_CyziZ7?-?U?S65e8SMJswk$S7z96lGp2;7VmX@XnOEW?h7Imu6Hkr3$Rie?8i!LY)D;c1HK?}*))-1!~9gfX|U@!U$Jd7C}paM%Q9zL?R3DZU^FW7_6?vYVf7?uY_88u83vH4|4njR*Alhylq`3<ZXKBFY>XXw{!Hq6h{cS0XnjMP@+GW;@tFE{Z(UUx^II{wBmRcqxd&V9c;ndiE$G^GHy%Aoh7$;2r@4p4|E#36U0sl}NJ8BY?8t*8;O6SptkpB*9ob!w{L->XYKuPlSpx0K_wE5{@PW36=wh&;r5JBFV6D`t*$3oX=|QauC1p+&p09_;?Q5CTN}%iOMI<KN59%A=2|c%@YB_X?J=dh&nx^2XM-BdN2lP3iIE@faez7#i~DJ23?K^xB&9d8$f=F1}j6+tVj$vcF6E^gd$isfZK8fmj;5YqMpR7)9Fb30{?bDgNbw0Y5*Jm5}7$%+ZmV%Iii0nG<np_A3~|{%!pyh89{@wxH&Tr5Y-q~q(?j&H>gr%wGc#}Cg%X`fMx|itRXkMOeCjYA(H#2M0y648PURTX;le|+}Qvj2tnj{FbQI`B6%u!x}wBlyb~Y`0Vd$*XfA*^1}hpsBq?$MsV;H=Bd}N-MjR7fH&}SwmJz_36M&glGt350C**+(LL!P1BDtV3anND`sAyL#NQMVnXz}~<Ua<#GDSk>Q&m*RjJMf#RgTA1TU$c^rgg}CwHls=EQJvv{<VTz|PYATOV!?{j1Yj;fU;qi#F@x7PlCo<M2!JwV88}1CmGaEN)+Mh^&;LjadG3>DH7g39$x90U_t$?gszed<)b~AJX<mRRfq6X=d6ESju>*tI-QRQ@i{j$<uV0(Z=7~f@J#qZ#m(0Sy6f7I5TB%ON%Sqz7QcmU?xk|ZCq$@R|Sx-F@wN$lMNj9J1<44pgnX5NyxkM8_QGhs}h$KA(oQ2s*OM+WzhPQxmfWQNy0Ej3N%p9&N<}9!WDclq6ES^{i^dneg4KpafU~3AnuPiPJQUF{%#DZ%F7Gr?HIKT*CwMJvG&$uD@0i+1U4OVj?UM^wB@n`4_xWx`giLpp<8x|}tKn!$d#2u91V~%G&(U=OPOMS#K*!f^+AX8YxWum&`Cuq<kz~3n}z!@ql7ETq-gGSs5Aj!A@f`GOPAPzK<5M{u=MF;@$iaZB40Ljzxj~PP#5zY!eT>-(%gLm^e$11R-rdWI+NO;gEBF*s#m&J--a8MvBG**~EU9(#W&MlgdK$sXa?#ykVNTh&dABYeUpaRg}k5F(8NWccLodO5HM3jJ{hXt|`E+-sYSPjM!azNURX+#9T4`^f(LDy4g!$2Mba7+xq1}&B_W)=|EkQoBHE@*xLftx(`2QwOw17yNWpn_V7f(W%dSP!sdKrYEJ0nk3p0%8mpu&g@x8eH`((B@XRgbDPAfAP0wEf;|QLF%j-Y6ao(gqB|v5cC#c%?J>k1r(WrfHni-KD<HDB3)#_Iv8^RHYwPE^Hcs?@br-8q3lO*hlu`DWJGZk115>Jh`QtqJD9ByS->`hCI(QQKz(7ABydA<1DQ@hLxM%3z*@+=6M<3y;U<r=lo&Fj6;{t+nP&8fpsDmHfI-fuK(agh3@i}PKM*Y-yyvn}0(7Xli2<nu*p`$T4LzAjKZgWKU_v2)kNSx!P);Tcw$*}x8;(N=N)sLd{s40Uk9AYm%Q`UUHJ1GXgoJlDFe(_55O<4H0<<`OdL}$J;32{sNiE2Q>K0*Gc{-X6g2^aNa86P{<g7_%M)qg2oX|+IM<{_IYYs?>#Ym0(kfk9uG8J)%sYXM43ke`mg!&25O2$|LVgl2kw<NN=L06y@z$U<5VcYlu8jZ{ro2_O&J(7~4yQT?98sYYoE8s|%sUcB8frY6dPoPf<a5-7DU6DBr_Xgl8EZN{-X!+ny*^HUNSSL2X@&VJ1K^EJZL5mAH91uy`M+6P1&GGUt07P1N?QT=RK}|^ufPc_hkYGS=hPDjoC|SAZJa8_<RjD@AiHS$zl#CJ;A(u-li3X6g4NuR3*{tqhdO`AtcOZSxWahw(ob8~IK1BrV1O9=W!;*8_EHeQ##rYvQK+5G&r)3rLo*@RQL~}zu7d%5HAlVBSIpB16D4SY0szOX@L(2MC!3KiUtq|gEO0sNJSRAGk>IYg}o5Z7LyNFO<a4hr}Rl+L%B!DPeF^J<3{s={Blc1rH^glCE)Ky0c^IJWmHGH5PM+m@O(5C@Whkzj88ACs_8i-qEUbhM~Y&R%k9CSl|CC>rj{Rr2iPtFi$ZFty|MTX3ZmDYiPdqYotS|u2ymnf*42(5(h_^k@lNI2~{;D0J!KyorO?27^)<WZ~a6vIl`3BY#{^dcW<?G+8x&H!mxsSRcZs5P<WqgZcR@;~atjE*NXbcgf<HuPX62(shD%#mD`4Ad5qiduz6Z)IZWXDo9ln2@jl&K<WBz{W<A6-p6X(AH9oI9gyvEYcN#IpKOC5?|y^D9&zS=!7J+d-&=0^7cNl;g#vjd-Dedl$MxKj?PbpCV-y_S^y3MrYMc1K0sYNl6HII(2_WVLk~DWd#vuEIjz~4s<3iC5gBNz!NH#uQpnaCQD0%jEOGEf_er+}wKALr8kW$Iihx1};ErrVWLp5E6U-LajSE@?G*X!oh{zeWVW7>za;y#Uf&k_Yf7*Zr#gCx7#_h<NxU&s;8vJ5_vAE_2Q0!l@1|%O84k~+jpbyx83&`JR@=z=Ov4;*Q3K)1XBf%C>o&e0Tk`~Yt$vDyai+XoLk=w{1KO6OeF@U6xx?C>Ja#cwvqW^!hRwmGqUWjl50tZKw%`JeWfL$;%vA{FdtR-*>0Jt%^rjgaCB8B{UZa9RNBTyUk3<zudZy<L#Uc}3!$O_2_m+VX=jMVu-Qs5PnM<y4G8FY0y7BB&#H<WP_dV?9s0LtUGx+wyBA~0GyD>9Y^yIT-Qo<Vze$O9QA`%g5Z>P)R@FbQVhh;uA0X;L64fnB5c4XER32X$K5!<IBPm;^Q&^$1FXY~Z5EI`WCg4IwvZJTC!AG|2190*TBBqd|^H_!<dADSTj3)tts|=Q%;JA=1Ig47e4v#qe*y2P82gH^Pb$;e==O2&)Z1Ie7wFo8m-%ekTZ#7Qx61lC!UcG*B06Cr?gZn>n(60U&JD>3LH%y<|sN+Cj22qwYOPRm#>G)+SQ)QB-+{6lCKMhWBN5ROk%gLGP4OMTPr`sL|TE)e}NsPVQ6*cq*%aykuk>=|pOMV>O}$f`%v<ZZ(%eztA!Q|K&6qUGKO{!SBlw?@@KS5NWjRHm7O4?l>ZjEd*J@{nqMf6KTeCUI7jqLA)WF*slRg&49^5;ifgJH9jf)=WNO((kZ9@N>Eb6>d1Y~$T+N|n(EMRgOLOra^fLt0XKzVA^%9J5gigx94ZVCp$<^=p29?ckj}>+XvW7$g-6*k*8~%g7Xt7)ou+PhWmuI_9^o!jwd<qA{f|(LQZ%&19(9HZVqi7lq$Ge)NaO-e=R#O-5n_2OLqP~q95R@Jeu9&Qq{X|E7qv=H_7hGgd`>{h48@$n7n0Kn8Q`{DY2=<$w_uZQMMQtNIi-=HVXqray8|`i;=*1!qcU1O9WdGTCBd4K6!LIYcBC(o7>Y2IQqlpE#2}I)csghzk3XTY4$x(KU!*Z&KmcV%lLB<%^!W#=le(vYsWpHZDhX9m##5!bN~sf06TZbGs8s?w#?$Y8eTa-;hN?`0N_V7c(DM;E{_zsMN!f<fMB*er<N~#ZXL|^lBEa{=3kN{*lnqXGr>qkT<A}gzhb9V50tdJ;#=akvbe9OOS$$-NU}s(lM*@FEb)BKLbBdC*?~r8M>2eK<x*;hF{>?)VOUt7=IPwG}O8sPiJmPf_z>?@^hP*f!Y@sNcrB-v<B-U||EPAl#DKG80<U^#ELn?qA7dIJ-qyaAt+Y|iioFg%TzRSm4B$w*JAfp&1AFwHL5}MO33<M+1NQ*iQ95T=*)>+XP`~+<X@<M711JY!~xZwqi9<>{4h+Ng(6~{*#LsT{yf(QrKMIBWn&hm0PJ+4^xFcOZ8b;wyK!6L;kI`KHpmdAtKD=E8yePz|c6rPAC3$(C+mXO|p;RXzJ!4XtV@<$VM*`3;`nsA#Y%yu=jru8Y{sHY(&Y)1CA(er=i5K6`yLRG^;7zj3vDK+zo|0#1K?55<WkWT7z*cF7^dW76csCW>1!V0Jlfu?Aj?1CFaw?NWh*haxYT|iI82^#@l5^ac`v~!RjNN?$ZP9-2Qtq_K%M<kE4XQ!TyCbQuMf>SnS1Smu`_BzEOIV)n*UX%5Liy$Qsu_2@;XUI8>?<v+1jLp&f8s;aYp8zw7w{c^cnPt2mwQ(`YO_obk5SIXhKmaQQO#-yUz;xi|Fl7%h0u%?f%K%JNS{99y10&35ab<a@$*U-7kPAvfrJ3)ENHPL)*<k8=NVI@$3n;Wxwugdne}`RY6aj#)omNz!bwv)78m)U&7f_N9f{L&SVhjTq=T5S=Jq)?(mfH+tZ$M4l#_L3`eoG|c^;}&gbSu}$R+<f>6|dFe<wh=5Cn_~<!9pcX#LI1>kSiyF;l$u9_!h==1?_woB!4JPcvszBD85U|@@xkjZA3EUgz=XIG9YUs*C?hQ!5WlpxpKOeD`!%rRJrj;lv1_lEWk9L$Q5&qHe#xDu2D|aadAalfl`gvzydal@fuNW)~c0y3TtbeA)UcY37EwK49uV`3d%T1bv!2w1lEz~7CZwT05tI-;5gU_@uVuBHE~`X9Tfy%UBOAK;++wYE)ci@qd4nW4M<sL6i3#kna|y6l^^|2;F(Nb*s;iveP)J=335<(6Tp;WMT8B810Ky#h!$W9JbqmWNY2&)mJ+og43-|vm=UbzGe0W1|3_UQS`(G}>pn1Ha|7llW=tQMhzP+EjE`tCG4ik?tj!QmwZP?lgC(s_;lN;#x_~nV4KBqe&{C6ga-`2k!2WWfgjzBoENED3pv%ECu$oRvjX+}v$D?85Q35N=a~K%}xWkY;kLw4Z6`;}eFmHb9Xd}9=6gjlQPZ(BOSd9eMM=wm@RtLzK!O71!%)}#(qc+S;ALpFH0=dP4gb__>SHRLNVn{MGSYL)J6lTK}s~Sm2d0woRfjJGx(ptLU!D#_7668$i0Gx&k6(~wlfTY$8N(>LB0vE6|Bx1;gAJPEo&Imja@d4B#kRGy>hU3RoMX%W`tub^C^qcuSIr?o{QMPRdV~)d-TNIl;DLg>2VqgglX(V%iNJLnYuv8XJW`RSul9b#bUZm$NEWy!+?J$XF@(c*BKa=8Q$o}~SMH95CaS{p43hpDxn`4BNWEOB_vfKuGUD$zD2#08^0j<swRsXOvU5-bw=%r*23O`Gap~zJQF2WgEOlodOEC1CPmWDB;csBIVm?4(sP#j=sfM_(-(1@akz%jyofYnW12>+A(9LXx#a960Q-Rvu#SF?gr+#ASmnnxwaIQ08?9`o++By&YOtsSI<N?pZS*l74uV~8sFsBE1oC!tQhR9vXEiC3$jS?=vOTvQab0P=Sym33%~BjG!Q(ngmrgT;$K?DQxVndsvhs^QLoUBxd@eK__bAJvG$kfCRkKmfG@L;_Y6eVCo40ngrk{JGV-L@-Vw>B^nVWJn52{-(y8e<Dmtj{P5{hC;WC<pB(40!0mwn=Ak`Wd;khRa_)5M5Fk1xoNnUgdNZUTO|pMf(sZ4AWcBmnE=ypxV04FRV6=ajrj@iAGC(CtA`BOg{;iw#B!fjOM6guSj9$wU2g%3C;$(o_AS&g`mC>1CInFhEXio96C@hSd1obPqy}9`elUiadzC%Z6zazg{PTzSK==cEPx8!vlr9qMP#RB7&s2}vQe46W8z}Xy<{$<ldq7ZVU<D{81Y~!CVI@Zx2~&m7G)9vq0>SlBOpnwB7iAVFMP0G+UrK(9z)vXGQ?NQ(>Gl(E;F>if)$w6KngHN=l^9HkXd7I&_5Y=zFB|w05$iOqLqpyK$XEj01SZIiR;b7xc*Z<?S~;o2aa7+xIG^3q&F$^2zP=q*irbJt*vZxDglPd^Rd>MMfc5Pn1*}S>Cd^MaH*vy(I$0pCNSF<`NmdPoYVhIG`y2sOM?a%&cn3?R>cUO0x4#_%H9_;YzljfS@kl&9J^gtkZpUB>0(_=<D80f5VDLLw+7DE!NY5XMrMQzC9ecf}8x*vZW`HMzw=F_y00Yx3;u-=TXQ6o;v<%=95WOC*m+$qE%-Hbc0y=&-FysK^qEPP)kbweVf?Y+Jkml1I1cBifusy)&kQvtu31AAe2-#2Q^~}F-Zm{{qQe6_LepXxlz96YzA`c8eDAN%Y<hTqNc7~qd%%+rMKvKTXantUM8!-bj0=)rj--UGv46jW@<uWitM#~AY{)`&{1FKpD4Ern5%oUT55~U>A%7FvI3dkl*5&z*4U>>{+PdVsn_u*6ERzR6A(u@BQfH4YlV=Rq+1L4TecG~tT_5=dfg@-5b07D)q`z|2F`2c--2#cCgaT~DaV9pB+CBRb56)ab+r;0^+0AMRlgG#e5m>i4Y_gB=27Q5(>iV7cC-jX5XbJ`Hjh)4iD*aMK8fQ_x2fWU0K6uIka!99%^3p@zVar4t52x3YDxB@I34TXt70Om8sQ0Y2dmrq2f>wuvGr8Zx5At|)T03kKUQ%Ac(VH==fC-^WE8uBVs3ejH#vppj?eu{Vy?hFF@G+>!g5&^#d23PqAOn1frNljrjh`AvW27<x=8-qagvoLsj5nnBzC=fBP+@#nM0~l-weOST-aKhP9xoBXD<W84n*9_pOIZD<#o?Ol+9}vUry6fn=oL%43Vh6S*up#>}riQTmK!QoQ{qwQ-LxhR_?HsVs?iL6j5vEITKX!jUwoo2P^6O*!xKkz92l@#G*3Gx4X>Nf#4~Pr4>vBa>s89wD0Sp9M=!zU*t=l7jfQI}{l;Kt~(es74GZP1(!Yj{(RPq-5xxbwP0R)6%E0szfo-38=p&G9@nzhsc?QZ}R9$>rubqkgTNU>WaDUYX9ULfa-zQr?!3jjdwp?v*Kut!+s2&4y;%X<h-dd9FY2Ey?SjU@K|h-C07Xi02i$a0q*1O*X!L`K#CK|l}A`isEkk$NjMgQuI&nDU765n5pEJRvp{2>a6$;4|pH1J(ih3HsY8VoD?{<rHW)pcRIs7mUDBbRBlIIn78z-k>y3w>w-x71`}d)j!(G=bIbp88mioZlt3}b%3ntwv|oA8*ojZfoV`de>Ezlc%uUUsyAx!Rw7lawc)Q)s#qz*KT??t`kgBl;D1|{N)i4yovJl*#oT+Uc5A+=gN5!^YcPfiX@ZCOO*ri8P`?SMT^;T>;j!z7gstt$xW5U%{df~E=SuNn_qkGUAh?-IrJe%sT&U%9r3L<WqrdCfRH}-gs@(u^Fmb?jy1uuNA8(cJw-RUGYJe!ujyHf^rHLT|0u=4=gc|%e7}##UDOFz4tNm@cQfp*EZxeWDbC@AKV1&s}(1R3!3x7Kv2|N7VO5xu&^x$}tu2ss7Zmn7F<{GIIUb0iVUs@$Iz_Z@y#-AHt(y1zegxL+4MIAl^CLcmjwQb-8>$f`7MsLgj{=G@2((z^yKuk3nxpD?g63LHS)mQ*+8~lpKa^+T|>GdxZ1+9Oqs8#)At@_fxQB|1!xkd@;U#NOS|NN3B(6zWD#1%?D2og?klXt<MW8nxsr#vwrl)3}xC%_|Tn1!Vd9Kf<GFrPHu2l(ln+~V`Pg7zr_+XXf=)ZTETxaH;yi)PhD;t0NZya6i%%xT+y?;m1J&tTf>hN6c=7f{}`CTD;Sz%<yw1Jgqi*yCX+?U7ajXx-kXp$bI74MECe<HHLm+R8xX2Ten4W)pQ_LJq`Va-Nx%j_?RiP@W;jbV$H}sel4w^5_tZG$4)(Kq?XF<r73wiyc<);3*WRVfY6eOons|euKS&W+0CT?QFQbCfw<KApHEoDC&Fd82WZMn3h1uE3=WvXqQG3`Q6!&N^?Ar<mj3jy9J%%c^DFVg>TW{=Lkz9$YT$gzKNAuk&i?lkOc@C$Bmr3?m{<|hh<kvGfmU}BQVabL|;2&QU(q6Q<oc>G#XKYzynt|jCn}&5^8GD<yIj+Y{uY}BV3fTsUX6EMmmC{hZjJ@Ed!;enXrJ31CYMf5Sq2Pr7{t?dr4|S!c&T-1t);X!)+_g24ISg=cufSnwa2WghwDB7Qo~~iHaG`!dGS@L^#hTlO|^ZE%{ipA*3xFg821|aB8Oq`hVE}z(p~kb~d<N@GmnyRSjM{F20s01Ff<yDVmUV_6R#%Vzuofu>&|Tc)nmn^P}>^@yC3MJOYM|%fIvFBas6h=k^MJ%J!kMIfO3DaSId5E{UjC3m9TBj|_}Dgui|@fUtn#8K$mqypG1UHnRT*xM~Xqi$^T<$9g>g7dXwPIRnuVh%O3oc2_JTcCo{6*2*8a(BSbI)^JGlj9UP*ffj&dHyT1>B0fs%BSQm?!Ac?1K!r$wE)?v8o_}zQKc|C1BusYbh;0EMQ(M4Km{}kPpyxaId?!CQYsL$Q39QwxPeJB%xJ9qB{t3ur4H!#jAWeu|Qg5^cO!y$%`V6JqbD?<wf)>DOmty=mrRf0;eu?&Q2Mp161Ha5fgm4fv*j-49sO^Wi3s`|OLlB=0QuZ@uJEUod|L4p%f{l?J!OgH!V}>}M!u@~){yFO)c{gS#?DT@}qFh7lkHbe$oPp)G|DEuE?+)kz@j<ke_TY6M0lN~C1B!@?WI~M~A!IqPrw!8N7=nIbFewmN{Q5gu$%^5eG0Kf|+|lGTz4TO(#L3^d=bUEJpsjcVcsOT+A`%!Bq|kW<j+>dZQ`!LMl?U4ld(P(5t&xq_UXlHOagilDovg70c?ZoeB+G=`tC%Ye+K;e5;UlekcXfnhE^~c3<fXt1tCx(XTNL5z%B~pxC3{`wzb^6FOfT^?-YaNy^TqWUYS!iw31nya%Iu#j#bOG0TpCLK6$kWNj{^_>0$u{&RSt-svkGvbQ~$vP2jm(1Y4`ih4bDh)Um1F%DB0&RO)g*nABP+TmRty94pk_?AlnX?y+H*mMVw*fwpx9vvAK|bmBM*I;dkNj)NU8-$l36b(Aj1S1nQ9(3`d&9M<FjDGN8S6$)Lj_u}9#oqUnEV%ojMu6MPT4qh=p?2qXdUpFoL1QGni>u^`wV^#EY}X{JDM0HaF-MbkMFCY(s1sZG9zE=9I3+sbm->k;%o6FWr<g(877KK)Gqe-Z{4u%`jo6k7JvM`Dc%<be3VWRoC1Vt-W&*!%<)hF~(E4gsYm7~>)g6m`=A(f0+L2X_5HeHp0<F8oA${{);iV5eP6)jE<c8M{a<DP(b#Ryd9)QA;Az6)Is+YV7X^WI|C`$RIP0RP>F+c@Q}%NNazxBS7%KdH~42mS?8zio?1Zdg%LMvoMT!Oa+9Rf<5%1jXs?81g4%;DGQe9Y*;BoBst4}fVKI7cHaUMZJ{opwigZ}U0yuYP7^_flhr6^4G>L~8GC6u5K2T^nzbqvv_LKfuvN6osyG5{CrKiP+Ma71g_A$PYqZ}PWREKZhws#ltpVFb)R18Vwz&X&H8*K@?gjHr+Au8QQFXxodhAC^N=lmGS-JvD)&(^wRVig5g%6yJG(9~%ng0Oq3Z>nZ45UB~X>s>U+OAkTJl#mSCv+fD{^vKbM^Cy8^&5$Is9jJBM$XMD_lLn?(ou!MYhWlb9GysjTI&&4s!JA>EkeknJ>mo_g;m+*YzPDpgh^5b1ehtjH3g_dNw_1A9M&)hD83I%8IkfEk#qV<N=f1jtAZ>{jq|8%7cT5sY}JGT<4i4RWL4siX3723Bn=U~g;F;pDCL8i3?#i>;KY@b-6l+G5)<Yj&GH&w)zEm&r2q0^Y$eO{l78yY(#8l?9g0^T2w+}bY!l3SV+QBplAt6wj7FiRS46F}`eB(3pfNQg=Y5JKt}Xv>l>Id^kYhEtm`_CukeITnJ6)IamQSk%Jg;g}7Wk41)UQ@pPo@)jELl<EGu1^g<cQ_qEraOoEo#^u0{-ojgDDIk!^8)G%J1|~PvFS8BAlaz<DmF9&Ym(9+#XU=?E_ysBOO*_Svh^Z6nEIbLl&X<N0N6~5=n<b{<SDl*$@Ggm{7J1;mBIng64phekdf2|2AtQ0fMOYMG*eLptdVNfL$+$AAq)kh~k>JfAmf|8fdS1|C@lxnp;G@+uxG!h=1}JbC@(dUdpuIqFa{O5pE!5MU*bL)QZ*(0c${DMGk6yJl>pS!j-8jrY)R`hC;eop_laHOcRNk?T>{dmKRzXa7KN!VZ++}jTOTmE#>W)wvfh1YvN6R8h`;0KVYLjX90f=mN1d|MA7sD{-Nn}7OvDc)a1W3ieI}M^s~gu_hha>LRNKqVMXFo|3KvSdaw)B4wCSalDt5ZLG?i|U;t8B4TsDY>U=?sIASWof}HS$O^TQq0o3s_?r<@d^!F7(HTIJ2gqGr@KgMOh08+sIhp+(W66hd@#_W^x$a+RZ#bBCL0uD^u(Em>sQ*N$<_WPm^M(BJj{(Oj#an)ZCr9T&eqN+N~8sZ;BKiM^qZNHN&lhjm>{vku|<IiPupd*gUks)oMLk?Nt7CnOW<d6&KlA7Q!Jwp2T)p`p&C)gNG(i+yea@>dWkJ)^_<8@pjt*}fWrS*WD$<GiWkPUjiQ)YM%p=200Q$x&}EeoDYl(Fk1K2#C}Qar?eL*N66b15jGyQ=WiwvZg~A4LW9X&IcSf=6%E%!TO|-l+kv#}joC48S~K&aj#X-j@NDEWEaEfNmFo40NNl8BtjMLTXI)`mv1{zM7%J4NkouMb$})QKu5sRYyjQY^5lV*9?tPaxlbWx*Bu2QP~5DF|8!wFPqmNdY$)2C#f(LoD$FqrIS^V$C82F{=q6+^HbLy^WqhwPN(_g{*t+<rZ=%yArn-m@wB>5r3cDr>h+40K2im_Ho+$6m!TJ~sgt-MSuhG}8wu_sS~K%SfZ76!s}MDfbRgl>^cwx*7;3G?A0?ci7HV7_6{pu^n+4KS9viNQ#?<+9X>hg9PF@=22})IcLmO18sa~!r8S+YZePhTw5J_azeQR(*qt4~+QgmO04^Ah6RgBsK1&fY<Bj`2aDNRlmOg}f>;yo&Jxx9>;CBqkVo%to#kWqU6dt^?j2FNZqyoj`}^5-hGw5?8X{3~~BDu}MaSKItk=-N7@tKc<_6!=%zRfel6W4a1fZF7RP0{aIOxL6l;b;wJt&!=>0JlsfEhk5$w<2Oc0Q!@4E&n52wcf29=$H`OZKk#D^=Vp|~v;H`9tC2D0xHJ&Z1v)bb_57_Zo&R@!uSOMMb4Vxj^ezc09gu=P4)KdniuC0}1^*-9>U<O(($?&=D|F93@Lo`y^5(qy9oOzZ!5rxmAfw2WQtAIjJ~@SJER>sGq(Tgm-EGKSPz~EN_-;%faRg!l?`>izC^rOPOT$Uxd&kj*a}32u=il*^b$*f($x>`&X!${!sKIPA^gC1*!}v;>BMBrD*f>&smM%h%pMYcn4!Y?>E{{B|lafYP@UbC8P+zmztN|ypG9C>wH~Q=2x7_v5B|g%gllWJBq%{D}8R^^%&~a7{UeB#Lh0(PDe&R>)hB7G$OQN#wM|6UN<u*vWELZf3R<0ao;k`2_CYN$49L!n=QB&MWJ`Tg-OYvZA4_VVnoe%5)9x=0-#>B&CQP4$(D5L<dbc8;zG@8PyB)qecTi{b>_^u*&xDY>NRQ5->z#d};fkSMH$N=MGs0aYY${BdI%8FkzYZ`2hxYUA?t`fpi(ac>p2e2j4X%sD<QcjyB=IBxp0dLefm=cLx4tV0vMv-cyTEc{6r308j?Uj%4P!>E;1E5ZU@Pi2AiQ-t#0&h6!*o|th3;6sZz9wBdK@JyJbL0lG6vd465e!s-0P{%NEheRq@E((P5=<ohbT3LlGRZHHLuP;eLGX#)ipT+$!bB)&fE^3`<3|B0)Jqo~(8Onv5u?l=jQ4}7E3K}~TS<^g!iHf@5KA<Q`J&7-p9EOI8-R|!(FgPARMG#vrtw_g!+RhYc12$p&vdPp@Fc>7N|4ktMTDlNR`Jw4kfk@8KbVES0Tvx$nOSh0<IJSNtrYnLFof|N1FG6PpQ8+G!7WUTwp{qqz`BYvsc;G%X1i86^-VR|EtIFXdOw9j^@f*<BoX0Y_L4D6KeQE-=z*q^Psgzcrpmj3s;K022>1?f3H2SuSJxKr0?gK@lqRK$E?gjqOMGx`ky71rrPfE<fQWe4*F>O2slFL54cInMB>HB739~P>RnFH6bI}JXQ_}AOdZyzJNaFLcyUGE?u);q#7ea%$ksKM6DykIfT`UFt8{4UG?_`OO6dFF}h9&=Rd~{D1`e(4=YF!=?O$Lo;H(!f)o-C+At}fL+l;S(z%uOBb2g{`{8K&qp)ZnlP==CxGx|I<F$XoV03tZ<QWm6!><;?PvGy!~#TWVP6+&aWf9KZEu;mO&<f1Or4iKzU|#i+wU<%<>xawy985=p0Nky$1um*E{Wnv<|n2>@_PG;$V~0>e*RaOmRzu(bFWD)u$OYqsS=HkKx(^W_i$k^*+#YJqQ-#m8)JE*;p_PY!xSitVDOquzo6Ff~!rkCXE}e{Qt#DJeaIMJ{NEQ;slL`(2VmTWh3tm2%v_&w^lDNe8=WqKuq~iqT;42x^IriU1(y|6P0R-`ge9o;|5KKeJPG%Ik+F2;QLbX4JjX6zt0a?WzJ}*RtV+73(Sb`eJq^?}N*T<>EiN#F#}3c|HM{MF%b{pdWnyft&w%)Fpo4LnqEvX$Np&)EyV3rWO`q3&T;)+F;hX@Dho^%a#7R<wuFpF+B3cz++15%lxZa=u=vnsdjF#zuBzJ7j8WpM$LRx4Ixkw!v2k@hx5nf_hGz)<eV4UnN|eidKwN<#u`3vr6mnMk?{SpOUm%{WWYJ%y5kHtKcwB?_~P4x-G3`J`{9!Obvgd?rMSEd^2sOw=~BEzz4hUj=7I)I27>s3W{jb`$Om&;9CMVaE5FHRDB<i&(UTfdZgMrWsw`o?SdfAqr2Jh$at2GMcks3>z($%|DRe3dI<Cr?SW+vA3o|s#;9r-@CNy-Et9qp5$SFJ}@qw}!>dM0)0#qk|bjkn)WAV*ls`Ah$+x6(+4w65Iew+(f)I|9}>XkG*>Z+#Nm3bi)a7XB~+s&7G6k8tDh3$p=!kqh~3iU&V_~(2t$Q|Tlt^xq;A8D9N!=D0%GWlBbJ0ebe(mZEVD^3uRpw%x(fil_7BH^t8=z0@?3|N+@8~qg@3sE+}Pso2E4i8)3e7iR&3S}>pEuq|ag9M2btbc@GC>zmXc+x<#N0eG_iFq{Uj#|t8q0c5GK)=t*P82@G{eE8(Sbya!c`4vyR0MZXK3!jK>qUio>136rY5^mRI9Vmw#mXo3@Tf{Wc_}~eG_*GknJ=E}(p1(ddB#g|;me64O>R#o>Mq@u^BYGzktBN{+)z5%;V~6L#8-c_z<9+Mp7m*QL*wA6YT<4;TtH=3uVFz<t=yCZ)0XHi8B~g;-0cIy(2v*%$5#%kab0|`pa^VR0v+f1rJmLyHnbvw;hlkx(ua(&EQEv3)gja!Ri83|{uCj7s|D8BNpvDb{8?vQgiiFi6-$vkBYpXqLEpWEc*~)1t#^SgN@i4@AjP8<9isPIF4ubCxThSv)OZEx?t%r)GaM9tF2W)u`*hb1=!*9ss0g9IR03cPA3vM|)Rt4iMTZZ*)klAi;NmhTIfx5J{Uamwx6mzYdo5}!*R<+NZI$9NJm%lE*?&?DcD>q7!U<P`L9+?}#u`o5X4dbD!(@&DvqHlu6S}URe>4Zep(m1jruOn&G>2|Ax6Ekoh(R}kYi6g|g&~Q7<o?oxe&_+}y(%QWbl@Kq5+lp>JVp%q%NSsn6rKJ#K~YLY>#yVZ^FtIR|5vJ$`9^!5<-27iiqyabZ>tX|HN9L&BN^fsoIY8cf2elfAH{6H^zzh<*8+rdyU^wLl@S%o6^*skRv)WnC;whIWr14y4RP!BbR{pa9v6Q^OiAKmQ6k^hCde|H8!GVN1w6mt?K{8Jt9O2(dv{dzfc*lX{oqX~G`@lseLDwbNKH62cBA=V55pr(13$utKAI^=`ESHoX1yko??9Ch!)a96D%9nqu&N*uQ1jsYyxjM^vLEVe0cIQ_k|?fB0p%?G3q>Oxb~vvz`k`it;Ob;_EgpJylBfdDwetO29rOQ{R75Zp%zmUj&@B)DScRNi4fWFinG|i{PGb!kc`cLbe@_4c2J9mC44)l++ahI#NVz?!_LPi*bHeH!gv)^=NXD&(jN5ZcZ#8DW`5)?(bA9}%%<`&=`M*<LNbK<hFfpBz>i;}tq7M@3Zr1!a*%N#Lm~_Q2y17x(QzxZ?B)rcZxM!D-kjNKzf|VCUsd5GELip8Wel491MCLI*Nb;e9ck|b4KH=HM!3stOx^fdOzNe>8{$_+@@Su(T$*wt3MpIq%rD%va{0o`WkNVSVpbbl-FNDBGEG-$`x>oqSZ-3Xthk`Y4v`2lBL{c8yTsd|kkqPctbPc@XBb=rpNZF^r9>rhIgY3`Gx~$-!ALLI`E0?k=7lRb&j_b>Thl>J2$!@GJWu!@dgkGt)ovg0ilyf5QfJ}2pGHs=eHlzMdIn;bH%(6JmeM2nZw=k7^AurEQW~UF(9-UE>ek0m}XGlocPRcRQ_)P8%XP`XPf_Nlsx{=HmvgKE=6FF6<2#Vz6z)|IAc)it2L$2bjYzi$Dh`aLQ6coV!Y~;Vq;r#NK%M;P~Xy>yzxdrAYQ#yfnVO}Rdb(7Qf{(gp<qzq!VOZUlLoUz0#x{0c+b}gKz&QQZ_`iZH`rW?zAbMdem`FCScBOkc`L_TmAD^^sCP~PC&KNk?3RzPq+6%g56?HWU3EnpG}8&;xbctf6=n7<aQ(*C2>C?DYesPI<v3=y1y-#bT^hvQrJN;KPDc=_uUbZJx<9oFcot6M`l3X6N-_?6~XS@ZSOrlf$aVM5dX1Ht+8O}UMX&{-M#r71!IeXANGKM>g}R%%Ph;<H@cJ&N>|O+%fazhVwv!&!<z>CwItb^6;Ju6jI{?<IkDE;>o_59fmO#rmmeZaw}R`kJPHU4#D!Z~XZz`1Kx$nQk{-tN|D@JUXz5=&HT;1xy=laTscoj+CB?#p&fMNzZ4EE;q#^S&8Xc(gQct7@wLblkV)$r9jiw!uYs|FAc$$@XOKt9Wcjn;M?a!lQRP`Ifb7y=Yf$=3S*e|9@F^VT5VKGy~ZBHlf7^y#eo7!19)wSpf=Op{cl3_Ra5_<C#Bq<iOZ>kM<peK>eYoY$BXc+3)GhgI1UqG;e@l)8YEXN=q+lB&=A&-fDcAv8c-i?9K7a1sW<%M9!|5um^zH<X#t{%tagG@>5eW?V&qbOxn6!JfAsPv(I`y6uyle<b5FA_qk&t6^vMeR(D1i}#Ti2>rxY)Eg=cK^C`ZarWAlHW3Z=KeufS>iO9_liY0w`gE6z8YJeHD}kRSYm%miY8Nfe-k{s;L4HCF=%&<(CeVerjQ+WQ>NZ*ur1PGtKD0A=0uknq;VK_Y!c;924U>XK_}MimFj#th4X&V~lj)qyCDLVsh~0sLYuyr@PvFW~qK<<XuF{~cusDct_w1qf9sqnz>5o7ReUvdLw7(luKsB6V_j03K15CvdhbgiSM=jkw5JXVvWp;>DU0mCj0uvklE%<{{MgQHtxEFRag#02bAx$>&iNqJuv5kB*{Uf{8war2sqI{ok?7<&dBn{FH{VnQU-#E0B<7emWnuQt6)tEiY(#6=@ug1RDsAcOL7IXmwg-fu5fo{6x1OeQ>7|^O}%6DJO**>N<d{L4EU|za4zVm|3L}E(I|)U+(#LCE%Kq&Hu>uQH!Zn2*0(T3c|-gDH}J0r_~6G2PW=fCBU1^u+fAMz};vnZt*8qV8IfnDffo-109Al4qXvX0V{*&c3=+^DX>rH@)1|u7}lg~Obe-00$OBHty{sq;CcA@9V`6$;~IeaHQ;75u=T;l9|F<xOSygZ7djURWj!=bY&SZtLFu?hoH9Yji{agva*;T^IRJj(9$P;Po*u$`-e8_hvb*p(U6wm7ZqVl89^s@|y;=1W(?>qk_HePF0~$6Lp95F*FUn&f+f&yH;{R7l&E@qvPZP2F3lZf~Z5%LX<*MZKig>&T2qdmPLVw!-i^c8VoohQQaW|g@qc4`O%lFa$G`)r@-A`jZ=NC<XbNl)4OLgU#awk_4eR~bH`DzLEh0Bt^yNvzxu-xTqlE25=aS<65DnJolZKUA#ay}I^%vD&IYQu3T4MioT%<bPx!Y_~XkNpElqF_r_F+aOVq~Y|7{l{Lzm6+qTgylbVFLbD$7C&h&LHn7bp{|K?_)r@sdentl%s<BtQSSSzxsZI+9?em$&U0xrWM=XuHAui+@Ex${z)I3il;SRW!bpJ^v!?We#&(bf8?E+Jc7;vDYWi8)ROD=g2_FP-nDv=iG{|2)51_m_o<T+=v}|e0B>NfYD>-55(oRPFO_;v%zZCDwfvUOx^5wcK4&J#BgXzm?&mU@CA{qJL7d;Az53V~FXZ))ivbbVmW1>nH{eN1U=J;PPEco$>aNV|6j)Ghz%Y`7R#AX)?5x(O=)s?wmK5IWw`JbB4S8CzEzspBwLqEf&%9kKXunfx;{2ntYf6{UV{6`+g72zoaQL5LH)Bq~)e^M`MOOgDH0qr${PX5*=x*Y<3JCuY0HCT~G3S;<DQW#O3(W58$HEf0lZ8TSED5nfy5i+YqCaAsSg@)hWhBvB%*{%TUFhuc7whz@JX><+-Vvjx#DI<n&<rs{ittAyt!&40C%dc>=YmPGrPg>gDobJSHcYIwrvGPw&tNh0yvoX1bk{zL_Q(k8+S!wsH3gr>?zS0s)9rnTjcQ}-yJ<<3!f<0@dJ^!H`8o<c0e@?aNr9<Po10<L8dn(2Lz4|?$+Br@CQWO2%u@=2>(0pn3-}%;>`?r2xz<SNiR)PJNq(h(HaV&NE2@alCFaMTrpw(WKqdzc#9BLA5Xe0@w;#3rK$Dt%>xtidm>%fI~2DD=Vy>V7riN77M9C4HOxHN|OiO5RcEKYr(gB&PRpmS`$ugU*k-?Kj~zgKzeMxU=nV(UkVYB^#2k7BWsNV!~vs2UvK1}_nee^7oXtoX%jCCCCr%G<%?t^u65ljO|LV&{(Y5TOJg7<FiUt{OIxONI?1<CPOo5@cP*1hz&2R*mV&{XVx0mBqFY`gOh_N=(<;5;6-K^pdE#`1$GoK+;^ic26&JPA}iPc5+Mi{p#QFawsTChVT)B1-3p8sp0QqoAK2g16p@(OH$jil850gQd6vc`CHm&XI$~M!@9!o?<>5|7v|%_kvPo(c4$sZr}Ob{k{qdTT?@$!m$~(Hzrt$@)y=QJtMI@5tq$oPV)Xe$m}Ksf@B`DPi{EKD|1iTNVX?>`El@wHm{>9B@UisOjb0DKynYe0xz_^*HHY%s=pX_<JAiIfmeLO-z?7N62Ef#1C>q`C25&w-xj-4}EOMj04-Fr|Q~LwZc)^vsMZkc1DCh!26Jk1i>)!$=1+ZG<1c3oQty=mjeeVP?i}bCN6Y{BFxHzGq<_7t{k?8cU>a+WS<vX}B2JoiE?+E_Gn=NH%y2~w3zYcNXa?3O9aCO2D?z7e?(f{CH>ytuc8GY#^AP|WB==>88=$yKA1uwek+5Au91L_6RCxa>xo6EN!T>PTJ@9z%({{6!Re_vNbVSwh}7ft?N+D|S;zu>*#D>G4N{K2!IEa#s~pMSc!LXN+Dg8cm6P4w#Q%jExzc0c?y1lrV3s(6q;jHCg9ngFTJ%oo6-01`-`y>9u}MBrHY@6Q?2zv))v?P4XK48u21GmfML6MO-XfLG1KqYWku;;HZVz=srV`k#S~XO7K4gLz*Uch8e<sGi6ds=M_hA8Z`MuiSIxy&cSLd}(7oX|8*dpTT0hKkd7Mvr42ZRK`c=&Sr_-zh^45H@2HGy_l%#B#|k7&TTI)%N>2NIRg9TYFe3F@4L-mB2y;!<NJhVHIG+3`(7a$k0nTV!;&+;1baQI(G8jXT<cI+hTmx8*xf#qeZE$2+@j6HgoSD^8ou20{cFcBhKW|w(&dX=kLlpNvoy1vA{85?gNs=I@ZRc)+2d;JY8~vQ+&h~%l(UluHku`!p%BX*;+b+d)f4Ac+n&1*m<H2!>#b?4l^XGQ-?t1c!e&G4f%guY5A(jS$-O>o>EUqoY;L)RnP`p8vA#_-@qWmPuKh-&w$XH}LVp!f;rYAK;9T?l(t9WG@p!mOxJ$F<$^LcbqAi{P8A~udHZe#KJ~KJS<g~RKVL*$#`RFCx?b;$~&VLG>=)E~PD#U!Du)SJt1}lac8#rzqgYhR<YZ&)^vu_&mcs^UPXGh`Hb6@E`OqV7y%oo@~`rxrtmk&k8)-JetKjjsuw~%;bDju%hU4$yp%+6n+n>}{%nzed{2Y$M~TdemZSKq%Ix<374zGZ)l4mL0B`$I6hkJujSrn)2U>a>|<F|x7bSCby=p-A3Pu?mMWbx6}<HB9XrcXrP%y16UG0+VH^&hf$OptFeYs$bPrD<VYh+f@7ISghyB-E`hgcl=a;yH9_PDOb7g6CYmoorBvrzcc4b^;o3gO%>-Izo~DaT-A2Ro80DDlX-Y>wHSBF8%e&hm0>fQudI91p4Th%67|yHxn2pH-wdws$VV~m=H_)X_Z6nQiNRg5-%UR2ubI5f#tAhpo(g#PRII;`zAj?!Uj1Ot&KI4p{e9fWA39!NXF5#S-#Urus_ckW!K{7T=(IYRzPqh4W|yWK<CjCmMB27<SKYhhLeWkjYFJg{wf>vKTunWsjy3aLi`~=Au~WzmwoS5Lv8-LW0aIx0ttJy=6(atfr5N(;NB*%py%VPav#r<_X4Y0a!G=bj(1Xpn;2u7=o`==Cl}z+Z-l5Y{35M9(!AUh*?~SkVd&v{5d7snI&1U1>Qjho}-PFjJ&A9W)eS}<o+I>A&lM%axSEGk@4sDsxyuS|}_M^qL+PCbP)w^T#*5TfTQg77Al?R<(W?+b!OM^jj?gc!UPKeL^DCYHs9@_N^nGtxeSEReq()umj8bpH2&2isYN2_u1ZuM5k^Ukc(Sr;pT`tc<jyEpFN?}m+ZZ5w#@Cmqu_U!MMKe>t27GS;G}<JpkTwZoBYc<=r4<`$tG*G_&#t?R&uOmK9?Jsso^i*(~Id^CSe+@3gBPPQgwu=?qJ3+)?QyR(-R1IL$ODQVpucH#GXSFjMWcXx*5UEh(d9z|2AMw`lOigcE!m9tV31Dh`IcK4k5k$r6JMvuA0hVCBRtgq@fBtPeac5SvOM7kqi%x|6h+*yN<_gk!=-OXp^HB0qp^}9;jYVu81;m^jK=ziTcvxdgQ;hFR<=;PREBO{Ydg`G8WrEw)jn}M<4b{JRPJ5Sk69=)kpr%~z6{YTmt?ymCAcvDPmM+tMUGmfSX^J+I%rasf2U@T!Py@&FR$u8i3n3BmU;4)V_Yt2sfJE3A(Y|5o_Z5npu&Clys!&J2AW`{<gyGYf;#*8gii7eaGn9o#m+T3EE_iZylc0>t43%nTK-e$`;TltvF(8*_pqy7GV)>Pb9-&pU<=JQ~VnvFU8ytEvY67NN0p)vDVTJwjmU2Z7Yd(PslQQBue9d8wKNKfL<l8Ju~`nq=GJXxHMY@IIo8I4cg1LWp@n=gF&Q!&oK@(v(9*?e+Zc{Lt-uMzXH{Bn2KqjR-%z%;LrFX8m4%Fn}An+-f^#e-sD{#A_+M{Y+go7+u}ov1yX5eG}QW9aQ`_Vr|+Fj*oA>m+(#dk=w~!UlwVLa15a#xJAt)S8+EY;6}CO%C#Wrn}9A?`X5>urvFOUGIK7Huk(!-<QfeIdiNUaryT{>%lzW%$|(f<}g-DTi;z~IH@kT#<`e$sqSO5**opuxD0n=A!&Uv8eGdfx2m{O*{XAIN^=9NZ??9Wh6iRk@6l$iHxK1sHl9Ynl(XDfkI#3V%;&N_cRzG%g+aiy_cXqQ`__8&MJ;>A^=_H_?046s^X>W5wR23WFZKujGC4W;UYz#Q*uYM#%}G=6Z^hswkfEE4g=pNxdNFT$&f5LmfU9bsRF9weXUC!!$mYrMW?=;LELh(>f#T*hE=F3%OgWl$Z|8~fq+l=3w%&cu*{&v=H9POF433S5%EsiN#fX>`*KXg&9d+lzg|BVO&=i@>U1E{1L_5Od&ESpk-oQ9fOp$r()^bO^v?KPr<vLf0Cj(t>nO}*+FK2aCX&URVw8zSIijlV8=k!E255;h;2Kv4<LOsv^ty2vS$J-7YpRvLuW#ipPODLClXY-4iv$l&Gs@}fW%~rTIb8O^2&g?SUb&{Tr-Mx1ga<nHz9W48=9@`OIb&r1$boZOR2a~uE8&2n2gwD4Ni|F0<uF+l&l2P(5ycOQ9ZhI_O+C1B8BX^LejbHVLQq`KaHO8&Rboy{y*+~CuK3S!ryWVrz)h7c9--05|>&YtPG}YdX3sb+B<8p!ZCN8>$E0JX#gX&S>tI@5Y;91t!p@YK}bC?JDJ&?ej$%E_gJ`S3fokIUEn5XZ$beVe|4oth^rx+H$q7|!e;0xqvej5`LJAY*{{M@^RfMYkPh=H-EUEF3`o4Gq_9(%%fw#X!V*i?hDaif~yBhzqu@_H|LW7BE)zB`MD-$r@XZr{5{o>6FPt$2H{L$kY-pEj$`T+qEDryef64|V1<L(dqmq~B-FyuVaTng{*KBzfogVq?_n>yhtvs+RURw``T@b&MI7lJ$mvDI8vVtJhxI65|g|=k}}q)!9<V)HpP71sw6tkgfQ;jX2Os4WA)5`|LWc)7;lv%~lVs?poE?c5m5?B@4Mwbhma`N5w;TFt>-zi{iFhGA?{>!BucL-rq-Fd;9*_$;TG)WX<*doZ5Bzdr!pPIfmno`Fu6?S$2m&uF_7MGkZHbFufc?&zp70Y>m48cXQKcsnDismiN`<EnJxUDqox9*bxJ&R5W)Nlc1QUw-IhQpnMLN%NoK|!av+N$IRGkZyhV|QE{B*y|qbU`<$%5)ElW*+ke+>M2%#~ADad{ws3c2vnMi{{#W&@*nD*k9;zviy=o{Y^Hh8nyf1kM$4K2!@7C&bX8Ph`?t8XszSC|m8AE);4?b%zNyBi?i#~>mIlF^t;NJaa2-DAVU&2reW;^1fIls@>eSM*1w2vHi^PMMKwzBEFb@k9o4OZ`ru_1o-_kkeI3U=W=M8-;$cfZ|E39W~$Ei!N{vU&DU>};N+F++Hs*{x}gb2{9H<UD2q=Ea&S4ej2z5d7Mj8qaQi+*<m(N#DD(dW^?H$#Nyt@~_5h>UC;vI`+)hv%9_!=LUNILam!K%N{6jvo-gY&E-=?X8k%~a+OA4nc_0rpv_PVjRJLQl!(m39S2E2tb+H|MW&Vx?XukAeHuuXd~wIblu2;QFBdr(am(3!wfE&p+<+WrBWBMy&%dy9TY1?r)tl5}S39tuLhL29&Mun1ZQU0#_rrtx*sKuTyD1Yjp(caiDiR#dSDAZvFl`zeGTZpN=6=6-rv?kb6|+RX*5%o{?DX>WFToXYya=NEp3mO}mZA5d@VO}tqKQy9;T|uW&del5W+t;qWYVpB-d$YKU<nPkbDufce491vzBE%?K2$eRvuU``THe0y@*Xzc>N*?gwP%{Ng*!lDhigM~wdgc`J|yl#PV>q$?Qg=9hU;Nov!?TdjlHt)Ry*cUp%t>v65OD{INpz5qj8yOY<3N*+1<3_FXj6NAG)XF-ayDm#<H6Sga0Mmh?ZPA?sHl=a*<^%H6FjL#JII$&-uAwmm4pR@ATGU7&I-;hrGKLwv@V|=rGD8y$@S=!ScQ=u06Ib_1d{RlAUR-Yrh}a?{b0KV&)1PEeRJLUYT3tX~&*^bFDY;F0=P8><`8YKHj++HSCO`5u?NPX1!24h8Bf(pb&AswX(jo(cQbJdv1r_ILsQoPNRk5Uss;PXM7<VT(9YA!5VYUs_RC(N(FhFwQim6HnE3Z)RW|UM*B;9?{2djVY@IkL$A&~JBV<!^XT1rLflI?I`mLn-99pU)63SRdyGe-G0vQwS^eW<FaCM&3B7T3vchm4L%5Q7J;uOh${FVklPORMx1zP|bef%H3WdpV_%KVqxYJJN&BhNi0dFIIjD1=5h8Qo#w%ycwZk%{YGR1PfRI}`S_VzrKhy`k5I_s}+p^fd?cSw~|FYZNo-F-{P+976Sv$p&9y`<oL38q2;Q@EAk{MEF1@ZLJ`yWy~7*nSlU&(-@`;<M7MIoRi~^hij*jG|3PI5KKXyhGZ*a=%mWhC$TOXga&AsXGwg`x0KqT_7XUqgtUb&SZnJ{$TpjsHUj#<S-&5<dMncoQAL9r1n}h?tmBO`D!JA(fTZS23hXKZ8yI5ZLO@?^;OE3scoyA@>p2Upq_g$HV3<;(S7hZ=jBcn*l~;gsLynmkvZk4I=jng*7rbDCb7TF?w()x-Po1hb;4P8&Dj&7>7eq~81X@q!O?ksNVbm=n(Vee*B--NY-8)yjL+t9=^;E>Q0+T^zG*d#QuadMSf{s1ThBbYyE9s;Va7|7_cUG2&Xei5*<MLl$E(9CYw0vU{maJP=Dlw(kF9*L|HWkD(Tr=}_qMBjgRR81&GCXT&BdNKGk0Yg$YnCQ_H)#_pG0U^cFr%{!l*y5P&UhGJT*kZE+M*!ywR(?Kn5QyX6JsHE6!>&Tdi4g#iymoz$VPbn?sY#ueKF|isw1+#MZPVH@lwEllfw@{dMe3G_bj7Wi>7B@2p?%)7d&lw;J13cl9Yu$Y!_uY3}7SFOyU|7OZX_I+29oE?RhC3oYNn%FLD*gXUX4%X4qT{mfgf?$i06-TYh-8eggSXc#p;#H~%|=i8_^$u&p4F|(RP3%SKt-1G8Ye~rCzg}LaNv@)Nrb(~=eBTw!@cwpD|aj8&z@$KnRBh;@oM+uSXhiiAMTHQHpj*`j6^gUnLO)|06!&~*MvoP**hk|H}E-Wz@J*rsW{MP7e<E=*-?%K~jE@AXKf`vjQ&-6`p%2V)C^igd0p6#wG7jZWtt+9WbT-ke@b|xOS4pW^(ZC)`p`5y0HmiGy<;LC<vbk#X3jr!Ajb2W3HG}{OKXRG|yO0?GOt2;aIyl#6|aw#&Fes34^g-Xv;SG#fVUCzl}sknK_*ur`7VPbM-J*$`Cu-Z?)2`^uvhquPOlv>_}tW5uXz{PU+xoF9lpUz%0X;+h61{&VY)*s2`Iuk>x_xko0@2~t}+ws8|nb}`F6}!{v_Cx~1NHe-Nrc;TWvGeNUxSegm#W#kL_wdGS?>YZ^o^eb^nZzM^-(4++Nuj>jjR#}9FL1~j0^zq!wKbjZKZV!8XX<_v-SWfgq`k8^j>Gr0d0eb74e4OK!>$J`>0g&RY~MH#+fJ)_A2Te<<Epikj`$YUJKx(E)3sD9pM{Vy)fo4TVq;Y8Sm=6$VKys6GDHW9t=e=kX~b4FTWuKazu0X);}=zmBr4&heN$bNv3I8!<|k2h7Ar;*omAt+L5|I-lxQsMoQF{MbC!25>ONu6D<@JhZz7%TF}c95@D8|V&)9jJD-8_Bw{&LQEWIb1Z=ByUDy6p7yWmTHNSdo7*2!n<p>We&YF6yka60M^>|*Rgl(h!)%XH339w&YO$mO)Z#Ei*8+9EbP$$Q(%UZU*Hm!0`7G%$*NzQo5O;V!jK4yr{*ojUN=iYUa}-C5wx`nBkExwdilIVjKWIHp{W<lTYAoM|7o?ZoD*ddIi#>Xa{De5<%#Uaj|WLv~)~=CwOgSiF8l?tJ#w_nLph)r}@9J~Of`X<rW_!(C;wk2>nhySdAI@EeWINSY!o@meQB)oii1>M;<vb-V6BU{bE<YlUjda!e<m8+}{e@XFYj=f<#LIeG!V84~HlzCRD}iN;cxMuqW#3Wj|)d%e(JB%Vhe=VlvTSk{M^O(neD+%<Qh$wMGNem128(Ugxf#WI<KF~2nB9nbacd#7jaH12AFL-V0k+P!29;c8*NuVjW>(s56<c7b6%_|<+6-dSty**d_E=tM347U&Hcnd!g>{HKV~(`w(P=7*Hym5-CF$y9V7;<1wN{hl;b?VYB{*?94d-4BzxYiJ1A<~@6AUtbT|T61IP^R<OYI#*uv`Z>D@wAvFBb!Y7vItF92#e3K8c-BpJ+6iZ=ma91X?aDUr*3X$(e_+hs&tI$o``8t$&B?eq5)8M3Lg+o)cdfFS_f@8l^f8}rU1yZ6d)%49mrpeKeZd8Je|(<qB9^@=2K>71v{k9+Cl=Ex(q8e#g*CjJzH*is{qk03DtCsoU~Q&rtIBrKrW<2h=I}XMyo%4JX|u`hf<8daVYpv?*x9qYZr9XzKXX0czRlk@or2R{3WQ%dhs{tKehw%1sjAnp6r3r}QCM1E6UXV;vUC}|`Frux=4fPByB=qc#a+qvBsVm-I%TK>3LDz88<@TR=R~^lw)=7nx3kn|yZ?D#e|GGn)%)T@E<Rb$*;r@SXg-JQ17MoP9i~9&#d&-!8#~JhHOm-c&*Z`NxueJHu{V{TeTCBQdbv2*Jim?J4;Axgy!XIa$ViV|5B9dy!Cf`JhSSylgVn>&j5aGZ9)*S;!~8BL_|sfA-_E|ISX(!EH{I6e7Ejlib0<9l{Yi!fc_2tXhr#5;xh>~}H+Q^KO&h+(G!=g?rK^^Vp~JWr*-6RgZ5BEXQLL=TX~V;)9vqk|u0b$1+AJR8se!-YNqUC&6Wcpy%38{`R^M5FjhX^u^VDAx55U(<9xQk6Wv?@7xg(1^pTqw)tGUzfpAE+AAZzoCvldPJqKs>h4<Cme)2{ltI(7nEZj_&`OWVdVm3X#JZ6zCk<Ozjd_esM`v{$?f)|=1ANYt`U*+a8<fpYN`&gGjjSvnrsrv`&`Amq#Mje)tn#Wnna{g8Y1H|Mh!%@03!U!i>LVUzWxVvV%T)ZeB?v*P=F7w($Ff~_&>bQ;lcJ1}ecr|I|pMjX|8fnjvg-Hq*I&+K*i(h>Q0<L)8*P&15kY5M5$I<{c>ySdPejaRzB3Eg*gfpqjmXav|Q<?2*{aBze(lbGxCz#h7WaLqiZ1%~zkx4)0KrmUm5Sr_8j*erQiy*%6%X0GNgS17eJ4bq){sXrvG^T^_1WbI^HuJDj~Ik>mk7MXH5<5T9Y>@qoOuPGnj_)Nwfi<+-@-?yjg^NlI@l8%|vzz1fw#dvME9Q0iMhkLi@IX9$7_bFF3+loZT>EeQqH>dr{{3YMvXo~4EgQnN-9P?k5wxb(Z2P*f>D@&K|=sYV3cQ4k&>%=!KW}3rKb9!ZIPM0%ND!L8EpI^Vs$Gc!485cc9kI7^iKHv3AZC7zth;19G5@qU<1t(Xe21zC}^o!m@q_mu|J7cIkD$u+?+ox>r(aZzY>DRy9l&7(@XGlwBp0vI?o6d-@vRoG3@8j2(Okmh$ov;2}u2t`^r)-Gl7xiUf6v=pxT(}yWjy`uqtJiwhZO+W|(7aZoZ29Ph>bXX(b!2f!cFV8f=l8q5o2NaShe9spaSQ}w&lxYX?fG^y{dB+8U(3;stC_Sl$kfZLtGaiUt;d&?@7@$n?>`r_oYgZ<#7v8Ny&74*^ts8~HqilgbB#1@dg<MZzm`6>U;0+#+Y4(M($s5V;8|~i^GYtiq%)1|CRoY@j~w&mD7LpR4P!9!VoHtXq5HmX@2aiaR8<f|9yj|??)sK|=B2n8hP`gj>^W^(3B7u^YV*a)dmmYhZf1sZ-o($HjpwjoA6V5|h2+d-Zd=3kV>~g=AE}iiJM2FX?E$gz#m>T=hpaE@-jDaQZQo^#?}o<I-r8U6XRlvPU&F~+#-E?NP#_f8bdM=hcl1RXGSNCU*|*L2Z}s-*zAaWo!CBpZ?t;LgohI0Pypo@rnxT@*;%@sh<8ih8(s6$F;{8Br`>?I;p5Jz(xOw5C5+(lL*sso4OCRMm*!V`zWzVdWuF%Xe%omF7^w+@WyBqAlR&Mns-oV718~5Lhd~){bH{_#<Vj|e<SA}io^~G2{9A1G<_AqD{Z3f%aWZ$Q!!MCx;vP)Y$72BbdDG%=h{X6j?+cgYwg(O9e25D=xUGJrrK-lN)+n0xAW_I@+3GRZn_}4CXXUbM*qT$&RaZZxsWV&3mzPJk2N#%7FslMOYD-J)m86;{Yy3%aCR?Q_#BU4!~?~{U67}=Y9E8AJUPzO_|#D{i=yTGh4C2Qqk*#6QpB$?Smsjv&X#_9NjZCkT)?bhz-=^EX+IQ=G+96saw{n|TfGtFlEvy>P%jI(kknfWZ#_x)lr&E_i18(DnYeZJ22bmw85cq>l%^kS6wn!1|fu%(^#98A%$_{kQw*#mulXbxYuiIV$p@WzkZP9^Okw-w53dZ?Dg<IeY)ANyU+)|YGAD$zsAA>LPa!{D*Sm~5%ogWFRu@xIu$WGIoZVNYNg9L_@H#$d_0J?2#}^jSBz-yKu?QcM{e+q+iGzcsG2E$3qGC=ToHaUnEpZ7gL=e&u{N(qGOgQ)7e0`*x;MTy6Zu;q%Le8$_dfH$Mtgyos;j(9=tF<~@h=jXaua?ZV7hE?I-j{c_AK-*^6weQR_SV(zKeRvp`cwUGdZSlj}<=6TZTT4aslOVZAbtIJ7ln-GV>{ynjI|2)p9+&cZ@3$xZ_@6FnqC3%m<R~ob}sc}C!3T|!TU?*cA*4o{`eQno#pXCjmS?E|u)n@O_iND(NchbvRRD8*rC)0;l>oZ%hMEFL-n?LyNDPKQCeznMKH+cWv^|cN4)y`F$`@LVU;b{r1PO5T9zK_@5uk7AtxpNiWU9o8`jzR+q<4LgXy)PKqH%1NffPQNo7OW#=a0Yh!_=Dlzo88?Pvv+R|OP;)6#;U`T<GAn$1D5wP_H5$ywUWLsyNhrB*WR0ltu8Gu$9FSX&31R}w(RmIrzw|ZKRGmfM~iQ547Nrz<?_9{?U{!lSKciIbLGnurY6m|Vl9#y6wSPS>(9K3Zoz9|XxA)kb`-tYinI70X@;yd<H6T4dbwJ$I7xBS;MAA7H+YU=M|zp%qFj_AUt_sOZ`l>6VQcI@9gIcZ9MxH66b;?iI_Z$eyoNt_CVzUIH12yO-44<L^KqYitybR#X<Ij5>}=zUoNql@n5OT`fZxX3mzAiqV9E>vU2|hzH+l-8a>rP$nrF#$bY(AgjHyaE$<2GMUYZmGx%YL4EHu{ZQM&7>M|Ug7!m-}%W`qBiX>)5jmy5&jXKA9TOf%Yu2BDo2MN)AfWSc~3YxmjT=YRH|nK8{QYpwftU)N0fy(comF_Eq}Dm%<u6g%stk@Y2htPdK<+xi-P^#v|=<c3<SVanc6u-f=O;-ZdvDsZ?o8RGPl(m+|hrv{PMnnC3(MoUa54h1F?BuoU+{tt&wvv?0Js-M$s_wFAuv2%Q<v&!_RLRRop<lVQw92W2SgS#JzhM7*qC$s+@z5A}ZSoYe`@Qrkk^Kwn}1_Gx5Wy85TX%ASkc8W2bB&=>bui?30zE!ty#YJYdtXtPJt<v~u2-{g8qI}^?q}_k@*R2`nyX5xsCnv8W^OCNU4y~hHNkZRlVh!OJD9REy{^8?JP6>0VzO=30$zEbPaK}F!|1^Ng9M!huytjPnL_};`<fB=6;Ch5n--q{d2PLflZC#xP%WnBM!V{XWhHj&e{gohxwv%EJkZO<Airl(y508LsHStR%ymu7`@FK2w^Hfy9nvcjS&_#DWTldkHtA=ZD3YO*-oL|H>OpPdtTvX0V!+17IX&ne!)&!{O>cU>6G58t7cG08yQYOXCIecy(tIFe|jm68hEx^<DE1NG51t8dxT0~5(|JGi~I^4I$H+>|u>uQ4sW!&2><^#PxSZ(3wWHht_6|V#QvK9P{DZug+xw!${=G6NPdL_=<rjnQM#>iIHo}!$t4mzY(K8@z+ctPOv31FR^^|LzG^jgc=d%HTB5a%VV?WFxV@>&IA20K4TRpFhNw|Fx^k#=hncgwSVU_02~T%ZH%314nXvsSCpqfkEW;!!w5&Cy6h=l5BwZVjx?{ecXc>#ST6ZT0wFA6H#%w1{iplRv|!=XM*g*KV}^BWQfez9u=~T#%<0^o=`+pQ7*md)UZv4oFjB%dh@K>K;eu7OKcDc*Zz<$&3vGQjfx<#BQfjCgA72|IriHd<Ww;Z4wJ%Cj2FpS!3(zXi7Tk&$DiU$hy49Y2M_;$DZ976?IjA8pfg90g?ulPKF->Ss+=<rswItidEGKX>V{eyFd0_5)L})T@wJg?5(wR*CSgSol-xedi!>gXd#(-h1O?=i*c;H@7>xezwTH16(W=76-!EW-l%Iug}GH;@bL1Hi=o%7)6aVhO!R6+E-h#a`wul3530-7X-6-}AiW32UAI39#Lqe&gZ-$|kaL6oW8pE*pB8dOl$!(=Po7V_*#XoTI|lIA#U_7r<O%O%xe_nEb?aPhyLHU!o}tgFaS<06*d6bpU#}Kvxd#!Kw>G>F8h4DnYaEw#X8apFunYO-CF%|sFUAk2Ew7tL9&Zd<z4AnD6ZmuPt`FLq9p&?lGT)@bcT75i+8ZC=M3UjH0Jofpr(H$neqM~DPxQV?uVI&5Sl;-3X%aGJ(QnqKf6=u)9kuRUu&GrOTWq`C<G8a@9>g#&SLu5Vi;ecB@xhV0>yB<`e|~6GX|aB*GR)S#--&knW-EjRONZiKpaQG~cby8fg{q#N$6}<oxBA<4=Q*yQ1sXj2W{!71U!<m+Hd}VI_b_%E6p?wh8IPR2_(9#`JU2S)(_%#)`OeHlm=zCCg&7+G!jp^Lo%X5D7s<W%>fL5X(=cc55Xc~Fsc?cOf7iTU)=a)BSt-*PUz^izw>i|ru&Scmo;hg*RRanAc63Htk;Uj?|7uz8D`MwImhw~MQwQ74cB^uYxKH|3Pn$;rdkETZbFaGgwgud8O|6W2P#AV-;HC}_#%@=}+WEHXvftW9AYA(e-~e@+2;6PFvQ{j-UqnKn0ZsLAb}FwqeGg-h8^E%4aF9{<cVGJ0x?#g+xu`IcGwtjK{DT)e@4C?50NZOkI-H3topc5Zsy#vD(`m_h@%;8zGWGXX&W7N@Jw=HA>U+Z%bedjr>h}%x2XLxnw+DDXODJBi{n;FfDDk~^xqnud{=s^!jCT5YisyNp47G(L#*?h)>xpuQ!M&vRqRv~UDf#WRj@?Us3>vanZ~rRc%ebem-=(Tu^#Dd1E9=vnU3@};;w!au&{reJ>y)bY+}$Pyw%ndNuRePhjC#{CN4x&^nA{1c6*q_LSs5#3J{=)K{X(n=#AH>@uO~drMAO}fIr$H-_5^TlEB064A@P%Rj0DEYswq|)b#d@+E`m9|tgMU*oux^J?w#0(`bw>$bP`iZPY!v^%^>Li?$|vQgXH$H%PbVOKeNhjgk|17l$V;*I$wpDeK_HCChAS*Q|0O%bNF}=f%Z^0Q~b_*&(BgDjYXWH$xg!0mi#Gex=2U?xQ5ko@CwV_*Q<`cnts0*qje%Vt)n3lsYhGwM4kG%)vC+Sk>uQ-zq5n`N{41H7_!`&?3t%Oy=_Q~`<)){4^K`?+NVpBPCEeo8>0C!WiC%H1kFaZ(qiA-<L@IMdH@T8SuAXb^F7x&@YYoJW|d{`O8@p}30;i7IdgpVxRX9+yiReX_MVl8EU@R})a)=Q|7pz0?_nl8_bKluPE-{2@A?ZD`aJ*SnpN?bwmgUsAY_)u#lQpB7xyy0FTka*49R1&Hx0n^R?05WF|#~S#6Hze`28f~-S`N%xuW$Ma_+gX$BWN^4YrG@H)B(>M-;*4);iU{i;K8a9^(7~Kj)KS8o>zfHIJDFzI^al?fFJ$a^Sw0>(3aKI)d%$?p_)Ep}$!LG%Y5=`n|Ejz1p(XcPg6|q?y-!0VcdADcm?~<L#WkQB`~ude(IP`2Zb(b5Pgnoec->)V6|iOyy-~v1H*`*V5|h9KP)5kDTD{ZhtxVR4FAixojm=xg5C3Us<Ai`xD(=u4Wrt@Cz{rrNK1S!1OHvpg>g;6l(*25>0Kq6*H9)G1~1J^|=EY(y!O^-e9|jZn9Gjwa)KVeLXYS0mvJX_V!_!zMAU?vuj+U@2c_V3J^;8wJ?P)PIs|84Es2IIh&EG&%Zb#Kwo!zF?-7uI{|~*-Z)Wj{$)Npz?{@NEO>|2LTdY_N{gdL^C~q^;)*uLWXtQ_Y8_{R#6H{5m#W`Py71_)g{u$UXP??}p};(*YrViqW8%V0NBCseI^KHPLhlCT+8DF`@bgsM2we1uK}(d5&&p{w8!$<@#~ZT~t(;EBeeYTRabkxvMMeJ1gf{u=d?X}gP$T@I>h@y%R?yn9UdNmIHnj%Krs56C_YeCqn47lX*-?eN%9ZF1$XkbA)}my{@YDO0x@l;QPMJy=@d!FS2?rfiH`aW}anvP#fa}ikYw=CU?GCNoo*rRqj-30GzruDt!4S?)M&YZ6MsLb_%WM2=TsLsO+4TH}vFg|QIk}*p<T4+X!M$32FNZsto_&E@O)g(AkDMgr%z5uF&HJL!LtyHZw_nZAJ6j}JF@BP!XT5D{z8nMWy-*I}@u*%uB-ge3>-nXadfp%XZ8eH~s^FhtT_1s3YqN{qfmr^;#hYnOcDLYJpP5s8&kP<FrR=E#i+NgjQ$irH8LcLRJG%7PbnbYq^)o^|Yo_<>B6J&n*MLe@dv*`7S0kb$Oy|RaeZwfgFQ3GL-SO?Ra7|%8qclS}m!o;MoF*xBI0as3dSu>}?n)LsB&gbUrI!17W%klLbD2XE{D5BElcbMkfI90!?NpvMzbS;D-<r9W{qa_6J4+Ig<fbpECI1`{n;2Y>i8*L~!Zrtej`6E3Tm9Zrs#RwrzJ<*C@*x=<g~1gh=axr~=gG%_;k{1FRV1P6)2p~vPp_f4As7KY(|A#|5ypOi!0?#ee%4&5#?}=|UULProbpTj(ay^e&&&E}A=$GGmDFvK8Iz5G3Na3zrFGMx2VX;4c^@qT8as}9uVZ3~oIN<=ULb!nbYuLaFt;Y_u2vcPr=cgr{>+?jJ3B}sLu=BDdXE#lv4x_nlczb={?UzMu!+K9_be}^O?7aQ!fKyWXDxR>zRg?CO8c^FB(&+#WBA^wb9Fyij;H6X*7vREwK$DbqvG~wogdssu9p}bRz5*-o(gv!-tZj<yuaa|W1RRHfEoo+AeJH@r2bxMRxhosu`iPIYd5+tb)NU3Ol~-^#2iM|-CLt+Hw{*^-?0KQEV&(6a$HXGDLxsTCV}h-1DwVu8AmD8av`ns$Frqxl)pWo7rDJb#IrwEhrX9vY*OE)q!m6Jb2x(Xj$vF$n?~19M0-WKsR_aA?^gFgVPhHiovl&ZhdR@S`x+5W5Htq+RGi5E+2O%NqZV#HJMY7-cNWa{sGe!b@4eksMplWs;UvT@oS;^R3XGIUOT7>cU2~7R)W!<s<`qYZ=?20!Hck`dwwNo=zX4nCTSfJNH~Mk7?dD?LGo=@;J-K!^-1o+1d;BccbHA}3oh7YZBUtm>-S42$dy|-ApSzr5ReA^8!@x%!82)&0AMTGTZ@j_a;q!8N(<oZwTL-^4J;;xai}py*fxT|yA&GTi)yo586&tTU2ue`+VRK&IyhwT-vx3beuJoL>wVOvW2kNoC2=CE>nNAa+hUn-CE_y|HM7Env8fM?C-rMDRr?JyNsrtbETP<I5XA&nNq+GosoulO@=W+CSG`kO<Xfaz}VxyJs_nh>nMQeIqdAoUiI-57kTQn?=!!(1QZ<B4BQ|+q2L;Y3sSg3nd=0$i05}?0i#5Zacf&?JS`wzA)rJicT^!H{$ar1%~1BM>oG;KF>YmaAxiFpWvy--!5Nzpn2q4F+HaJTn^@VIvu@S_;~RQmeB^k>X|qB>%%KRc@qy(E$BO&NAtm$QtI`?Sbs%T+~O%C%(P*l7zNt6gl_e)x!Qg5*ISFuj|!+`3ls(C3LEhGAM;&nP70?2(I$`LO6~kornIX1|wJE*@@LQ}5P{rC>azFD-=#6~4PP_U}$_-A}CC9k&v7bzN6heB*azupZDojVGf~+?ufXS~%IpqgNv|BK>YSB<gn7qzzOb=w0z>=tr8B!6Drqf3Ius#s}(ay4^fR-POzgb?3*0Pk(NQD_}b4ckNXNqV>W~JH>dI5a&n)ew|-PSJwy2vU|SITMuQ{i+i<Yr3zL+H0-?Oc5~{!DjITY+-RVU8$Zj8XPEm}+GB3wy+57J>RYppy_#r}=DyY&cCd+-neTOV{aTg#T5DIiX4szVtrp@utweQjIh}N>$5redhfKA(uYm{5I_9IfTm{IUc^{7iJN^m+C^a5Fy^4{~aDzz~Iu7fHt9w0951U+{0bnDkpUb9p!R#^l%Oq}uYAixEG!zfTz^N>jDTjLIo{<nr@7evqc+<kyh3e%WK$7$CB4Us40uko$DW8cUdU|qhd@k0XX$yGZ`7ZSj470a4uX&vq5YuSUdQL~{bu-I45}_JkHX@h+H!9t8A+6m5C-1{ryGkCH$!E5<`>b&XcaJrU)dm-CJ=w9pIueBm(OG(EG!4Y0c3@vBHCSEV!*v6#j_bXv`T3>{wb0rHX|zJijp$$=4x#vsHU1L$z-RZ}>#yF)J50^SKGPdc9CPy1-wUhVTr<n!{3GqZmfdEP`QRIlZf<fH)6Qag7dQEP4?hiIyfpW|rX9+)VH!<o+LtQZhKs?2QtpMxPoP*aX-_@{T?Gz*nVmAhbY|)~;N9a`bwOhB7MyAGXRV7=n&4z1o@Y`2ns>({R&7l?G3zi91wT6Wzn9MV`Ay$7q3TURmvmNs<~shajZundp%;4rvt!GvAVa%(c$vd`J6M>7;SEXUthN94XZ5J8`t#K)zt@Csn(fn_wFXT#)<}NY+`!m8q$rHg!|Y&v(*{X)`9!&VG=v!KKPcQ87Yn`y6drK>e0!bG_)V`)!$+asUA&5(2iCA{-xQdlzH6lp%Z%6$&S~);KC26#grskglgCc8+yG*3+Xq-C(oBxxg*?7Aq1|tEaQN9~&w|Hd_jcHq=CVi?;BQ_mYK@A$d#~%aEe}8O$KvzR*KtVRPD;{+d}w3L%G%eZ=kZ<M*Qbb$Oi`uY3_F#0gWAcxI+Kq+a^zPhC18g8k&S_C^n48}aA_n+EkCNgH0n21AgbZevwrr5$HU$KFiWrcdhRJX+1237EP79&m+|`TqJFrqTVr5*yS&=p#nHn14fM)zD`K*5mo>fi2xoBr##~j)AW}^l!Yk7_Cnvi#O`h=Z$wPaaL_XHZH+ro$CthATaHq(Pw^RA%;xBVQxAUY%`(Gk;D3IUGyD<gia}Ao>h4`S^zVbr{i~I{^v3LGkIel^V`ri?^n3p<hzGw>E9hH{7v&SBF`l_thI)Rt1L+XKNF)e);`&y;_bM47NI!5!HZ4m{us$6q_=Fc8n#VlU9&qFH>yEgs%W~~n|<8C@2ykylEwG|;&yN}sTEFM33_tY=kKzs$21!Iy(sJxWp#1r3qvx`#F?Z)Q>v|0(=iMf5b!FsvR9Z_^_m-;EynM2GEPb6CibybOCJZj2+&-tLeMh`i>%UKRyGUEzBZySt1?z%@`zh_=sNjUsiR`SY=0GDg9opHmNH4P8f<FS`Ecl-3n>|y#b&OXO-VD_~uOGT!e^{&lyS(Qp;4?pVnj;$||2hxqaX*!!Fg@5Svs-ppaE|8yj@95P6p2epgH-xb1Y$R^$)fc_98R;d{TUUY%6kpMu*0T2AS9WuLj_<`QDSU%;e66@sD^YolH>)k*{MNRM46E<u{L-{<e(?<Pl1a}~+<!bMaU@<g#gvjbzHce3O-I}_x6;|IKo%W)FD}pxxDQ;um(>o6)a~{07wB-jUO4mTlwz#~Hm)B<_p(i<8>F&?hpTxq_*9LVvZU9^<35;Cgu$<s@v00{?A|^!gk8Oaj$vHA_iM7I9zAK>1@o=ksd>$!KP(?@4PI){@ODci^-z_^)r+)wa&_yZqegJ$R_52TJnVCTr0KH<+1G3U)}~g3N?~svU7TNjb)!R;?oSINyKa3a%WGbKY(hvn9W??9Ur~6VoYsXse5TzXQ!UQye&Gi(Ga-E8eW+FWTl<U37L;v#o`x@a<p<BdBKb?TKShbkuYpw_aO-`lzJIN!D$wYQo_5QjDHpFzcddKT7Pu52IH%p&K;Vvp${XspQ@1_ta^XvuuYPbLil?+ZDXY<T-YD(!z&><|xcb|(hva1sZ_Z|nKDN7UT<uTQL0>UsXV!;4#-mQoPeR~(gD>hip*>}N9J`!5$X15qldS<ZTc6L%Z5Gln$Qe~NsQm47%j{~KfSA<V`a|uxF@YMjR_(bN8<IVkl$MZ7!{VEL(--9~Jg!CJ)9a+Fob-$V>JtobmL7a8k2#tTX3LX1-2ie&a$Q8}pgmqu`yHhBxoJUi7|kCOw>q0JS=5^Vy|XFLCv0jq4#HlYH`=Xl)2;oK>GuGW@z>u2V6}UWo@=NB+}>Lhf}%>@7KVNW9TVK#*fj^cs?YD&nGTCb2L=0MeOev#S^ne%pQq31m+CTEUVr!t&6`Kkit3(9zh=cqQEyCqyPO(J;Hz8-|M*?9+6wLA>Xvvut4K%=i@A8VGD=_^eh7oro6Z?>P){0psi(za9)341&u;b9`dfx7_)>S<S$`^vjm^&$CSx<GTm}(RAI^v0ns5x2#$=F>!|lM^LDjN$u8#3qh_pP#dMu;QiaC3zy{wnM32))ngD3KCebfD7$oT7X#cWXTYtG}kge&h`d7iD$9XLCGWb<`dU86>J2S+sUx1~AquKC)79#xT_b6TD@_5hU=5<T@>_UR4q1~4Ahoy}>Et?RmtGL6eCjvH~j#fp1KZhZDyZjNi?S%(Vl+uo1%yOzKqvex)IX0c4qeZ{7azs^c6@!xDbm=5->Q_HONf&8~mFTGntD3Q1nyYp_Nc;rVA-?xnB!$>aN!`f`uVI4(?OrgG2RkL;G3*L8vGCH0cUwCW-c3*~6S$%yHG|8Uma7OH~u(pnfdyCtOQZON7Y(2dl`QiRpWs;UVxTkEv{`FX9yC;2A00YX-8{G!w=rwP+9MxB`$qpP&w$zm@H(ZN+wMO4&b`)2~=3WI?4WT-yc&Bj1uHx}_(xHY$vh<sEr@8is`Nl`V&hg_5jFuuwzR#^<QsrucJiqyOVznUZ{nqyg<80*?7%%aB@?Q;v1iI(iENk7mV9}jEwf3b3TX<0I927f4PQNbB2VsL#N;O{WE+@mIDwFM;Szz-n^xj*X!^mT}O0OE$$<zbKbB|yDQ2z}lW_+Ao1dXu^Sr`vD+u!o{CR$aWXsu>8{G5Kecp@|IkLVdYN2@A<XyVZH%8yBKh4Ho{os>Ut>NwL`m5Q2=WW|&yGf8)~65KzFu&<iT2pjh02|3@gKhCa?m*KAmk%5w`%dR<5o#fat04msOT-7__m$50e-?egtmW}Py)D{*%@QAM$>oP1rS~?u=<N9g)xgUPUG!l-7{i5GaK7+M-PPA6d3q;s!Kfy=Ti8SzhUL>t*lcyx6AuG>@!5<j2w|_|5LAnntP`^y)=766SjNaW*Blpk_x7I=b&5_uFtz#9c97(3!>O6hjITgAhj<M8kbbDcK6nLW9hz^sJxM-`3fL_~{$ios>hTo$w9(k3U1lZ(kvxYy2fouH-aR<4kPBeur_g=DIOMg!;7lBUaAFGaYZ1}eO4QHjx^7*Uh)eU#QVb8&XqOLd3L~1h<z2n0R;g0Fi{Cd03rYn@&;LX-zzHr*<Vw|FZHH?)-wePT3DVaAfnbj`+pV(+Mb{%zGtMANpLJnq=Ux*^J#-DnhDvKIB`5RdI&m6oX_IW{_xFz&OrKMee;&7rnCmYuWOLxb9pTZyJIj3snoPeyS)azXJq#0p<W|Dio&HiMu0_c<N9=<3pdM-r#uI@SCHNDY%j5Wt{MpU5B1b+h)S}p5*f4Jg+hdX`=J8OoChlxqs#Rrp;?;=00(>t)0&|@-@x5KVHE3#F)W9`T&gr@nrl33GTOfNNppa+pYObfT~1`s?xZ%d95+wFpxI6c{=f0v?Xo8h_9Q4;Zp$Ia8CpX|SLFl?R1tQR9+Ixc~T7Cpl}@0$9zm>nk)JG}C;-@HRj0rYNf>^|ieZ~y&ek~6vc-Ez{gs?HvB+)#_~<!!gzUz&ppwSpS{`!uTix5Ega$5yj>EsS+{G~7&7yI;pfd9Op#zgQ~unZFl4_pa%M1X=-RYk!_x40X01w7$JZ_#3=KI$Vxxbn{?f><gRZ*Jd?mdfAxzQp@A^%-NOTr*`*KF$FV@`(5qEY&5-wgXuUt-qyHXZ`KM_HY~X@g9GGdq8K`97vIJtdhQ$$T|X|P$#p)(>PV9JxuyG4@rAFD0LTnAzdaOL!rs+$j`LBwgU=SMI{j<q!yMaLvOk`YL|zNhXOZx;$+D#o%F7i_-Q}b<ux@@&>#HBQpU(2|IbEOO<6v+Wn(6&-jmzZdxh^r?NJ$YZ=;&|pomF^R5;tyg^|GHKF$b7(oK{t~*LwYIMNu2@-;15&+H+kc!&5HxddKUbd297P0N2c}@ECXIb?Vioz+rN)7ebHh5_}UtPYt0Z?|**F>P=74Nc729_3E|yhCrsHJfZWw5?;+ZF???;ywg{kTk$!VkW1olY%dOR`*H1JOJh0iI=j4e!N-4m#G%Asg&zDi{KlUMXP=M9V?_7CCwnrBM<HLq2aQ}NCqcgtDzd9}I_-sdn2v*Dgu{|=HRR3cqpo_<cA^Z#TinTQxJHB|JMhOx@UqD#Ug~<V%LK48n)KM^fSmUc2HNhd2M_)RLV&-jFR98GYRh^I$G-jOyv(UTzSR(saOSs_CJIx*vthB|TAhQuo2earI~aw=$7=XHi*D}yxdOh9L?8(tzbasNxHTMPH5Qw+n36i~Su^nl75(X(!s_xyX5O6o1?$aG9~SFGy3^g_yg#Jp7PU&5#(dnWiBo>Q@6H!O__8a{A{%gJLJG!yyKML9+Cs>dG&uF!6=iSTy`gX+#rH0>`Jwd+Vh8+dd(pb|_;kb|7!d0tu#dN89X5>Z{Ps23)v7qPA8}<tcvh>s*9TtPc0bKA(5ngO5h7vX=-h-3=9-NB_OZPoJy4kFOIs+@DzDXl6R7d)8x?B;-{#NPd19PT?tZ?j><>HC!c<ShzIZBye#=~xN)gQiY65bTRPR*ZQ~IFrP|$V3`Sed#n(Sh~g3%IYtkw9qyyH7_gIbTOgjj36-!|6!7g}d}I(WYKNoU#tWGh4$Ep8^{opq!hZgI2<LEz>!q{!~yn+Q2tF@D98nc22I^#?|`w&e5Jp-(4LQWrTGM2*1PtdwbvF4hUXUe+Jnu>mh18rf?wbDo5pp9%)@{&2sL;CZ@xRmjl>yplcX3$kXV87>0v6+Nfy*cr)DKLf0B9SkJ5YGSCC&-S(33xDVY>gyGJt%PtxqDdkhV2C_0J}JEi>`gn0s6q3dZEjIsDI@eTK$VevNUF)cRS8IN^18g_4<H*e+-WLx@&R$q6Y<K`;XG5JP%ij^zstVIcfdD>eeA+^w?Jn;A~Z=xY7RYo;M?0|nlV*swpbsafomVO^JK<-Cj-9QmmVauXJq-NNDL<Ln8ym54-I8z+q1^Hjg%whur=xx#POtGr8N&{ihXWfE>zFf3`(<$6f>^W@Hsvl$qRa=lfO)d`^>T19RWPktFC2s`z;X`6yMAzKX(}J(v^=|`L)&_owjfnfj7__Y<o<&?EvPq;(K=IkUhyi-#xB}JZHDp)L3cf+<-=I^J<&k%K~t2N4!--w>!l+-3Hj_RUX>Ox6_+Rha$Y}FQyHI!)QutbZgl97Oh1zVXHtS5BVF=EuGT8y)N%zHJJ0A*$Nt;9N#^4gb@dwf4jxUOP181%|7CGR@~^@j)_(gmzMxg3?(vnZQ6$~ng#GSMYo6VV*L3CSN&Hxr0>Q{c^h|Rb^XIxkghBr9dAykRYZ##_S4QNc`HtT@N@oD?^A`oGkd_AtM1vW-&k)*jAm*-(tJIW8v<&+TC-EqddZl(lxz6u)9u3678r@G$#Fdh7uvn0-PuS!??L$nI(~7vv1df*e-@9=9i*uA_;I5v96epos(+UDUT1RlEv9JXHf4`IRsv~!?e5eVIxJ9EOHMs+(3w`)W*@(W&MdP>b8(Pd%*VD-9_rtLdZuf{XWp%YG(Mibv8w>y`KL67hu`s%L-7v4FYb<kdA<I^?r)v7e7kJ}=gd>D!=RZLy@HywO>!txYVzg<{`n3ZdEwod*R!z#t34WgU|b1!v$M4B&jyRAg!o&kBC+H!+M1f)(AT(X=leg1os`GQ>|GkCgHOoAT}zx$9>0QZPiyV>DTQ=urw6bF?gGsIrs9ITK#fb8oYJ<-eZ>^Yg_a>@(@C(0qSJVLE-l8Zp0Bj|TCD$$AzRg$l*gUx_%Q8VJfQ68#Uf~~>JY+J$Yid0m@{*cwoFYekl(DLd>hxGhG9285#2hxm$bn*FUhm##5EUw-(N`JCpG4AJ%i^8y~#*m*?%nQP0$%QE;rKh$)iht!NDVIDMvkktJ&}39%AeS1^#&+H|UR9111`@(V^zMddcE<AOW}8N@6xq1VTB4Q_JsVw~a?&mqTq{u`X@-5XA5hFJROGE04RNVyk`L4&-U;YfE4G04a@QYjY~o5rTB-mAo7Fwx{CR*F@C;B5)fcYQ6n~)6yP~6vHNKBi^i(6&I(+4`boN_W1%T{Uc>S>)3w+uPYN=THQZ|N7{K|wxIlnT?Qry`#mClbdFPcwBhQ?45F!QwaxN;dOipJ*E*Y4k=yi;)<z1shFSyY*ri6x%NZHmi*iF_ZmS~cZYe?PbTy-l&7vpFs2Jj{=uL1+ClNXiRW7ARXBCvvZ?lUoXAKbdxRy92qBbMGtykGER_S;5v0#;}pv8Xn{^|hrl;WlUW}S3%AO`bbtB7`jJUw-$J~69u9?tu(a$e!T#u`*EVxu9M7xGYP^>^d%F1nAQDWPNVyWQ5alASJ3k5vy?a>Vw0vIpK9a#rhcR}Nktb!p|B7=G5<HM;khuvCMq?^qz}HxHA=juR|_h&H?N^1ZtpG?5j{R}fOAO+3u(F?Dw478*58@=^H%j757*+=K7WuTg4of%9+Mn~g`)dJ0!hhU5@P=0ANrVbgtW2p+&IwM7Vzt>7eV(j(6+8LRAL3icXsx2zEu)jczYHdLgpjAt#rWdoj^IW_jg?xsU|Z6B(hF#1X_E*xBF`bmF(Na^vD#jO|1fZe+`wtjk0I%!yf7<JMDX=fPi=iO+{uP>Z|;{D&3lyKho_|s#m{lkX`?Q;cr7M0?Smk;gGAZuSVS^^;k9kxvGQUEC+7lIchauJG3TFfOen-Ex$)fN5^lL66rMDPJ<-d+!1717OMy?=L}&+A+b_WI%rl1SCt)4kj~(=lK(gU-4-vT$@dtvuCpFJJmkwQt?~T;*`&DvdP!CEXp%LL@_IKlZDteM9d*E>$Yt-ST|&n~p1z=<7lK*M}x!@3Pb;k4P>7?t1MG3gs3*d%wx^A^ItCZjD;sKY@c0TG5)Lk2OU<L%Ff>#Y8%gz4@U_-0v3?#>6{4ctj6uuaRY+wOiC>#3E*QleMJTEiP)ergfB&y^nz<$h&bijh2Q}fmn{2ml^uRUP)L-+T8{uH6Yr*yMo*!I{4#VPjP{mQIki>ORtAG8uPP;zwVgwx&3&?@O9lAeV<oT0g%$xy}Pmw7~-YgX&QMqe%s-U0srO(`sZ+=C-%49$(0-hlf$iZ+bU+m{!H#g0p%8|@y5~<4Q!{N!2$0r?H75@^A4y-gMyigAq*XRE6tF%14(>Q-aYk_({N+I&zA^#upq6WCtB=p*eL9UFYPtlXiZnZ#6C65m&0&$u!uu&npK(G#6nHK%(TsYqoqc68kwtEh74+g_CU+pMo^Zx0%3(lc}>9qkxs^Sa;t}{2=MQZM#B?eWaeJJ<dcD~DMPlKH%klY%Za)U%*WoN;%nDlgukHc2y}uk_(z@FnVfjvnQk53*6uQ@8bIsL&uY9jDYCgF4btnr=|<^5&ct@(x!k_%)zpO`y+!r<+j$8p$o5Jx?Ba*3SlamZ&Pu00bd(D}`nKK9tMk-9o2xuxH`JfU<6InE{EN7N-Z64~;U5@V?_#&u-9|F?o2VHkj0ty_HhPQx1zoYC`udF3F){{>qrdQiX3#q@`L!654ugz&CDfN6yUTI*i^n5C4z|_aaAoLwwQ_sk-W7*g_)W{K?Hk8G|M|ApAH8uJN+*G)08i&C5_&i<sj<177`^%<BVl+Rob$_ehb{VjS3jl8baF>q*kG_j7g1khYc2Pnztit+*-cl2$if)MJ58OABt3q-gS09GZ2enY|G1xjb_2fS-qIz|z}Bn5hePaV@_gRG)n{@T2Eqh2^%YeB#fy&LWA;S`732P$-1}nvEW_vJFH9djV*}X0WgVNHno`s1pRvCzn_RVUC;m>^8<DOqE~~BHU6_j@aN1cPSzBa9KwCxgPb#O_-L8(Hou1&<yLvw#PP;{xUAQri?p$Ko#owc#Hk;Xj23KY_bkyJBdwflyBx|e@<Gx}j)I}v}GsxnmsZP0}AbLB7yO?(YE641F#x7h-J5l;-kkHAu$rPa1C09(n!nNGyalOyg-Ls0U6AN<B$a8OeAFe)fcBdz{PMT;$hoGlSBBM@6_5zHjweK@HGPAW$gfj`7(_Q3%iwPSZ#xfwoD`@4mbzKb{YqO@Fi(nPXTgLnx-FmQwRE8oBLKEu6gDL|q1gQL*^48}HUZV?2O#Xh|oUjpb*z3_AN}5Vt?`v!O$M=84{prnL6LMj&ry<b0Qz04X?6wny`)=my?_HFvBq}K4R(C65-*Crmq1Xe=Dz~{|Ib~T63L$hxRz~9D)w#GPE)L>r1&$wm7-PRL<zndP8@_LkZClF&dA%H^)C)*D)^(hP#-<-S&8Z5n`chkkzz)*b&XIIQuSauAncfjEM-DMll^wplC^O;pA(O!U^UU>`oD;W2cl$g){#YyjMbC1ARWG2bhc(G1Y-v~jc6GWC=_z=d@0LbK<tFUcjUAvbQ0AH&I-Sk)xG2lGyENunarj(z#OeSK!G!4*eTO?7`53m_YIu9}=i2u%3dn?U@~q)*Qa5d-8DU>{d%JiP;!k9O>aHc>2lS~U!OhLjz#<9yYAZ~@$SqrP`aNBC(fH8k9&A4ytaaS32hXrOd{{d7nm75RhHk_><eo{r_BC(AHIxQ#!NQ-x^*!#-RT5~5oyyFaZuItU=)G^iRmH-*+UTP7)^lm7m{Z*L6oKp{c#54^rB>w|MTX9c!Jd#X_;p;w`%CL$CqiA$Ne?d<EoM#vvl*_b&?fDvDVMWffLXR*iFwxh$G4h~oO-u1a)pj7*1+wx7oI!F-Z=hAJGwu_O^&}p?criKa$nu;U<_6ajAtIrZnw@6OR}n*{!T7$bHm<d4H9#)T!a;Jx8F{L_VLVL{0P9GtBzl1j_5dQwAhhjfo;??p}ybaX>Bq)Zdz+=_>JAFz*lIg+QEm6Uqj>J-Ppgz1~kW{?E5#4^kn`j`V-sl+lytx4>iG$^uGssiRNZoZ`aBOi-PNnc9_{A23@we!lUi#1hy)7*tX(#?XmH24y(Q|DGYWEp~`>1?@_lrgy^Q(0Jw|t(zSW?hqI4#@&b&j=j+Sr=Jx8J8$#C`3(%W-_Ktnl47LMRcM7-iNyS(SGSl6@MCr32_2P8|$Aa<0Vv{~qfc8T-XyQ!X{_2mNX6-#ywcUjO<BjOeEmlT_$WPK6uCZL<&Eh-w;2dFeM5e9-A0JP3euh4Mhn3dKOSS?9I3K>x;)AZNcUDu4N4>_^tGtxWr=301UYF`(1ddLXL+6Qj3V66=swU9{#s6lVKwZb&u7*CP$5P4XvESMB^zY#~`i=<qQ7Ov*y?Uz4)&4`2cGdOYd>+xp+9y|M@+lwalXIhI?|j=;y-mp9VOYNG1?@bh9<5V7X6^Ts?O*y+#ZOyMdJG(DrDDFho0e45(dx3qCM{|TR`L0s&Qw2=D;tHtwau9s3TrMaamIgrF{*j^Ey>Uz>1Ta5S#*aYFXa`KJjSB1H`bY^tDCZNuP#nGBE{DOz4<D`@nu^9l*0+;1&YefGs66IKeq{J_L8ymMqdpcyd*)(CZAa#Ra7Hnr7&aCe9?SsQtjGpuvlUWl8m#%;l$Ue-T9kYtpPK;D+hGz&5o$;Ws~rC++AA16gyXxFk!g%=1(+te2HWJ_}^s(rupjZ5LX&P0EDXMJ*Tfd8nYJvYCX;SwtavPng=n4IwVz2**Q&uqSjS>1Z)TG`f0{k=eM_SC|F~(rF7x?;l#{%mNc=}+{J|(Rt?<77c|8!)!KYoi|#e!Z%7NL_`a;*SX05J^=4gGeo9X;ghnHcY@IV@-EBv!l&vGL6<CArtXI|+L;vRDQrGm9nf-#I2Y!t|8*F%;T(`8neC-?S7%eQLKV!{FVH5(-e28LhNt2;5cAp3Mj~QReYmVtcZO1&VY|#GZf?KoujcY4XX1OLfXz*Z%gS#C1eWTsr2Xj1!Gl^h3-VbYt&KLsPxv$!<q9RG*uy5nb%51lSD9%~C3JU0}b5d*h*5+QeSM7TGP|^kM*V*YM|4@bwag9)}>GwHcY+$hLEZae8-5vNoH$&iDPYP%}lZ*^UF(0*R+_BrXk_0#HIy{D#K5_#N#&)({z3UYtSy%Luj)>0?Fiz;*+1@fjJv4W%B}(j9X0v>T*bjT+17{UnZqQ`bTU3b_zFmE<GR~*LsUK0$1%8dLXvNwbMRIfs-yF7a>)+>0t}b~mQP?K11#b4t#;WWranJc)7tnge&maEzDeaPdR<PAkXK>bPtC#)UHa8!&3|p(A1K#%|M#tav`<o^Aa<f0-FR<Za$-^0Abz|Pc2kc*Xo<~m0>ozyUg`>LzqUyE|x>DnIzZq$tJLbgmAhTXGcyz(qn->H)IO&4%!DM0aClT<n=m6guKD?w*Ptrd`kC~NN#hIZOWy#*Yy0976uXcZOY$`V%i5gT=%d4H57is9~>Q<pq+R{C*GrOVt?{@W4S9jm3q6LyI`Ck2OK9s-gTj!GeVrV=Mnk<`VMYl0-LP}-40p7{=wAQF$;(XB%`96ENhIF9GBSA*841LEf)vQC^>H*D$32%uFx7n=|{D!g9^{SLM0n{ml3V7KFv#1l%$gSyz!WPoD_ck!BnPRLAlU64ou7tWNm@vT0Apwq*F=@iiG{tJoa%c<`Z?0n-dw;I>3d$=}bQKxwVyzzL$SqAa-}bVU;n^r>E%7GGID*R!ZqMV8OdXq)W$)Z=*XV1?B|8lQ0<Tl1kUd{lJFeDG>0+}e!;VI?uPt+R9F<(^_CUrVev!iCEdNri9g*qvQK~r{S!eBH(*tf$q|qBATnBS~E@0Ow604PG*8}L?HgwWQ<I{sF_nz{}xjmW~Z2YP9i|Y}X(HpY8Zr+8)K_HeN<g9HcG`Wi(c)2?inmJ>-S9S|-x>(kh_EmfEUi1+TQx2V^YyRNZd__4o5Q_8ftNO~WH``RyS?A^jpBlth5gZ|34IrQXgeLP%IfxcSMA45toMIuNQ#a_BjGUls_$@>A>y<cOkJr#t$j8};DTor=Od1QI-u#XL{5UTbJMSirGJHFR2a&cPSCO_czlRxcR<JR5{RL?4IlxC&6%q%d5-Bo~MW1#z%b<@ktn5z7whS24hdE&1wWg7@C`N;+(@v8$3VryD<EJ9nign}bHhFTDGEeXEm@;Ao+U@M3I@--9GBzf^v}TdoJIju**`Z$<Hq__oy6GT9kzvPar3<fGo)17rSpQO^nYqFSPf%wRwA0QhJblR{SCkIFEt*x-*SfoOQ>?QVB*GFNch4c|?jn`P?fllNUR$jQj*;%d;6MFm-sOlW?@T30_8g=0zB{!r2k_~6s9?kT_69UB@8xauFRpv8Yt`nX4??xt9+Cs;a&f{7J!3!x!&Jd$-&3W-v<KJDsuMvx>qYo$kh3=pX13pSV5GkR=p3+cUg@?Y)2QvxlN>AchQ1NY8DvfOW2Hx$2QXSTZf~qHxn<|kt6zhdzi}Xp##5-Xn0<r`I;X~;5P9u2dGY$Wo?^p|4*1;_1FJn3s#d?-{88O!e3~itJaKBfPLYuM5~>kh+4Y^vYXC;WX6*?WlK701dt4K6Zf(Ab6AIeN+wu@K*u%jf4!fsrn5D<pWRn8tb4hLES^0Q2KEOkJ#WEm{Dz)9&*r(`t-I~epXXEMHd9^9it)f{-G^*dBT`Dz}pE2d^%X9ihvC6OJC)btdO2qKgVWi+vsX;&Iv1pFEjjRhrBdlfx?D|l0GVu#cE;|Wasy#L>VZ9<_2Zm>@Asc_4c4fH<+x_2&=l*ak+lz4TsrKKGuy1%tuVT2>BT|+Z_xdT=0XQIj><!oF(Q0)TebV>K%@Bxj!m)atYj|fTSGWE92%U;mWErx0<oslb+^ZyYAk=zg5<BK*;#rex3BPu$FI8uK>e9`zV)jN5yg~d9pmq|NA^B+d`_7eZpq)u8eF&Y2!?#qOTWk|!t)&>2<2*|*+1c=~$%S;IEh?_@xxZGW#ktCk;y!ivU~YC2+7Zd`oWHsK?R>?&4$JFebHY08_&j-OKN%ly5TD6|w#^V@EBcSNGuc^?OV{YN7{v(01f?}1B6ffv*b4&?1jGhxKw9<M&-b6IK_zvjXX4CYfcJUUihDx^LY|`+9S`o2c@g*{GhDpO*BuaQDcL4dVh19tdasJj_H#646>S`($+}GgxV-lEj<|!onm^%gb`6qavuRgAv|NPT!Q!8HK3;^~*^WDm16A9~h$E~8&k1$wsqY$lvoFA1AY;3h*SHLEvO*m!peOV(@LTEOL!u|yUW;a1Bd7NgPpt__jLYqSe;dH$NFpD^-fXeP4?kWr&IIz)SMwd4;c&T+-_Gx~8$QLI9bIn7?vi}izP1;xu-VxEhSNg_LKZ#sbV(YYWtQHnX@D{keTk(@Joi;=alsaBpY3Gd(7jVX`cUdIxHvLg&^37zq4Kylr)yQ|QqI3DXDX3val2XtZQ;G-<Ex{-3l8A4S*z>Sp|lu6UjNn=lEKNe&<mq;D~h>cq3hb0{xxxQgWq`1Calu>-0nK9Uu~a3O^iDDHwxwu@`#;3=L4^IHo>qxyjEAn`L@IOAye|ho%@AsAOD7Ix?1pp6*%wDr*dQQZ$6w^zzgf*++zt{chG0BpF9sR%bivzyHLtbA~g#T3n$BK^?*L=b(kWDdPA%X=dzeo6FDyX44!rO2pH~*PyGi=`24{?H;U5^=3tda%MDO@Id^g?^eQ^MB!WgV<>N0j9sEYsL;iT~rL1baq8+$O48a2imtl4Mp)(>NpVVP?f$C?Su)h-Y9+z|02z_|fv;gln<WIVu^q$1*Wl!`f;byEyFZVADQeZ%q2K~*7Z3MkG1L8srM{B;J85mw;RdAG!CauRWne)Ti<s?l$2VF2khktaI)o8k#*bNc_wsP=_XV|s<u}8_R{<a!Z*V3i5di4DO>S24}`g(l*n!`Q{V4x9?MSH66-Bsm1n0;(z(|MWxfN88Ms{n2f&Z|vF0bW<QrS#b&Ayv^)RSCmW=Qq{z7H|&ObGl!~C_VXc_V~1y&1jEIKwcO+10!H^Gzoa57}zj9I!sP{7E1Q`&8k{l(J~5l+B&u)`rP|DxY0@@P3pu-f__PT8YZ{O7>!pRt5?np*5Y#Y)sHX1nxQUd&znQaLOuiaPI4KHT_-{09pCGrXVhup`V2a{b|<JRI4+#(>aM@us6^%SX+LC_4nL|{+LZK?Nl_yst=}5jW@R`CO6KCh&+Yj|j^w$#fxF&#gEhrLeJyo)WgYeq;nsLp;Aab@p2M&2W!)6Y4}c~6n29em+eM37e~CUo(%Q56OxJGe13A{XJ)P`JwWYQnSR@~+3e9g2<I9$W@!Y*SXbsviwVXf0Sl$NGPTiYLHi<e~^e*V9Mpt3E(G89Z`>8cL&i$42oBT=zn-e8tbd&LCslFNvU2ij5F1akG&@~r+;5*5+HX1nGF7U(Pp7774<1)?11s1OB?Xs(UpKn0ti}e9%r{7#pez*zphif+V=Nrb8=qiK{Grmc*<`j`m=hN#QyYNOf>2F===sk+`dHSFNue;M!o-*#MH==gNZ-+8_hvm^%d(;c}$K5W>K(Mazbv^)gbpr-p4Sh{ZyX*0`j=1wK9|=IuTYZn6T;Cb}!(^35<X_W_ZuX>|KQEz($Qx&{pko2&)jxW2y2}^h*U{+XC8xtF*%TwT&k*FmP)v7VCScEeTOKj$)~k_wcvj7u{%wlLc?VnvEaeS=a(tRU^rq{G_i-Y*NC~@}zLF`2)j=d4!_dWP+nnl^!+dwIH-5#}m7@Kb)`&_n2<rXwV^eRByQ&4hKU&fS@ADEi3B9>3oKTnxaWWA*2DK~~B1s8NQwvT<VgA}WFb)wpD3nVI<Vm3#S%U-jlO5*N+1CB_Y=%+fThf}r3J|xk-iAJ*=yXckv}-C^U6ZByw6<)$N^g(@537nZ50P01DKET8C&Nh-$2Ry9(WN^dfzZ9ZDyI8MzhkFcE2Hac&R4o8`aM+y;>M2Ht7YHTdS_l3j65G*@4Hl<A&h%Qt*WsGU!Ahux_79NprF%?gt&EvwNk$tVYl;lI^`?Cj!W2s5eybW+S=eRGgFqcR$G0<KiA$x4(Wy^=xp?L8;@JeRAq5}7+3q1&8(5}x@r8-+UHaQyBzZk#k;kf^srr*@FsjTt^i=(q&IgZi8hVxCxkck;G9CUiI40TlNV`p*U$O!Hap2jaF{{^iEKtyEM}Ojhmhj3;dm62O`|t|{==MDoB@PblF83ef6Wgq!}+@s1z5&FhjZnxy=s*jsubMJX7Ji*;m!aMa~6yJ=zJnG=JCLw(RCRiQJ-o*ck8<pj<4;vlxUmDpTv7O{X-R*uVJ5D9oB~KYPwdH1yrn>9hU63l%dT(c4Or=W11*WtW5|?BjkBI>HA*xd{$uh(KQ2gwDqM@U+zPeON0JuK<%dkquKO!wiqANzRa=6^tlq~Jl2DHM7-TKwQi1`;UZKU2Y`6~v(m>{ET&WArv|NbSc}I!<3ZGB;KDgS_ud8`N!xTs&ho4q+UbCl>faW=!5{dKZOowkr9gm}oX>}%DNl-Y9#h<b+F9i$?;$gPwP$*@w7PW7bJ#jw5}fQk^v#wl<$~z)%qiz)emK&7KU<0kfpR`ZJ7w>!FjP3twdbx}TC>}+L-_9gV)T>6DC?!Q^!I2apDCYKhA$@mej6|2z52DI=aE0wTCbC>J>|zyR*fqKRkzs{Hbyi4JW?3Ac~?J6_;_62;@cPPuMX!I2JU`}GM9>1_$=K>8NRm9(7mdb5LH=8S}xY!h@}NMdSHUGm{fZY)Lpiz9&&8lWOC9>n)N>MmS2rw`_-)9w@;LK^_s$FAxROp0GickG9QG)fig*SEY>nXi|m7DcLv&1e2<yoeJ}_HZ%AnC^G&6V|FzI{+1a*h)7A@=j&GqyR_?!_H7vUjIcu(ZTF3N?%9wf_5Ov1=reE_u+btGSInlX$y1+haXPR@^z<#%>%AlH69U{@frrjik?L{T0Fw^qrX!`4tHnl78^}hK+D@Nr{wI9D<UY#J_gZhV<8~<*acJ;`5cyzxJBUGsAjd$^s*r%+-T`m8P)t!^G!hz;fklPvdHeNRMKe}H|kD}`fhj)B&!zt5@wxdnjt{hp0V;Ty?$BCRF*4Y6zPX9hw&6dh0zUv6E?-<c>JFcQZf8Cy_L=S7w`%6<j;qD<)q+hL7?*$-c3DjAcPj=^)hV4Uw9k!ouS_E}!v<b$mIxyAP4VV|JlSK7LcDnG^;jAo)D!Mq+j$cc5KQCvOwI*A=ENkdaSzaQcGs|W48h+a5ywQte4cVb`b9<UY(Pfh##1t{w5v7sWEr_d4!%r8Bg4)Qo_M(84`5TnPzg&;)WE^1ca2I!0<qVBh`8&?M{;(F!db?#)<nZYzMu$6k1_8IChhgXMYXY|J#N2-`W___4VG4D2CX@AR9!?e)=7%`3iOZE)XND}?MeX$md+A}>BGdMlZroe>7-R46j><<;M}hT0+U*XlZ-cOUesH>N<-<dMwwS7jzkk2I-&lRq@!0cB0#+~acUB^`>8dlND=&HL9G<>??Ntue{?3fo#{~6O6*X%4CB%2RWHEIAl#Lu&**)*3lbxWCLGFnwl7-!@9d!A-E;@orZJHVIm+miQ_The4D}UV!UxD~>b^P}Q0WtbEKfYoQZVj6(bbY;AVR~nUMp;=ek``HBidIir92UfJzAu<o>)sEL!=_z34Zfyk2ZLd~b31N*@h<HS-MaPCs}WaTox`S4X`eJ7JDpqA0F#13(AS>|$K0~4dh+T=e4<ph^m?bc73PI3`2M8XFD73Tk_m30RkIt_96bY{)J&RukM3o{bVFy6y7gac0^@-&g{7Xkv$O%?I0?lf-ufQzwzbvHRxg9@ZM&b0aAvk=n+xv|2rF{;vrFRshNrXW^61?s%zbvwYu7<MLcl-zRBcriZIJ$6XM69M;z0M+mNVDN%hOUC30;mxRZb~z%XW3{H*p1;Q6UqhTR%^}W@Y8Ob;!yyqq>LvZ*_6xL9bfb`~*!?>M%TRI1gNR)0u3Ysy17`C5P>FLw^AcPW@i((aEnaJF|`*tXI`5?~Cxx%HtK=?EuU=)MRf0-!8SeqOpgHUg^xlL2H|%EFPFw&}Gn@o%sF!f(aevG^%qcp~zX(h4;t<j@GGQd9?uxj*3cyTfeH`!L|a?rA-Js>2%Y%@!#OL=Ev{Lztky!>UWP(T50sjgiw~<8x_i(xtpKcPc&^hv&otDhvH^)=BXJFz2nlYH$N~zPPu4xcvN@w9UwG_>_E0yNX!l0Dz}%%wemgFRCk%ikrdmYb+D$V{`EG}_Y(rB)wUA7dK;6UG-`bEThfo9MgM-9Ck1f{g6(4dij`sTw)ss~qiNB(+R*|y#Pz#yw9Z=cNN0e0^<VPyK73D(Tzh2lwL|*2%}p57)8EMTrG(~)nV@0S%C4SSbc#J;5#7|!+Mo`1HbAx4FT9%dZtX(DI;CJ87iNubuCg?J*%HL{u%cjTlieSa;wD9_TYWqkqAi4p=Rkguwuc-VfOipGS8CRF-X)5B)uHXuv{&a2NOy<giIi)>Q8)9n2sC5Y?6%U@qJ8QQIOZ(3M;ob`#vuE9Z>i60&As|)A4S|b?Z41>u$q_zAD+ysF8o>ZZD1hp=62jH;yU<=C(T3E-u;c@*Zj4gx9Hf=jRk!@WnuPaqze4KrgD%hB(nI!VhApK_+QWx*B4+NXXC9Xl+*^jY}g@Aial$J3|YYDV~fG|)Dvj!>!7;v0U+&HL5c*(&i*}fx#wD+;qhOGmmK?`11@7!aO??w(qG5OT-Scr<?9LBmJY8Y1Fh9|B>e7d=4QCBp8vInX!Bfa|K}beO6Mp4+Cy|#dNI~8miFd4*`Mzx^aQn7Zvs@dIF_=n^9rmB{i%U1?cR)xgQl%dYhe7CvAy>M1s`92+Y9p84uw-pWNi)wfrGt&+&YIHzR$C*WH}Z5{r-i|{h~IdgX#7`M(gtxYT`9;>-3o$eD5Bz*Y8;Lap-V)a+}&ZO)E2l=sYHXJq4<az`xFbKS}c8Rb-7tWPENsJYVRH*~*uk<u8OV#QOyOz72C%-8;$!O(m6hVGFCf+tVvHkS0f}TI;PcXh<CfADnKG`}1(LJew8%(WcBbs1I&8X5vEbqTi+EIb+T2-L-r%T1l*Rdh24-^f6DHEVz|6sB*HX&QKwyP`k15C+=`%uggEOC!{)Re_;~8uU}Ys#g=vQ7fUrtz0Py9pVaH=RX#ESypH+ep4$}tD~sn2a%YE8Z_wR>#b)EEdD}i0+t2a3EJzGnLc<3o!F|Y9Jh?SF5c`J4eLj~K@e5uW%Iw%FpFAQdB1d9ORdaYO^&7XS^N26QxYES=I<WZ5GON_@3)*4*y=O!$Q+rfaNu545#dqs(vVUPmy~@uhSvNYBi}d_?6_Ah@FKq^7wVmVUeF~la<&tTh#i8SegFiSav!;2UL-!i+lk9b6zZ-ObVYmJ|GIoQRJyw0W_HJ_>AV($K$gQp-8h^k$x{c{e@Ux3qk{vk>OBe0;e$o3mL1kRE3M?6|T);xSs;RHAAEvk7YUOTNMs58op5k!c`A{^1kB<OGluN}fR!3;X6$3!&-+|Uy|MWXM?pbRNtbwsDdG2*7N5HA#KmR&j*E*iY7<GxOZ`RcA)8jvi1(f~lZhJNGdedeY78=aTnRrXeU6ObsWEY6YQZ8E~5F4{R^4jSr_j@<Jva|h+6%4VnXX-_8Sp`jDxn5LYzSV8~iB<S?21?6!xVRWgYAPP8H2=0Wz8=xSGJlyTKcOb`;b%`6GGs=_rPdk<HE1iQ)%ph?P#?c<OEQa>!L}1a@xl8p;^Qu6d(`H|Y}&*#z8_YC<8I_?e|Ej7X^WdL25TwkzD8>DeeE3Q7BlhY7;N!H?ZQ{h$E;yphfglWenQ^iR@UXmG~cfRX7M)cFGsIWekX)^doL?IarA<mq*P0m<<lm4cjEXV&9bUUEbNjLmKsfWRItzK>g?KB1IuMk;x5Ty?f=E>C%LN?4qZBiGOHu>aB1z^5ZtO#>Tr2=TjN<5+HpgSEAE>of+Zd|FPpx1YPYN#x9s#L7vFZL?-SJgQc$3m6R)PtAAo&aJ;(haJt4V2rMWE4q0{!%;dbMzMBIQn&>!F+QgPf)4*Q5neQ`4qKVkyOUw*b6aMcOjC7B6^qx2GQVGpg^D~H-y5a-`Kr%F~Zr_VjOXwGXZsB$(%bNgCQw?|_J#%i+28*^9>8n^w|T9>qCF2yXL{6Fm+YWBqTfGLiJd<kEtZ*gx?H*UoD&r<@2yfFJ&=Tl};eD~rg^G=-gul5``RKN<Znh!|V-0L`HCzQI9iiq((6Yaju&)Q^Mp^z0>08BRk!k2+>J{VqK@r$w4aFb_w$Isg>R_LVS!yk34^hd3|Y%J!Eo%zaaP`jBE?fhNJ`iOc%E*e!mUaWQ2jd6>DP6=5RxaY<t+MBo=>*(UI%C=wgy*>YYV!h`T^X~|LL`;CXSwAXzTHTiU9ytN7q@KgylHi~B4t5sYKuDkI<~%w6<v`Zu_v8Bb3{|?#E-6fq$Qnvl=+b*S<a?JWrX;(yTm1y~p6Oh5*39Ba)WvR{+%}ZP^6b>ApW6bGp|7g(C*^z&rV1Ku&gmZ9Zh?ZF!_)5S9UKO`^PRe75^kac@hvi|q9b0PCt5!V`)MaG==l^QszFj2K8*>AyEn9Qv=9S8hgQ!3o4>|1tyGdqeJQ@s3Hv@vn&fd+(YKzX*S1Ypa=2eKYPxT|;x9A4yi0puU(x;VN+^wLdML-0*K^XGnreGYkoTqa5@N_U@$H_szfF2RejEJ<c{{(l3UsTl<S<;)>4AO6F(~zBj<E?dsWdDtmHt#;`cC$gKL)+}MM6Bgx4k#%0oAY6AHu%qKnjW{Tydm&H8{0$Cqpg${=zqN<tDx54$ao>!?YU>4xgD}@h@kpv_|e6Fpk^Ju2_Qn7`q-*fAHH+#Qu*R%Sk1tca^6r4-QH5^~{|leXpJu_kNTF3@pKV<Nh3V4s-j?pFN3%jI8hQ_onum-|U_?k<{1l@fz5T<@Fi&@5VBZtnsAmOj5m+^Qg`LHXCGE(PKxhjjo@(d%4uXNzQ(4$Yd{!yxt}7t;gPvYxA)@#Il~gH0FKnI+EXLx!wlp7Ugl<=x%pg#YK0P(`dufj{e71n|^0<L!{0%-L)mp7T&5zwO)q}Rh)+x@bz2!wcb#sMAUqn^oZf4)y_Rq8+yu}piTLnQR`%}+sf3@_)JRls^v6(Ir$HC_Lm!@_F;eNWLJ4}YIbT(4V0SwJyMp9B;L8us3yA8C29+g!5=DL+kJle*AHDo4t8pqkUMCEjv7h_X#r51d$;cndKWIBJ%AXrJ=8x?wGC00mIZacy?W-achrxQfp{aV*-IHQ#-l}hqmz28++Gg1)EB$Ka70`6ck7LPr_lH%?eOkHYaWHaj&g%8rm)H=hgy^PW#VgLck^|9E9OfO3+|%_OYdV`Eh_wbLoIZDCUpArDW|LYgL>ndK-0@AZj2*Dp4||?F5N=GZchmYT>JK0M5qA=pFhGWyIjauyN{xsg|3Rq{P|gJ+FWuQRS&ez*yp8ld2Dw23fAWJ3={k3-@c1nF(=XYv)*R<?7Pbwu&!0Y=jq{Q-ZidwUf1J6y*+NkbUJ;{D$%J-a+T@5vFqxjTJF3v@sk0fh9eF^7&Sof<$9ikw~bc+w#B!y;`U5%1701;*{zGX`i6v$r3>OCwl5F#s=V5-O&HHBoJo$wtmw$);d0jO!Os5t-GYB&4z9A7T@Z%;p@3CLP3lXP9y-D6=A4bs;M7?lz!=uoM~d1ZD-&=G2=W$u9GMw!kZQ8xo?U&wpQy15+IDh0cslgi4v}&5YgJGbFQ-E0;K2>Ojh39kNY{=mWGJ0;wMo<Sn|%CJws3l~*j8^8lQKd+<o-NSN5veU)4j%=kFeHJ6U*I-YLY7u{`;6*hKiuod=#H=@6}!S^tN8kdwKc#``p;vi_NpN-K$=LRYzML-$2pK7BHMIK&I$dPUq17v*}4ZM-p`S+ugb_l-FWSi!aX&Jf2S*&st^GNIeE7uWbl9+y{hRRiU%H8&N^C3GDSHJ^pRggBK21W67kgVJuZ9Bw+<Tw?`iM_&hTEf~{TC*KC6v2QZ8gm&N)9DSqUaNi~{BbqXQ|?7S^Lzh^_tnMZoXCj)4t6jNw$TMl^7b!2_lk>BU#<U8FeiV5$BGop0nT?$+Wl}rCe_hn5wWX#Nz4nP};Ge8n!sn{h3*o$^vhgb>y4A@>z3(dmh8k+Q`x4NUa#<SC#Udgmwi#rW`Y8w;E_L;|e+zHwB_H2P|8X-P6x6TZh&XD)iexp74QS;mSerYV$?pdmf>U-693%@>PIAfH4%OzD-XW`KiW&0Sn``rLv@#$k+`{Bg>(EM#yBLdDGe!{!1bj+^7CHmuX*xzbl(B9Y|1u*Q_ew(IW;n99yjO2_C>os2s-;3cDA&RO}vBcntrtx1PcX|kl=zT@xvnYFW+@u>iT0%IfkNS|8EU(jfbY87Kb&?E`2tNTd>RUb|pH~lK`fCNi_i(T(CvsM<t}Kr1k=6A0Ge*gHKdp;XrF&kkbF?MU`_@L0m5_o;ykFTog1>(2TI<=KhN^o68rNF64zzc(zL9%$l2FUFb`6=I@Hvu|-Z&!WR@?Y4e7=2cDS|WMrj^Rg*0zu9Rff{7f$+`kqtXfgIB;VWEp~FQ^C&;KRWbee>^Elj+QC^Z;SMm$4qEq7twQc!JU#WaY1JaGRhCXrKgm_iE?4W{+T)o1s_A)t)epHgnCnOX$zEZw`wr%37I)1&7cID#f16eDxDh8m(ONWuIX9hkE~`U7t>S6)!auaoXw>+ovic@8Y@N2J=W0$3jMnA;>d%M!i0{49iC>HqS6eIrYu)AZ2I0%!QJ*0Feq&wB9*^lqnBFT-50}Sq0{jlo#;raI3iR*?SmLgjT;Ea7Q>OV<yL7R~O@$HmINQD>fq&bPK|QWV!|>TimYiY#fwE*?R6xuI#vBOIVdMN9H-{#OxrOo*V@sXQ;sLjO&wI7mKo^!rZPs$c7x<Z(Q|Tbsw%5;0yF*mJO;)@1!~=XU#%tTwL#a0>?gS@x0}<a<EwjEe+aHOd`q6#xBl%F?jT*b!CrW+c*-&Mz2E_EfJd{yCJ3yqC^vMn1b{=#fKPk!-yq!2YPRU;P90!_I%|T~UJW=&tMD>L@-aL!J)-12avcW`S8`$S9y?g&FT0_*UQ|<iNd!5602E2N2q}X7n-)c^)<0;A|9ky|M0e6VW{n((`)!SQKp=bxcAn9}tSSR^IA<*qW_jd|xdg!#jxWQYx59pR}cnUX=h7c{(>OGaz?MRxfCv-$7+T%Jp9Qu)NRgT?u^7|Ysr|LX`!q;G0#DekO3tBIKGx05j<hXN`^yBc{_-hTLOm~fgTRVPQuCprV%Iw&gdT%uW6p!r?L|roF)8u}THS06UZj-Yyc9adYoW=KWXXW&%eWRICb4~o#i=KTbWVCX+V|7xsfw(_s=S%(GKWXJ^3o?U(Thygka9A==K|w>%;<Bie13FW}XUPP8-Um_|<yiv)_4i}30r3x;tx#mt+BRWuuMAF0nQzpA6o>lG_j13_v;K5~55NBR_oz+6X7eEQs76T7$+=-wHYA}fXDxEsd+P42p>6cM-qP)lh*<EQfX9Ogv@+^t$SA?$*yG$hy3EITDZ}$J#FDU6`no)P3~{x#y*)OAioUM?p-X>Vr))w$N*G;^S=6$N2VJzzCjZz-!!<)*sxStQCHAFm`Cd5X7k4xI2u<dq&D%FD*f0y8U#I4Lwf*^KU78tlk#-beWg#y*?iS#k-K@GDcb!HvW<2W60;)K^?pQWa<!i`Rl&72%@z~OA8#tO1{=OP)=5_G%%lh<JZuB3XxvJXKhk5B=wtLkNkOFP{8Xz=bqj4OU==yh^bO~WcUBznLkr#dS<8Zcvj+W1B_oCkM^u%os40^v<%sKe=J*P)Bkxe*l2TJmHMUIx&?avBDO4qJG?2r0a;saHF&p|9!{Ag-5z{x=ltJ2qfl}Le<wNaA}R_tYXOj--DG7g5B_v|hfHE`9tKrg?nCD=CYt5+*uaqITj1~1Hc>UOif^uxNpJz9hw^-h-E#kneIS0hd8Cy2j(B-M1wdM0Kai$ELRxLyI$cc&2etuU!M_4OBFpT?%zsm>%QgPUfNLD9h<z&^P=OwPPBK_1K>9G`>vcdBXWH>+5!yBSPQjjC~?Y<`ID%14dfA&$|n)pt89d;l!gE|THc2xn8h(Z0V2`_A<2D(>X=++>m6gRW-%MC!4f-|%p2PB)$V%166}IZM%AI+tUFZ{g)*SIw_jv!jk=jcB}^@$hlpb%ADrHqZfBO}}Z_+YWC}anylcv-dPx?59`&zpMp6*Fg{gw{DY?!ishidvJjcHE&HNn>sm-M~f9WJt@D*(yP(Iww&%Te><;h%<^NtgYbxh+1gbS;KeBSqZyZ8fy>3S>K3(j@3-txS>*6XVOWQjb8(IO`tWbk--A`)U(4ab>1zkJwGy?>Dcy@zPdu$w0Wz4M=;0QrFzSvv9;PDeBTwZthP&6LTkTVyRr(RMCafP96OrRq(lF??dbc!-@8f!s#k3=!j@}l%33*v-vhBuyXqUvW=1uhH8b9J@pXZE=RjiM#w8<TNWBjoy$+%_KYxvcijqa`|HKbs&Xz#Y~^t(>7Vu7RDcpihW3kMr$Qdx-p-u#pF^{#dUlebfP;~75!sEaM?TmK%%<hq{}4evBeHk{7Pzi-HTH63VlQ3&YYi-Vf9Jfn7R<N%J1B6s>#)oPmTDUg<-USTVnnT;lTOqFO%u+jJZz0|zyFUe#Ezt`AZ!6=~^H>&-=vRuW&Y(y&jdhDgxGI~N-Xerm~DpGETY5mB69zNYcR<WISf{1eTwLwSz(Vc1XHu*b97p`{H$d3n!g=ZJ1COxEImCZGm1?=k4I;`hoU;~n)Y14SS?Ex;1IeIvqQ~f^-!w&~SGhkvpv~});*<DQV(>>8PP=@R+ZrG19q<&VB{%V1o24H(qR{;z<hg(PP^bX~%x|;>$M2u%+X)z;_U?<MRwW)j|b;9oFiam^o>4wCVH~SYThvgs~LVKputX0c}3r;=iPMh8-L{&lQ7Ki^_M5MpykS6F)QrjZjvteD}6Ll`HA(z6NyonA-Jy5mva@i*18&V~#w)FGX$b6tTJ7)hU4MzY}5aGf(Q6@Jb25$VzyB@l~ZN0rEvxDOjWy(8IRZFsG%6cp3Qu3&{?=!wyET>JhF(Mm0U~TYE`d`mi*1k@1R?H39S<igHopjdiycrJiNnWW%lYDV_#)!7vm*8HFjt)`*Ur&F7gk*m=NmP**%HqJCmbz&V9>Kv4HrWxspTE8G?#rj`%59x)eYQO~bs0h(S8-;+5qvPP_s$~O8Ht3y-KL-KETcltGr}6ZB=8RS{M;t(m*UF|DG(h`j!VdoD14RA4ecaidur^}w^RAl^cx!O@<Sryg2c<c!?TUAny<5)mA4MK1@%w>+!Eo=M&-o4bHGF1P>=esO*^N|_iX6e-!{zm<I~+!DiWSvTZg|I?Ox1V?y<%7EPoN*7-{=o<bH_!<PRg|DBsI40et7$Dnpsx$;TMD<i#<uuQU}(et(jUii-}Pg=()Prt7gEjjcUl>%wY(V5FWwM*A7>pIX6kRsWiF-I0tH$1Telh<WHNtXW`ZAg^A6b|7lmYRr!ZqwD(j*KL}r*3Uz_$!LSzola#nQ)&UO%Fp7Id#zl|kM5zuUCNzFIFGVAHH6Nejz)ej8(hXm<vha6LijU#i78a*c+nRR$Cnj`IPdqkOnIgL&=K0aquiYP)n|fnRH8F<p+5D7_!jpS3OW(OH-$L&nx;ADJs$&2sxQCJ3S7kt*f3xw+bti>;`9I~<Nfkpfe>QOwZi5`xXmSp-<C~~mNIel6ZUKWBBfZB=?p|2X&)992bhvm`ehrf2O1yz-$1F=|0u@2eVd~>eKZvTI8b!$lT9@uG~>s(RjF5_p*voN#_RWP)@62Qu>DIKy6f<j7W@}J?$B+j9eVt(J;OV2HHz+2X$k=W3WcW_I&-g=sEmK73V3|%_V3Hx8nxRn^7%sL8O`p0^~GPqb-W)p&Gq0jSklwuKzkQby}OwF!H{7C89&vg5^T-lCiW4tu&Q#+UQfer{`5~a$#v<7acANh2c3JG&BnOiYTOflaW$I$W{-H~dcHot#MXTLv5FyHbCq5-yo?^|o2GZ@a45&U#McJn>KqOYc7LJMOGBz7my<_MS8@+p9Jep$czbi3<9DjOugUD5{Ycu%20rMb!sD6M{1ebz?i#F=;K2dD?p2T1kqVqWK3|bpd!F7HAZhSExeKel-H}o|4cq-y!}9x%p<PVpukF=_j(x|pp-i{npU5NI_vJJ!&!fKpmTK7W>&9y`ZiewDUt;M*#*kWfl!f*WEsBH3N!nuAt)sh38|Z5syQ`Q_L^+W+WLl!sFD>CGChS0r>816x-^AzFc=OY~DzBg>v`$I%LSO9V3;48}pKiM6U}xJnOJ?S_;ZCX#vymc4?0!G{I(nvGl;mlOc_#+fL;k{Un@C=P2KS`qH=DPwI_g~BD9ynv6qMV632lL@*btVTPHhE7t*%o6>$`Tbjy5o7FO|9TWs_5)KReN*VBW9x^Zl;8e5J7VsG9uq9Omf}1Z#(y0M^$QfYXH;R+-AXDpZ_kabj>y%e2)9n%5q^6*6M1-W+ARNu%?JvH4Vi+i6wqFE?S(9_P;GC~cUjfjid)ufZZriNCuh!g!e=<fXEyG(I1BM|ksLsT#dN?zO#QXh+J-YhCl%<Esh0q?QV+lZ!+vBO-d1We=Vu0&Oqe7t<O#uoPIxop9{RuDeQ}(n2SeW0U*y?gvQVb&f1LX%jGJyL^Sb6Cm{gQJY{r*Wm_tA`E-;;y8*ggI_crH?|w^6)+!fdBG^ji0X(y@^6G)t8Mr3?p-8u7gwkZ=UNgy3;0gQOhXE)x1Q+y?rZWsY29`4e;>=-QLY|F3iqX`bRVtc=Fip*ekcEdeqBJsF~M)U<Q%OPXX`3JZq{ZRB3zyD*k)eY-iOco(N{k^Do38&XgPmOY)W-y^_X1kcwC5A;hw$Q#WH_v&llZ<ei1ZzsZ;E8sEo&4F*xMdTvc0++GvBQrM7RzK-fqSYn~K+meiL|qCJoqEPinL>(ZOQkq&KBzn~?35T?VE-{pJB&f}B1_aBe$s6zGGFKyg1Zqu(4_&2HWqAV-qYVd~!huT;n{n=Sbg||1OzwpKl)}!Oa0fyt=*a2E|X?rJI+t{6rSTAfUB&9u*0o{8%8rjP#yAcH@#Cexu`e~y!l8wH79>nPeF~P=A$5Z~=yOKT7Waqio8e`;3WZC%5{G|*Y4k>ms0f^V<pVu;<FOGzbsrcTN_qEK=zz8@w_ZPtru`JBeO^1*m@;UDRr7#DtN6e9{J(bI{6`*5vW6TJkur3#`m3{uQlm(|t^`R=Qh?>VoTg$Hp?H2V~dzeAsM9GxLYV)d2aHqw&4l?<>_YZp4nYFRo5O<b9^qz-FD-+cza2vMgQ4FE+wzZ0jX{0Yn60si*c-uKIK}5NIgq9g~U?l(#)_i0=_D22Z*t$3pWuFJ5`FB+);rC7T@=lFv>lwT^@DQop^6*|5;$MAb_N9(#?L7q;l|#3{N3x1N6#fjKdw&fFJ#QVs8z@!fah@4BBL<QUrGDtTS~17>=6kZ|llL25v=L!&D%ANT>ng1c(u$`Gw2`0GQSs^Bu%5l_#A|$_NDVzOR;Rxa$&CH-pBIH0wM*rXKBu3ZJZ%BO9?{>yo5%X6o>chn1kp}^VMt<&+uI5^LEhr1L2D8H8TNIm-7kRxoRZTuPK{)(u4&iswUHgSK&?LGcQ`-l+~O#*ei9=tPu1IbYWi!p0x53X5g59iYogvjN&be}V-kkO9Y|;evnupgM2-3Mf+UXTRZVlxXtviixN&oYYstQI-OfjeQftp2c1U^cY-VcwHERy2MHRs29DC~nZs$K&SPf$)0Ow@K*br=^H((ZfMNJlmgIi??KOCc#^giG0@GADGG(wF{@?h*LTx|8&Di2#&Hy<sS%$rq{m=F=hIi0NZ9x;|&=S~ECYPA$k3$^7a+4}t(aJcZO`z>%uVYjQv1lqP$W)KZ4>|OmsommlE$Qs6f*_W!`x7x^R*ZskzLwrBqNw)*<isZG3CWGYf91(j4py%DTq8mV?ye{j0#*&|wKdf@!RlhcYh-G*5hzo1Fx%`b$T`Owr#B;^vc2|I%jxH<xZFZQvoay>p502>04{sH139Rn^EIlEac;0`~QK$gK+Np`rru9mm`t6iM3ng%SFYDAD1DESKIKfi7y-I&68v!-samN7op0z;XD7bG^vpUqiJ8ZHpj^7<(qtDG@Timo3m1_6*?A%UsvD7z}zw_<><mG?`{(fOJ0oHmMbXBP3UWUN~D+;Uq0FKn?XsImyq-SimD#^+w>rB&J0=a|L+~2wGG^mm#7@nBh`zm!7NxIbab$X!TcJ)=CY;#HIx@ezl9B0k+(?{EAPr7i0msXzg=GUFnu08BA-S!_<MSOOEKPb1ZKC&Yh_umSyNW;!96JdSGKEYt-tWc(g?nhO7*4_Pt<N%MDbX=)1I@63~8#~!n{54^40JkH%?bNobw?uTMaf^lwPt(m(ipU>ty+aT;JmYXfn_Rn6>3hQ$QNb3y7X@dmvso74hRAOh7G0Rh{#xh&BHp^|ZjWQ9p5Egx_uD5~%<6fkBrIH9T+ZowagJ8ghv`>qg?Q0NoW)np{4p`)MYsRnjNM~Tb?<V=iDiG`Du__w8Mer382N?AkpChVx}1wEypgmnJa!mNB^hW808kZvmC8K_OD-!vpERlt801%iH~r{tA639mss>bCfQMILMm&0VZfC~l%oDU~<i3P-cI$`Rjq0ZiuH}mkeVNV9?PFcd%dwjHc7TFnClrWw<&aN47$?gdvARe3Hx(x%0Jcm3+M4PpfIrf2e;zBZBlCvLEm++9!SocgL~<~$c%w!`C3%uzL2PrCOzj4Jj;Q8A)5#yoF3$_jd)upwZf!gWK=aDdm?_0|R#qR6;Yn(`#zjzW-OIN)Q}5Q2vuAm=na?e5c`o6_)KOlgQ0o|zK0TS_jqyXQo{py?694j!I9+!1XrirguZ(1Hymi1YJV~Eb&@^FcSzq5E=@-<$EAGSE)!C_7Rrb9*rL83D%-@BfJb?ER)c`?|+7Dl&`tYf(zE00%OmciWy^cFVd3G>8+I5Cb=a83F@<+Se9V}~f<%xuq4AnGCY*4}E$c(RtFf1!-lMvSp5#<dXZ-coti8uQO*nE;5Cvmzbc<&>B*ED#ovdZq0`WK+KLe$TMpP%<@52L$Q@u(xPpA4X}SmgUqXe@^FJm}_!vD^5d@$Rz-ZYY8sGtf~_U&j|}3LTvZTD?1mi|elz)Cue_qQab_<OslK<*&wCf2#GqNLKq8ocU_AyK2pQu)zgNvof^WmYi|fX8W{~-u9n=IfH7?*GCO~09*`<Y4WpqmX<Fm*xK!DwQpa@v+BcCBpK7*1mS%WqKv)mV!Y2tt@{nTFVF5XecuyYSU!CYI{?0QJhOFW_}p{~=XG6*;L;0t%p}j#_iaVRy^L$UR{m8t@4@fty;$(mV?{9;j#ka@z8O1b@fC^hqA`Y$$8@mAk$pIYGWY(#{6i9e6&>GS2|WAq$D+BYuxA%gF#A~JzxD1io>d(MzI9dM>Bug8mexZ|hxQVEHyX-HZt3!+f~v~NZXrPr5B=ZmU>aCBxYYRIw5_Q4;4wQ!6E(V|O}g8~QFxs0^C#*eIyCy^qG_zTc76q?>u=nvI&DFOZU~r4J`plhV^(Sk>b)XE8C7-HI5vlKv6lv3yK1+X&_~^Q<Fe|}TMXR$oj#xx<MK$|b2oi+<-Ih*;0hW;LN-5d#Q2MZ0lK_9kG%iMlah|-pXeghk4U#xhfp{{EEn8L?ATDw^zkb5jZxwmdM;*9qS@#xa_J}Aqg-hXFO(9IuGk;<v<3!eN#6Jq-&Ji?!*cWUI?PYA^?f0nqS@|Jq~L}rgmwph{2Al(E=#u-{Qlf}ezsWLHq@1v`?|d)3+K<tCf%5mFT*`O_8vjKpWU9>*rwAv@pyM$mv`N$0qd_4Up0cnv5~0trmH#}o{O%K?G7pctk$g^t4>32PafjJ=<~;w0d?<@b;VBt*5IeeHtIG(U@e;A`9zRdwOviQowXo}Kwp*<w0MT2K0Nb0KGFB=;1E#NV7fT!V1MTYEIF>Bakc63yUw=wA(i=5Z^C`>-~ub_izu+MGtbP2n|GyW)9~aby6O3RBRbdTq_w@ofK?kc+zrDlp3#$s=SYZ4^^<W8ZdB89*r52B%6Gi$4po88sHFio%jS7nT9f%E95`=44>EMU^LxE)es4^tbo!<D+|Ib!ALZ;0qPOJLxootZUni4+J1ke=61T+1EjUtx+h#RZ&ZG9N-><2I!&0ei%=wiZ4zcE+b*+wvf^=iks4c^m+s!IIt(|MIup1Ko>I@Uw;e`&SA(rol>9lfz?iaKdo!4brnS-qGRHA|D??zc|^fhnRm=gZ(f|1u}SNLtKUWDp)ryU}0*EzuhgOKTdH)vO6)qx>$#>niJn?I+RR}s~pyU;Wns)K4jJO%IeFZ3}jevoxBLj!7}!ONS@a<9kXa~Li6C60*E0$%B70=)$UvOZTG0Q-cA@`h1lxo}8r#7E_y8@i03pu`Q7C1qW%sP)>f;ZZSF`YSUnw0+5Srnd6JJ0vJTwR-lrf#LNL7{BEQI#t|NaU<YO{`?a(tB<A=?F}L5vqkk*^`L!#P#as10w4DJ<vqPuw{jV?R5tP8Z(lpPkA8f0=QklgHEJ0DGm4*D#uVEreXP+mtI(6IUB@BM4X;tt;f6j`YgB)3tNXRzcB?TO;5R=pGXWp;x#2jbXVMqHKz!sF%iKiUTCk6m28D`8Zj25T=0cC~&w0@S&U2?e{IUrjOknTZo8Pk$Tm_nSpfP&|I#}J-NNayd*P1k#p3n8d;naht>J4+mm(AZ3Ue<&U;|Ryy+7(@kT~KYOBQ!DdxoB&^^WmP6)a~jY0ebZ&JxO}p4_@<6;f#3O`S@195%{*5$aQ0IAA=P3V9VS%GqvRBNXwi{>-ae$W*RdgzjHMrxA#-wf&O2Q%`^w348m{T=gsqcd_p36<Fp)qI+BD}4xS^q*!!AFHU}@)%GbUZRzfjW__5@yd3583*V~uhXf;=&_m#D=RmLr}wxk)%GpaZ0F3-(aktC7@QBAixwcZJ)p6z<CDc53OFp>Hz-t?2t*C%J~gAvizUtP78A+-O5f!9;(*>&?x%P2a*+~yX0ppu2CCo{!<&7X6i2bd;<(O9m}cSCHae(^HNPs;7koI_+++^ywBV*<uz9xK>+@rYo=G_lp?98y4SaS??r>%7?~rdQST-d&!)3K|tNYv@_~c>j8=o|r7OAG`aFz%K1AOwH8Itk}1zTeF7LIga=rrp|3wSzt?}&te3k5E4%yMg*0Ehzf$B!i9j~2^A2LgQ(BG*T1{d={v4Sg{t|@G2&XYmRC}!!fl3iHhZJ>*1V~?LN&A-5@>@)cGD^%uHiqP+jLukPADh5Hn7x5G9kn*{al!vU34iL{_~QxRT8|UxAtd?Sx>O@b?W9&X+BEYBK3Fsv7x@ZCnH%hS-JV``>Kl<#B;^HI>7Qd+{vplAKTB>b*Obyw(0C^*A=l89>QZyg)GkU?ACaer?Ap24o?SIB=L3~PC5kIKD?;fs12`Z-*LgiJg?h#eYeSRWU9pRQncd3cCl;WK{WCLnh1S3HfC8QbQbXq3>%dfQ7|h5Mj1~#YkW&y43kH4Lh{Ix1`HsNq@H#B_O5lG`l8H++jhF=I#J)W4p*t^{;HRbyAP#$GBvja%_S0$0|AM>KYsefP&M2uMAMy4zB6IbxcJcz9(0gaeW*(iPEEgeNW5j|MQ7|yR<&oU=|8~cj~yzE;Te%0UN>QM7*;>FPnh4$erQmY!kxD-hu1@#JelR1Tgb<Tl2(G2$4mvN$PzJAw}a=j4$GiY*>@#~6SLD*F5ggg2Ycu8JdU=<IB{L|8P$OfhSn$7W80&iP3Qf48rG#6_h@gzuZ~Mh!z$Ac$!xix5jJ!p6wcM)SV{Jk5gmU<)Jdg|T?~{qz13~|*<G<=7l5edvX$2q<b19@K3cRMO`fE-`0!pABsVUOE2vR}-ornVr(0$6>ff(ZulAyTbP6E%_k0EoUu-4)>uj6yo9XUm-CB_Nmrm6A29MaAI(*7NjfA<^NK9@Eg=9c%!an@sifZYw_mFUN*;2k^qEF9BZ-(T}QA&<r|Dp}{hh1*zQ0-iL)cz0&PiT16UvPH^ZYEa{IRYQQ-aT(bokqKDG(uga|A`e~%QMC6cy1<Y3xL;QT$*9Z)OAqZr4PtGpcx*&qaYt*Xy|>tFFLtfys2d7nF4B7JlE9^h4dn#JzkL`UIChG<B?2-mz!*=^ZCI>0?0X9sS!<7FAm=4R&VEAMp?Kp82NTgy7y_{2L}RS#A!xYy?$C`=m1=e7UYW+k3Bwzl>=P^z}f3H>Y%PQ;x6CL6_$_9Eu#M3>dhXXc|=)es&GKrO2~HdrsIYg+P7KCJ!ei{`wRMOf=z!D?8D7>I2z>mZ@yISh-J?=eK_1UN8PnMv(L(Pj<-6RHd$+U$!sz2xIDap*MxPNrvzmiB|rDb@Nzp^wRC?nl&ZM9;ESJb%#Kj%T$Fyk4r-qMRC!ig@i&W_8R}f0k|8#Hnvd=Z17Cj8OY3o;n*m8HtQozYS2Z`TQwM0g{ZiJ4V6y260}A%imbk;@qyAX!(!ZL5U~@9_su;L4MDhOa_^{a#;9+g&wPu@eL=#)6RY!L0bNv89YGof%IeKeIq;jzxZY<i_-8K#Y^7|kz_~vK#l=>75KGxgE=`r=wJ(D|s>9pQk^CxkEIy(q?8lTu{y+=3t<Ka|vRP5bb5_zJ*L=Y`8lh=WhnWK~jZKnAc?wVx<PuG*~_r_M>+N7+=<z-!a#3nrM4iQo|QebvFVblI{)AI)~uQb*j%X>fGE9Kq0C$Wc2o~_fc3mu6=P@7qk7Zm6XdV%L&%z;2yl=TI>`Tf!)Z~d<3XvHxy;4TL(oN-q``gTT7FvXR*F?~%#@pgLJYEU%6hYP|yJ4y2BC(CEK<`B@lR%?3u{FG=@VdkPIssZ9#i<zi`&kXs|Oq^||FX2tiSFp~Z)tnwS)CU~m%5%&_nfR5*qpwI6M&4|co8Q&hhrM=UUKaGz1o$a0ROc<xoYAXs%qCM-w(voG8MJT2IkKSaac|hG%O4)9<L>h#P8@1b9^;J+{bK(%{N`*c#E~5aH}vvjpoihL%!g)#P_W!Ev69as`XhN{*+w&C*`l@1WnZMWU%EnV?K&`GfV8_lZ+<gV1()m8oaixnojM2<@U?+)?;O=W@z>6Fy-PHbukR<>o5jMoU6SiqF_UFI8MQN#-d&V!?l)%WLhY~zwDtRA(bW>mPxV>;vU4rzA{Y48{|I7uI9I?#z_`c#-Qte065KA}wl2QGg2&9p-1GLtXfL9KQslAUH%dq>s_pA>b);^czVL1?VQJskD--f*Hig>+ptjS~_ba>i<!^mw_Y=QIVpT}fW=ZZs=r=S3^g_J2wxs${|DiO%vS*^TVm(vPsMz$`R!iPL#r!}mLs}(j(Bricx1FYHIrrn}CMK;_lT@Xff4ooT@ku+wc4q&f%Wt)xCo59>oQs*f>9`N`lU9NCdt-F34gt2-==YqrpX4-v(_XvddvBck@KJrNjWfF$i$}c!3BGo}AEfGU-_bw!>*CJT>mO2nCqxyX#X$Y*j+?<8v&{&=Vl{Hj4H{nXnzn56xAkidQD(Z<dzaI)(>YX$PzkH6`D+s5>I7aA-|nJH%x(V!AEwizU*620s;BZu{0glWvVetveph)f82i5v#_`**T(@%nEZ^HjXM0<=lUSO2{2$l7yBlWJNe1~zYoLS1aS_n?bJ+{E?h<}2_ZIq>IG>um3kcV_MJ5Nm4X_^0t^Q0|3gc5WPM6lv!@6ajq3dm{#Ae8T802lPwwZTvzwLj>Sy7!0OW96~u`$W;vT<nZynB0sfo*@T1+@eS-bn}5k};-uT->A0cxTwrZkwN$H0M<JEXiDu`nkSo=R%pXWDV*B&ChMLn{L&E(e~z0+A;tqZw%*776)!k@G|)8FM9LN(RJBs<vbhDeY|4+==Vxv#biU~6HJhMvUS@2o_J+6M?qi738KfcJEC8=XqN3ISckBHfxn%}ve#^ovTyJCXSH&mW;_+@M%tNPnA463TU(NCy5R!j=8!x@p7mOL@#(3zY}3#%wSBlDIkyh`!w1Xcg;A{nzR+WQKHh(7&u0(9h3%k*@OWBuX~8t1YwbqY8rfD0Wo%n$6qIJmrj$1HmQj5HbwfWth0ZAa)zp2|OuE&Zzv#zzDDsYNe8*t^)P9Zc#xA|yIpeRT$OB$8KWxewwexhSDlKlfN695WoUN3{u{YNCU9&L4pR*YUBj`~WEWE(sZF^*Lv&!f?e@vK0>%Kj-xxhUr$dH@)6Zq3$hL=`pH#UFu@);D?r-`0Q&8Y+mV0<@ze6~8Y$$r>ndY`9LT_zK~qG{3Pag&?}YVQ_P&J@XktO-FARoY=kic@c2{}BgIa%dFX&cN2Iaql(2hfO0iI@FdhBixBTd|lK-V^~p}^F{k^?tq?1!z)fyJA)LSzQAZ~k#PM!`rrk1YfEIHFP+xjV1I^D;WV-z14aK9*=yMo5C~!m!0Il`YA|wDcn&mRtVM9ZOu#;=2`*}@o{XO3@%QxX@s%cnY1KMwRw2-8mhIJdPKp}P>})I9B1Vo><{jAB+@JYf)&{zbI`Ea4@K2-R)pM{^n4Sp#y3fL?-_lUE*3G5ScnrrK?MW$*f1?uK_5%e~i{SQKx3FUCvwL(@H);koGl$6c!Sa3;<G90yc~lP8iN5MBm5zHIWdBBNk@B=%uU@>kRg+3`%YH2^UiGwdhV-WMxWTW`$>lr+t2%vRHfPMH#s;Mkn=Z49$pfAmOOJso?p{39`)8s<-_DA$)RqX=MDjkQYmcw;`8}@Xa{e2&enY=o-Ht!U7RjA($<3CLo#379T?lJMN6E!^<lox3Uv|1xy`Ca1LWwquX@AVW<9*SrauQH&Jp-;aT{nf_I1Un3;{{Y}1vYA|>V7fWkxQpdRwV8fG?*;=NS$X31BP>%U#xcrYH<VP#TjtxmYY2vTljRE;{P4fWRtDo9zO=nSxs?{KCPI4f$hVgT7Q9r3o@-&-s^@iZ!-yZ%)u|o5sg(JqAoc4;5^}~W!<H=Nf&W5sCe12_Tt}yMV*+%>4~)z6p_*w!akU0woe+vuptr^t^Kol2WY+Owxg$9(OZcL447`JH~;4v`_sq`|61%Z@t#;`Fr9)*dsy_#Yqc54k=9(F32~^BPBdu?T2DDw@dQ~%!=>GN4?61DC%5%d6Boj3i8m*U4_f}&wF{XAe%W*f$Oqo9AY*_uldsMh=#ShVJC@7<1N$~S){js*#Me(rz2_H}6_OKpnQZ@WYLoboFR+L0?OR;}QJm|1OgsPMDqwSaDT|MfTmsPncU2*9-|eCa&A#@w(nG7{5y2uvjBC>i(#m7RT7e4*<{X2^+ll+83_rtn-KDr#eOWrVW;=P^S_<8((z<s<zkKI%5ynF!|1vUCc^ZGNf<y#~(f9s7u5vY}ZFa>fU3t_>WbcHPtt8m|N)jytt{D!voQ*0ILXeJ~{eGbNmf97`=9fDx=MXcI2IG3ScdE53++&28?>l})JqW{S)i%NADzci9T^yIeRywbDC3Yvad)hsC7R7Zs8z#*2V>x++*oB3@D>I^yzFJ${G-So@R04NKZZ!MNE%`btrHYQTAY;q35B@#2?`qcEXzy={1T9$O(iH-Li+F*U#%=<g(6krNk=K!3jy`GA6}?LHvU(k??;kNH9|y}wF~2%)Z&B<K6h1ZW_9<0ra=dO!x5?`m3@~(yU}?I4&bI{l*w0#wt?fHoU|;#E?e8YqM5wom1FNS~j@99Q5Z*rpBI7PwjI^-!tesRX^;2oFo9_uOV;{c0#q;R(x9+f0<TczsRfKvii>a07JV6)AcB5#K5hCZy-gb1Z+D1(Dj|-;x@BhTW<au5Tr<ZWtc^l=pJz4D@G5nSt1BBx7%LnaV!+tzL9t!8zcZfI1DAoG!;|s2>4LGQaX~^$ADzK~s$?)1Pd;6w3Q*F&hrZ8Zvs!IyX+o@x7Eh!P~hpgByp)6PB4O^Thb{Q~~HM4|xUfZ8eYJZkC=)swdZ>uYVF&y76z{|_@lh-^roV3~26);xWy%b@agUcJ~cY4Xbtv#-KdRnM;g5fbgnwt_{O6Y7oBM#+EdJWr5k1~wvYR~Pgg%Aw~S1D1T6EGB`@|~^?6jom?rj<SP5w_3pbr-iDH}N^-@aw9ciGjphAcf-Hnlb4LDvJ-es|y9Pdoc>Blhj*YD4vQx85u6)+JXkULhJAs0?pd*kt=QN_Iq0SrgiamdUtbbom6Q&$G=?4E}_4QsHpdzq#m8hlc2X$l9P9AtebgKt`vl}WzZdSe;H5pl%7z)J$59(p6>|P3q5$egN9fUy&EFv<N3qz@Y(FOns?2ppH>?CMi$RjhScwRe`L+F)g@}RoBU<fT$y0!F(!Z~!#HI98;&4!=inRnj*7E4*0`wiWjpi+r$8PO;n70HwNlITsx`!?@MA7<gP?DB;#&OP6^!MSuIs-|Z~tDudfNm2?6>3y);iEuozMf>AHk`#d9A6t-v7F*+GBkeM(!=!!A-f229BU*%gh{7_$1o|H*xe*k!#LXd97)nn#<lLWyGG-%-Qf^r-F@$x8AnnY)6&LXff%~eaOU;<oGPh_MyTC>)d>vRPFR$Ov!OX+tJ3#hTk@y{PiP10J!>b{#44->mrU_ZE_%vG^k|lXuN+v@m4-};ZIGiAyf2n(|QChpBuX5jZEb&9*W)-@oUq95C-<6re)@&B6VXWP%YCtec!YBvAaDzYsQ$b#$2Npd!E(a;z9AIskMX%cBjo9fL`Lzo;6H+qr7%|?>0^tQYAtx`a~0cv7kBXsQkv+uLEX%noP9L+pP9CB*T_6znf^MH3!w;4^9nkpmO=O@b_XuJ+Wi*N$d2W0_*4eHO|%p?<~23LB5>9i*@JV?zq}-rhUSen%U}BUHUMj4pYly>3k85FpC@sOr$&KOy0ivVm83!tNuXE$45*%+r3LrTZFlNoCMfxBPyrrEIM@{Yrf(p_=d}qLo6cc93++cWq_6O-->=W^MrS3y_aqi$GB|HeVd=dygoe#qz9z)hSjRYkR_{6<T@6|%y9KP9*CuZP4VWs__L<-R1Ct(^Fj72zL-R<n$h}j<)ZegMU7gg*DBsIP<GEJ9c`o5Yji_Q?A_wwP5ijqkBLz<f@sBQU*$Zth)jm=L$o9JI1&2}DX*IS_s_97TB$&TerWd~ZcwTB8e48=2kZtE#xjVNAmceKq6{%XI%w<*l^;wo*;Y^RRrgx6#WMAjb>%9o0DN3|VcC4-jDL-0{1PuA!EBqP`K?$zjo5<Q?YB<8y3W*_SnFYPO?P)AKBa&|$87jk(k!4eGF0EEfP206$6du{#`RUb+r7LxQYf8PTOqY?Isu<Qvb9SbvHW0*sjZ{FIaK)N{0h9-5+eBHchKszAIn5-l(XhG^j_$5H4*=KuX14feZ$K-EebT$U1>ER^I;|XYHWgbk<a{?p*D3mGgsukJlz{2dpUe#`n+13esvQuFkaQh-Su!bx{Ibhw#A5nE*ZN=yv_~oacZ+MUBJ%6+{U9@yr)p==_<_*H_n<_J29?%r0wRu%`ChFNt!b^?S-Sv9+rOLtb&QE#nowxByQ7w^Kz5?$s}Kg%`db$xw<iJY{?GTX7~E@w>;?6Z*)8^D)0Ha+u>_Sc-~7FEb*8=x2oqOIQK8~dp6AsqI^s!mnD`yYM-}!kxk8fty$;l(5$y1gQWi=avy^>PTO6ovzg)UWaNTNrPaO<d;KGv%CFtH`Sq?BB;Z$O%Scp#_MLE@m*4tfP-p~KMh1shPVKSPa}Jmf1<7gt7%UFeLI}O?q;=ZpAv1Y@(!hS0$ff<fM7s~b{H^n8FTc{JKR&whu_5#FWch6sv*o<k=d&t!?DO{_q5{}`#LN1k#wwXzze@D>8Qk+KC<mTzp8mpsyin#|>YN$3jt~!irEE1XyS|U-?NVwEuaNRr3SIVgpH`sHas+TK&}4J~x86>*c0O?|K3*(jy4L<me0k!5GAE)MK8BUvS;%B}LjI&zwyPKqMr@u^_D1viCrQe<J8-p(X3b$z9H&I|=holHjo)XLN)=jfKhD`9$1teZR^oaiwK{GFmQ#S4+9Svoe#brSoWoo7$sFb4u63tI`(CX~5?c-3Oj$P~k>*ssx<HF7Ml$>3il-7?kN3#k-@T#oy^rgY<A+xk4eIFoV`(2s*tg0G`+AOjsA2)6#sd{jLv&jB#EoNCD%@?1H^YJ59$hGMbZstf7$^}sB6~qn>*%@VT;JR0@S#ms8=}?vfaA@zozxhQ8ddiGyYI<&qUJwi;XU)t^RzLV*7NSKy`L@Z@#S*{k#ksm$pX-n{N`>r*i2+92)B(Yv2;pYT^SB~=BWFnV77(aE;%uWK`UhV&|g2-EpRz@KzTxJ>c}hHh~Mp$!RRur=(bx#8~{3nDy`T<AdyVlbi(<V#XF)@RqcDLAer;`SdI}%Q0eU#?jPA=xL&pn@PMWoRvj<H5ZJ(toGNK~bNWU1$$+VjUo_~q5I^a;)r9?-?z1Hy@MzkB`(4O>+l9V?h{I`*b#Av!)9SWNz!wR}V=I$!)zFs)gIueJ1=gXP#Rf>k<W;v0&9T1*<+r}>_bwrB8|b=?w=Vi?+rR>{>}8F}BFe%8j}R#ADo$_IX%R{5)kG)zS=zb$1?YINlOuEf#2%a1FxV=&Xn&bWX*`2bHBoUPXx!21!=iiM9_-hz<CTFIe~;U5{%3!4#e<v{H|!2JPh-8X46x(ZO65a(<E(n4t8TTS3M{$GO3M}@xG_JD8O7k_xQ49PtIR5)oA35VJ_Ng_n?W|D{h^M6;TPuK>4x$8qievQXrE}i0C*!gi9$Ya@U|naokMvpD(~o?_c4RseMuonkMTea?FQ^#{chKJ!ARTS^rv=utTr`ri`X-8fSmg~@>&;{hG-p6Cz_$9z3}Cos@Xt=c9YC2ZM2&vjef&xbP8i-AGiDK_I>9X+8kE}K{G3zT|eefj*s$D<3Md%yJwM}REql|_fQ!vYvaXf?CrlgDoP|yT%ZHOqmn*0Hj9l8)Q$6T65ngWu;!O;Dh2ze1g5;0Wf=3g<42H?%(u;jTkLc;YV)q^i!fwiui`y?<mC4aKdk$ye^t&k<w!`<HvRSOs{NVD;K84*Yi(2KtJ3GUcnEL<YZ2z^Brp3Cd5PM$Mj0(wN3wyLRSmn>dmp<bNPTp*HApFk+^|l#gh!F2jINPlW0xHK_rj%3b|3f#0|ukkAa88pHXm`4dyj&ovp8<lc3+}!EuT&Hw{HKLN=2-*X;ypkhLa7Vpo#Q*3)X)yaAI|&Q^a6i$8O%fNWJ<7irVroG*yx*)=iW;eWrJMCJ^^55SZ$9kK(g6Ks1dPa`i4F%j)xIy*oqtkoe~`)@3lJEBKPXS0t3e2Gg5ju0>UZ`G$m0`J%<jY}9y1dUdbw+}mvaCEsWYw)7}eo^>FnYECgH8vUSZVWC@pDI*Ch)6~S%ErfwY=xXcwOn7DtUvY_B%YWT_HuG-t?+;k$HSDd;xFX-$C@||9NUz^9u_l#bxo5oq$r>BqO|sPjEJmT4B^e9Avy4B<#oxW7{{x`MMHu#l#JoG_*3rc8@)`r%gl&;qbuPUil8Ef6`^NM&)vdL1Y*O#~f(BQ)5l9a|(fLgI>FaQw=}9p?Szi;(2TeW;k<y9VMU@>7KoVnjba`0qaPD~=(-!B_CM;fW8XZ5Y#Ful`8BZTj7+K%%XoDMW+|EZjv-(cWO~eF5*#<Y!Q|-XbeehQBYz@_*Zb?>f@Vv9;f`8mEx3%H2ysS4#C)ELU9iO~DALwchbSAaUWe2KeAoYckT6h!=3LK#Q(I88&#(^2tr1!YQ4tt}*IZx`=4qbF=?P&M8A-6R9=p7o1-imDD1ijqN8u6s#D<2B}lL__(Q<TmwgsDzw+(sZ0O4>z;|87Z3u*Afn?_9trYD3m+7H?@l?yZJhM-tbj_=3Ala=JDDkd#%wh02oq!yQZhzQ}X1LwiING33l#fc<vz*TsQLpME?BXS)4%dfn#n)IZeGRsHomD9`aHmEY+s&wl2|-$mfp-yW;AohcuD`2M$1*yfb>5col?hJst-<-(`0=s0D-X}^~jd`8Y@_ldYg$tA2i^hOX6+-rZx(|(u=M0sxa^&Vxiv+y@QF;n4~Dw8Rn)!kw@dW}714uthxeV^wC3R_3h!6mKrhEhl)6*}q4pQHenc|L4#zYFa%1jB+~_e!p!xM9WQ^1d1_q2jW1`2}9l@*nan9l0@Q)N^sUgFfDz{icm-dX<{OgS<aZjj`qyKct>jW=&(02Y#0t?906o(K&2Y^tw^MUw`C-yQxOviFL)g+)j_?pV_TfKD?#L-Ga5%bxraflbg6$KkoFY@%nv#Tcy@Cv9Y8u(A)2{q_c>E?+@1XwxI3yZSE$o0<(1vIRLGe@2eGpovyGSosGv%Uk?i3^MNyX^Z3p5$K2Z^al0-(vUmEEbNu!9unv^jk-)7hCU-io)<POy<PpeKC)&2eLTH*=g1(`b+SR^}KGYR|%kS%bpl!`NJ}lbygshHlU<HJMN!DGJy#a+*EJoCAK03J(Umdm9kP_q->}-A>4=6m%<VJ&O_P-<Krpa9b`ST$N1w>ovUr#S8x<Dk~tb>PTckYY}p+1l+1K>S9A4ds9e_YSZ8V#v|J8i*w3SU3^fERBZb{?wjQ$qCW&9SpJ571XT4~gmK@|TU^8$5Rxr+S6^Gk?SnqfN-Z4?UyB4!ZEsIbz)fbOmMFL|kQC>7b*T)~L|%tF7>b>>)P~fW8B-$>2#;|AIklVvo)+bew=a!f74ig`KjVrB&3L0kHEM-`};-w-3ro_x7ttp?>PAPgd!TShB%hYHpJxp8E^(e2oUH=ijoc21oln)DazKt|`usDjVzN*;7j`4r~$YN&Nfb;?!lFPIZ@@NG;+UL9-bJ@%n?hQ_A2kO0Kj6%#-ro>@!ESy;!J+<M!S>-Npn56QgA?+qMSyukJ=KTaW86H|^^Ow1S{n4X5h;A<<!uuVEkBPd8B%*PBt#6^PY#b=bqs`ZByvq2Clcux@zxCfPJSLanu8&qT^;2F*q7!%0Bsup`y-p37Qa6#DWm$Ogs3xYySHFz7z<9o48lH!oEs0K_^PAAU)QxBK|3EO&lily|Pa72Tn6OuB(i3kp3AU@;=b{*~PknK&W6__m-%2oryH`{$o3pGQhy3p1@hrT?P-%^fPrvoGnZYJXl6^%p6g)MD2UJbHsQ)zbwgm)LBRi6#48;rs7vWXn~-AUfW2I8WyEAwK+W4yW)vXHbWpc+*QJ=Zhybx9C&ZBg|hVezt%3{cJuzaFDVkRl~1XoSsjrQZ<3sEs=BqF0p((|7*6RM&&O5MnDn$PMJeR@ouWhP$vSa{nQJ?5QDGU9oCK0@7~s*OGVth8}h$d7zHe<pQ>Gwl5nR&26$BBdhuWe!Iki&qB1=`PRcdeH4Iv|fl0;tE-sx*AZh?4z97FD7i!1uFtvPGUgY@<ViHFRU}A|0c)Q!9Z}Z}I9U#v+5+;bgYeGG}iHxTp{>BugXnfU0Sed^2U$#8JyX;XNR(@_|$!C@|+av3~=k>U;$|tw&1QO)4O99Md$Ia4<^J`Lhq5VQUX(-pfCp&#;4r`?IEcO);Mh^2fPQGgg5F`F7$onikeMCPokz8_>-}vY!jap*R%%V0r;V^E_dP8kkgW99<sEh5&-7}IQkLO}02UWkhfzXuCe~{B(=Ki>x+4tpcM-9W{^!eDmZmmdva*JhqHJ&#QV@mt!BH{!$BJMo2YA=MIuaq86!aDS`D@i|7?MA^ZD#@nSyq^LiYByK>=m-<hV2b@&kAjK)V1a!^WJSnNWNEr8ea<2YCNF^%%@)&vkGht9O#Q=+n02Z{KFOBDFR|{lJ*wdlu3&Dk^&)t<2ihw?&;i&?gjdOrn#r-YmRrmI1a0=t!%gT9qS(n+Rt9q3YTzDpHW@E{e{q3ymz65}2U|<vAG2&=#9KLkWT)ctE?&8`<#?<u-jsRWc09%|p1?lFT7hyF*23;ViJux^%{DN_r=sVtqDw%t-d~c!Q2T8kvKN`&2EBouRmeY$VGrxPt`3OG#f@XxK^ES<fq?D)Rpu&Y_w3}~Q`M8^X?{Ch+h8444#O6ACV#B^fkL_JOsi#OsAeB<QrWM?j0@_5A7{XF#X6)gcSf)v@!H|6ZPtGcv84yLeC<f}+6+T)b$9EWCT82MYnwzJig>S92`-<(sDeWqu(L;vYM@FNp{|nf$_H_gSUB@rbeUp6dzTM2Rz&AQTxU{vslbuxB4OErpaKB3<h1kWzS^wxSym!<Nofdy_8t_~${&I^oX1R^<QDBJ>&b=gTaP8upz_|K^;6B8w-r~J&Q~ntAoNY{J(*`7Sg%<}^!0w?2cgqdf++!q6@<Y=UJxW49t1N*7W!KLG~^96DM;FGp!z+1h?P+bz{S~NZr6DO?_CvNYqL*$rn)9zD$f<bpOApOOw*L!f86#k<^<t-+2IwWs^?t?`}Ye1v$xw{C46lMZIoUv{l^sB=k?3wXAGy$A{d+P*LG(Aoy1+=XWY?#W7l{7anwB3ch{59bw_f1#m<-IciUg#<#}NM#{;t+8MhC$WG3_bzWMh1+YWWloO3-NB)DI!??|hUR4RX4xeEX4K>ZH=r25*}{HWVmwefc6$+lfP3pxE=fQ?@y@J;M%UBPW5O1|bT+!Er!WOtaH1#E*3HWked;Z{1F%uSBmrt;S46{CY|_5NWAx~-T)QID4H$Y_7hSNl=yW7OO=61U~O{I;|z-S_*sHs@Bb$jP92Ek<XCtTk6F{?6rJTKe>j(-K(kqdyA#e%`D1v40*sq`Vw9y~xXZvseymy{1XMXFX&b___JlA(zJI2w&}BK9#_R^}&^q{Qhe7W|eo-8)&Jw`gif*z~-bUBKu`E(bx3O8lv8TP5+7w&1AV`lIl%tj|D@mj!l2Mg_~ZVK0KE5rwIFpBsO|8f_gWWzj@QDDx|)%yWjKAhL0Y5<<_oDp|yv|3iEqsMGz%<YEAe1$M#e4{gv))^z)E<B~(arho%YW&t2u~si?D`pbQz|?wC|Z)m1od<vu*%=PR7!MZQ`49u_?lqrG~vJV1|QU98W@rM?9^x91Ujj#V{Bf+}c&DLQkzYi05G2D5v42Yb%5s8fhl#5eTC)b$%~jG01>0H+GuW!CD^{DNFm1MJ85D!wXv;t8-Pi}WNmwO%jpe$vTeCym8^Ii6H@&7%liKye5#iq@{4TeGkl<<y{fVR>5-awzU=t1qVivR%vB5MSi0Q{Q3G+>J+B(16=dU+Oy~3D{M-i;ZuZu+Ufda{+D~?965rpO#4ePxIFT65XCi(LPheC4-;9vC)~otA$-lvtMQB>wXi3okqXrI1nmO+!0r)1RSnJqpHg0!HF6d_#CtFskvL@lQX(t7}KcO)bjy#b*k>uYv-NT<l;j&sua_kaXK^YQGb}-kLfEy=J)Ofif-Kdp3YvugS?&E^xD;D(nvxta-P+;ck+a?<<-YIq6XG>y&Y(YjpO`z>h`^txGlWj+$~?0D16VAo{ht2Ls&`RZg+tZld1LN8e47-H{`wh6<8$zDr;_2Wp68D_qO?<A9bH$e@_4@UT*IAA-=cz<sZI0IyG%T>kr?v$8`O~cY9Z~jNIC6C>QLS=|8hUB1?1nlt!umIG`xTxk1IgHCy)&i$ZPOtkK-Oc5J-Af%f;_%NLKIiM?O8HUoCE=NRQS#&+Gb4?QBX8wfSyD1QK=oNDO0IWT<UvR&&sk`(t=v+umtyI*E)tB9N9PiUBqdh6T=J9c+DHZYZZGA&DB`Iw2Nvr#Ra=B_^7b~WPKw<?UfQ{YdfcDW2yTeEZf1a23HQ62M;nN<wusvQ*h<WH@8<iT1`%{rydcFD6{p3`Bvi3DXiTHJRz(yM0~bH+b2b$qwfV|alcjjdALtI@nZ0TmrC#qItu;Vj`-tk(5#jhxNiU%Fv@^Vn9R+cpL(=CK;))8LHGZ^C-R8}C&)nCRdSfA|q2qk?<(>Q$~&f0u<Q47qK2pFrYz@NG_aRDCu^JR0(-)iqePjDUX{N_~G1c1C8ob`uTvw{ft2j4Dvj2h?mZ9UN?CU(1zZ(_PFOGpzXIo4nzj20=s%tx*#vuCNQVH~0gBdm*)gY2~$%j`EFT{S1RDdy&^a&E+#{7WJ}{*R9>HHV?O-aWEPMhBOV2`(a&<=7X1|OokQyNnY9$q>c&zYV(Cy<<gdA^J)B+aV-wD^J}Hyz0W!+ZHJPN7cqD!(7km4dsEM)TieB7izM8p^ZS#WgV<Q&-CqWNbWqV6m;RQT+2~)BOTm_NleQyFI?~bhmb^}nlaHNGRi5I<*HZf16}yzd&ZpUbYSL&s7i-H}*c*s^mRBm(ig=m6zDHhRj9UMZzoo}RM&5h$?2OSCcOdABakv`ZE1<X6qPpMG;O2Aa*7;)giuX{EzimA$O03JZO>Q0u!lu#-M-zg%C-^<w2NH`~o6gQwuSi4bLl~GG8rJELRgTNU&4Iylb1j;j+S}vLzl4l<q4SakBLJOw*Y<8JM)Qex3!dm1M{MdRD3AW+AZsgP+i8)oJt9u6eZKum=K}k)e?iqk<8ashBEM7X3}flaQJQEAE#rfs*(Xfw3^4G~&X;U*G>ND3lA1}EK;t*Vk!5F$egtrJZ2OJte*C--BE*;b<s!0%BQbrTeD9suLiuZNlEGK0Cj8@44&I_pRXFdWG?#d>oUwu2uvdL>UqZi(8?|D28B_<&nI1pq;J$Oi;VUE3RrK!^hSSsy>*jKN>AVN?8xhIr<pVa~S{jYVWWyHAA(d=V;$6K6m+;A=KzY9Nm`G<GB`cAM@!MhaqvCaDDSu$loet!$gRDHI8cAF~-E7b0Hh%%CV$oaW{3UmPdR_0IcJN0MfcXJ#fRK<g`_x(-I~c7IH;nGE?uZq(PJgld%qQ$KYt>8@g#D=t9~qYt@9(%F)A7){ShZ<dUN)m-K%&jYn^31cv8W?Yb7wEYOIUdA96*7(!R+?6z<AGLw4bX_1(wRkzNgpJ+s6eT);HxFZ{cm8jhOJViw9M%Z1jUUmtBozuj78C3p4}T3wgWgppEgtzihZZ)bRTTjM&`4vdo$DnI0A3^VB~nz|-Rpvw(~CsY65f^Je$Zihb|ypRyuVMHTP8`<KkoJMF#GP_l)oQ;^*~0A{%1(y}a-Yq=yOAb91~MPmSK(rt5Lkhn4QcMH%3$wgw1Vg7A{vK`zLdP?Q#>UxLX$Y8Pw0?Xk0OD2vwm^A-}b=bNN@)fT~=cxaOfp(<uE~(hh>IDgO2~q9KtrBC!YQTK2R}7@J(VlJ=K-yRUcWv_(+czcNtgdnrw)ZQAlqOS_@3g*(`}*6S75}JYV6v@hSG?V9LAn{7*z+FU1YZr5qa+lDeJzN8Sd*xCM=ct4J4EkxO8s6h*-!;Bq9x8A7ZG+k7!~@@+@9YDaD9{*vIbfnsR=CoV{d#%nKYN4z+ReQz2`4P2qqVg`|Zn29zwxXEqgzi{KyGt=`T*|AFfV9)*mB+oDZy5Ze&le)|EYeakSdp@%#p?^>-*3l6XD=BUBWK>cxtMutWIB9(}=qKL^&8RU0`$-E>6m!sY6x1!WyPYyr_A1)0BYV*qw)hYz=#?)x$SbibX)#tDpX+PLf`-d)fDawbpScO5jo)8E6zhMf_i(QCHdgJit=S!eUU*%`1;P>MN*E-Wrs4vNBDZf;&R36luxZN60JMcN8oO;S_)vzgGxNc~C8w&$$){=_5N$gF>}BUYkWBO+3n1p0V%AL7aWL`~&vUT4tpbN>2SfWuy6sxG{D8o0UCm@lgcqv4qG53PZAKD6LB>8qFJb<<c?_A9HO8|rcO(x<2CfUj>-p~Q@rERWF|k>pJzctb+DZ|x_}C~4)ber*!;)Z07R_59gi^A^rgbXDWfuclAf>mIAl;C6G#Huom-iojLc)R%b6lf|Q>e*lJ~U-4U9hW>^rJr3P2jSy*1n)U6-60%dW>hZ)HJ5y0T{6b{Nwe1R7pBu@p-oT!FZOO+m;hdrSa$IVLrYs-qXCb9}LuyMZgE`6XUekgZT|4#HKb7|ZdN5gwF=|yYK&ggTr+YjByMV%`xrArb!bwt45~_uUNgb+yvuWIR54rQz<k#j2ej4vmt<TP*4T!=$vAbmIHEI?fwh-C_wK42G9OZb58*ysPjuneOe_M~F)tUFaYV9tDadswl0jfS%L~aAedIhx0cTp`$clP`_&jCErDEl#Vyu(hR#?^K?P0H)(*1??`Iy2V|{`&YVFY)hG8OzY+HwDwDlDF4Hlc=;So9gT}dh|59?ah`qWNm<MO&T@Oie4PDM>I}8a9v`LC!v*zAs^rB(wkdR{CQN(m-1@CDcQC5dXH6@VLRd9$-iEfjp>3$F#B=NCkx8&<6m&Vs3%~IXHWy`m6PLi@S^j<an-ooWMMa)xA~_n{0TYXIp<!{Z!|Z*5P5!4&02gHGeyUWH&qmExXw(?n5F1-G`K(648O{B``4eM$2KD5VA&Oupn{`6h-tM-+jB2zM-r)0C7W2R4SvCSpooKe{EfG#f%9(9cVz{>ZyMph`<)NX^t~1h{6-xm=y^~Ywb|_JTxg{69tEdgq)eYD0_+BA_f(pBi#{1zzBn*Sx<`@IFgliKkWl9h6}Iumn_}$BpL7Pxcyge<)q_L_XmmDU;BgxK342Q<$`&Y>(592+(Pu#RK^W+{Uz`lqVn^=9KBVvF!@jc-bZm7`r{e0k(o1j;@upS_*Jg|1UVjH}-k4ADOPq&e4%i&WkzXyBTs+uO)D2v!@->TiZQkr&ukQlplAHBlpelfU@;CF^HEFz<K&#Tm*6C9FdhLZ#>ovybn)l@2+lg!h3UbpbHCF#IxrStwAx1t9w<aY$K{gn7u;JIGfxm)bgnVc;TKelVRX<DvOJ~dK%1L(B6ueB~`|egQ2iu#i$=5alIGFN2Ex`faoj<vEDos_Xvu}5tJ~ZQsorgE8DwY1}|4<r~@$#cMoklMK;I4=#k9^C(CVSnMr>wPZ#VNhQ#}~Kd9OF$l1z+oSZR`_{JnEgkH@+zf0g00A<Ahn$4u;NA?D+`s+7@}$;u+R~Bd#}F)?4O5Bz&!kvlYff?iKOTVbc)U4j<rnB%q@|JlH~u%YD@v;am<V#LQN|pS*BeA>E3v?fn>Gh<uUH^~&seobgzqL3&a<HG2W#lJhp+7>ue4;NfnurPQ=NzUOEg08VZ~WJS`1pt$*Vxcgbr{k`5csu+`LHy*u!@i~c_%TP&>wEM#(%xuwOx1u#yen}XXC*87ya=LbVL{=kfxgQ6(FP{?x5A*eR`Y8g~-G93Mby4`^<=)9maUtN&w5@7c+u@}VdmGQ^j68_1BB`Bzd!jo-*i7?Ft!K~PAAqyG0PI+F<Lh+iGI|P*;EBEA$Z#8c$B>sjvo?%hMf~L%#DbpE+wUZ)K9WB48?`&n?LMmDUh6UBj`H{Fp0drkN!84Yeef58y^Zvejt9^%Y7e?@F>!S>+|{4I%j<O5hp}+1?mlA&6&J`1Fz>5pABb}-=~wN{YhxiR+=uOr_IEgz$a1rq7Gtz|&^h&Gdx^e4qY23;I}`6-2Q}yVU7lX|e7xa?@alFnH_)l?HCByPbvh(gSNJ?3I!I;M9h@I}%Z<Rz6sXOm599)S;XTOx=C!?9pN_={LQHn@#A@zuwzxZeu=X1MyF;UB5OTsa=Dq1+KOI%ja*+LfQoa8L2pvTG{1HcIf82l##o0Rrbbc|vcH3FFN;eBaqGpg_d>o_-)s+1sd^?Ye>8TYrU#}`V#Bz~J;pym<fA!8T!U_$2&Qb5Nw7k`-ulnJ537JN|S6hN|zKsHfZJOA?GI#NwZs6N;H#H#lRh6giuQtg|Oru(MWAer5$xZ|w5aC&|5&onuKB>BuKH7{4svpesS8aW%PwW04dO=_<oGoW##l*|!lm2A-#LWBmSmOc%>MY8joMij0-E<mh@9xj{BdR}z>ff3J=fOvb_SKFF{&vmI-+734;=KMSC-U}Gb8T=%qnGoMkLBuLh84MtC{n@az}Ij?eFLqtq1Eb7<L_+M+EZ)qR6%T^&doDbyH{m2kbzJMCzs~z{;qpU_HcgqCaAxThNR9dhLBtND;Lhk*__{Qui5rn)MAPC>UaHqzhm&q+jmZX--Ej7<n=a*E~131iz@xICu4&Uo;LQQPwiYaX7rY&sq1S_PJ47^xRMBVv9yf~T=`U}afhG9GyB6m`a7<czVGhh3oe8D6hBWZ05m0s{P{u*J1a@weGW!}C@FCtRIAcqTkPMDN5P#yK-cZ7x-pyN1;QA?vAiEU9H$M(Wn9=S7|&?S{$LNY(UQXccRH32@OPVGFMs=b=!5t4FqRukBq;hN#d)j#;5P#P!3P)URpJX*_UaQ388MwF>5T39=#}uO_1aPfK^;8ros6oxsOOm>g$(x9dpmPd6?zSmseSq4#&Gyn#mfJ?b#Ps-m*BTE*a8CpBM!?STH|`aesWSzo!CTA2fWh$>R3xgH%4FixbrYPe}E!Lt<zsp?}A3F9&F0j)R{8E?4k9&_kMCeRhEZop49a1d-qcRUMN>Lczo#f{Nk|b-daHv^7A{^b2pKChdTQ|BKjETuxbXre5&ov|Do#4dKOf-W&K+SO;lncC|wk65Cy~r1w<|o5K$BqQ4z5D?dO|wt(~*?wTTfJZ`CsrtQrmS0#6a?-WS`#DxdSO8}>I*QLiUfjc22cwC?2!pb|l$LMB~oW1LvZLyt{7e}dB7aJX0r+btDH8^Q$^3Z`5Pve!PhY`EaiKKhU=vnsVoxSmTCoj|kp+$<RR!uRZdi(0b~Z@;(w0(;XsJ-(fl505*_D0aPfgV<d*X%+EMRTIsfbh#XpZgO5V=ie()C5JPrv)8v|f8hWX_j+q@-b8w@m>9sU#LeWw2D+%|?PM8Ed`23KZtXgLu%ll+1ci4Yx=q#2<iGWgYvS;oyVNJG?eVKVO||hzUE~PH>lOI%(Dz{0>sAFy!rtLK0_XM!nDv<|Ru{;rvoH&#l2>0PF`PACC;#;!VpA^vzWVSnrW+|1j@8Z|{OlU}>({e_{;BKlICrSO1qbm-aBjw1-8p3Q!Z2Lm3Wd&1<J~hkTOfpfeXi?7Py45E?=)L9t6P?j2jN{kz3xhVR#iKCUnUJ;VnS<Z$8<sidQ3ZPf>WSTVZs)ztQV1Ef5*xb>xbjt4F0C%>`~igH-FdYS9-LUqmRKQ8bf3AO$1E+mMpjAGS`k9w-ugP10!E(v@8eO%A=sU;H=vFHTNhNM~Y{|wyeN+Z8NryZD97*YXbPt8k$1zwe@w>U4!#^{2~;2I(P|(_Swou%b(VLh11&g1Fo~5ru;f^Z4?dvboZPXqd!IuZ!GkFyWVUP#{XtwnXjYyi|-qCvNhMU!U1Vyaac36h18-7$!ehwv<5x_xb^AU#V+hsDsmH$-zkT6)?$NV@wUZW)liL;->jiQY(ZY=QG><+oP;b2o7ULxzS*4l)REu4Z<4(j<ct?7l`P#-?MFo;A9~q1_G;g*=+JQg5n`H;HkTPc&4<a>?$^oj?#Q)8wP^qXQ}4}Dz`~_TTdV%Av9Z}-HC?5$;XJ2$SKJgOm205$zDN`S#B?_!VsxsFz!KO7;-D{#3Rkda`AnkkPh`LgqFjR96yzg(FaU-g8ll?o0ClU07<OZ*S#+iPR9-w2DlSZ=jx(`*8;71Tt(0S_Z{nXpnYP{KI}W#moyYx&*=>61{(SF$i$ws)=sO<E&)-KnG~t@oT9l}_)%2=*((fI*U*%B$LcZP{n1&rC&~<mcK|j#M2uA3(3Qg_V?`T&0;GNE1@lvhJU8vNlq2Wbqq%mPnS5J&4AI&&BSNkAd-E<NrkJ-3&61VqDa8b;T6}(@dX##6&gr6Qf#a#Nq;IsCQOKUxb=KK4o9y2{5UA!m8JZUX1i#@a`n$B%@6dRi+Bd}?;t4$v?+0n$L{|pF6Xz-|JAXV$o^ZYem@lVy{y`b+7fS6gN#UH`|2ToAJrvWy6G@tpktqXL>$+gmFm0D#!sBV>?Vv}I<Om}CE7wXfey!vRh`<3_v3;Ut|aVA@|@=e&5`cHzozn<@Wo8LdZqt3i{;k43zpZ~yDamdt!>A$NYEtFL;Ll|FX2B9Kj_jl8QCcFLg#<QNbop1MzX9IP28)wAKh2~G|%y=FNs@Qx(-Ym}ShUjp7^E#dFhy(J@JFRVLx%1)7rgU<I7IGRkGfSvzo%>afy4>b!77-ZuLzCs!UHVFD$c|_ttU;z-Y7sx*_tBRl^c_pAPwnxf)0<=FDVTEJwFR-4;q<<WheP5!yiUurP=(`T`OHXMeh#6NVn6{oBd$H86GFhf_NZ@zdqV~r6>{O!QiVFPVHTDv=q9%Q?xQcvd9^iach*^Ncj7uRR~n;!zq7-anlfq%<;pn~&{AB<?U&H;^{>>)FGT9Cn-8FNsA5Ms!I#__BB^Eb%ct;#6(iB$jCK9jO#^zxK*STi?k88fltkUHk<qKko|omYEGm$nN>}!N{FPH_y30Lj+FF1K7WHp7Zgh6aNS!~eyoDT{(tils;Z$+;FGgv)4l%&)86x}Du1eL2AKM_-@i!iNuYI=a8lxLnZ)vTXon_3{Cr;40{t!CZELk1)Bkr`mPd(XgNfFZB9niYtR6VOXoweEv7{|v*rjt6aZ$KX%vaNNaZwhZp0jX!-Z>^8CyCi~Na{czz>uu?`%E#HA{3i|zMxf^e+qC@0-Pp8y!Ze@!TR3QR>tkd8s4&)}&jjJQKb~Za(2n}{5_<ixX38(q)5<iM(}Am!OAkJeST?M}Q0?9dp9`Sea0qv;Cld5-#WCzBverDPy=_=;Ge~H^+^$cglL-0&;aw(_`Yv7u%sG0z%*HP+z$*flBpfhgvbv{;ng6Nlweeluh1E-{SDTn1t|hHI#5hM?K(RG@cmIGoYB=!Bz{BMt>wrQtv(@b%PHFoJtu~?WV=K+EMsLf0((o*@n8(bJp;hl1l8kLac90EOvN=tM{poyf;l&xSB~|Rj)a&~4AOQpl5i$0~06L(R8~7Cd4Efh63@nevaN?dH%8!)4TVd~YvEL_RAaGynv{<aCA#?c`v+wrgdZ^_9xx7?cD<sX}aiML*a_Ci4%I&NEAU+Pw@*r_S4JFSTF5Yxd86`#+r$4*@VAF8-7(8gbT1tSur*RIY5_@WHIs;JbaDAb_?hLc@7%ttz3~i%uHQMRb&L5hcS#Od;VgjDk%Kgv47e2(%cpP6$HM>DrHd{hvoF38o;dh>r^t$m>WyM`tZ*{WPPb0BzA&|F|%6*f0hT|q<f~!JbB>%i1By|w_#0_RujRtkNugc8%G`$jJct@d~3gKsN^Xw2gf9Ns27@0RS?)uR*a;`WEvdgB~!$wUvKDml23_33voA-CEb@t6O3cB6_r*9_wRK5~}R_Esr!Y-f6z*?(=&kWq&-gDm6<U6uGp@LN1NbdA9t#^eMl?jcCx$)kvA20lva~&atr>!nrU15vqmbmdL{cYFp-#k!+rWkEgCWrCq-dZkbyukcPDn48FGLsfy7!t7u9VHDLlpCr(QKp0SwN{Y|aiDd^!8q*V%~`uS=rtgtCRQ4eTFt7``6pwqwOMVy3w%;KYjdjax%*0QM|ggf2Xn3p<-m8Rp|ds}6sSF?uHxCB{M$9yE#WlAs83s+<&Amo?hWJPUChSwMhWrX%-<hdk2ASG4Y}MuFaUoq|JyZ~wxS?M|Hv~f2v2wIqgmt+xQ`4she#jlXupIy6>??wVz}(}GkXj!h`#=1dOEq-RxYl!L&xGpYIv15>)PYW4gQe#!TbLvVTI_0aP``5H)H5(CY>qswz?T-(I2%|7rVCx^}&4TkH;+?9;ZfQK@Q<Xh@AUhDtr**n?#VeC0bK2lR2PXwg&h#iB6P=ul^2PjV86pDR8_>`9Dcmx6j5IkC%fG0r%ec*?Izx4}sLsr?~E6;6<=xh+agyY%${L?e<Bo<&3HC3FesJPk`hi^N-VPXxeW#+z2a4SU;oTW^)Oa7t?{bktyaIhu7N$xRfH;24F&&x}{Q+UVETdTW=#z1tclEH?F`4`ojH|vGR2_U^amC;G(t%40GC8)+$PUKqLIE&UP>$_MMjX0Nxm-d@4e>rOaMN=-pp@O&BfK&_x|HnGJMlc=b8*&lKENlD;3I!#c8iz56G-Cu+*yTvL|;m-ja@Eb{OhAl1X4Mr4QX4$i5nUajl)*<U4YcD)w;Hdr!;i&>eh(cdn4wg%E4zW&p5dQ{qSpS<md$*56VH27M)D%|Zzd9Wv8yMDayz(+CCz2cIW!PZ}XZss2?2W@RZ@j;;Xvs2uxww=58sC|2|Tv`FR%)mp$yY@9dt=3QC^RXth{kqh<v;&w#bci<ROYb0WJI5gms?MTqj_@0m(D&wM+a@b1MU9!Ng}kZL-6+t8l_Au>HG>`HkzD^*&EV?3PgEKi`~GTWF5uzp@ogs{*9q{x9=^UWnXQVm)1rV|^Fce~FyV<Wz*E8GGTjyw_D7s<4P2(j%l&^tuvnkpU&c?7zz6Z~Y3I}aqW?pIw_=q1e?zb+LGiBR-hf0m6k**<PxW!NwKx3ZD8;Ls04*u@WonKuO2ZAa&$5Caz|}sSp04%j{ZVUoz{n|H4M4JIC*`y_V)GsF?w=nF@@m|KqbieGhjEhPcdeSzmd^G&Cijk~k9vyi+fp}Y?}8U9^hxCZ$m=^uyf{__ChzJyf~s01gGmr;jNA0yR*nO#&nE>t6bpN?xeVMiwh?4M{^$dBxoVe{Z7r`OA4M<YBj(82;8Y{@-fP(Kf5&bz9yiZ+-RqX~$!B?kx%tL8kRvukO!ebTi4SS`q!w?_IL<~a-<0BZcB>6Km7_2Q=FZqp#a=q3e&c4ib;rYT=gnrHh?|e@;?>4v_}uM}hPQaMQ^{wrh5XU6>j5PuKVR-T<A544)(6>b2DLjUHGkVvB6~GQIVifkc6<x^YBJdO2aP=#4JD~fVzmYUlAPWUK`JqlV}k8&mxwumn}0&Ez$#(&5$c@C^Ve;+-9F&+F1cUK`S$gmj)H$fuzD`8O*^JC{fb<3Mx#O(N6^NsOTBlujhWvc@%i5ntaci|%YQ?#tW%f3{u6?AUIR+S*cDrNnu+AUAy|d6nyfqeZP|T$x8AROdSvm`M{*ZycdF1qRDfJ(FOk41mvv&SaZ-p6?eS)X>?wP??Sw*iSG@dxYX+Ck-~zWzs=I1FW}dbkz>0QOWT^G&qxV<8jIOA%WFPYu76t2NUeE8|-jE}$dg$!my7kG0`FRjH2jg*kCaSgw`|viy0<eI^68FZlT@fF~oosT{*0pzIyyN!)OABW&qgp5pw(I*2vYMSv`40Kv*WIZ#k~vo2-G6s}pB%`NsvE<lp9sG8K$^0o#N5m`_jmH0-^1Qr*7wDUz`tW=ysylEHG-fNGul^>)q4E01j3k#;bB_izU=557sXTy|7jAFk9B9NvSy>u@Bf5gAwX#pkf3z#i>#1F1I8gu;|)*evR(gE76m@T!KK>lv4WOij3*74>uGWO@YQaE)9yZVX61`}J^v>JD@e%kj0;Y4yQ4^xo7>O#{84<`bpff3<(69I>A#V?r{f^=pX1AR=B;xM_%{R#M9bAqnj+Kayc>DnY>o{-W6qA}DYxLXkXp2S*<Cdhe#k~I!sL3b{}Y160PNvaEBRHc?5-a@GG_&TNj;A8aVJsB>ZD#Sfu!!%-4kf*d?d}<K*1gkY!GKQqg3^FcMzXJXwzG?h7|gJO+-v(sgp*LqSE@dwcX`U=+_-3&fY!8TfgtW#i83&>o1c^rOU_@lX~L@6l)PDZa2Z*Uz_u2`=03Z>|FfJ^nPF?7!C{7RbRfT!kB@zhsQd(-EBf)b4L+dGL>Ckrj@CI%y)^^&izX5^Xf(~Yq@8W#0D8`*qWmoya_|_7I=1@AlM-Lr+2A|?ZwxxO@5rDDC1YdX^26RiJwb#Tv@{`*xC=~PBc63<!0Lbn8x&x1CbB@s^oMm_U1=-x=N~jdYul|o1GYU@5e6Elj(A{YAwsoCv}qA+`7J%7_U5!Ps*Tn{V}t4mX-CuV3eYH4<pfP8T$XK89bv?|9@%*E5GW(&R74|4AvLzZ#b`hUSn&8e1ktcJ0X|c1ouOkNRrbTM(6Z3tta<tn%s5Em`I-g9l~(R&8Xe>!QA&X;TaGAU`Lp%m1iU@zl04WOK`1dW`XW%PCMB}VD-flhd=T*wzU6%doAHLd@8xV>S^%uT@~k*TE#W;No)@m7Pm*SV!0(WmebnabqP22!V4K2zdE{QM#0m>I)=2m!`v=rEA#?Bc^D^Otbpht3xVf7krAiR=SSt+n9OU92@Y@U3jV=DL2p%hk7=8*Yi;FnfT~wJHm&`D<?x<Z9Ay7%-D(s04vW9&2!#3+uMqfC#mfy+@rOpBC9yM><XY=AY53*us*kSK(`0=2Q?xQvs^9!NgiY{yyA(|CPfhA;q`6JG)znkA$UE)U=v$JpY8OiRuh;MWs#q;}57E3?MvvQlTshCUn}nbhtV9<tF^CE|E7bPpU3vB}rPJ(B<@#w{t9gJ}%Y%t$oEHtHd;ylWotvxGqpMa%KlY<OuEwh=J#F8pG#=}^+g?qh=V!QL>%rj3jP~n%nH(h;6>%G5noDf<G;6E*ai=!O1cRH~de#UrE}lw#7<RvNE^=*>mtmn!KN!hf1@eWgve&7XtC4(uw!Y}i%5=@XwCA3sLNB3PIb{<&JaBUFSf!>iE1tCWBb=YZ(od;KZy#H1Sv0^(LF+c<tRM9wGwvhSwB+QdKkV$+lMZ-i9q-4DgAO%@B4W>tw?702oqKQmd}}uuFlC|AAsS8NmUIW*7q$FJs&?oll2pvy@Rg(PdDxfHi}^%~9eL;Z_?{UyF4nY%H6$EB{gchhrMl@K-3bSByV~%_G!^XnG~U?RmAF+^RrqM^)^blpJ;VO?x{iRHXl=8*rMS{wk(y~8$wT|bzgX3{DT&I~?;|#NJ41bv+HCY(XWKiqT$SXE&O5b7>$<o*&>S894i(%p7jx4iGNy&AztI`pLJ+=U=gj$B@iDCMn!R>gx?^<4naXT4Q4u6jsY)`1)O-xlo&9lJk5GNS*vvA9-C-*;;PXjz{FcTNhdN$7vc{b-1*VP6yZ($u`D2pH<u++_%I#S`=4+^3suq2;Y?^5gUStior^n=NB3LnLb-{2G1W584kAr55WI!{n4sP+POo|cDRF3Tp|J4@N-z!(D%B{~&(x>&_;^!}PDyQOyZr`*Q#o%;&Sas|Io*X$LS^$R@BCY`d%8q?}3mH!CNeK0+b=~Fr9Bge^d{mCeHMH|XoWg&U(|eSPk2>1C1}a(az`hqcQk+vq!NA#ba5L0p3ZCNmacpzX(NXykhgE9#r%moK*~a*b_SScwx_&>$DFV^Gv-c-T4|1YkZV9m&bZWEcZ_ug@X#s>2#`As`-HpwRV%X<n174p4Rj-E6nzz23MEMf7)ctXOT^!4Y@Y0L*?0`U1;y3PXAH<4frWK*H9K%`vBzW5YbPb+x2Xc~g7BQ^a);nR?ezd~T?YfpKNW)MQ;P0V-{Il0;qzOv9=)&t2k0wqSv~9!F-4J<JBHX%6m#s0=XwhE`?a*XXlWP}Xx&ci6<67)Hl6I#(8Z5C#Grjhh{mm}K`)8Oey*%xN#Ff{c--_l=uKQkhT!G%Gp+&RDu=C(Df-J@#A`mSG%b9%hf^I|0jpO;E@r%!TV|vdf<#O+0Z;@`Q#bXJccn27Xm&cX-yHeDe>WizG?AVdP4~yEY8uw3>2Zg!xFsJK-WV^#!E8J@)#O_krR2qj&QR{2-X`Y|uiM#yMH7JHnP<-nQ-Mcd6*Q8F|crslgx@~Npw-MZyAwGoOi*mOq=A?^;h)Rlw=i>{it!62O^&zAnxj%zS`LL)YXzBF`^<cVw_7b<<+?OB4D~EBN5+?I#Q>!7hTo+L!=s$gb1nTu^06qL)>vGXrDknFtozb2^cjhy!6W6N6{HzT$jIT|n7)tjpFhlQtehe`mk9%@we#UxO4EfrXxLh@U2*pV3i+N}HdH+^VA~-IS!OZ9+r)hJuLBPq}L}q?$ROXY_--<(9L#A_Ef@5wHF1cyg$|HP=CEd6)EmKEipGa^#ce*&JMwJjQtSrdN%OhI%>loP?VsG=iEQ|2zb2bFNXmjw5R_O+#SPo_6w}U7N9-jOMb_z<lETl&rw5ZvPWHU6_Y29pUdxw=;E;^5q-<v~LFKOK1ygqv{w=sJLS~9tu^;grA_Bff@90Co_<YuG!<Zcy7GnuT&hyFJ70esk9b5^^oN#{Tz_txnB*wtpcxvV4p%Nu>Gw8c$d&!tTr1MS>#?w!Y}HeRer;`^uP`UeUIg?{STAK<<S&y&|b@03K!q<vcsqvP?TiWlThF51kq7;Pu~F2!z5ufA9vuLBG})_t%%XWy?RUf%uzLw`M74{)pT81$&oO_lfdF>ZH)Wb<?{3vi@%YWWaf3p{F{7>~dH7>eZlWUW$*0#T%D#64zaWx+m!J5+A%>I)hdzan~4J58F~pjRKe2Vp0!d!;)O{7Zc73&n7#B+K}GMiqjP?_8sfELj8LKJ#R~T)u|yUunNu2p*YN{t$ZI!W~xx@6JM}EkC(3N77iZ-`dn~)P1IQ`cAx2B0H@#V&?#+tOV{cn6%vq@eW}}Ta1|Z9OzD-)?TakO32+2Ea8QI-I&AW_9(s2`E$I%K(Ie<ZgB7(R<_S!2UT9-WPF|cF)P+{vv7gVt5IE|ir1{HETtEGr+cf@qui}Tm~Uhg{|fH|A#?79W%6^$cc%DjBwR)>hs>9wAkf$RBUfv;XhC!et(l!^&plg^#OTgNZWX<6voXUaf6f9rlg@<)QeNLsRwk$?tr&sb-F`5{x4pMiIU)9{SYKBonD|ukR{i$4)D6CCEGf6AA*U59tI)aLc9$!y+n=1j!agcSk=<%IZ=Y%`spEOt=?`AV>Gs5gR@psbv%C(iwAh|jk<Pu@h+0-}W2`?n$a#L>pLXcR_L6B6leaW`?DHlnoDpC(-A6M+8(ftIS{<Y}{2ceo3%p;0Q|vVQJ0ZE57aa_xgzCD@SM9fBZMc}~qd)mjy5Ez6{wJKNh~L_PTEin7n@v<pDnc>nJMeDetniMR3DSU{UU{)6#ufPP(BGO{U&d6|cF4*I@PDt+Va+#hv^<tl7$F{O+Y7A_==q5B?0Mz-4PKLCwC9xWKW`VxXSfO6kkzMJ#GvKw`ZzZ`&#B(`?n9O7VJjO?DXOVar&^~zs_8~{-n~xTrL<=&p-e2UX02a?D$Y5lwzB*ETx}aLuLP|wn~y3+KmC%}pY%6fvsGfJWD4XAHxk~saJa48b|(NW<o1c6xm)G-t4nsP*5MQxTrB4YX1^bN<<Boqfv+pX{tUXkMRz0fQtEP?t_24rX2YHSnab+6PV+uAf5`K@U#o1@DS27b((;GLtE$yr?#G8l)@^SlPPJH$R?#R!ch3@}V3KSQp*CsCVU?229@nFCDp-1wa|B=CWH_B1x-k9T_tqTU1k!UQyzXOS-_D-$hjZs@nassAPzLial>_6?QEJrk)<T)ouqZP(0b8J3iB=XIevvvhJ8RCg-zK^%uci3}*zi?W`hoo%G~%j^RM^SYYZwLZY=S;)l*W_1&{VzW!{;X(9iAL;ci>th5h<b1%?FE%*;LQf*F&}-k6L*b+YO=iApsQ?h-Q%)33wgju6?M|JT)!InjG-kt#bOl!9MBa<8b_G5!L>#aeTH>=IGkJ$yvglx1aJ%H`VvzKI**0KNWjO`H&uQZzI|LqE-B)CW8LK7>jRFbu;Q%TL3WH&E!U(G#x%PE!M^+{R-ADGued6d@!pbW8bk#L~Zh3b@h2#<25El?5vPC`^iK>gU6r-E$@fUsWJ1O)e6pRjy|G~_H+GvmEXC<YLm3TFV#@yj!S`D4xfZm`DSP4gqhPvlJW5}>mSqGRh9>Lp}6{kegh)kP56s!4(te9XQ`M!LF-wqMzxpJA%3}P@=dhS`J10-uS>V)BE>TsrW;1Z6}p=muE&Q4I_w;i>qw^!V7c1A#s1;I%~x%F*+zqCty#Ts8RXvZN*i2KAyN8Koo?=)etUj6{E4sx{XW^VZBf}K&1*5K4c5tailKP8;4yJNHUk(KuioGiLSY#nv8o14(jCTSWgT^6@bLX}l;Tett~1PxSKAkRYe?V<>bA6Fw|3|9>oGxn%Ding0RsBC;7bi{HP)VDC;e@p8&QWRf5yLQX0Q3yZF<lT1&lEF{94p#ZvhgoJp635s`Os7*Z_qz90hQGn<0YqmcSXEOgrnIyU7;Xu~jTC*T)Rwi}Pu9Q<jq}roSRnW-c?0%XOktq0EpUF2il9<0IiyK1cm1-f3?K_!Z%80J%XR>#FlPK3s25+ye1F1Dg0Vg`U*_clN-Hn)yK-G`1Q4sxIoH_01@G&XAp@1Z}$s0TFH9t5@4zzuI*6s6&V>bz%Pd`cFTv=|`IL)!Sy*e$=!tU1Gni;%~Zi=y%2;F|g;<cTSG}c2SaF2GFtOsV3(d59xUz#a-g*{u_DT3%s!mro;7RH5<=wlvrP~i<DHU$kL9IZ8m(d<)w>~WZUp%s(z-}8<-BsKbY)|tpSj1Cn_Zl?KV>dqQQ8Q?vG80z>)vvGM$&TyNUGNBIBD64&UqtmA4NF*;<D0P=OYREp~t7?t~?W&z|E6il>#TH+-DXVpy>@?~!RIQrTT-|38;$xHUq(LZ1o%Y_#J2=riGTuxm1F2%Rc{!6{&CHRC}s*V_KjQlIShu&(w5=G(o6)xBH0w{PATOn&^daHE1+2u|efby-!5Ywv-&`!RLa9UKMP!3ul-9#j2A(~FJqY`xO#*J<ME#IXD;*?a5zY6QDC=igkW^-B>=`!;l(17=Km1VIt!J^q^)e;2f?XRg0(t;SVNtp8cov3X5<i)q|@9A{6mtSyk{NW-g<J9;S7;qTRpl4m3oXT6iZ8?;qw+hygH9^h$s`Nmg~c<hEr<lPiM7V~>+(|7SlI!v*lHTEaYwgQT&F|n39+RCr_bhh7LgZIs3vY_%mm3U)r9!Zb%4t=uyjy@c)f76)4&%|Aq?b9*Ks^xHzyk_|0-!vw@7g#!nmh;-bX-tnim1^T%bOWvU-f>=*zeTg3Qo;0ggO@rYvOP#z)W6R-JAS{8-2GEf+TjYkx$HB!wNBR-xn4bj_y%Tyrw*#M48`8BfHk*!qtQ4icqMV}@GT9>{{9(sZollcNxPo-Bm{3RkXhtKiwRZH;C$LQ(B>@U8?D|sv2OMfJyMOUJ+Z9yR#@KYKHoqX?#b19Ax^+`dv@Szd-}{DE>iR7X+==|wU|`;r<v3g2@+yT*RSk(y-8LF63{i<@rETk6YNKCsi37ECiSSS_l`OiukcUzZ(fXzyk0&Yf9Kq2f1f<ES^^z8&-Bwm?aTMcr%@f@d38tL7UK0IHzuoJ>)e8HcGBDIl<tpuR6mw=sgL{}aR?(r57?)ARm=Qlj+Qv2ul!w5#_gYcS)xng<QXXCN9o>hNTP{*)_E`E4A>5PN9o8+6nW#-iU|l-q#b%Sm%UXv8t;tdrC#|}%ig-8tR!<aefaOz-{!TPzAo4EQ>@y7CZA{X`k+Y|19Q7BnOdj1*=-Kc0BN<4a|<}kBn#d=%|CjqK#!zJ$$(1>Jt51#ZD>Y!v+YV$1*NjjZtp7*8b<4W*&UAU!YcXszLDRqwg^gXb<`&|#O_y5C6u8%>{seKO)!orQlD7LX-Cb*fMqKBJjjDd&oBh6jUR5h7Ix7KePJS^8+iPJ@|s)wBL&zDT&e&>d3=8_BK7JrC}xxU=(HkcfY{_MAfE4bRz$r2fOKO@@?48|{<5RM%ZAq<xY9HF4!2SJ#X9h7!@Ef)w(IFJI1SxSawy^!B)fk)`rYg8kG`jxDgADuiaC2M&vK9x>CLJnE&q_N^O5Krb)X#q4lTyRO-;Ra5ukgnHqyqVS2=a^{LfxU*q4RDpWk1yodbXich+#<vp)wgcnq4@+-`kso+>jtzDd_$cj~4$x(7#1g-sxgS0lG`aB-=U?xE4TItyOPEn>u(U49GY+Ygh|$~aTdbO@~atM$5*Zg8`KHipNmjfJaieH1rF=*DZ-swSyceJ8DoHYnj0*_pHP=P<Lt>alOc;<!HV@0ZkiPP8WdA74uA-=?woZoB@WjJnO!7ZO_+c^nQWqj2(J_9~0D#dj98p2M|^Ezov<(604k^S8n4;J)Vckza3X0Jt%!?p|5($cBz^F@jBdCY>R1;Pf*4Nz#1Y?*0OT8*r`V&37-8bdC~MZgb-4>q+e;-wFj|;2uq0_nXSBdi~?oX*@U_gsu#N?>`GDzR5>NO<M`?<k9!8+hDOg)!Tr%rncw2)nSieotn=%WrK7?VO$xn9vNn{m(sbVsMX!q|CUPNA)4l?dF%B=2oZvbKMXB4xK!Nz!>_M_dGu8*`TJxx3Oz0WSh7=iTA6lN7i<#mTYsBPv*rg#vyHbo3=;!`x~#nT+?s8O$LH6OzNr5&){wcC?A)N+=!IIx3($M9Bm}mFQ3$z;P+BXQEbw?%dp5r^`@TqYr?1$x<oK$@K`S2p?WV4HZ-M1CHEgbmi}bcKt10`hRl1#9wxw|6756beok*P)3ES_(g8=EDTQ;zK(0Q^o-ul!^yhgj5JVuVq)vdT&vtBL38E8d&y{@D4$m_{}4~qx@r{_;C8ji{N>c5qEouxI^*Y^6&KZHrEuqMWAP*r-w`&NtQgm;JcmMl9IX=N3Ar_aiWIri~k$Ns#h+Z1qd=CQ3qWaC4`N+_wj`@8gp$Ms}PC-Px#f~(ppOoLzvDzzn^A9`PHwD*MlKsI>QB}Q}bp}qJJTg9<eA61v)=F}Tzs+%QD>(s?U_OUTJpwH;shY1>a<qU!VTCYBNezXMB;fdOOFZp7$zkkX{8P(}z@T^C3aFeAjGA)9w@nLL)pc$mXtX<i_SP*@c%)X0e(iK|oCh-2(Th8<<yefp&W$4_zrZG`Egtm$eb$p-Jk&~y~WqjB8Tljj88%^dbr9sQ1O0yoDP}pP^hi$SQ^o|vgoCoXaq&elpoN<TWeo(6G-KhsS@*;mfHuDBO0%GN;)adz|K{lZ_%1^Kj*Bc2^)V@G#!HCGHjm07vD(u4@N%(W<*EgRHXR|1Ha&vSZ67zN%Znb=q-FPUGzrkGSf1BT{eLuHVl|BvT*jcYJ*m}1pq3K&4sU7$DXQ>RINjESVjKs~St{n8dai-kuvmDN?+JRf{8)$yMiuz~g^k)-q(Mv#FG^%pX>1a1^?`Pn+u{B!}j9`^xbThS1^yrjmcD?;doUFf{VrDiHb`nZkkqYi7ef;p}(Od)sLT1azXdRHrEj*mK{hya!ZWy-{RAWB6>c+a+y{Rso1f5bBk^UV|O|hZIF@Gh{>AH=sHut=>`K0^}7X-D=)w9o#3DS(-@UK2Jug@X^iSXl#l-6tQewd<%ggI6(<TnC)!$*1<tkW@vyvA<atzl{NdGpb6IB8<yrg<b@UAitr^;*D{n6Y!<%?m+0tC#a{8y>oBHQ@vQZQZs{>=B(3ni|Og8n@7z@|-~e-aDXM?K`xoC8cyHhftNby#Z^VG6<U-suq>J_SO?0*+2ZvZbP8+<AW~2R-cEq{hp_Aofy>K)wz1uzk97ZIX@gX*NThF)6I7En_cjsrF(;+F~o0<r)=KdVXxOkv->gqgEMr--BneT%jYTttc$}^UuhN&BV`bpoO`pO(kO0<RPb;7*I4#WgP0x)<D6b9@@x6~9;xBj)7o*8oQ3x}baf0+XaD@${Kgcmdu#A85UU!bFMV7yJP*CWslZc*zb9WS(z{f*^4oX;Tw2gJA0pvwE`?LyroP_uQK4mev#iGL&f$ot9dEXno%`>(W`oQi{$PZ)7H;G2JvctY-gJL&tCPSOce)12mqjWLNl8h1JNYu)z`Z$(vXK|_pSC(>&fa2C%xfEEph4<_pVQ>DQUf0AL08^CG*_n`fiN!K^|&PZXN~*Rz>N@HM`SJ}uk*&Q71!myj}pUvSB}sxSedQE`+f1g1Iik0!zXsw#<~@HaT`tHM>g%z>Au2O+6fm76m)nM;prtXnwjH{E>}8HK4?lK2H71l^7*3pbpw`I)KVAA-&^2-+roz@1(?2{3ZUm6`u#i4;JOt2+oScFh@BRZ_WPaApsfv_-Hl3K_VUX}$yMI|8xo6fd&WToi6n0d-7#GK<vVBs68#yh&HY^m>bTnB*0rk0-)EDB8$9Cbq|6RGxLGE<;f$8vqk_R^4Flv#b8YGSMl|CCW6pUgd8p4Y>x4?PcfYj-7xuMB8X_jw@EH1u?)fsB&-off_-6(vx-26M58_){sxs$Ir~QgpK#l$=M6(c|)Q!(@&#sSq{>lR8!NbP>E1r^2OB_@v6py&L^Fyl_i~S>&4cBN_mIr4sA9Q1<yK`B+`&u;K15I3^^m!lR>ms}^`*mkFd`?*~NEUYWYd|kR`(|;y?o-%dvFnJ+9%njqt4nO<TEx!l&Y{YX9^N@+^`y6VqG?(nmUhe0MQfl`O0)h}6YNY<LQpBFd`zDAmy7H*U2sz!ruZfQ_Sb{2%`Zvm+Z>eGnrb+)GZtC}jl37nHn=z%f9TewY`n#plWM7|d#OQJ-@vD`{YJ(hy1uK$`j<qP(vY9?84c%;%ALF9uP@MrG_GQXkAb!XDeUXkDNFA>c30olWPGBPleZ6Pa&<B8)>_kQFDIsbAoqkpzTZFv*$Y1S>^Yr`^tJTi$Eaf8N9;+RzV+tLBUdg=ziGE?3o%>0-2)Qb_m0;C)`HQ^$kSu;>imYq0=6jS&0gqbz83pbH(sWGa3#O*PZPI28=D!OkdZ2Hq{1J(0mCi-JTm@{t5&{lG8^p?BNP*|*{CIa&{*@BL%D>8oYesbkY(#pL&^pZOuJv4OmOuJs{EYm?Us{|$}L&n6Jqi1tXhxcr;GKGD`L1kOOXca(ldk~uiU!r#T`bD5xwg#D%vFSNPuaD^*4o_@9o;{==59I$k`u}?dFZf?RAAO-=3Nl17}&qQiS#9C25X2xq>a6`om|^4RST_Z+Cn8oWC$zmFs&5g$%!A=VCr1Rdcg7rTeP0Ue*u6%|=~!KlnrgMWvfZ)+$Dqyd7H0yxPu>$K|4N6>qrH#!DEh{QSwVac*4Ox8Yg90L$_I79sKb7+c7;<-K5WEBw~O&U-egow@m@`)TgutXyhUdAo#vatw@z@(psQI?NKiM!%9>-1BPzd;HtZlL5Y}#7wimM>xD99`13`e&4z}j---Pf<m)fTRSN@EQ~KjDz5KrM;lNOaKC0=hB`1J5698{T8Y(oz^pH=#koumVBgpKbl25r;9|4qAz&yGy?szsby~)QkXw_JT<jJb#!O3A@U@p7vCe$7vm2x1&bik5_Bl;v_jQ!Xr^f46+f8Q}+e;#C@1@(}`S8Sd!vm;9HRwDZO@&dd(Q0oB{kfNw8Zg8!IDbZ=kVMJK!oq|K)O)>G9^`gLo-;do!WqLnJYjG22j)rW8kaN17A8G_e_j`t<4Umx(tz!x^Ep+8WYVc<)2b!hw;E>#(8*+|YoF&6?{7VtXe(MZHL&-Gy0eWnYj|3E*)@hqWPe`4>)munfNvdO^%FmyZ~oY_^YiQ7_h8}iZN3&vdO9rki~faPKNsKopk7TM!!=Y9YV`Bbh7wCS_`QOs&pom|>D1ZFO8BO=tkZlGDj*$JMEA!(c<X)(cWIshpH?!j{lQODOLRE4*zKV=H*3dcm0%Xz-S`0Sk?adt_!3sjYt%YJj}I|%JF_I(j{s?PYQ%qtqjrST%sSop9W+m}mhB364?W!{T?R_(vbQ|?pxeGQcy*^lu&6h$0HYohP+M$I7$fQC^GLnj`e%EZHp$kgFT2Zn_Ge3Hq(%HwO#Zz1J%#`oYr5c|_ny1%>=$z_L&-QgtE~EB7g$pH9beG)5r>-ryuB6^b=sq*dlH|SWU(@@%5>A)Lhsl9o}7YKP?(RvsUUur-c&lvu(eCjb-1gI#WBMm+)YrIO>~@CSGO?kv)fAq*xi_pc%Vf}=!uG_b;F!Gy^?LEH+}JL<;<7P;R||PQxBl{yP;%1q(c4nAoJZh^|Vv0J=Oi8@p4^q#2tdr4Ytcy8XfGGeH=q&z<ZQCfw*1|6>Hiwsy$r;9nibBdg-4mU0ihOOKp$!x_Oy<#+5%yzy)uQstlku*Q{om$`_w5>C7dl9{k~>;xKLQlQ5eBfie&5Ny6u4DpW|i$hdCDMPG0o)uvx|trhR;7^w|-i0{(U@q}G?oWzw`k1=rlfv4%_r#siFg%_l`dVM=fM10~R3_&9e^RDv#4C$r!ne}UXL#OJ?PKx+q=Ei{js@DU(eL&2_<YeWYT&L~V62D?f+%Np#1`?&)XzY5o<|;;el>x1ctD2#`?fD&dmXQMW`Zw6=8doQlo0DU@|8i(;pd-%Me}2o=a-Mz;C}!17u&z1JG+O;_&~HW#dl_Hf-fe1yY^V9I@Ok6XPQt6TJvGR&roFUN9adJt;!lklEvQC)@{pB8(C`Pm)Ik~#!!FKo5281ZIcjac<;(f(2}m7&XT8;W{O#SQ|MU{{&uX2$Pq-b7Z5YRC2kK{9IkNUvnpF16oYBf~u$>wbaVri^@=_w_G3G6({p66z9|iM^#ph3nE!U&lN2q>?fCe!<)v*q{Tu(-<*+ovyZH#|x-+BkqenJ0J1qQdZmor8&`!&%~xqBtk`FAJ?pC)MLJkqTL<Fuhe{j&iaJdLd`a%SOHQ|Nt!XJ6|8^~Jx%XAaGF?UVtZ)1VC+Lgw_{YKh#f%}#Bv_H%S330HDHdb%%Ww6TrKao$<I_kP;@@d?d2<oIh;7c}@cF$j(iANvF?rJ84KM(g8iZEu5_EZqjyYdhc1=iO)93W+K6dZ#O^)%15u=VRbuTz>_&4QQ{es%7*^sD>Bb+0LZ?eT<iMpMEt+7Y3IPoG2FEW+yrwYx;?@-rd6>c<fwaSS4og6%%3f84X=wb7`8@`JV3&mJcCGx5D~+6tF;;ern%i{IO2%G&pz4()?}#AZ=tfj})!haDA}8!^V+6>~v{ar#o?D&#k~3BM8W<D`tB0r^$Telr;1M%6sgx=SreBAm18)H0rf6v~4sDDt=b`hz$hj+F3d1tySg1>+5m=;NYYE*&u)WvM<@>-;IaHFP}t@;<i3-6w3~cQG<1^wfYl%C3vl3x_i^)yz_Lrii(5TDq6JQWoa&1`pysDC)YIN$pGfenyXWyud+14K96gp=fEdwo;kv0I+9BG2+|rC4uQ@5*Y5TV{d^u%dRLR><1w&Wi-<L$<2@thsFE#X&J*{;&v-~U%ly$h<@HBd?^Jt}_p2k{$ONN<I`>oXgI#Ot=PfYTq|->~-lB_t)hj+0&tv+10QOBuWP2iA^A5`Fv5G_qs|=<w*{GcRVYkwd9$llcKl2&})b7H_&rFlsBE;V)>(vU^5xPtGK3pLgV{omksuMmuAf7_q{+nSJd=6KBUmyEFk@`*)ES-NC^lg$!TN5Tjux{;=Hd(lzy;Hnl0J<mD-hE-%7lIz1s0qie_&(8k+&j~A9x_M6dv`)1$lQ{YrDp^AS^RN#-#bs-r&VRkuUo;roLFjZ3j5W))J+j-MA$yRe579^J#2o}5MyaZRJV-)=+q#J+@Y<8XQo$sJISu}bUxvgG+(WHUv@fED`wP%kQrp_!!1XGZ|QTsBitEAw+XghGhFkoHp9x_xQmhh!HLtC@qo#knmVl9ZQd8n)ENdnrM}9!Z&pl=@$3}Nz9rDE;TAA{hr;91<Y=;+3F2v8U*l(<Tzq%Gu{6StZKm%2lJU!}nbKr48k`Xry{LTp1ey!x6VIyd(P_;c3Lg!?P#+iEzR}Y&;?kQFf4`LX?QMpR>&WuY*0A#?uD#cf8VnyDY_vlaHPYJI-$@s3&-mGCa;q(y*Ak(v4aWT9OJfGViZN@}A35qp&d$#I&m>s)oE0)Stn#GgMliFXYc{pxiU!sL8O*YJRjqAbu^B0Xmj|gUnpImaJ7Ne8huUzsvxwVlb1+)Q?G?L~&>D>+>vgD{m*frDz-Z-h$U9d3ckx={5oq4%YGcl~#p_CJUq;DdnT<hbtu@-z2&~J%be*UjCXhB4f?A;a>%Ac^5W}uEr5(uZ{~uXrwzDeKB+zFefDws_f{F$i1Q|p`K)~Sw6&!E?5d=}xXYcQHchY~-cixbgqIT6<v8;Wpn&Tx{NZ4Gs?O%2HapZ`xLiC>+KHIiNm&s#(qOnxTeAh=*o0O3UH1gmX)ytA!cN8}hxB;PUVc%;B@wT}haj8wJd(r$x@{5{@zQ(iabv=nY_rGpP?1Y$IGtp^#Zw!R93V-l6Q+tqagEHcLe%Y3nvclJ~c;l_a&f1$;`VdR?=C92cKaiOh1i(x4K~u?CU_6Xy_!&aIiqyzD@F^MGw>M!H9vjMRqqPqA^N*#+xVu>AbW{Sp*>IET%Fy2QdNgcL;hU9UH~2Up!WM6ndbDr+TQ6hh%@zVJw&QBUbvEzZF2t_!*5lV*DA57i|7<(OM2E@Pru=4D0FGPd-zFo+U?S1hj{_~ntXR*c6RT>dLC1%#SNk$*9eBCRI^vc*T(MC#7o|$IR~D+)it%O-?9*FKyAF#L+<Y?!tS9|O%9cg%z5H`=UQGb#qEoh8t4%F6aGRxFO)P!VxX+0%iB4c}*6nG*MBkpjFFH1>9;(gWXn*l`GI7y$y!WSC&urv6JZouXf*^AZxuTQFhIwtZ$bZn-J>M?-!XXu(`wgmfj!7x0nK@wX<1;ZgT7&$4U!_*t?>T7)UF+Lx!Ee{)vS03V%2t0&QBdK?nnW~sNwy8XefO@+0Jx5Ft7?PLABfeVPB}LJTo%7@!dF!|diWeqiWPFhUOnpBxL@ex)8wtQtG(i)_z(t~t9za;heq{duHXK#K;8e@Sd^`W`~qcU^xyZ%aaF+k-;pKI4eA_W&TAfv0@TdcC02S}yZ2{R_1hPsrM)ZKT$-H^e%Ln!?=rYDO!AvF70igNVG&*4{3q{cGy3{JSILn}(pow96?fL|VZvQ}Kx(+20Cz=^2#yN@`1*H3{!Vy9Y!x`Ps=+J%M8a9GJV}r%K#^6yj?n$V2G0V~(Vx)cz?i3W=T-Fi8+ng@%pj;eHB^<|V1HVk8BoO?;rve9Cj9rj%aHDB^F9kNU$^lU(x(U0G8q0<+a?Zsm8W?5O`lDEmU_7|59H=zUi{1h4?Xpc_6G58ZIl<%_+>22#QL>9(e~WyZoRK)vS8UiN-P9_RRXP9`WJK0Vx~3bLlT`HmwSHlnX9wAUNF5wn(y-!IaYm8`jDTMW23*~Fb4DcF;}{4^>TBBP7cq-@!P(>=5fKG{+BlJuvZoE?_qBGyP4?^wFk>5Y48gUW!38MgY3|udo(iPgYRD_FD_jpNAl{y@Agi2*1(TvWsk20AZ~p`C)y#$1K5Ut{3)rZ3*B_a;9_xF4=yH1R^M}f3=#7$JvE)xU&)%wukFTOKYTqwJPcxwmksI^K7A>wm%3ZqG4AGveA7Qbw1y42MYy(eNR9n%sT&mv8@~|rE8ec~tT_A|E%i<AbIp8j=?sJBXPY6~+YkLYZcR=^wlOw`s=+s#pVbH&yvY+5SN3CY)f&v-Hv@7nQ)(!dn|4K*&Z9wSs_pjk@@%hiSJnT-0U*8DUtmDT+7B)i4R7IUlYCu<{;dc*UhzoYn)qdg4b&hX6$FjN`Udnk@-6~{!NAS&#b0Lr8(D%d<xOM%IkFV0(SM9ANleN8|B<Eakl|)l{oj!#vC$>Y_dTF5XkYPlMjnqQs=ZZv(y0GO2M%avPG|CWl8=^l_sRxyUH-8bn)RmmA+Cee{ee$3^|xDXy3%25hVrJx6N7!L|Cpy7^)k^OVXc3=Bx-)N4$xXI*Bo)qpK08<inu*J4dH&BTwD@r(lC6istu0QcVmp2%jf*ix&oLW-P<MCL?=++YJvWGxGn#V32o2rc2v;zB7k#s{M`>;I$Q@9T{p6+WfpS1Cd(RDX|&GM%~d$zOJ&_3!~udnS#-B)S5YiD(Qsd__)mK3H%Yd*y^mZ`x|dDuJPk2({Ot9gt=#OgNf)H;+C}=mk);INuHVfGJh(={4OpEfId_6;F<qOR!%f$3$Y1?jQ0kw3zFs${13f;*Qc$pE`!n(`PEYz{P%Hm5N@3~GkLCkfb8hulx`BJk5y6`sFCXK%dB0qlTka;s`Wv#`b@Y3kj_>*nVQ1#o3!9_V$?4=>-f!OqGiZN13A81BxWlYxO7GodO8<!rq;$-XPGTQVjlKjMKx?{K0=q+ob)K#IA1!rHyH3$iGAM!fH~MgS+?rq{W3lfxEU4ZxI?n%XEDF8lQ{R@#b;syIar0xgZjJI?kXfWOSk*Da4V3Sn@Wi|2eVyUr8doMZU+P$sih6ZTK@~)Pw}O_oY;^gnTw(U@LsD^PzstD|r=&3<0a)*9bvN2yhr8*P?61h8vXitlL>o5~UKnB1nVtUQD!K0tQO*iw+U@KGu5NxhPHVRzApQ4Ma@EV$eX6sf8T{K=q}Y|-77VzELHV1LA!n|dA@b>NxBguv$CM+)9=>K5>kjxoN?PFq!0$D6V74Jw8UN_;6&G(CX1_&t!uJVZ!}fMYVQ;tpTshC1ojvIk;!$|Y+~DSE^<nHz+^Jf^q_Ovhj=%01^EZo+E_ieSzwW$$-tKmT-+Io5Y`xZ=!id~eD1Z6@LzfUn`<*fc2mQt6$t!4MYRbdGCbyoPcf=koe0cJB0Mw0qrOaw|=p>5~#te3st-3ft8>Q!N)5yW?EAivTmxq53an<NoBmQwmQcxO9`t$tf6TJ}(4<6UU6n5=LH{$0%V@Wc*#p!@1Kz+eOSKjuI`EfzKw&w|4Iojo%no7mRbMj*)@g|Gi-ug_gi@s;iyhagRj-%DqMbC?D4nXe!ksxA5HjX~psP4B@>xuK>rY8|KWBNwcP?shz+vZAs{<Z(o)*!ln+E>)myK~tJICQ*Y$#h%WbJ+_YbXS}^+;0@5UyFH#R(j>EHX&}e_4NH&zfW5&Kt`7ncHe{Scew+G4qNiupW2q)Qqi3=x}E*8bm1aN9Sz~F{4|}T%XA>JyTSNvLOD?JsH0g#hqZ>5g7+9;<$x-fcmVfWeSnZ+cdY}BNU4-RRxi&Z3ldSyjPd%ZURo+mR{dK4f~zF;Z^e%GZbN@GgB1IIqax%hUEyZA8@GLG81cW&+zX{Pa~}UXFY7ha)~BkzfS7$_<TSfJfN|au6aNh`bU_&avtvW}EzM(fI@I>A-<4(Z`EU0r8waC=A2!45Q@O+NtMXd-wFUh!lr3{UI&rwuj;#-SAe!^IKMETDcM#z~YXWqFbfAyP&8j8Yl*p<V<l-<qbwc`wb7oF*8?8vX)k*8+oqZ1i_q5qr#9!g%6<Zjf(^wXH&oKnOB~KfphSPSM(~mwK)EKCI@6Pr<=#BSX*R$Fd`Wo5SM+z(!*PF_QP^W&lYt$_KZ1azj8r|Ki%ica`>0G<l3E9|qoH8p_0DA(`7osn{<;&V2Qa@*dlVq^#?9gS<WOaZ0n`is@7`IODBC`Bco!!`8`?D}FP0`6*OBI2#Bc@lWmScxXxIFF;vZE9;;ADnW*Jz15;%!A-V>FQBgl?<j#$`E357F*36%m3^UOw~osov!}R;22scm~eQ?cOUbuY945nZ6$2qC?;9UdGoa^{cupH{<1Uwi_x&`?TF2u3VLEcx0E`%Xh1M0vhsbA!;ghrsvaAZ>j5~C>mnB^^C={g)F*kHwmVhix%E<-TTWZ?+>Ii=@M4<UJ}~Y8^P-RQRqWn<!Bw=ARHToe`f_*f7nLxxvReD<6#HSXIIbvOrZ1RvD!@fO@4II+c^~^#G}o|Lw~6~jU5`k@y5Ivk~^g7j`X4}cACGxM$vx{%-@<<z$4Hycn8Dvsp)!;OVkzO0jdlm>gJsks|K&Z35W!%?UZV&QhaPT@H-n&xvK37eE3&td*@9BI%buI!?CLe*O|!ruFLUeMXwFUE}8E^x+=ek=62x!!H+|1xi*z<2A_F+)`BKb@J#d09Mg9|W7o^k0~^3-cy_D9Csy-Gq5`3Ny#54O^SY16HUCZJ7<VAab>&x3zr|}Fozq(IK%&nb<hRxF;(Hls3In0z<AQ`ABO<lIy0((X*`wKx=GB(NTf@i$SJ6nb3LCqc-B*WB=da?hQ<3rh?Yu3HG{j>N6UXy@_p-&VFal~b&NWw)8Ms%uXx&o@^B@OJzv(m%$KB8m@89VuqxoEDU+NWt;NS^^4v8D7;rFTX_8o4qJGE-hOf2K-pWLA<tH~nMVFyJq*~!mE|5lSuoB8{P(#cK1QtGfkhC3T+ue!$i>CU3P{6H>sN5mJi))9K9^`zepW@C0wGMv5wYq$MT2gN&XXSL5|Wz)?f9s{ptCNV+m$nGYx4Ocjm{@oSndn<CtqV3bb@Up5nk7260wr6E-DD9hP{(<)XkY{C7#-aa|APUl^&^t|(Su2aiu9M#P*ZZpEcci@}yA7IPq~m7hYgd6!Ay%E@76rjmAyMaTGG6)2wsY6Wv2stfYyQ~p4NR5W%svu=XMOXq?}_if!|wVdStW=(6qTM^@?X~ilo28Y*=@R6g8FH(H<!O&fE-s%b4=NGY?SEs%JtfH3HmtQ_fhC{;LfCp><<sHvvE3~VVl2Cz7}0OzLHseF*Cwrovr)h@;sbeovhWOFNfY{v)*2>PHjPv+-TMkhhE+?;(~{fYPdpYZ<_?}Q|k2%vgfxkd;C0E65fsV#!4qUSSf1=ud*?rVf%K486uKh5q9$s8$VxW{1OcU-a2AzC3iz@{gT_h4qRL3kZC9U%D!U#(KOqQ8v?uf>!!HVhKJoyb?M4o)EGAK@otbb&kB$9xv*aw=x+lx?IV=~A?Sog_jd1{T)Hv9;Qema5I;1dcK79FfM^KDr_3Tj{R)QFmiA*IoGbSEaUZT?@le|D@f5xGfVuJRCqpv_V|x<;Z+ZBVF7_lOyK6+nC$pfHK<>J+sh#e-{AV(hexMw(W@83F3wGKMX!JMJi*TJy0GJpydM~!2yi)9KpO$}psA&SNRt9#=KK9CE5eUljrL_vm1i!VjzYpBu(XOBxI>ZV=T;i*$S8fN5_HVz8XvAV9P?a8lb$d*d@b)RMQ}XtfV4@aJ4wh#xQgP2f6N~jHf97te;{~|=QNQ(2P{S?$*Mq7p@%r0b*xm^dsy;scw=HTIALT!|dVQ*#SiD5etnex%46xj`OH)h|W0W$|d@$_YzXL1NhmDP=R8S8bxQp>-^o|fxoq+|E^nX)Dy5WPyU9Nl&=Ya}wL(P`5YM4#cUQ&^|CpC16MI_+;SrA9ZdJUbb{p+$G1OtCsN5+^p=yhi5x_Ec#?E=1|i+%&G+5OsWBPfq+^z9E-V05ieOu4#Ue*t?9{z%_ms-3dCp|a*)x=&<*0c3ZNEfAR<!a`^oMo9=kzvZkn)^`yeYOdKs<q>^(Kw+<~xH5zgnAEe=#bF<dB{^A3m1`R|=`{^bdc02#FH|LKSVHBPouU0i2drAZH${PlxZPs6>-Tp?I|qOWJ{W*jW0%EQ-tMGMk_5#rWGboq8e%#1@m{Uu__4fK{j*@kh%|~>=S{lng|y(pykk5@cbSY`M&6^dMSUuOi(dPVtTY1gmLe7I?90~JYqzmBR2zSRZLN{C_W2d{>gBxy=FFwNJ!gmE_Q}|HYYMe$yqe1guezZw+dfB3XH<vkHZ%%#<5%l*7#|AeKDk3-Eojb3d7&SG7TKN*{0TB}+H4~jFVO_^zTNstvZbV-b)x=t-5mB+W>^=mx7$qb7F$Mbj~fT41hxWNqYB@;H<2d;?Jj+c=zgJa@n?%sfP4cw?%)%mzeGFaaohESIORR!>G$`6sLho}SwRm|&xehVC`2Za_u3VHNLsbU*oUfQRk?_NN$_y+T4e@B6w^0%Rnk0>Up3v<@3xK3KK*|;1}^E%!sF4@TrZpK<FOTXtCq>wEQ~RNo^=84bpf`?UCw>ND}j4^y?;%Q;&WHNroT(5Rrf?U|5~>O)~xqMyY&f^q4ZjSgJ^G=EA~md`{De1jmvLQ0%jq>&+g;~kRjdaj)KRbvxYkN%AdvTd)RPPISu=TEe#5_mm5&XFL{6W9mi@sqOugc)}OF~Y;6qfJWo$)NKtgFsZR;(vq>iB@WNC1J*JYkxSB<?fN!2;YgGDO09LBlwae>!Co70S+FFeS{{=my%}!34GTn2{FqvQGv6D*7SrSN?5nriZh-r%8t?DkRfX7yOX!HjUq};@BPZ@XmFnT;!4;!sNX;f}UAu1li;onThxK!9%iy2;Oie!(?{<e%I*_HTQck5^AMYoiVw19eAbXjzP)^4zw6T8+P6hGAU%P?X5eh7DRO#G#JwBK<xu?e?_8@_pr=h>`F`&srytbm;Qy1IYvj^B|ZmXbGfg0;NVnTP&7$KS!X4nr~=^+THJUAyuY-PYFgn!%l?w)&{vmJjNZLKN`nPN10~82k$exYSf_rmXkb`S>@pkiSOvS}TuZv!-u`(Khjs*<~Wbjc!;rF?hZM;!J&_vOd3lb}X8;yUDbai1U0B5R4G&r)UnXUj5pOHdihUY^FT>1<&3u=eN;xXeFN&R6$$WWopM9S*VS6|HOgDi+}pc!mZse(L+~m>TDo|HJPkd>0uBE%zz!5Ep#{OORL@A$^W;ANLHT5HtqO5ZW2HB{qF0k%(3g&KK5rnO4kjfNbn9746?=i@EN3oDhHzj;R4xRWVc0*y;QpSr}ly^eDiSbQKQapHQ)U?*~+}67c&bEUDw6N4W}pSRai?y{8*C`YOz3jopsYZl@IWIWAB(ExJ52Dk6|I4FJY@4<R||uDzx57M$Pu!<?*a0!`po-^?_vW6#%`hIbJn;ve`uFQ~1|T!0w&q9Si*u5IAk-+~AyF&sUVFTxO32q*LR4dVugjX9WiX5_*j~Hr@ku5tX89TeO|$T7`$#7yg{4)==5@{1$;Y6&t}=e|0h2$+LH{CApl_m(%n45I5}#J9E$Qv6e3ee68?rBFLp;Ses|vWeXqNrDXtI{ng~P+}1mj4u2u78MYn&W&7ZKtw#@Te?ADJ&A=ZE0vlKrEuDBUah*R_*5RYmGoCyOh^|uVs=blTnQ1<;rB@fgOgGCk8GaDco)0cVp374(oRGLwziq9iTXZ(ZiQE`;^4;BUAJyI%f0?hoI{$p()(+IMZUBd_ZT9uQv_24Va<M*nL{Ev)BL>i@d4H^*Q))FE>7hpbCBJHh-hJ;qn;<XMMa(1#$TE14NPw$0_*wB8uj)oPs0}l~9`=~S6#3wh%5QrS;f+bOmhR44cj?RS>sFoMbrhRdr{~0I3$-sQs9#||9=w@(Evi=xgRh}@t6X~Y>jmO`v$L79C9IV@CAA~dc=MR1AaMGrR4R1+G1XP-HJ+{`d_T_l@FSizQq%4x+^CMrbVg`ZYybAX%l^CNG+VHz1W9cTr^L6u(RH(d9Qss*E<%4G*6pn@&X(6%F-goOs?>TXqc0&MK)7;)RQhCt=y}rN9E(%DzT6nLXLgqXmnV9wJlc5Du-~`lws^=5!rdRneFx6JtI`P@_H+#X*h_`)%o!EzXAVR5-$RnT%9G0BFNo(gy7htPxtR^vpF3~Bm4alAQz8=Xqb<^4ZkwMi1-67ufepKOP41H4J?{gT-C|9fFKil4txbO?<qK3N2V;BVJkzt;Txxw$b!WWWyGNzkgXb|*nRhDjw|L%q=*SRzRM{$U00|9{sO+c>b!+eZxfzW8vGM|9zuL9w5V>SqP}DbxFdR6Vx|PX?PHWs+AfuM_=Ns=`tWfTHyRlDB`RBX}l|P89<6a7LI6qV?HF+Mva+&3}&})yI`LXm(y8Ul&>abqhX!o1=S%3bJ28~Ww5O>kRKsTa`Aic)VvNlHBKV!Z6W42M%bSu+1Z*1$@{?~+h>pJXa2kEGY;e;&#`I3zny<&pjo0^HM+efGmt1tA|x!wxy7T1cmj;P`WKCd!0(2;40jZeqz;2!xsA2jI0Ai3Na-Tm@Cp+mY^|CX+h`=+s22X$lF;CN(*4nE`VsWx8VxA*<3Ym1z$Paj<;Iwf=SH15s!vJL6&pmN0F_xqTC>UQf;587w;ckZA!q%An>!%R;aEwkTBN~BteAZ+?AsH?D}Pu~9Fiy?Q8)pho}paGM;&5sNY;l4i<JJm?Zhn11C&guwLzOLP`OHbgPOT_MnvEQ8mfL*r2y8Kh$UdWijIN5m5=#~Gl4<1-Z8M&Pa*rYp><_928NaKA6z^+NF<+T6M5Wkg!=5o0lTKERdb-(%1$JY%NjuIK{mYam#l$%PnvZ0~5y3QNN5xgQ5oOvCp>ZW{iDR<hNoi*d|oF4=!=cFHHm;_4Hgx0;B^p^sBTne~AF*RVGKOURn&^E$zC4Y=l_K6xIey1mW-K6$V3%ch^O&#FU2f#llL1(k2Rcl?&qJi{2kKhaajLj@^Pu}W_^Uv18HoF^lMv$}gknDAG`#7g7WpF&ym?*YhNL7EK{vndSAL5j<gTdgoR+=D^we9TQHm3Z<*?Q)F%Jn2>K6v&U>KF676oXQI<^oks>n}sggEoweA_-k6mZTd&%Uby4swa5q4(o<~pWPGg$h|69#ue7?!GV*H{*gL{MddxOUG8D?jXfv3_U^LssNFCHtJN%}L<eKvLB<^??J~n8nbKbCMV0D9&*|dO)KF3oycD1C*1jth4<PJfp`UMv8(Xi~0`uJqZ276}GNvmA1!yGP6WQx}GoP)iJ}X$<3@%yR9j1@&YISpd>3+8D*XP@K&|Iuw7FSIc5!eaUnYc~1rX&-8BnZ_bCiIeweTN{LR<dwXcKv)-BRiNrSMwZ?+RFiLtmEWy51T%S1qFGS)&6>gM#(A|8kIoK>a`(~Jw4PgEIHHhJY}}aEu?jLA2nkxaxcT6<bPgsuv;Fm=k{3-U7C!;fl2Iw8E69I-$NN3a#R9l^7OY#H*OY3ve2F^>E7pj!g@-y%G{}3ksWV8v|m-;jaI+?sID%TOfaZoUuLL}j-OB8pGgYMC(`gTZD;j{AFsEcB0Dv<+#N!Fs@11CY%?eW0=D+jj@M=Y?ufvuRK#}VhFC>KZUuPU5;u&t-PYe@+$~btS@$E)h!X;N3BjmR7faX|L=)7FKm^4>yzlG1{mepNx4KzSmw8gg42s5Uv*zG@JkK@u;XTzf7O!<K!Fz1^U3Rn8Vy0Q?O^aB{qgXfZFNRq4y6BeNR;e$RE^w$8ogzB`)1|j~Dc4k!q1mD8h=ny~YnHv?k7v7h3zCrGljir&`N+XiyLSY01T;ZvBM6#y2%z-4%7>jZmO_J^T~OBw2YfQa9<&rp{T$bqQ)X?4IOWjLqly4SZ-mNPoaJ*??@y)=ud{gI=aeL+7LZvT*dmA3<c5!>zwe57#wF%_d?U_Jo}kp9=XE<hovb(<Drqk&7^>2-^6+#O+{+Jyu@fLnFuo(<A_Ie~#`1TJOTW5I%rjl#;MZX0otNTwaQ#@fiAB7tFB)oY+sooAF7bH0VI=K>-LK!r4XajUy707_{ciW$?do+u_~J@?`R4kfNS-S%tPp4C;EIm$&h7E`>|Ljrh0Q#A!#)0#0>*<oAMW1BrZ9Ql72B-#d!5>f{L=|o9h#FC)8m3kEqPqKQ{#EPO1Fg_1N&-Ap837H4gt5(+jSOWmr#KXhKr~Tez<Ncn5f8StJ298tQ}d)cya1&`D<U{?O2Jyr#x`5*f%EX^tBb69ew$|<c;M+s76moMCX4HKyQ0vW4~vI4HR_m6EiCll64=4yOKR1AKN^nW$qBq`Wof^;fL#+07ruk^ryFIaP{^lu8;=$=}oZrreDW5UfTJ<I$HgXkdXzRnP*xZ9y`;upoJ>2lSv0pDu{cj%{!;m05_G6|2&q~;z#rBT%4)1jsc?(t~>NM_vlRBD~%mBIsl*X^8yEMC4SLFqjvkFxcdB*9Dk>MEnE2B&F!W^icW*>V7GXEPC;>At-<KxHKeXneJnynD|&`!wrL?V$5c^WWEHU&ijzND`Hf&8^Y;8ya?j#YfT~45mLS@-X-+fH4IqZnj9ClMC2L$-yx~8Tu`oN%zo|S@X5aVqllc_{K5Tjw_|QdKRl+~q^e+dVW?qJqaody|VUKrjJJ;stW5wx8WQ#sUN~92#@GqU}7-I<OSi(PeBSLijTr@q8rf1hdmkI~Yv(;~|CiP8jnO>`|ZSzsv{Ww(Ro<NK8F}rNV^>Q3VzP9WIuIG3B@vZG``jf79y18(r`;LOzap+abDioeyyKF*FC&_2iRnQ(<*VNbSQ3u9@NW33BCzM`|FLCaGqg+edam$R3{qY7*=^&Uz*Gd(OSb7_7dRJ~-rwhMkuGYY)HAkUg%@DHMdG;y5T)WEiCTa9kW&y|k28^#gt1w*CW#!iB=-;DGP?+^aKdU9EF#kSZ!MgT_{wN}sJIZ-Lq1S;{FcWC>*seCu1qaAU?*7cfdeFf{`Cjq+R_`(P=R`PcOn!?+=XFFGR;~S(x2b)NAH9Gl1{-B|tbk#uj+bDTLcq*2H;*=v`OUAD5g6~zG4AR&Pn82Lw-3sLv2qjeSsTY%^;@$2gT2jk!t%buW&5xL%^Kh!F*u@5)--uTrr_^vt&_fRIH1Zc1zxVgWPd@F@cOLv7L46}|KZ5n=uH1+j#10T4NzFD+_V?SZ?;;vU|mJx>#0w!fi+)C%%WK{p^9k3)ytobpE9x6iuBgate3CfEj}v({uNHg%|pE;v4h+{-yCk}%UPY{PuAc_EWP+ax$crW3hh}F1qH`?(f#Y-15*}$wbn&OgROWKvZXvY-}dXN`j)m)|B?N5dg-&g{o<KT$6$HLu?4`jHB`^7xEFTt5fQ%+PjWS_xer#GwaQl5Jm(XG_@1r%wmMtO=~H(`jvT;^yb)f`1v?gB)5p5sc82%EqdQx$3w#*V!Me>vqH^A!0jv^qwX9}|a*vGm%k`cN9SKV(a0RO@+%_<j51CH(UiP6gZY+mGWQeRRO$g@=%ZMeiGb^99`-y=qdO$DBYChm*Pj#MskCgcK2n0DsTdjgQgJ)bZ5^J6t_G)|soI`iDu5{f^=ekCiTaWK`+&4cRn($gU#_RZDE+wf>h>KRI(HPHiEw18Ov2%vqY4v9>0uKZ9Po?EHmXmJPdrM3!)rF&-S&V+FPulDK!rKg6ozrNE6j`yP$kh_zfi~Ba>iPJ~K{0mZpg#B>qfe!7K1XzBV5{q$sH2-_MLb7uhC3MH<eeh6v#Xyw$JX*5R4;dDV@KU-U2(?t&S<x{v+Az(lMkz9g1j~hLavDxnAe3#;7pDe+6}%bpa4X_Q>@I`hpIzT=~Yw3-$BTkshtBjmd!lo#ko`B^o-A)@(k*uke=LB_BQa#ZGADBZCC89lUQMOu!3bena}V=?<Q3{Ix#&=4?``Q?T*)vQK7&N*yumUEATe}B{v5RZ0CU72doM&e%L$!#cA1LSh8pUyF4%9qqo?KTwOq34dt<ib8ee?eg!hhpBts%<Jho@Z29cN{--6g^+pi43Gb~N;H>lF%X0=Ej&kjQUW2s>H5eX<q=d(J`1~=pIx5&rsPPV1Z@a7QAJ>kaUNdN&AoTHRg=q4tj<)bOzM3761bC0(I?|O1oef-FbNw+Z&{|+QUZ0C7%V&rycV}ogs;-gs>=$&~?4OLPSJq)yPC&Jj3x45Iu5(khNk#|yC%0bDP2xOR#PrkVh<>E~@A-i6(=}ekq^2#S)!iSKZc|ZTM)$7OvJrYh*XkT~p46F!e}&RR^qQ1@)B>pQ+~#&cl$H9?+M>@pRW84l-IPuAJWfxk`7w#Zu7Zr}yXGh9b1rqlC~nQlpMl8!^0c`*Tj=HvPCD+NJylmCRpqlT8t|9J^(~DW)l1WfD?yzdzxQl~7l_vGm8Idn+KAQb20Ww3h4IXOC3bn#{ce2ct;OSu^jK5jRJQT4p^@>sIh`F<1slsl`0$R$1d~q1>`w;;Vh-K<vi4b=?@(i<E=HqPs_ehD?x+^b&h`0oIhPsHUu5v{V#rQ}rTNY?){ose+->&Dc5lUm_|?77X#;Ott<Lo!kJi=6Dj#FrZ@cAn1ZQ%dP?ZYhJ8u7@+oachcNWidIJ_r16i*wg3S+MBN)FgB{Ea?nIQ)x-B&Uz-#pv#&(W~=EUs|1Q3d?DGo$+%o5k-p|9X)nUb#nGee#vFtUMOK4%OmSNW_~XkVL+V%9DVZXmxtNjZ0xhEZfzIxeR<x2^G@q^sWyMLs3NaUr*(gGMbJ!?0Lz>_3R=<Y93rj>|G570_GVzRI=LsN{u?}jfY}>KFpO_uZmVPGjrwDkOUGB>XKAk}shIwm|FD0T&mTdDPwR<sfPw0K^6%c15hhLM53%7*<MXWbf&mhIHl^~tiw-&~bDikVxzF!i4Q>l}d|F*nz;HpW8{)pv?$}4Dk2WW3z-vl2f*$HGYVHoBba4sdq3Cz3llm;wD)Yj)jmW6G3=(7>_vcmJ#Wk9}_f}*Gd`=ljMtv@^J}YlhU5()Qz_{q#f2=l3e-2ozzC1kKGNsb@iuV2@H8WNh5`10OLC-pMe&Du}@=Ig%lwUuxX5`Ht__1f&t6g%=OYjCDy;k#3+TU)dRU{6&C94L9e8|$W#7}K52rIY7BE~4;YPw^Q+{C=U`>TMHUO!Q_h}n~>D9sy-Of?S(wN>q4RnO!3GCAu^I2@NFoesvwaDxPtGt&FCnSPQvXOa!X3AT5W#9*`V$^IdK_vhpKeFusSE7zOg0DcrpYrj3S(|I?Xq?<@~PW9*J_}JaA^VVRmJ(CMVzEm}Hud$QL-BYQ0KE0&uR9LX@#zFwC8W7DE`tF_L^&k{1WO5S93)`(Dy-OdFw`;9VOwa~e9ia7s>{d_g_nrXpFRw0B0G@lyr`(?1J_W&@%L~Sdl=t4h)`nVwZSGe*kfUI{;+K6*?Ed#S%j!8GTRzV7zE7A$d95#w>$Do<Wv#N7X2kI`Q-S9EQi7I?i<NCKCT5{JNZ=j1+Ldx(Smt{9V+&EF8PA7rZOH<R@sd*-{oHl$ou$$e=-$D+(yNC*)H*$%t`=3|2ZPT`56z6j;eOu4{^Y)CoU?5h=5{6L2teMY%XZl8nP4IAmgsQv`ryxWtWO=hrR?Wn06WL+xdPRg-(UH?nL-15&jqQbY|bSHX8rDl;`NHT_F!ytpH?91chLt$V_|V&JZOXC{>S^uxVrsHzyK!p-0rY&@Z+=--8oj?mCd+b`EH=|agk;gcIV+IH&s^0O%>U8E&AC!#4bWldVvHyuqiwRALle(@YJxM`u3ih@7?43)@4_>+$-mk7c35LlZ{aCPl$&wG;m^N+St8IZ-xh2kmhTN`J^}UStA{y7KUH;@j}}F{$9Al#9k34Mw;Z`b8O7>w>E!jJ$91m0iA2%qrUoR+HA$NC`vDZ<<B#7qH&tjB(~|zpxR%l;8l6Ell{ZQK{IVDyJr9OMsbAt%7C6GiO@Fb^>2MO+tfu`GmP|pZR&^1sR8d=NC9QAXuI6kZyavr%lg&-`fa&wylWKkK5RCf;!af<&T@)cx8g5wc{N+j($^R+O2^hF^{tn_-q%g`$19ON82J8n^IFxA&hlq<N2t-iJ4n^DbW}-9CALeokv6B5Sc)p0Ca?#W#ZTQ5OaB!v5$b_|W;T}cL`ZZFM%0N1ReE<|8@U>scE47yc2a*NJSSnxFg`Y3P`$F+z}I6Q)U^{7oO-qIbPo5g1klF#%`B+>TJM*O4M#fLU0^R9UvIA{^RdRjgBRl%6vWDIw41+~f!X>)pIRuGn9y#je7C>7yW0#^k87kZfECcbp5C>Fy0=D)?Oom-PILI(xJFR3d`HaX6i9e5uJ2d)cY~tn4dId5ya%!UYd?+|dtAjOba4E2gzGVuuR>W=aLy6l89%9ykr4a6X_=afr-d?P$kqxcw*dVHCyYmqQfK4t-F+QNs%`7EvMRNu)EVk+uJfT3(v-oVm7RM+G&u)C^|FQze^sk7iHk_Bk0f<{-H%-2g*_cu?l6{^8*!R!Ey<U=av|c6aj~_B5p(CV>v7!2(c1&;PcKLFM~v+%0Cc*W2fWv|=jUS7l=gVVIU@|WUdBkHIh2P}a02My)dkvvJ=Az~Fvy^V8|#|Y*J0yq>p`V>S&iL$XywWLnFIpl`m@H&IvkRQ(dH^QdfBOqmkGF22Yt*?aGqdKmQdLYZdzdsI5kN3Qga9^FRvy)IeE7d)ecWH!T#e4KN%93nldXq16%XUhqxYXR!jf+w8GY{QPZZceiad@!)rd&O@~};-YAS;s+AhDHy5Bvoiip_qCY~K4-3aD&Xzs<jG!QlkI<h0=ZQ6`Cut63p;P>EVTIWM<feJnuD^jkuxbJ~->=($O{8D@Hbpi%k_`6ekt9QLBXB8KC!z9Z44nzW%PpueZ8cv)==}R(C41?$(@=OowJP>y&u8NG%u0{+u+1v%cr&JnMZK^KfBCc{>s@s8q}2dj#k~<e1?cL`@Pq?zM#$NPpjBtonSz_-)Z@-ZUCj$En9jESpLvJR>#cAs3V+O|zs5PSx#{g>fk1vd{=WC&z!Rf3IR+5-uwnj=9Hi&523@wtclv84%Jh_vpmZAN(p#m_Ag?X}PehO8kDCIecH`ATFrXG?Px`=0Mb4Iq8krOrXor$7c1!q*&z+&|#Gx!>o$s4-Z@^IG&;xfvLGsxZ@l#QpL*HE<r;t^+t_Yx{|6ck8x`fYjGq!{1Q^9AY0fhxrfAp?8apVK2X|-4UmLkp`$?2I;d$QMR3(rPuL+nuW0ve^|eb{BDG+UjOFtaEa?>K6E7Fg?-Dgv~}e<e(<DDiWAlmMbf?fnxl*SdKvSIKO7A&v&7t&S_nnuC`39XU%zy}Qe@B8(?o@Iznw%Hvq)n)76Fy7tbY1ykMQLGw%b)y6BZyI1K)#$9UQUtzx8pEX5letAN=-wB(i+w;B#O<K>yO)9sdH@$VOh0kCK%In50apktPFxrGZ_mOrvA6It)Ywo8Lk{H3)`oP;~gVhf0+b{I;YW21%yd|-83m4D5+}ME(d_b7Lm^O~x7+RnShHD)pkjQfWtU;IOd!!sjy8}X!WrO2QFzQQ-3H>|%G-%_B)u8I*m3B+m+$%=aX?Dgv#W(hm&@Z&8yE?(0d}D#50J@URkK7DCPI=)fdD2n-qM-1{uH-EF5$ySs6{hcv?N7Z<LFhcq4)2vPH^=>LY9$S+wwL3;yNVXjJv%-$opH)xCDVb`0&av4GtY|;4b{Dj$*1YmW%v^pcfH58@rfI+U#+<?d=&+6_2WapC`q}#(3GE#wSsEj-Sd1np5e8whztgB{jwvSgDI~Jv)?3pP#?K6-_MCnR@kDGt(U#kWBur(LG@3g(0aakG#8kf6bMM6Red+}Ib(m?kJZ(&?nDkJPEGzGRP;Rox#$hrglD8(J+&5cMV6luZtclS^O98?5iMA&%UXe0J3LHb@SW*;e`yo_YPORe34e|jsio8e1>ok}^n|Rnv6MJMmcLe4DqHf~RT<RQ>5TBJOlABX{hE9yU%;yKw9rpI8FxF(FZh$YdWQ%Uv!cAC{quoPV-kdh;jWq9XS3;OYV9^btY+5!GHSKrUVhG+H$C^Pr-%)sJ3(K+wPl~fua(IHpbJX5CTpngYxUlpz=Y%Vc{`DvE{|b7UME91o%jpDB&dXbn#cTVTi1O+w6gU6@&^}78zB|wiRvd8?nd4pwM>yhH!zHzt>(yS@fB*$a|-Pa!``QddVc<Z_MQ7n?FOQK$#+f0b&u0=8r{}Zm6;50>jLLz6pjzb-mkcK!Ta+E@04PyfHH~8eocyZb+ZcfHTDr+TbEPQKsWp~pZ+y(QFR?BxM9YRdeEu-2O4GvA%u1{YCed*#3JN2SqC-?kw0=bYrIUTRrZ>(thcvzyX*}mpf>y}<Sywos5f+hRsc1wsE1QM1BN@hw63i~b53_=XF2&w>2rAbQuZ|XAlG=F!zcE2aKMGMU+hzoyN7hoZ0x%Hr1EPGVz<9kUH*9b(FIHOqt|cs%Ry8DX46c%b32EBHeV}eow&b~&?OnC)nhl$%-VRzccTZsqjj7g&m?HO5gwddG+@4J!S0HS_vdNV;sGGuP{)XT@U$0=!i#ukzN}kaHh4`wY$I#8NI`3On&p*M#mC3EF>Z2^u=vg<&OqpmJT1UV{BiF3yXQ-YFyqiP|0r_V-z2OU{?#Bh7nYTaUs;sUY&g0raX|;)XfZpKF-R=sLFKuEZ6ZLh(dFe!5Z-wRNw-*cv}IdTcGoUAedQwC)$_Yuz@|ysyd<BWJ~91nkJIUH*=bT#zc62`>er9pe4qPttHn%G{8;Oj{QPo<i3!<B`TEdb&nMSRhm5uQCpDwYsH*57hWKbjn-dTGcoWZ=qx(d2j!0~S()zKcfwVA+GyO+(p2X>s7KrNim)no_W$Tc-8-Upn@rSL{2F`&|ArZ<}_Z!+yshGe1+z!HBoEu|`2kL^Lb0@9w`L%#Fe(!o_D9g&I@_uZH;paUvYTqks-{}QCxekp+ttJuI@nm?ppVQXr(6_;AEZJpy-u;wuSKO)MQ9*X++W8?@8*kcAye!9nD2{2iwy5B%`++9lk;r5W3Ta8xg^wLY2|aTk)_rRfS=gJ#1i@!LG&w3<rC<`Z7lX^o?lX#h`y7-#Ld5Z$BaOuDaSDpxp)1=D^uf~r6bGt^@iOU-z<QdOjffhC;0Dl~s-zc3>0;!AQ@7fyCAi21F+Gl?F?;#)Ejfgb0+(go?;<_6diteUIM&gembKiC3gNFTFz>byIXmUlFr1TjANKtD*$b~hvYd)tUp^_Xwv6#A`H)AzC_hqKeqX_1g>+R8@wl#VZQY1bz^?>vZJ}~&f1|O1(n{gU7kz5YnBAfe=ESr}&%}}YoY(#W9UAySd~0xHQg#Quu5)1_=R<0buhD6=^lv9|uKNb9<9WhCCp;N#Qc~SZ*&hSAw%fFvIsQj!94RV88^RaXbxCNZ_@}n4*D%M8*%1@_#`1tpkL|pxNJEpz&b!OmbNXIot;DZoFFRM8cx}~2!zkU_9VoGSX9FrgaCui#n!}b?{W~7jCc_SORnCovMxC<ny^Sb~r>)0lVyOu)3dvq{@A~Dc{Sond-~V3d{jk(`yMQ@WdxwqGev17mGw;mIKKv)Inra7BO?`ak=c+wxH7^}cN}%;-Gq0>PzIk<hlZob&hKx-b>qE>p8GBdat<QU-909mNPj>|Dv)lFPTiwYhB#DK?JjI`~;xS$5YTr?N`q;g9*^HgeQz}^xBX2#7aW%YSlPyOyKO!`L4)AF!a^=bD{a(WJeH1Ku%U%0(*#F?rSqWNn9rT;EhYUcqBLobsJrFIX6t~?m-v#z2hko$|3bX=oe{vjQ@+d--hBP;x=ST{}T|XYtwWnNnC$HA((Hv>J%8Czm<6jD8<!RM$7t_NQ+NXbK!PM7N)UY1Y%MX%jmDgd8*B}%?=>2dH>-2QxWv83Aye9x*ma)LV`@s!9mfPXlXY|I{O|flS#UG1DlXifX)VF4ux&+r+EzwPBI&E6BooudiC})1d{^d`tiLG^sxkG)h`xM~wihw>N5BV+zN~cj{jjK>)In#3O_??M4;=$(Uufwg)LV29BLI?5mNL?%}MwZa1;Yd6mZwaj{{TEkfy0a`4Wb11ofB}q&q9O!P6u|)%L<aE(;smISq9Qn4d%s@~r*rxW*m2ihYdxk?#og52%xvF4pjltZ1oB+&vcs|Hqf1KSR^2t5AmN9H0Bo^=>FR*siE*jOmG%9#c3o`QTeg?j(mQj|H_DcCa`jg~_GdJ*@|m)=qGnMpK8O3G4GTXK{~B9ydh0`cB8qe&8OO3<-rTHL8F_4?AKFPbG{l3^y%R3hg5qLgMBww*uH!4OPVlEP2pD1vZlF+IO{%v^G&*&+-!_X@#}Bu9T-4UEzUiaX$mzBSZ>2UG`163N-44n=iP$@Z?BR|y9?IZlO(n?sj$Ugr-=u$Fm`>^8Vw!El>3pOQ46K$~Of;X38;$!ac#o0x8BA_vN3Zn>A<>1LI&G<y<7ThT@WZZ4-&#*?z3PH5g?$}wfNMBbZ3K2bh(#Ea)7``C=1=hDqxXn|nP=mF6nb3(mqCjOFn8zxTY{aWt>vGlU4+HEgSGAWSvkEpAJ(S|WMua_+3$5fIvZ~t?O3#gdGB$=jx=rThrn!5xp^WO48aHX((7cxUVXkVZi>xkpl-<ydftSZjw>T+PFT}3_`N+T_U~t|bjFXI9@6c2^W`uZh`Ihoz$Q#ZE$@GRgb^@gdHayeFYJXT?<&ms;lBCXA-9o7U#d%ntN0Z<Qqm)XmSVIWem6UL8NMd#qk^#nM+cXdWhxECCBKL9Tb|3yzQ7KsmO2~Ko;vf;if}v8lhjoXX;2(C74EcLL~a4HSVz}hT226m7te$En+?~;QLj`3JfW<Pb1v%E49KGKT{E-v)K;i7f!*x-s`uRwrID~JiAX-RKG@U_7PS)m5<U*F=oD0f?V@|c9t9e{ztCj%Z#0!REg$0ksB(2PJWh)R;?Ex-MrQ3*(&-}upvG40vq#Y}VPLqIz#tuWh-V(&+GPn256@Yz91RkS$C&ArR;Hf;4_&&|S&{6;;TL2k@};xiL>NExXTndl1Ir`(i%FJX3`W-mz)clT<~2D?asP&#Rl~5?BlCC3U1=}b&SqjhozE4{`g;_QAL<DFa}G|9wToWk740^Aad&Pt($_UZ<;&hl7K@hoXe&od0mjRJ-A>3jtq<_k42&Mj$PW`@4L~CQ-lAZUXyUee`6BAwO`KP=Qh&P_WO_Ov<_;G33WHu6PnnYj>QY4Z@1*h6pS$C6DK=*_A?wvvj%jR99W*5b6Fn?vbp)8G2!pK28_~MUtAJ$oOS)FSCuR|gLIc%RBk9kGZPMHCrhK97YaHA!>yHn!gPNKZzuvTS9Z{dO_Y66zPsJ-rd&ko^`7xpGqJlMmY3hL3^P2R!fhdBL-Bj5f(Z}Rn+1#YRvKx8aVJ9AbE5<#fw<Nw)9_vl}7E>o%2|D9M8iDEYUV0zH`(Zn0o^}bq?efg-%PdEyV{~FWpX`p_yEF$iUwF+8TNGU5;`eqE(517j%!zZ@GDp`rsC9lPA3}!wy3`i43ei$|lCCVIl`H=7Fb%Pd-ikfLx613LLUR$0kGJP3oCrP3-?o}c3bRHZ=|cAUeIEvG;=PoR`-zoqB>bH<o&KY~9mIk?iHrNh-nj;IICmB;9hc@x^UP|5&+796I}y9$zE#iJ<2z=E!BQMVaCCT=CV#EnJ5u;5nC>4)`80q{j6|0ce7IbXAfB>GKR$haV$cXE&*d^gl|^Tv3yNmxpKe@-5xZCU$v(ccM+}BI=c##X%qqnu`}N(2k$Y;ZJ(|fQneA^5#b#s<zv1Gto<FC*SZPng`Y_3k6<6*a=w4r}p3})pcwLhgS7)E#-esB@+&&?}4*t^D#C%I13j3{)(F_5i_KpDHfgAZ`m18=m;#g~8-yCapU)Fw+wym>z<WHOZt7Y*`LQ~h!a1`CPY>?0*rm^<B?0lue4lL_P*4KWu+Ng%kj-@v8vpP_)v~nf;o%x(<?KA2cCRW83`W-5~{w2#g;5b$>L^Vr+qIlNu^yKnwmAu^?&j-hed)%#N$seknrc9HXN-7YYpmq0cD9y(R3oh;NM50?ab|L59rcpAzJwBuu@29|z!|eF0WN0%!E@k?2kd$o-pr*s}1M(t5O%Be5$K$asn6KsZhVS0nK3uNbjIPqqQj3W+T}{=&eNXozNtk8ZU1znc%+{L$&(ipPT|@qGZ7Rg8S9YzhHurr(V#~B8oM&4vco^L1f(1^02Nwwuxp@{f(!h@L`g%B1uh;cjva5FcFpuBSpNUV2vpL}OpsfDzQvY+z{gol3nXcTm@kd#8H^?A<;#>bRL(NxbP}F1ld0KodU)HY#?RzZ;TlI`VfPg{TS&r>@eP-Y>wG$Bt1lD3T#scp0K{%*9)ap4<Rf4LTEu2iCgAbz|#D9G^Zoz&Zg;f8%HvkG;y+f<__odo$!=051NROgZ)SkJeNkQK~%-ieXbWr^xn9s!1q*c9~^ubGCvNzYAz?<pY=uXsc?WE>P0TK4tP8Zkr(Cp3I^|{WO>UrLz*<i>D@m%SO;P7<z7Ye8J*yDAJW!*=+M!Qxuyavt{?Xvb72VWg0+CLcNorCK$@NbdF5;tn&Eh>usi+@H}IBGpX&$%C5@K#YCjTxqBJ^O@xmU$oYrj=oWbDdeb-iqf0x;J`T?B^x&@6kxsCC93d&Sbl9Sln-CS006&+Ouk4))4ciVV|h4^n;8BnxZZ~ssd;~O~V|?2vm{7)?D`a69+M}qz%Ljg0T~$iYR~h%yJJ7m17%j-@%MAHa(Cn`PbNGAkp9G-J^K?5Lh|i{*ljx8BZ7Z86R-jc75EmZe;&53xkF~4fT`LTVUM!3~RgBJglC1{ZNd5(R((t2(kg^#gsXyv-P#V+Jev=<yr;(p6QpQ-Rlxp<q(Vc<a+RuWw2gJrMnsCV6j<Te{wT<p8Vk#hnsI&lZ@NWZPH8nTvqo~Z!cVum7?9FJwwC~jSh)FOPTZ*y%>BQ=uY}mAHn6R_Em@AE8i%&6(SV7s&C@&WygIrx<!4m=tgB<eNNu7RXHK%a!}lis5(>hPJ^?2m&nGkw`-u9UO%?}#BTMM+FMtU?BCB&*%IjHY2|m4O4oR(=U4asUR>#pKb!R(el_~^%~p7!mizj@vDz!hYu%IK7AA#PT*)iR6$sD%tS3l%cV%ww8D<d<CAQh?i@x`iwpcUW8qA&RQ1phJ+-8pi^qRDa#)#s3iwy+{{abtefLp|^pBM)^XYRwKVGeuCARm0N$_KwJ_XM<t)~XSNP`Hp^V!9#x(zZHP^YL2Fhr8RK-d2$PuP*fu-`lc)DvkPlO+QAa!Pt+-tV&49E#0vmT>FUkIPP@q+^a+#ZsQqKn~vRdWE_ROHU2Zp?)E*JMbh;RBHS2sD!Fpy(*nI5c>TVnIvKwp(`Ai*-C~q}W}w=W27*0%PkErey}Z_Tk@@S_;2sCe0h#zIHGZ_l0~`+t2VZw}q~kr+4F^5(p7@QzCVv<R+Sg@N_cK<snt4S&tEb_I?^`#B{bkJAp5ut8zzuM5HiN)=NK70cN=Y}hK1zQQJXo7ZSC$JXrlJh1-#@PK%u3T54;XMzpAcKGjtyEhB<K3JoV!6vBH^?`<50_??=oTTXhw0{691JKBNFh%sw?DBeN<bWHM&Wi)T*N~Rpzq~!A5i7+{ZnB<zsI9)<M4n{LMlE;TumFEk>Q4m*vj;EKNu!oj*H2rx`TZY~Veb4Aji8hwW(dlIC8}w21C5M+IWPn+?*xP!K(`k8uvuy6F9-`_68NlRSvLq=5zHH5ssR_bhQmR33wGBXu6^0_hUa4o>5X$(6XX8LY0qo^Z|j2_<Fg{px79THR+=4}dgIwL?iKJCoYK^+KYWbJ~HUualu*9k-jP;w>5Ku=4Do>%YEURpfS3rDW07L<g0cWH!PdC%LY=2=7Zx98~#k*75wxo`JdO06&aY55ONWHLotK#&pFx5;fV?^F!4<rIX!qZ)EY(#jHB=tyN)#4{;^Md((7}X!INM^63dXuBH>gST;Jsb84<vVK_f-yEv8jlU(S=zSXivbiIpIlSj0r5$NxbY5|c*odTGtE)O2V9Z@4pCFok-LLO>(9u!UO$Vz9I+6_istX6HCq)q%HV?a#4G>uClrFDdHn>!4;^-)ZqWuRBuZdk2M07qJ{mZU0<dXtCP96au|X1jD@j%3etzY7A&Z=#)^_{qLCXWKS?sNQ`Kr*Kc|cdk>tdKun$%quB0SvE>DO7GrAg<JDZXw+xC@w(@r$z#-7k-e(l_#C&^&qqZ~d3C}X*9rUOpZMYIV<q);s$FC6SBkGC^<Y|E`pzJK|A1GQ|GF)*=`dDH@AeN5hW?ZNOV@exdYhzqyYAPN<Lt-@eO67Wcx|<{GyZYl`NY7#n?)yJxC7wV@Dwq)hQER>|1+WFo0~J?wbSrF7WfLt+At0Mj<sbvn_%q?wDa|s$K^z+G3H4BH!E1u;#%euW4qgyrbFU1ek1$+(x|qE(4&=<XZcXOvRg2G)xcKqRaaj;bxUB&w^cme@2QgKsvOc;3Xg3CpXClUjbRqf7*eag?hA>h6lBG=zJXAE-$jOiHn`$9f14R;kB#@A(pkaflcF`oA150V=X=S*3l^#}MwVZOIHOhdQ}%J$gLNNYRo)(qe3y&5)_=SYcmnovn8E31co@8orZQrF^*@%{RS&PaH;$yAh^+n3`GKZ`dacjI^vRs4+mzK1`hDl)&&C{z9u*70<Iuq<>9YUxjn6LSQX!~wW}D{tUB&DyNC6%VdPtAk$4T*!^FJmzx1KAPxYUY6^Z3{`?vB%o8SE({1&|gdgRx$CceK>L4A*_hG3&P{JkT3nSOrG@YBJw`>N+GZ@+`cbWBlhe(i&hA;nA8aX>Y^X&QtRH;>zD#f_ZiQmf@8@yrx9`4_n;`l{Viy*nPpz$Gx(M6ncJTDz{h@WTX9Wd{|d_9}9S%z4}hq8n+Lv7t`sy%@`;WxE)QQ`H1FJYuCt*(8K;>tW-SfOzsZ%UFQsl_d8KJf{$izuAHdTJ=f6%*FJ47xEu(~9avW9swuzKKFsvMG<q@J@?^aw&-dHT3=*b|01e~A4&I^-0g1hIyKCUR33zdmV{q0>6mfT>Rt4h5@HC6t+EMDDA8)yK&B1UvsYEI`u0#VFb$xr^)Y|f`4Vh29Q5Pwr*{h_ZjRX3`vR$7s3medWQhuHSJ4zmZ!qYL^2B_XMdh+-=9dX;vRv6c^AsqFL*Bm;KiT)#FvnN3Yxo%u~W6_e!h4}k>)WiOK)6IAL&A8wn66v*By!zdV)qeFkF<%X=(x)D$^jy!lxJo(vb4ZcgdX6;)-VJe#&2WCSW}R<W@EPzy?p7gsSL2f}U)0IfJ&RXWtPbOyTZ~%^<=*WfxdyAlJ1$ZjdEENZs0Gv;U}x6j6?mH_m=Ios=(6cnxA?ww1k(q#?Le2ysZ-!b$j}}$f}RNtrus}*Qmt#)6Q%{v>+OO_s7~!7y4q^fpZ54ge7KU$XQg^!0raVj>nfJL$XVMtVqIz9!?p@KwVBGe)3uJZGfx$DU1ln@77?9ML+ggxvdmM9J&!{HUbml2=*M+ds*LX76G^c_`Cbm0<zN^&jol%cuUd`n+X~GOWcB<#6~foY^k(0(;-b<A&KUG(E2<>=?X`TXbyiDTfmz>^H*Ui9rkBH~3wEg~<x*YYO9=DbW>`D?w$qK;@BHv(>Y|*y%v9PQe7`#|la$3=ugBGCf8rPO_b}kj8zDS&PPVZKRGQ5ipIhErclbcb`QJ9x#?R}0Uj<3sZP@Mdg|Zsx;T{lL^N#d5Ol#;ofA{DK8dn#a$Kn-cN){EZMn7%(OB)zqfpvpWS4Ymz`_KFXlJ%H<@&4+q+}Jh#%+F5b*q4rcx()4oiyQ~jbXAAZaT%9Rhie#{v=~G)XUO(x11)v^sMx~sTpFK)LQ6P4BxpH7F6iV6P&>We<6R=v%vSk2+@DS97ET&uFs6K@Mm?Tm%l{p9_&%MV?z?oT-7EXnp-aa@0<i0n3iWn%v$3TzY+6sAg;bdpL0^L+uG#P!gS==H6SAy^Aoc`lFK;ex)B#jkW$8jax8e<Bq|a>PCxksO?Ss8}IhV_{E2L!6YHeh#!Ii{0xAcVq{~5sW!<wJmzrfI{u6#t^@LHaiXf+~iGOq>1KJ()K*+}ODJR{o1<)tE5iGF9V$xl6lRj>p_R8AZd!DOsPf&15fdqW27uu0zpCo*s)Z`#)hKkS38n8xcHI1vEu>)}0WLW`?GY{1I}J5wFu9iKz+nSd3^G}NEg%g=LUHGW2)n{`&n76o(^q*u(<WTXqtQGUStHA3Rb*#QkMP3V;J1Gcy3i{V4}cU#yr<_7oveesb>Zbqdvi-n2$Y{@ML_78_Mhjfqo?(!Gj0qHud=YFrH6H`*_JAgzE>>=2dwsh^rq2C!doj&>Ds`g#i?i&x6vq8I!2v|KC_1EqCIvUwPN#!=W@848te3>nqc+Q@6FG53B?%mz?zFfYwvLI{GxoaXfdhFfR;ph39n!v@g$mXdjS}#B_7#7J#uNa|MgRJs^&e<F$#+A)*-*NZZL|%{A>ea^%JYY3k(wI0<Y#u1{unN-KmwTs`z|)@(s`#$8;R~)*v=`!D_ZlLWk#G#4yA(K@)t7Cw955Fes8oe!VC2MKn7&L-Qk`B8(nDw1H5c(O`>5AbLy&099Poekeg7<G6ZL%y_0Bs@XmQi7?Sah-At^~#r7FZsC~;eF@tT}ZCX@H_Tx>gM{->h&GHOlpv8%@Wdf>A5)y}fW+CAe#=66q*)=L|e>AKt3&3Dh(pO(F1`M3{IvA^9!#R&Pyb|H<;eM>%gclkUYf1BQWZKDl#w28h8Z-Qv5VK?x*=}*AN4XKUUSX?k&8*rXx$HGRhQ0>IEKhb*Ivy9$Qi;;FatOu%8i$7twV{HT^y3B5f#AD$>#Qx~JZz&VSdp-B5do5!b@6~(B)=HcMV6YZBRYt|v(&PFP9Y3IVY<IK4ngGdqYaNCHUrZVk$?U|~7j&IdZq_>Qs^BK4%Ii6=wR`(_MVj$*X3(ph_8xFvbyl!F8T5~v#mPYJfF)--+vlT=^4#)#$nNmgD4Z3;CAO|$?D&IFO57NA8-y?khny2J{xrC;<!Izih^Wi{X~W@twVxLT7gA_PY}`30zLADtsKX&yj=i+gY4@d`*EB*y&fsows2@kKZskM3JM@`$wPBwEhw<}w|J9ch>6SfXHFLL@ID@{D^Slf#pJ+vHDZ!}tAc`}Yovr{B-D*?x=H%<k%zoVG6nDqIE@J(F;~=VI#gqTTW~?|31)drU+>x$h0j#fd`f8J}`RmoHa@T&}`LB(h{drVTqc(=l$NY65Q3~8;C(m&pW(6=GjRXI(k{?Y8^$2#LDU`C&V7XhbR^D@?WlRO;&z~0QsO17v<#$?I8-N_sN-y%l2mTA8R^2P5%JkI?NztmR!G`PSPm;(qb^UJFr`fW)ik;mf4T-vT29I#-6FucewOLn}$Bs1|XYb^}p1KpcNd)M8CTs5%scAmc&fZ<i5|Xtg)7x*pV4_)r7cy5pwGkr#Kg;iW`>HpS%3z(LhhlT`mr^Afzxe)T|NOe|XdOq8e(bXQHepQhg*Zm5YhrYVuD_J8wU~3JP8aKI;;W?~v|8?<-T%BhZ=ercdS}Ogt)Qd(m)@)fBY((84*&4ZMy{7dl|iv_6zyvc=)C(Z*}{j=47o19$sb>KG|O=L+zL+fWoyMx>|m>-s-q?DjlBPxum$uOI(km{kd9OD+8XRyqSMmRWA_%xHpvYZ%t>f2?oxy3q3_Tsj><N7rf7B~ERMDBxv!E^`g%1SWFP#bHic$6#UO^HnPT`PDeUI^gWGn);gg^9T+Ehl_%77v_jiK5JIrs4Cr@SWx2$WYW(KoY%DdqDFSOm%^)7Mhx5`~by7ML~ynD>nsI=;r-;~f0?=~Rxr8%Y7(}KCvHh5}TIFm80dW0{4_n*?9!6<%@FF`0G$M-O)o%aafoHhgF(8v1Uk41Gj6mH@_vfKc9X8YCIYB}h4rCN2SjqV)?sI)5X!(Bmv_C4Gib8l1L<V8Yf&xCq$^RO~Q(;P@Uy`{gl!Y(+%R6b?AyiW#-i+Jx7f^@8t;`x8@{a9Xa{k;-B`t7IRE*9`Iq$Zil_uN?V4vh*oUZ!l@pITvsUDuoYOYY2x+o0ho3;xNA*!fkLW8mn&NXn#VTqWgJ_rs<GcN&%NE+rSGpB}X{=^kxi@LF~{LSpE_=K=+?qc;foQ{PsF)@*)1#66_ksiOmw&?}VRQq2qYxJ6w|U3i;LdN8cl@<S@h_FzH|x4_X?m2ShehfiWZWvFTLZqcP)V27lxzibxDh0^hvHFwXmmYp;G1<N+Tm8A}`vvc}ZZC$h`ClPsE)Z>Clqh~n*!Edi}I|PXZXl|{(cxKDPju;!kd{rXHwa?9gVpjvDgud%P-Mq*C6?=pqS)#7ut|5iqp`7*NX9?f^4gm<lDtQ!oeeh}sx<xg{9A(%yceziMg8`ND<Vq)V%lZIV{k^#^@0T5D!;sIC>Uf!G>y8`jj_WjGpX3UJ*Mr|&PtKd^h0%^gh>71ks`wiNmXak5p-n-qOfTKdyIa(fA^*Ni7X;X%Djd{E<sUu|-opIPJ8SOB4Fs*P%W+ZJ545`uWdLoS3oK&}Udq)Q=k@AczR~CL0Sr8+vHEPER{3tsrgL($rML7c1{dY22Dg{syeEh5tUDQ<y~v>#KLe{ZY?M24HD!72e$H0saDq3!xpkYX>SjZaUhsJP$Be9D)=O-f*E*9AM7YELbIMLuDfT6zn>@MfE}yH<!_!1qS4dvJVi4ErWZk2}8QO2kJ|=gAc-ANPK=v9dscn|7bPX1_V^6M_@ZS?!=<=V@-=<srI9s8B-%M^W+mre??}zzkg~!$XT3o!)b0S=Py?dWGLjJ3tdCbXAt8}{cwC$?@Y6Qulqh8#F*o9ZurPXUk!Q<w^jlWRs82XU_sp}EDPr0h?$Nl~DaL&s6>E|-vs-*!)D8H{;lOikA#_&B-`#8)t!YH3tclUDrXi3pGF^3rPSQ(ve({8(gIPOo)ghlqtN+);hjO-dq>d{6xuEf>3Q$02RqV(u^aErxMBsFeLV(786*z@i*dMmpKZoxOM&zQC95tz;<<#A&LM{kU2wd<$e9SY%`XzyxaV0q83F{OHc_9VPtgjf9@uitNEH}Z+>Fn<umb)c+h!ILCxKljH;XAexjH+;Im-BHHhd-dvJC|8c0reACCULQ#-$0=<!o#Tw1oC7Ym86MH&iKb01!&j4CPhhzT5B$LB-oRP8i-1Y;ci&oS2>cp7g&Hq5)~z<uY+mxJWs*75?FIj{+w->h-`$=nwSXi#0S4ryntnb#U{68BQMJDiiLbu3;i9hLwK+>P2^?2PMXAR`tbFNvp4v`+xR-x7fn5V=3AaziD?(6WBTr3j#{~7qIDRa~J{t9?%3rmz@Oy#Tp#y?_I(&EM+xRE+5OM*Cb2t-jh260-UI&Bm{q}tZ+f6l=LD9=jlMbS^l$*Cjp6FmdzVnrue3SR03CC|GsM=wdEPw%19qYs<OHueHFbHxw8`x`H3hs!YxCJl}(`)Mx%GcKX`iuAR^7Ds_=gW7QoH^zGlzv)b-1#nT{?D|$C~eg3!)X{r9pwL;mPd2@-Dl1z*m3YKD{)+_&Wk6qh12A6$#*;W<WCz%XSLjTuFj<A<pnLLF5Ph{-lWZoCp~UI9{se^|K9CsJU~@!z~q1K_H^cqtHG$Y#xrE!n9Q}=d}G7Tv=k01Le!yTM7_D?XRr95V+EJF-K+X!g)wFHtIE^dtlRNc50V=2F>C+3+f(izKYvyaLg?`jU)HEh%GnG*LH@H-p6TlS2c-XRx91uH_}tvr^_G85?gyHSic4^v(Fi&F<S>tdmI0Ets;3<(0HNMQehSSS4Sx6PYQJCZo88mkH>r{S7BZ{}f{zH=5-Wf=m<{gdQ@ISM2O|;yPp%ZRR!O7au305JXLOzfX|ZPZz2@kAo2IGV`c!|=;(M{4J2Mbz)c8Ec5Lgz;A8rW`=>8=-2!w^(c4a)5tUvEyDS)tPRvzN_2QL&U`Ibj)F?5?l0Y4Ts^ig=-KJ;ANSqQ?}chA=+y85RVxaZ@~nuWh}F?QzG?jybe@%#t<vW}_J{V>8DZ}MYVraP=@#7|qYdnf-b055J;I?5it_f+-rO7MCA_+A-kw_gsHmDzoA<T^!Ok;Z6*RWo0Tb{gvST&6yltc7`vP&pezH;-hgqwNp(S~csWTW%mJ%dXexl~}EK+uYPu8eUJl?^iyA%FhUg`_YA}BHxPkeUz_5<;=#`xOOHVe<@A9ROnRUBt_WND)nzNI9KS=ZqRDn&WfZ<`#U`t7Qr6FRJJ(RtD=HPR-dSPO4WO$HV#$DK2u@9mSmnDS%)bn$H#3ed?CHhkMQYo{OxwxXT1(z7ZY-O7NPEp_D~W^tX~mSz&6XYPuo$qEdcHIaJiZEcvezdGg%t+>aWLR*aDsMKC-t%belos=p)W4?0f^&xbemY%|><$9SZxsC3tTgTc77KX^>Oj>;4A2H|Nv4ic)Ew%PM^i-RhW_OyD}8z{@wwc9H$=lJH*)3151_AM15nudG!Wu1dD{7}R)3T>>G{;-uJU-pS{ykgv64%dgu(@uHYsA>%a!f=-|>=Z#d@_7}Roo1?Fx5vzVV|EmfiCOe;Kz+cC>tmcb&)#oK96B+@5Uee8tdbF@@?K5y9&aF=B{RH0K#WB+~mbSHP!}n-b^2T#iE06tM#MBo^|3O{}4>d`W60N(DS?Vj0dgrN8<#?Ltr#B|Sr(x+;DfC(LB%5!Ey-PbiX%N=xBe??@nM;pMZ~9g$Pw(-#83)yFRwsAvSb~}ZA6zl@c(4QU?D*<M)O|6kzE9iQkQX+ijRh-z+2aQt)(xj}&sMzyxKG9LwVD}htuc$U`{~Z$ea&`)+HGZf{Cm{6a*K%Dxr^en>Wyb2wq=de>f4%y_^XlV(YSmUqg?SzbmwEW;7h<)OOWPBzS;~Td(nSF(1!tUxP&f+s@7A6vN}iVH+;X^PYK(`8x{G}!YcHuP%WICuP`TDoNL|JKA$Gv)6%gR3EP{?EO%J(&g>3TQjHYm-BS2Qf!o=+?*QdI{we-W@W9j@L)GZg`Q57Ftaplx_q5N9svYeqScpI2)FnPt=q0L%oU~e|E;Z2holRlIznwma`jymi^zmP=AjMgAWv-=Y?~TiPdjM_;ZP;^ewfle~`*n23|Fodkevg-(^Z7xe4-a79XXUytO@l9CwB_*cJcx-qW*>1}Tz1%;R(8WTGDE-$*N`ro1${U`gY$=QH!d^B2QU5<cM&h_-B!hHx;Ar)Y79u{vbZg7=;=N=ANcVV-X5Bie2=Ngc&>cvI#`?LiuB(-UNvrYp!Q$zO>Jw2G`AEisn#yyB4VHEf*VrCpEZqXyR~*lB~iaCm21BJJmsPzi9Kr5>gGm+|2HU)?UQj+s{o1Dyi@>=Y5$a3i^SDZ>F9<>KkGYqcKY?J#FoCfkUR1wFnm%}5BI6dQL#2zg<kS>S}m+*q7F7fouNCMH4BR5xjQ$J45UeZ#`iM%l$bxyT*5D4W<1DhIErUuB46X9>6jmCo5~SCphZOO%`q5O5Qx(x*CmYGiKEucqD&^=EkJ)Pt+u6N3$1@Fh%$zs+jRQw!;`xzhNp$^n0<|$?$<oy2Cp*FUme%jX3jvyg?z+J(C#wyALeD<{_T?0+-v^5I_4vxquIL^Znsw+9bb-^!%2>pH_g8Yt~G$x`nj1^HiMnLgOX;`x$NtoIkn{9O8hdNWhU#N_M<y*#qUA!fBU(PA?Z{Qdo=HU=audLhz}_X$dGO+JgX3w5oFwR6j8oT_=8Q>3-Z1gcbcni7TizzGU1+W=JmN>)?zSc!wSd9*Z2B11*pOBow8TirX9=biPJV#O^<8hY2v`)#81rGdo+Kv!}suT9GCXAObEW&{7f}<BDn@<^_CUIYU5B#pR@q-`EcurDke=no2Io%tbsks9t8l1o7Zi=DtY%gGZTG&e}D^k*k<?THrou`cq|Q`R5=*7=O-<^HZH8SZQ2)UHChpiN9sED!Shc4eHf+sK-_VO-Pq(k9WSf5>w^0(rX6?Syqe1?Q198DXRFtJLp4k*R+hB5MOJhI?)vJ<lE({O{jFEo;S6qDE|+o-jK1jJ=hepLMc^CG;c{xD<JD5`MCxrlT;%=L&Ha!`DLUEUc@6w=<tp!?qB^CY!VYQj@2G{evbn{quDqlfSb3wH(vN5igg<#s3x04OH^^tNCcQtxLfgm3w%Ip7*p;eUlA;6fxK`5z!d-uK>fU!VE3=Xifm+o`){N!i)}CAB%P)m?wvJI_vk@}*9eV5twM<CB2cIcDxWVb(J0IrR;je-ZT1K6xkGXq=vBsMziv0$#xWmEHTA)Cl{Q5}BJ0Z3Vi<kOUPu8hg?x$&na5v!>k6QzOcQMZLqJAGBcTUWHzjl5!Gk9eb`7I5I%V_1146+1@YDS{YcQ`&*VtZhX`BevhXy}7s`@K%}cSHbTar<TM4OkT)hq|0hHqUXs`SavrOR9VOY|nEamL{WiVek+A$zj!3{sdO!v4J$-n?C<l4_8F*?!C}j8(1B$D%aWMa7N;K<rG}LZUp74#i^I)j20n~S%YtZuV1?vyM6DcJ=47uMYE_hmIrfo8mFs%u!4o%T2Ro<tsM6+8^y}nkF0umL9O%i=ZA;hHptf_FXz5!uRR5YuY8{@B|sO~$Kgg>GX-X<*7n+AE>Q6!SxItm5lHsUrbs50pxRuR$G<4^^3n>-bS4$<GwqA>4fwdnyBB9|ztueeszzmV`KGLOe$2k_dM$2KLtFec;%~!fFWoX!i`#xdg`M7P^T$to)8v~+cI;+tj9JKQBa}I@Z&FaeldaS!sb{ehjdQ;)yPCqa(Qh+C^HzhqqKF7BmbmwqmV`Gv14scTwZ{S(S-v_KLuw@8!sY56nC8$FLtyeaG!X)-Et`WaQoA2)JGkyp=*TVfMY>+4q4a%0*!Mecqgw6g-NV|XI=7)mFDPs8;iAuM)e%%{&ZA`ltsQlZNgaC_*ES>433k^;M*!bP8>r(p)gYDwv@C&_hw^ix{$w2IA^85GIjI=Pn$E5ECOTiOQ-Ke+bnWt;1My(&I>11v<IjCKV=qGUI)VoOHVx3T=z`Cb;wPc24OXM9YS>a-MnTZ#%%H^}*r<4NGxwc-{dAb^s-Q$ob^oGo=<<~ALsqCiuGGy?q3N?j$laAivF{lmbWUMikp{ofcN<Hlw6a#SS~xo_rIi6cYa#BAh3R6{#cS0;F+aCct*d`5X1mPgTzE#8%1rMysEIZ`UhmZ42_l;f^U!~0uI`5^yN2>15+heF=-toD#n2C?vs!Qzx1Csa`xBd(c71+yc#ONJ7hGpSh6>ivd)D5ges<|YvR8g8bd{U~*Bl)`uy~+OUc`EG*k{gKy2veE_5k2%JYGaAkQ>as6d%g?Q7bg}R@TM{=*kQad{uP@<pz#Hdj)D~)?=l|n{dt<y(jW+UzWNpYU<D||Beas4s*ghWsFvDcWd3oHqnRwos+}Uj!(toY_yihCVAr5HcRU1U^G;N$(^co9&4$GYh4(<RI>rKOq;u?7%IA0LHTxfYM+!=H>%#Gd~b4RvFU!(u8!TtsGC@A_IcXkr(qRYeL*9<|BPYX4<y+`pmp+0?)D`Ro3m^=hsT48b%uXaYReCWn20c*4f@A@`C*X#_LMYQs&HvJk0kw`Cn8quom;i}5Gd%?JXI#mJM{QUL8Jcd*j_T;d=6|HeCe`Ikq2bcHA*ckDG#AWX=#3Wvq2V&*y{5?JnIdgrydIGS`G{6FT&WH6e;a}TgBv8+KB17cy!dGG3r^TPtDUQ8h51Z-S@i2xxP5&1)VUWIqQ0gPj0V<A<Fk9JnMV@a4kK#rt+%TSSJ8Cd;Pe$>`uxbqPoNH<c*uVJN_*Ek54xv?w*m!SijMWT6`mJb5g14%17_$Po&RU8%OjJ9|OI18&7rvIM(w=R5@KZu``|{OD85&BX1ATtC#Fu^wx6o)AcY+I#fG44x?4)FhR<*5iIZ8wL8&qGPcRdkkTh)xWi|LvlyBiIm(!z?K1;GxxNf<A-z+*GL_0-qi?&_8uP(6V-LSffBhQfZuLC<B<N!Sx6>h>9IxF)2MvY#hUHObmFlAs4tRb;cjO*ANZA-TVI^JVyB!<UZAncS*#L{?5v%{Phs0M`V|Ri-x;p@4?6_|gwHN3}w<DZvmHlAlTeGK@JAcaiUezoQ42Sfn@!oAnawp4Xuc|egWBDzI<!swS+`Vwkq`hFgiQ}akvM;bQ&^rW&vKEQF4`wB1D?h*A_1!Ff1MAhnxfw6ak#~SmzbRpUAi5Bjw_a`lpY?9nt+sJ`?4*)7<{}sUQ$W7H3c}@wXh_T33Rxd#p2MkF*wh>ED69@23-<f#=NMHLXRb?*H`ALxoq<|g=!?pa(#Kq{fBMYT`$L)cI<wd5Lv2PBaAJu225YJ9hxo==@s!9CHO8(w1;4`U_{yQxa)meV!Vaj9>hU}}7FZ8g!(nd+e~JhZwrCZ&J(su1NU4Nory9zS)2lX=${tWZK^G;|4&3q8BCff8oQ~rO_pQaXj+*G`h0u9x0o@nQZs(l9*09_Nauw5Y;&Yn^8GJsw@jf71I|qhcU(~5J+iA?9Rk?QPUbdYl*z;PqUA&=}i{zL`;+B8G7w4FF&zI=3#4sW;#AysX+v@5pkA${UF-1JlZdwO5?kZ~^H_7tYO9|+LoHzS`Z5CCKkgjEo(~;A#(Q$CTxV|g<_RkvA6V^UKy$irk=WggworbH-qic{n<!#Ebnvc1M3P<lwH!lOVm_GaY0i+hPfBRIqSg@t3bpW053v9&g#kg8rwszb4peLaCIH*|B2GNZh63jseYeQy{UDzyIPe%yBbsWL8)<tm+V=*oZBidhpW&x#z*|!CS*i0me<fk#$ySScN@rpJ^w&;E*H>-yprJt`DLGGDZC@k(gR9}EG`CQA>CKVPEE;Z&YWH(Eepe;|q{8sBMp{vV1WR#j|o5}I0Zokcup$q%!jkr21vMR&v+DY`8-;I))#WKCyGhv@L`|@<uEGPF}^_HdeN2P}{(=X!=wI);*;%c*UQjQs&cE{GIa(+eR<<=Tg9M?EW?HPvi5T*ZD>lJsHn^~jQU7L-F?MK1mH19iuqDDUumFCVhI+PbhS3cwf-pr#<5`#s*0R%hqp)q}<yX2A88l3{mQD;_<(kb7q_O|1t=r>C!TFl`4ZSVInvx}=fVL*-IGzq=e{5oh3A<G(Ed$+Sane;{;v2L7~ypi2Am9Vc7C8+Vp1BE`Kdcy*F55w+6U;~tWI5a!`uBT7pN7;GV<#uDOd2NJ?UHw?{@}o^?{vfWfqt0A5{HJnOci%rsMJ9jGKd+sD{;JxKKK<qtt?YqSw{A6iy*#;I?i6&k4<VLJ=DEVQ=A&(xGw$qq9Pak*Hwn~j212`wNw<Tao8I`>*e{b@2Y<?CZm8_p!e24RDBNW_Vt(%9=RrbrU~0X|XF#!};Y;Fqa+=jo!&b9NedKESP6yKD<Yyxz%YA!5_ZK4@oHX{P!NaU#zR~X>>k-p=X(dc<*;t!$(PWq>ui|b_&NvCQzl-5kGsG*_y1mp)$hGmtB<_(R40U6`O7wO(n;xK01kC2rv5{mfZTjx*+b*FdI;Elsf#Zb(8T1-kw`=H0J9m_8yl!~XPMmH>-4Uzgz_~`e`ZfRCYF)#FhQhG0S-7)`m(jE<>>TaE7gl(3p{(49qpz7KpXV|sb^Ns7)@GBmd-?sIoTvel1DbsgiJ3g-`Fg$af^g20ubO$u@$vi{gZFORq@y-xDU0t<J?gdfbG-2)5EsbRI?dXa*-_UTBF#|fGlfL59=zWTNidG+x2kP0f1c{OP&UPd6?@-evb!C^a*LI<QG-h^_Bgv0;r#qd-<vL72CvpvW+yldj3#&2sxZ~DczWt#`Fr)c{W;V6@n3eJ5>Zzk*YM}co|w%a)O4Eb+d$G+H)4g36OCMBH+|I`wqqau$T6|Q$tTWL(Fp<f6RPSDv8sNGdt-3@luvtJy`dd=39J~|pG5~O$if!_fLQ_7TjA+Rwf8#M+Z!cOpD)k<*?$%P@&f%o{a0skvWqlXk5BK@?_K)V2xGu{8wosr#x%E^l>=rvdFt7P4X}^PY=<Ah^BAmsLeyg8T_$_fbJ=TWPuMM&+Rm5TPU990k8IjU+VW^LAKxEc&Xbm_@Ci2sa9KG_VL(6@&&L{=6+6J$1JNrl23zRN1P<0}O<(+u_K#&PkL5ChRE2IF7v$zzfDdpvQMDy7fq%V6x;RAXv3b}4$HBc6W=cMFZo!g!W(c}7V!ok$fb~6%(*gvq$H6`zMlm^<&3&S>h7XEi9IJ-^@2|0lnHV}&-z;vU`25tUxDxrIr8Zxe^g-&oC$^z?j@TRDtJ7V+zq$rDG%#ZRDx(>C>*mc<57MswiW<evWm6C7JLUfZXKS~j{skI`ZVII4G?fmSsAQMC#?W-TSrzhiku~RbM*r`Q(lxk}ov?pJ*7uRS1vskvsI6K6q1V2;PG#trZ!hYD=^Q*p3suC`>3JBURNu5?q>&j@XMG-J!%r{(UZ?q`0;GV^Wi6I)Xt4HElWVQ|*fa{^kYi!Lc`j;gL~OkmfM0U(ZQW58x6BumOJ{X)vEYzQ9r?TMy+g=(vOGQQHEH;!M&w_~?w7>qr@x#w*Ip?*`!rVntn$!XYNs(hyHc0q@|XX_Y7M!5Udw(xkG6}}pIa`yu6V>MBZYBo4P(x2{`dgC+I@tRNpNZA_w=quX=A#{tk$M;L(aw)!Lf$(ojJ`?c&rHf-Ki7S>CGBPL7#N#H(bmw)Zs=ee`jZJTGPkg&x9wq-DS1_gi5250LY51QcnEQmR9@lT3+@DQULEE6D^19nRPWCFnna6eK|6>bvQ1%bt;|&S*M8^G>5+OcIyW&XTdu?tZ$2BHEl3wbKUt9tAI(w<K||XuRjY>`YfJI)vYX!+rxb!HsbNXo7N9{JvtIa1?UEHXTjk#*rYbTGE~0~hAoA~{J6i?#Tt98v)E;v7rhi3xX<bSy@FqAQvK?SsSO|1+IjTei3>gIA0O~0d^I)GXW7>xMn)qh|1xGlvcdi%pHKblF>@<}*U(s8MX`SxkN;jf_jmO{cp&?Mz7pi~5D$7iNa;JmZELXF%bW#**sIMR<<DqF6X(q+NhEP@uI+P+-zCp6U3-E?9g&vjrepc9P6r{+wxfWLu>lJhKo6``HF>ptghm8ylAZ&p?Q2v%{V6cpAx#M5P|7%8_>an;ad*Q&%43J9a)YL31=j^)bN?Q99qvL<r-1O*DMP{!_#LFf=iPcVM!7fmT*7CC)Ypv<;noK9A0=4!<y0fKeYJl1$Y6~<XLHN+2kXIZPX^v?{k*^HPl9E^HL4EF6kD<{XMi>Mk#1F21J=~k%1`E`(~wu#K)Jn7WpOKS$-95;{s(Dq+NLPBv<rXVzalM%awz*zRFqbbK~xk(1x2V`GK(kzGC1pRf61)s?y9|e?{l8><_E=Cu_9Is>yC)L@=EEE(Jd)ZP0#3v5CU{|rQ8T3vy=LZd4{OhQJa%%IdRiFIL>ft(uQdCLyt_oN)io)d^^XzPB!ANGIf6|k6Qj3+)Dyex;uUBcCFS^j}7fyXTs^Y#q6hxTuj9pZL3`1udpkum*?|Rd?zxwJoRX-3yz@Uebv<O@vWA<N0J-g(Gya=@aK3kJ%G&owuYDu7^z)`frr~+Rm#q>L#Jd6VA3qgbPd)^B9gm^(!HoWx9(M6<POL6Aym;LqEh<;(be!txtCw^)%Yw(FOVBnXeODn%BAsqfkS3fy_`y!;Fg*O&_cW;CkeF+7!~I0jgdr&`s=vWUb=-CQc}m&abZknri4?!<(c!IUl=954i}7k2DPdLaIaO9zQr$Ck_O*8;P%m*`UR?D_dB}1%w?!7b)0&-qAF=S)F8GlV|uB&n0Fsgul2~+Y*QO&$aL!3w8$olXRLhgnD&ooNi2ekxKI*&xO(;Yeb~S&BbZL+91e|uK{BRRQrT*zsXix9dwopzSehPijouzj=7yoZf9cIOsm_q&e9#^r5t?+divp1N;JnITDw4QAB56T`$@K<e+D*^l%-;O6gw%#Sufc|}vjihuPH#4y+o%t-#%M6kJq`=BUAr{m`$j-Z?VK|8Y#e_ib$f8RJK0u4&?Y^Utq}m1uNind)l1sAqc710ixr99XuezBnEtD^ZTX{|kko}DZ(G81-(Ht86W*I@WHWVZj`x`g(YtVsG#u?pK)0gG_37}gs-v=Se~sqVS~9vFZlJG-=AoFvAfbLQa$f)Vw8h6oYUT`ffQXfTZNBoOU|o(JGc|9raE_`s>gjg%G&#t)5Z~xE(k64$EXXYCNsqp$X$^HT(?^-3*Ur_U7ch?6aBSb<OjcbSrBbRI;PqE{EdKbS(rPpO@kOQ4yi=j6;`#yghroJ11(7w9I}hK#sB~Ye-k{XTp`Z>H4smVlZ(n0W+D-bS;3e<{NZIxxzah<xgSo7ZTfG^5%<l8mDSkLxM}eQ!cIBDlaMR6%$&b)g+3K~>%!w58akL4dbRBHZU9Wfdl<X2T#im{EZzn8rQ_sCFx0rh)Z|C+-Z8M9Wg-wOpfdmT9)6L8=6_k60aM|x4vmsm<-GbO}r;MF&nbEJ6fh@OBg{2FPKx!fGeyD1P#}ijoPh0PAVs0OcnSQDwTdnd0iFpJc;)mM3#M9t?)5nbv<hG}(kI32fVGeX^#dd0`7jW%W?N)-K2RTH++U6_$b)jBY9IHVVYETLi)#?1KirR(=y!@<&cGiXN2tCB0kf|59>JjbwLF@&!9<T4A)mXhN&b=zpCKQ!-BesxW-cpj}E;_*L7)Hy_@xE&yq<WioYmZm~h3aLrHY;^!0BGyOWNdmngpk#=b`KD0F=b2s;!<F($MZGn;OG(9x8Q!3eVzx$^|E}T)SP(1q*_KWE{um~r9vY?lVS^i3L>$BRqNOTwNby+Z(mEBIloK<UJIf(IBVPfjpxMu-OF1Xqckdn$ck`1SN3n^JXS<qpuc_rcUZPl9T|>Q5Y+Y;58%?b{uO_ul*KwW?xl&IBBY7U%J~Pr?qv#6Y8r1IoA#yTUv{0OS;eW*aa3HoG1?~Dia$bXT*nhld0S$V%d}o$gHBbTdrNw?J<ICAXtR9xUMyV#IK2mFBStwQ@>B52tWK^r71#A&f(nY8OK{{&rpgrfEO*a42!Y+)!@y><d#JnW0_45cSV!3ub-0}k3Vz1eqs*emF{-}eG@fcX+gaXZrrS}CZPR0PzD1Y;yf#6QTov$p>oK_*bICl83g!AH-Kdz(!LKsWvN_EgOL054U%6o}o1KkbNWV!GgFK&0ilGmosaG4%6AMnQ+8qI?p9kGBx*lZrwe_~W>I@F$Eb~NesG^2h?5Y*gIMd%k?e)&t$WRJh94QNfnSvJ;DqYqI0wzC1P8DG??1s#&S76FjLaDh!I+Zr`R4oYn{Y$y$MG6Cqwh^b9{fQ%OypnEDi<Lf=twF%#g*?ZJf$Rf+W$S@<xI}&OPG|b$j&&+8G{?}oElwE!m;hP3w1=?^bqK=liwS*l7X#bYh$vH8J6`~)NptjAPK%Ghu(754e9^3RIrA`Ywm{PDRpW;F5Z$sdsU>ZO(5l-bq)2G9lhq7Wu)EGy^J5<w!Xmy{28VJ{$(6_fteA&gR9%R3*Iwi|7qn4>8{w+P<R6We-Y39<X(5fK#g;EgBxqrJcBat?@CyRzcE?C*)}3gI&kfsmFD|JALbwyg<a$#A7S;gU#&=||4mZ$&xIR?#*U<zL@Cit=M*(7dk44PAiJ;S18*h`OJe-jUvf`9v_&BGW-Her^;Rc}Roz0~Xfg}9-s#juoE;OeKoJ%d;@}6BkD|`KMy`arQgVsY?&zSKP&ATWM!$-6+NGCpP^_i1}_s=21nOa1T?}e9qIeE?6?7BY9eG^<wjoMDA$lT>w?6uK*RJ9`xbsI1@J9ZX5zU|cZ-acEWR8qmN*A7=Yt&}!Dq3+!^WN@kcD5cDf0o&Q6;G~mBq!a9=mxGJckd>D~&nw1mI6QZEYej<kO>#y{R4&W3Pt#&6T!2BlXEazeRUD?Y8Zqp*9<B7HcV^~L7T7zD{OZP6wpV9S>>4v9RjNcswLA>P#RNw)_kJ(KptGBz*3QLz0WYRNbfeglm>KsMf(B7TVxK|!@NPC<<K3=CFn&ozS`HNPCYYgT!lJWI(Kn$=JY`{1z_gpyIh5xeFW+efImE`E4(Vzx-#gqmvCyV>jkI(^0aknZ6oyeFU*3)Fkv_*L?{Ro6uG`|G8$oar;<C^*x(<Zf>^zkeUTK}4)^%UNlv{JTej)RRQ_J5|gC5sDV%qkVDVB~=ERsiG$i<E6y~EWxb`5&DC52m4ku`*Ju3ry;RfVIZd#b29q$c-uwqJ<Wbi7$0>oIEXmlXv~QY<g49DY&CE%)@GnB8D?=$Mz@7^vN@d^50M$#oH7KiZ6{h7r8HGDa}DXqh`H*Ur7aYL(`%v0!xjV@P%jeAqnu1ti{!Tu1OxYjc!W6vZB|nd@@fs%AD<{veav?74thR9rR3i_I#fuE#i805L;e8XW}ccSiF;qvLB^VcuQW?z_3FRqf?c>xm-;ICTv<4<1_-xn18v;n8h0m6tZg7#2GRc9#};1@2u&3^38r9+OPW&&=**@Ep|Hkzh#xh(XUsqK;8pDdC|oc)hZN_IgPPS!-4wHAekh5KoIGa&4nI@=D)lURyuqFHrNz(B87t&FWXIKEOh>y2zHvyqgVr>(bRTx~DLe+aF-8{1V6BBPd5lr!4`sgB!sa_1b=HqS_grW%t^_TeUYMn=!N9Q?#7j^X>B_e^LEp08Qh^<OUZeTbE=N_>61SR<_3G+R6yYdJBF!Y@Q2xiL%<mZT7(oTz07UJ!xp~nPWZA8smZ_Rm4((8`-CYz~mNAPtu%xZ)n|xqO2{qP9QP&&<Ly?w~*vfqctlGePUd&PSXo^Zqb8ke;Ga9J}OoEuM!-$kjD_OQ-{?(=C(DqgdOn40(#8A=av`pvnI8X8F@kOU0u#e$fd}(Is>^}@U#~*=;=bun`Ge&_tGWf1tR#wzB`$e<3B;~1fV&wd=85CW!Hi_FHFX2dHFTl(0U(QL3#fc^v)WPJ0R+FCOv#&71`)s8us{GaLZ`*RXK5SL7Pj(Sq9ufsM#)7A=Dox5nbZB5rsCa^G<g*#zlFdrAJ6w%?+<$+g+;|Y>~VrQ)mf!r6Jvv95#gz8;J~}qa~;DLr;~k;!EyJa^sq<s9UEvv-(XL-GGg{*EEAO-6<E>Q66&|7@FN$bf&kc7}Nr$&$U}hjiW7hpZVuraWgDA7P_?V#>1Ok=ou_hI_sSH+6p>7sZoUp)nb;t)N8=9QdCm6vKM9N4&199#8I0Z*SgOnsKDZ~JydoJD43buEcjCKRwF#nC4EF^TN$9+TP&K3b)||DgBoai3Q#@pC#r3zl{A=oDLOhH@4<eyk%b2R2Gc7a9=Ba~sG8@dP{oYaJ^KKwNG-1P&i!>NoC8)jrgoXur82l%7mu#DZlw9>hc5*UUP^pT=|pCFFJ|YvihArOh$80s3cW8)I|q(gLr`i5uB?}QlB8O7;pS84O(#lq=+Mk=v{UQS-E^jwu$HXTlMR4%C**l0(>?Eja9>QJ<&Fg$_+fumTkXeQd)e}+47zI(M{&{C4wCuE`o;V75gHBe{jaAy$z#n^sWfyYs%lazDu2COPcR@~ty9md`q$-bn4@U1PIc$Y6<g>=?ebQ`i2JTO7GiY|Fk$aWnWm)>%bRd1G$iM^Ki+H3gRHeVEGY2%m8li#=Icg>VzF_q3+S;S8%#2G-yf6u%ACUrd7lYv($P4bu0l9e?g5Q>S5w$y(%d~)IkHkL9&Gj22$VIn4J6J{q)v5UrdCQh)7*|Mf7W)d)B0`v2&7#q51_Y$!ynQFIEUr5D{=eIfEfXJYd5<UllcOyYK3fV(N*@N+4(k`hyCZ_E~BT-g9bF|9wf2@VprnV-jjs=Y((7+eA3pJsV<dOs`DA7&QpU)+u_EIHMoA_L92Wh&@x>QgjImBo+@qD#-NyT++8u-y7hKEJ3AEeuvVlm5DsNvr>dCzG#uWxQ4OFP%Bp^@lEe8-WDvF9A)<|^R6+%QK=3@5i;viFxr!Oee>NbpDD9NjP)#a=<&1E>9yD)`E7$6c5S}i#I|-J;+EZ!G+Q;)P4OQT7q1z6jI{)-r2G5=!M!XuirM*+y67nPIt)61nm<#S@@DkZYdr<G??u4(VkHFJpw{Y(Pbm`{HjR)rO)+`jRT)w;|8n=1#LG25TMyK=~NrFD?_x9L<f9YI7t)CSjV_8-??2oScuBKbvx>+fY0-$$uyOprlDr$|1D5W?3<++9Jrqa>gr=dL=cI6YXW4yakVW$2thqo4(V?1e9YF;Y4YG+=zHUiJ=W0!y4j;GeTUz*o80t=Mq()h*=k4Di^@w*ZgOZ7A}J#I!|uAd@aK63#{L^-TDqoHQ9jn1u=+TDwAm1PqYYdH@oR}D_7D_^P-wc)8Oy(U{}fr@DnGs<C%?u5R^kDgIw3|WgxwA!75H3Xb53M2%ZV7!L)0IAQ`g3)8>7JVVe(U>~~6wqBZtAu>Qcdb#ZM(mL+@2^7L9@Gi}rf7@tMy+DR`~*-UXgrI=T61T)?H#C&R_QGr_W_3?%%n(dCBmzY#oK66xFU<y`AV~;-eHebAseVnDQaHJW9Mgezp^^LID2G96Et(Y?(G(NevF)DXLQVV`t^Kt+p=_+X^y&lv#6*eO%RJ$1oyV1R|(oAGykz(!~nnZJayYm-oBpm#fJ5)1~U)UA1U%sVP_&cxpdv`{mge(d|b^FC-E#<E5DSdylUM*#w&Zph+roY3#02`VYKRP-(4`ZTC2{M8l15R*A3VAX=V&=Qlb&0riFa=6`k+=iBUG!NU55$Vb>b7#J0WZfSVI}8MkFa?%vsA?2MeKnb}3lPPTfiG;Tz76gK?QQp*CkCWl;1beOmh?(TV`+>kUTEXJu+)@+cGC5F4kt+Uwkny`7a&e_PEx4VNRmDKCn&w^*PkZ0hiTP?&>>IIK8Lu81J$kmo|#cZL3DS88>9r9JgXsm;fh<3jO-3c|@yK`;3Jd1J#*!ubu$?021Ej}T7l`7D;1c;4_HDCMpoxMgZOtFblrH4L(CM_3d{*(YR_D*c|T#DIBaKGi3r`T%0Rt~k%3(oXzRGBXN3$=+Vs{ve_oF~Y6w#jbyu66QErjqM|Ei~GWm;4F8^d2h<ztg9ycbwF4+$FT>bnA)90KLu(x)h<>yG$w8Ys%};o45JnEUM&f@`4Ue<L+g#o0d}m?+UwZXN&F^0bbf>Zq3s6DVHaNN$YBNEOX^@xph14Dk#f8XL4|P4v~-vTD#HKA~OwiGRd{@o7KO%VBc>}?`2j5Sl5r(m$-G2F>qK)aeb&?(umjD%lOr7pY&7ryvmJ?`&Ks!+42y)NE=tOntqb#O38`JmCf@$stpd}Ve}ZJ@y(Nyb<#+ej7TD5AQP^8lV+bZ^TtFK&T&@gkZC)+z)pQs4gufDP3J|J$N9>7(sFKVn;NAv;CPog^bzcp<hQd3<HkPlgf!7I`V=FoO?4;TEc`aJjBv3t>bq$_kJ4<1+3irgeF7k{GtT6&<#@<aUZu6a$aFV95A13ybyS-c-T~QF@43ydhcL{{hv}wr>N&u+d%aclN)=ueukcMN(e3$cHENdWXth~i1GM5c&Bnv_sLRmFbh)lto%ESWjiOg)2hnfW1HO)IRcn<vQg8;tId9#RNd>u<53wR?aGGv7{Ri%!E{Mp8g9U%>jTJ0!9O&q|92Q-?WwhXZ57-ZDq-5l>LUeVTC5GXPkH~al?ln6|dZ{?ZU)Ld(54mgC6E^LfUMZwz(5*Tyw4qZrIA2?<q_{nb3J1wslE<fhAQk|t62h%|d9v><M-$j`Wa`XspzUV6i{|YbRUk)_yq0-YE)2$*$SIG($x|m_)*8$zyM_YrgKgBAY!!LHhABcEl$cDaw%qr(j{zyTXBG#Rc-bkjoEPGybauTnW>Vx*TdH!nL(4qtEn1yz>C#(}lY>!Ou@E9ULiPB>B81bKDVEIFw<VHYKt=7fTmfs4j>omleBsBH^vu0EBT!f%#bc^Bsj|=Bfko%J-bsn9^Xfn!CoRFvSD$jo)*jPlxjgER+mcvR_zbmU=RK@gS8bu0>dw>w|DX-Q$}HJiWTgZ15vIr;RT_C&L>!jz*~nmAbiacEd9vdwnnoCIPCm2jYl!NX;^mg5cCB%L83s;uZ64)0=fHkiD-_+FCYYH<npz;qRUcLFdw9L7EyKFn9n@D>(^#(BKD?XbP%~AV#0G$D=D4>UFNV|f$SXd)UaJ$pjs30E%!o~b#p2?t?|1S=orH=&Fgzx>m5x)!K8`P1ZV?qmdbLv$18@UpI)t8sMUs=dzCgCZP;DEU1XebpaJ1}6{V?;Y@MzBB>T`TRW_P!S7Vr0y--gkg;H2Vx(vaUH=VbOsr7h&qM%&31+TeJf>f%a=B=lLQadLXC_dm3i+LiXE47ZVWsMA{;A7rJTh}01N{CNKo8{kmHQM{XjE>o-0LhkH_GOxAkF>|?Q2a};w((%)v63@q((oD72v#hALl=_C3F&;PgNy1)2vtUCwS87Xb0ygKwJlKgL38Wt2rk7^B>(1hm%Z{MTrTq#U3;Gq}v)<_`pF8_UuBLUfljGDc7u)UEOdv8FGzE$Hs4X?t+^~3WSZH@#Ek^UI_M8q_CdA+ZdzbXSoG!d(<bar_D{WV4pZY4Dx!3UkZ_V~>XHtLW+nfaMIDbSS#dv+kbCG$%?L$;q9%(c;XJmF=K!M0b^+mmUjIP3=Nd`UW(9f=$mq6wErCd&2i~@=Wx}6L<!E3;<@Zk00sk?&_Y9(BD$q2q^DUV8cT#U1Nbi^*u!A#*?eG{2fZNxb{_nBp#?s%LJLLr@$R~23^J8*6DOxuf_l8V~?c#_Fv{d#XC&4c^0&?(2Qqm~*B1|@F8SZivgHa&u2b1R6sHNiD$)z&_rr+u(YAf;>$7-Mn2+s<6!Rxt&(?D`>@)UeU!I$6OXe$gv=>$1XG5Kx6fyj-;F5m2TsFK^u5=E&}-JUEG}v;)yx$#5Q^Q+)7jtY_xb**q$AyAOCLOCCf1AWT)o77UH<m0WwAC7y+%MfBrYV@j+bZ>~|4u}2>pv8{E^=*FM(Jp06#<qUf|bGu&KUesusAP4eLd2Ib6p?CVHnV3vP9IMUOPWM=ogp58!in<^^TX;Fz$-xsmkEea%0#^crU>g3C3v#!PGCemNEg-jxOO`83DL!d_Q_ZO}2x`q7U(RxVdb!B0#-K20*y<|OPH9MXw_9k#)3?z@q)%?ro=R3mII8Tn1~=1cT+SVd{_$=>cgrQ42figNQh+gb_xJ5sZ_)nNWH&3Mfck4-F-fQ4Ou!f9#cR;;J87rWCI&M(ToJrgdvPgN9j^~o-#2YNS6>NIx1-$LyMS=aks#|0-bf+FEOT!3j$Ok9x8=qb?+{L*a6A+sj(eD<NWw~}3;h8aEikN0cUWlJE9Nit1KTrF6grBZyQH+%wAuQwW5DoqKvn>~D~aPSUFF+u%ZJKxH`)pQQRZ4_eQ0Nn`4T<p&kkF?&Jp}$G|G*bN(4^k#&Hj|Y^$;#chJ$vThO!4C3CPSbt+h}!{B3id2f(_pdRMkex4eSCwi|4nELZlpt-b9THA7KMy{Smjcbg|sY?<IbInz8h><ra#5+Qtw9ob~((a9A24eT>C8#*r6pB43xkIzg2UbnWBfC*hZSYgZYhC>*);%vD{Z=+x&b8O+a;DQYF21xQy37pEErIlAGQ%yqvr>--IR#B%vzJw#qvCu2`zV1%r844Yb8X-6lS-$*TNlEaQVvrkEP&Vk3nn);&<vebw#{^*TskbNhgYE~tz;on@)vlVf{Oc_H(&JAyQr(#!kmJh{E)0Wsg1bElh(!LgRI`(*0qf1O;(fUw5rb(qP^MmD!t7<RUD~hx;zOUuQ03Gwe*rlEyx8ri*;+G6mX|KyFhj-F70Q$KdV+3x;x?KomEG-hPU1xa99Cf7F$leEnbij-)Um`4L6;5L(u76Yb$e|^V*P28t_T?cE^JzK+_VGO&{`=%DGXVRbQ<XKtB8$IZOJ2)Wlz%lGc(bue<H(-D|0$R;(z+G~b-qwHkPM9Y#`lLQ$OylquAuvpdaBYW;I3e6?B`w|r?q%N@0D%o`j$i;*f(Kap7&lB;$d&GQOHO=1gVpTqM;CBcH_T?b3)kl#4m>^aChHkZ`WN<G=@dHdAFK``tcns$pCo|>TXI_hE>UI4P&sERqZ8#LuB-(KYn@!lpvgTF<Y<K>i1xAGiHoU!d1wdGnT7i6Kn)WFk+J7KvBjoVHunmq=kGbz;zMbFu$NnOl)bg?-*zLbj~IcOM0wdw5~SB37Bp5-cbd&KrL(hSiq=!{F(nahQ4nKWa!X!R%$h9r2%YW)if?J|RUd77r=8|Gd2g)L<%h$cMk+6Ih<8cn!HaV-Sq77P7gkuJoY_2ZTYR~=yq!EDL|2JCV*B+oKeJ%X1jJbIMmVbG}>?VVAp)i(0%88zw)c7B=YTIymgMpGyRLV5liFhgQ%b4feXA9dV8g&Zkq;x;vl!9;1dmc0hW0&B}gMitY$^;Bt=qC7#^uIjlPiny&3yrW-6YD$PPBk%d#3!Iv9gPdfyS(h^ojc%K2=40AXM|HW8vgkgf5EZ7}Y4TY%^FXr(?8{U-Gfhhvt0M)z6h9yA&MAlIY)Q(@Td#Zgs<jw5BI~P*CGS$FkuBt`&f;<4r+W8}VKh(h1i9_+{roAW{iw;Wbf;mh%0uLS>DREyv>l)3=TdjhbIc|lOrLRcUG5CR%Giy8si)nhz4mIsHCcKx_qxPn(iA<cP3I;(bFU8x*FVgeYj%m9VVPJI`r8gn;k6}e7^9q*1Jh}}5iGKOY|7Ofy?}>})Y^H}GzvX9Eg$flb#ITb)muoMQ1fQcO0c2|b>@jp0>0_5CWCD4rde6|wH>c1BQL&euWuRISz%pGM;>pTTuDw(7ODA8n3>9z*mQ3S`h?UPmbzS@=aRTs?Mjq!D$ncnHCQ{Ql4KhWx+md5j{1ZC(G9l-c$>!8EUTk|RvWHIei_cRF-M8!6@h|_t@JV6FL}!u<bp*hd-wXJqXd`Ucu_K}@{~Nsfwp@p)mJFM2)A0cYTZgRJ?+v*_C{E!0A%ax?lzfh1~OJp^)RV4LU#f-7QM9cREB+KKI?KQUuRS8U9>`RHDK@uM_liw`YKt+bEcW@-`iHNuP&v;t<P+VERoAo2URJnD5O<p(Qg=wYx7?7+UGEfZWgn#yqXAj*@PDSbb;(g5!TQe7sSRMkVo|HZu*oR=T0nz19m&M`h-i{6`Cb+r}p5yoGo;3;y9Hn0O4bFI8@8aN^xwS@#gC&G^7z{mh`BQSxu?%QKu0C7znu9jo08M$VYN#D=lwnQn__kjoVcjbm?tg99<59=50Hr(r!i9H-wx=3Gmk5S>lZBBf@&8nJsqGXk2ge$CiM7A#<sl%QRdH4NjxY(IakKJ7JPsiAi&Kz7r4Vx$2h(z2s`Ui!oH#J{SI`+-fu~-TpoqC|~1w;j%Z5wx!|~Z#fV$&6S+SYcnI3kL`VJ;I|6Z2^8!N91~99r8t9UC4%2CY;POErSWc%bMVF<oS|6P8UzWD^ixGg*LG2`Jcy%L@kpnyGf>kRVb&iWxYKjqRhIEi%V4x&ZtZ3_?d1yRn#7^>T;Nju;c-|_s$0bd2j}@EH61-u6yHN^AMn$dIj%val*GOs_Y_tgW4U^^ym*ZS+s$StmUedgHE+`rF)Bvy1`HE<Dx;(;+mlRdE3sb)AHXsk4%-BvGFUX~^3!3pwlGuRix{<QJ!u;kl^AY4Z<k8;{OEvE=86E1gRoNiOnb9}ni;-VvhQ;Kazz<-xUHwSmyA6<SKqH0rN?<$VH7sSOfsKzMCp*er&N{G^~$a8?Sy+=I3Sq|a-6TVb09G-`Nqc`(Aj9S4t&kys~i(k8@S4h{mf9=;%Q=@8&+C47*8J;U7|AVIET-glaqAbQ6jUU*KVhCC$F6}-3``D#+cH%^z5F_&kA_PH(<Pj-4>B$R;w$uS-eZ;{&_uQ6~2Ykq2oE<;fpiq6y@i2rn?+$FtBiSUFW!K^Mw-D0P2_eC3_POvp#hl)x{&t%xOFqFPL2s>FhcN%KL@qWKXmw-nN~)AwAGVg}E_I>g1qP&W@|utyFmpQ)*pW@=jw5?nX5Z7jDD~7Mepke9>6;Q7naB$|uW2t{}|z7rj}eJ4@eI)+N43SR15=4d~vgb1V{_+L81};6W7S^tLY{H#%Rh&X>vH!auhcHZ^iCpv%A$rCi_M1OFB3s}ppMOG`W6qSkU}%`e~$-wA3DF>m5yH|^~~On-`DZrW4wRNLG=SL0$iSC>=Mmb%TPn&ocG^<jcd(|GayU!OeiYi5|u6=(u3SR~Y&Rg+G1)&e+aiSM;M1D>FPkfU`uhit2SkddhE)Qg&xyfDw_tCc5i$gkb8Krffh*;}~S4K5{(Oks7oYgB{HqntjA*eEJ79HkMJ9khwFJh<H6Ya;eg^-BiTbLP4|d^D%gkl(EiQZDwBwgII!tx~#oJ*o03e^?oCx0VtctGll@G*>gyP<4ET-G*1nVcKr`#Ao&6s*{<;ovH37cYY13%4{R?xOn2MN=VD#D#cebujMX+yH92HtWA(&zdT$bj&N+9r?)kPMA8XEB>!4Dn1|5{98KuUXxT@J?WE6CXc5mfCMV$%G(iTVgUk$nO8e5RPEK#|c;Y{+JN)b)j#Ca1S}d`@RkduYQH2r2Bh_@JuI<K5=GdGBnPyoSZze#$^nBJS+wq-YVMfL$KNI&3t*1WbrxH(1%jk06-CcrQ6)I8N8k2?KL*a_&jdrt8+CD@;zma4OIG&O0kRjK5G-J&np*k!NMq(6MrN>}p>abi5Yw@Wnh5?%&37%MGH%CmicRhh^Gq2}xjDy1>S)~<lynO^aW2{&RLmlXA(cwBOi&%r3)R?iV8|4N^X@a-t{q<BQYB%K3i1&?SyIsn(3&n#thc{b|6)|-`$w%jHz8wx^V6f?MyG2VXOxBfD^*Y|MzNpI$O!9*Ek$4QD!@yx2@I2}0A`tI85j`-QUX**75Cvuejj49EP7S6f#kBtfohuj(cF@6tF{YDo^>uD3ZqVM#Kb@#5GReW~qJ66<EmGsVrBg9I+TQbX&c~nL;!NKr<>5S4j)wW!e4V?Q%y_hsn8vcyAW~zJlndp@MjhZ_J6-hd(7oG9yYr2sjqDsOWlALJGM9(qO`f@Kr65+eEfPuD3ch;uPH+(JdX$0rt$fMl(8((Bsn=r0MrS`(jhte@!E)J&39;6I#!ZWN4^0&=+}vJFL*OPXU6w;R?~E70ZAWA}&0Vau`{LosrN^qSm+4EKb$i%YkbtZmEoy9fT}qvX@u*}v;$X<+bUQx?UrU^FGGqY;*+yG2+x8|fN&+M-2=rnO)Q)g2JEe3TeVWHE9p0#Ay?#hRi%3I|y9IXeosxgKu~XUU%~gNT;EqeOi;k+@E0aEoj==OK4L^en*Z`+<cadtrBguAhST!Tg`n${}=WgHzMY7Kdf<CZ8DPHy5hr`#F_Na+0UU|5KERzw`QuJ#VD|1gTGfIh%ZA*&JC19Kk{ic$QcuvV5H?MKcM}kZx<%^}dk<Qs*8<Jq+#lKLu+q^#6IdTN<`RzvVPj=CQryj@kfyUSuxUJ8WYju9niMnv-bj!6L*9{SJmsOY`0_mQo5ta%Aq14qOluYYJoV!WqlvgZI=j%$?&O0g9UkRy4jVgEd9Q(++0|1FK^=iC6G^rF{tkeoh@0iQ2Un4+X&Pem3Nab9*)SaEUb_T?(Vlb*9^L`uGNi8Ec9^>a4hsi3_++2s75%6kXTj)CI?dNvI^N+sLbBoywUaE*pmbi0!5z+gQE~o5XKD}P@)qx_@)^ln@z6jJ8H*Qu1$BM2kx(1PMJ)&`T_uvk%;{xw!1{tq+H%kect6T!-&pk95?ha-T%_tB6oM+$}p22L|t_i7bJsu1Ec_8Bxv%av`D~%17pjDq@;<j6hrLdsuu@&!yYK8C%W*2~TwMn&Fb^Q??PeC@hR3vfpaZV!wA}aHXRXiKgx1+3z6=j;WPZ%6~ohwHi;v4%Ckj(ZOUThMvfMxJ%_D<JUvqd56h9ycDq;H<HbE<oP!YE$2*^mURiTeF2rI22_*oK*5eQg`(yg4+BwSza2)|B74jZCZi6w^$vvuOdAm2Ks;_-;?3tiEVJP>EmXV#s=;cUSMF*K2$_F_C&j*{Qjd(@ytJ*ygt2>S$K%xZ-6GIr4O-2u^H0m}}B5$5+3ZPh+-kQ}B?TUCxUWEcjDSe9BQ1IhM6t;t)r|t@S#YR6)>@#2Yz9i9P7j8+wv@)t5ki8wQiP2HFMzJ|6?ZS<zt$I9+$A!shVcfewkDD)_o0F+?WLah(Odyc(!}%c05;Xjg^h{lLG*w^47vXf1q5=6NCGynn~fDKn|jr)J~8H#v_;?{ddz*6x!yIBCJH8DEJiJCgt=h0fwE-jCUFquJa^R%c%U!V+7m6ECt-zHhMsu#uMCc6d7~b1DV&3eojmzis07ZnGG=@LE?c^Nbg-(6V-3-k!^1+p6)vBGrhDQ5Cr<-m+d48rMLXr?C8X^$P7~J{(RY)S|uIa4n63UhnuYx~6F$Z7=<3RT1D-Rj%pf+-hY3cU7lH+OczR@u}`3CChX+C2la++V%Q{QJwCkQw|Y4V*}W*S|l25kJaF1=s4BQPCUIHwYx6*9iwRWircO~KU5YB>@IR$O@FRVr@J3bSFQ21J?@`3lJ?>i9_q5pc1D!jba6fmmq2bgu5;cktf#D!0!l9Jb|x0p9Gn`Z4u8IDFKSePS+NRvQB31~CO5+5S}I-QMt9d~QvG>;E?~tDp@}nf2fetR2|Y)<_=r=T16pgevlu%o%{^{i!lC+j+N@5L7+Hv-Qk-ztQ~`qpruy(LlN)9GxnYaWV~khSR=MuniOgmT<U?lY#b`;f=lEiVq5z#a)`D(pQ!1T_3g5__ouyIDT+cumUKNYaakf~#Fu106n@MCIW7yAwH;Cx8j@XGmBGuJ+eJW!;NWWycCtIF%;XYhF!Eu~24|u?@krg+G(^Io0wbUycUML(53ExX5I}fbIXWeJ^;$T3xQ<l}P62x`iTui!sZSlOd%G}WAuD8~;dD>BSeP0?&x4z#GLaj7gT$915Cs~olum(L!<-{UZ9t{1Qof|;vx>%g-IwieW+#awav|ef(Y|$p1rBz&=ow3R8Prcd+ZdW_|Xb4+dlfTrJ9m7aFK6X=?o$cV!E|(G)H|z**7<jRxJLzS*H8!=pHw$5Rm`?eY-gNt_z88Q?(O!@n8|SL?f`W5NtU(sZ3`f=jT;9;bBnz5cBY}x>WsUdO$K&uZ5dmwH_3I!-!<q<`r=8(RB7w^CW^}?vAKQWhqbwM)MCdCo=(WXVYI+4&|KL?}#;~pmG`G7W!+nDpt`7T@6y7a=+~3|3n9Faps=%FuQn@bUE8;eZ$Jq;v`MF+1OzKyMrTgMJf`DsZoekn-N_KU7aA48avkV!7)jZWyO6T>Yqi}Lgx|f!9wl8&yo5G-*;s}Uyt1q6F7I1dU_(fx(XSW?{dJnLEdwMB9F9j=YZb_Rzrt@<EraDltjEolV?gnEJT4m;Rp*Bu0Yi%8?+BYIrsdae5JJ;MK)3~HXahZGMqyyV5M)1Nas`WHFWE&dBi%)jNoKKiI=}=|uKI?i!p;x=+Y70eLT;W*&q}15RznT<T6vw?GSsEj~lsKqW>F%;vw&~QhTskz{8wfZRrk*j$uTH(Ad0!9OeB}@ZO$*$ly1BU0T_CK>J3N`5vX2>?4i{~^(@Is-uqB2uBV1>knchvK#d9?}G&ghX)!S{rgI+CNVY?TboV^9o$%HRB(1Gui?Pk!MJpf(q*d0=<=P@uee0rV<iQZ-y6}5@ITAir}ItTXo8?w~$6|>t@@X|$aPZeogFASH>^Y}{XP<6Gj&Emt2H-JAwlJQQdAe+IZMBLYU?iOyEiki!>bChsxPJzi*yKUpw6l&kp#a=~EtT?7tO`=zvv1V-=^>dipC66oq%Fc#Hdz$9NL2)rFOA=-8p=<lKL3FP#D~;L;e3kn3YUd#GtKNPcfwwc(Lnno9K4cbaZ|PL%UL!2w(&M<~4{}py*5IV)IvD}pD<a6cwuWs){4UZHF|}_$1_rt>9Hz$cmVd5CWB-`pvT(pDjcV&qV07&Q%6W<y_BuvaxHlefa`76OH8O;}eUb<LA@!*3EVI8kHB;4w0=gp#>Xh3jZ8tfy1dO;RR8(^-meKdJ2MLM|Vv$}kZiCxY!iE5TU2G$L+}}?!O&bvPo>q;<Z40SYQztyzEY(|Kn#Q6CIN%iDH%NUxN#=}0Bd6TfyQAG_UvqafLrP=A+n<+2qY?Ypad)ZS$O)g05lP$M1`~`aUr{7ZW`9w;K-@utSBKm^_TggCJngON-MPC<D21p8b+4M4mXKQBkrBaSbgDMFE2o{h!9BO)X2^bMG?%MRoF}{cVqI8ySe9D{i}lVD=+4dM=LsZo^Rl$*2o@jpVX>_a%6xwmZA@nnY$&o3Z;5->wmLVyUn7~Z0Mj%}6NAaZ-MT1&@q9`qYQ=u$I+Z-+)S++7%JOA~s*}r8=8u1#Cjj^U?hK`=#?nD45YVsJ-fHwJUHs!Et_)JWW~7Gw*eKuQ{SNR~n_9!D7MC--uVH2TJ~NIe)MDuz+DeBVz}{RQ*=-Ck*GK>6$*p@WxS{jd@f4a|@X&01;Lv79omwlgx?OjTCZA$MrE*XTmA1u-@@rOl=5uHW)agmy2Jz|Z4u%lPKW_AvbygtbJaXv^S{x484o2LV%Jf#MT2RIx4pW;Zy|e&5U!u`z`t0sHYAH&O1+dt?J7kii_lsI2rpwQTLg63soL%rF2j45d%Sro6vsmxzXH5;w&)=T^*m%L49N5Me*;gZd`)0due=&WK%<%KK-|Wadg!#hn|9DG&<Vf{R*s;SX|3miI%D2MD(ucU{i>!VK`q=;bcmA((^bMBP?*e}8vfV^NqP&o-yz~6SFr7cu|3l_To~I^*<m8{Xs%yxqV_WX$-{1HkHov@mJ|~Tn9GJdp2Y(Sm({W_SRE@t#T5MQmBwMi>yi@zC6TXuJ&pZ4ZQFqjXFTV>3f1(<RJUNU!eg6|NGJ}Kds);uH7~6&#*`E8;@+R=yNDksZY0rNo=4}#ZdvUTg?`feKoIZAa2_w@_Uew6^;bD98_D)0KYQ6rJnEvK^(c1^ZZ;2`S&xU#hKmXu;{|UFAAHwqc+Q-Y!!aqme+5P<G{w#d?9J@|pMB=uO4WGY!{Z@EOBKGy$Uy|OHeEs}!$j{&4d;T+yjmf$AIWbAZ<`bFY**Tf$eye?XOBUyg2{DwpDa-#mNBy0nkIfl!&dCfmACrH>)Ocs22vR2IB*w_xke&WJ6Zp<FC3%^^`s9C*F{LC4`w#n@?+hHnviy)D{{v6!o#!V~!tlfNWD)wC#h*<7dyaoGJz=W;JJY|I{`ZppH2qyh<!{#ivi%Ql{mSsG?e9|l<|x1VqY6Afk}ZmT$IO2u%-7$6Z-p<F!oOzAM>>9k-|yS6=?m6d{~>*W<}dmCYt{jOqafMvlZL>I;=uh`=J&nthf4sj4#@13oFw1@fBpIyUx@N_&h{3>NB#*z5_^AWeEahD9~Y~Yk5!JO>7U79<=eL(n3?E!$E**a`;KS7om3|_|ANUsSQ2!E4O@T5y1yu24r;JBBlznFj(UX;==8oz`|J1#tbSnP9~=K6Q_PfxlX`2)s%5%=BJh{*umb;pmoLTQpYU1uzUeo8tA?Q+{$|*!<$7Uc>*3eGP%JP*JvL=C2)rOce7N!*uD&bz8BcW2-Pl$fe0+i@p8E%AMZxv2h^qR>@`vXCy#23C@{_au`Qb-&zsvZbnit*lqc`&7Eg}B?ue|rq8#OqXNhSFDd5Uq|=Wky=ibewCUz5l0&VGMRVEE$_BxPov^g8f|fgeu({`9}K`VT{YlKFqK_ZLSccSU}`EV8%xd0zp6o7L7|Z0|oO+dRYaGI{+)gZxoO{|)wiN4+NgNNz;o%dZKbZ>+*Ef@9Z?LindD@JA`PzxZDGo|C2<C(cbSTmJnJza>=rT}xQ_^@M*4`%U$I(>3y6KlkPd;C%aT<~FH3s<Bh`ckd=LzI+JNyrc#fK5*jWv+bVpcHzxTYUbM?!Tj~SKYadI%s+I#U#dT1{+&?${a-@=vk*V<cBUSg2K=jc<Ubca4xIn4?Bl3^cZ#onDERgcHQ({#y<7V7r<f0X`XE&b-@dyw`S9-5AGpK4d*^T2k;=FKs`9tc3!?nqyuwM}q8aL&_j~z!fC5`L!w*s3*Xp-KRMGni{BYxczD)lQK=D?B->U!k_gnjq^c%UFmaC2W+rcyJjjey=W%&CWwmq1}8~&b7Z|uzs+HZcWIkx_$tG=q)PLf$>_@)P@k$Ay)(@e)Z|CWgB9qcHu?YDsw-wf~c#+fJ6edFJC&OgrdkwRnDvK%u>Sj%?Jx8(LEd$c%8&idsI^VOuEEd#$*eoIhbF`;;Ql604!WZ6{@rYwK@{nH6($w|K?zbBBK9Nct1J^x`|Og;ZO{l-mut1_K<)(y=vWh7ag*kuO4|0MXIZesT(3e@*d{&AT9dE0+Hluh?!Cw<`|*%|(Pw&d9`+i#(1zD-y@X$gLdF41rQbU%IiL|zh6Z-)8Z|Mq({KCamR7^bdXlPw3;eUDHQ)jzZ*t>@uyHCBJC0lzg9xttdVy7|GABueZgfir*euW#HC0~+9)rnl5az14)wO$E@KrV7@zmR76P3{Zud;HHvL8hRb7L)AKDRH~agSla+KP1j8mY}Cv|!{oh`C1>`7_mh8l`<}6H$rSqK&G(XMU%%V$IZ0d+2PA!bf>$53<fw1<JL>38Y$W&L!wvs*|LKyG0W~4zWztho?Bor8`}EzQWRYVRbK<?b`8a>F^8eR_4~0a6U0fpV{!jNk!9RU;0iS;Vepl|Fl5iQHzBgo_Hg*t3^3^sSLr%JtgbXCqgL?UemUP6QKHQRY3dzUH&n%xl8n;iLYk9V7efnc+`0_(Hfz9XaAIksK4gBBhuKjo2_5Y%qXBmuV|5bP6?{&BSMtAF1-PQl9`@PBgQ#HrWF##v%b1_-@$>o2qyIT1h<sZbKv;+V5+8L5&C;t}k*8fI1@c$I@=X?5L6N<s;pA&GvvK;y02tC2&_n>|_B8-Aqf4{>C<rBfO(@$SM*#G^E^}i3q_AgH)b;wABy-OkogHM0uA1iz~;!oA}e^;H<YJM>%KmEtN_?swf|MJ66su_wN|D~H7lH~Mfvm5_Q-QcgfYyVZZz_RqGKZEdRjfUU8{s4e4f8NF~f8LERzohDyKT_w*cQ1bVVbPZ#(v$Z-Ip-;wlF0d|Z|{1(4~sIxF3F!GK4ItlFgaB|F}PeU{OOUT>HmZL7nMwW@E)Xm64%O~RDt|KE&oCOFzn-(5#L*~1oXmCj%+6xUOu`pIkay{gZo3kr%(N)UL=6*nn79jg5>S(gZWKOrj4ZLMqBgkUDB8Lre-2=WP<318T|Hv+wUT_s<TOq`b+lP|NPGz@O|w^+we92!Bl?F$v>V7$&C8`{O^>%_<rRneEZWhJN%o5^X=cw`=S5)n49nv-i^5DFJIoS--ogvkMEP&e|6ViUUkj0oRqUfryh9zm$xJdzr4L`31#05-gEK~<+_o$<)h4eeP{jl%aMNMz()-BwrhN(IgL${y!&b8eP6Qjq3I{`hko;q3-Eq|pBB$aC!h3Ja^!tq{{Bm=pM@r!nyJT;eKM0)=m-0Ez<5i)eaKD_>s@S8vg7yOLJi;kWc*3c_gnbUj=!Vdd#9@#U;l)>Z{OSe>d(60Z>D2!%+OcekJ0Eo&&$>Sur>dFhSJB%uh^S(rb)iM75~NYKWo~UK@!CzcJC)lo_;v1{Ks*Aji~C{2i3`cfBOjPA5m13ApQXBuiuydkAYOb{sdp&em>N90q-o|75y4h{Xd2DUmX3PgSq(#WaU>s{I^t4|7)iF>BB%)Ppa*xnq&SQ-}^Y2giU@ICu^P?|9s599Q}Qwlm8en^Y5P%STQa?p;7WF$s{=GDU&<-eM5qRrG)Z6{UsOm|D*0*yVFRLMbYp26=nCD3uP5DAX&0YDxamcHQl~rTfVY;de5d=90VncF#=419^<Y5e&dzT$^vBf%sqG3-fmQtm5<2C$au#vzcsW6&;~!qvZ$Gw`N+=T*PX}Q$76TX_!Y^MkGmc00%%TGbqBwrqiCG-40SsZTTwq?Xl%ZyUKVo0g<4y9s_q<Nm83Dk&$tbaSZZFHxAU0`Y!JACK2GO%br2h`;RyQxe<KZ0F2i1sJ?|j&dkXqbXuR%W^m#tL0r^%f1scx!@B!WsfB<G!ZZ3g*68vP$&f}381p#@KVGq{A!VQ(C;oE)eY+txF!o#{ECf9I14D0aUgoN@Wnq)1#8<Xn}K1A5TZzGq(B?`1<;&ij<f61%*xhQ{r6iv#=NN2_$yOH5YrXMvW7e`QkrC&O5Kz>%w6ks*JW7(F)bU{RW-xHDG?YlB`g(Y<5_fTjbj23qp@Z=WCe3(!1mW<#a$Ps;tDDp?GknVRPz^eXuGz95*G#=-l`(t^7Yz6QWfFB?TPNt3NtZ+^<Dw9XWXi_ZtlR>WZGlZCA&A|8kumocMdpd(16;(=8poMy_Y!&&@!;?m=pd((<{{_JC;tsllrl^#n|F*h%XRU!Mj@}<;%VDxW<!tWI-gX#ih$GqWJDQKc|4pFD1e;vk0@4n7YVhOEG!dBAn|S6Xw}6}$qlOffU)(t_Vrv>c0Puf5MenB9IqDq%2Nu)l?dhw(oxB#l5#ZM!kEb{J5Ux3<+JpWif*b#40?pwW+o|mC5ZfBedr>b5P3;^TA}eyGM!T6V$HNFNSGo&`^Z+^mBbQ$TOKSJ{2}#g&ma}fQi0Z^#CDHH4pH9w_C|?YAV(194lhF`oW-r6)Y#ouW^ESkB8Ds37hEt7#%*V%RRXZFQH<O4w%>szp#U1=V(=}e%M}U>RvUy%i$Jcpuo+BKH8p}e+0J;->y3FsQWx=m#jOk>22TYynv5&SEy1^s0zNpdAoIe1%$bl`VjlJpL!NI?oz+-6ACjnueLzfrWUiEl^GKIgX1r9S`$n1c`OnHVlE_{0xk!&Y>j8+i4N1DiTGSB;iOJqY>5w^IT7P+%w2G*#YwqY$Np)5Pp`Ao;j15N-<Co>lj{{7fGnMxB3h=E6(u4lX`#MbV2k8{Iw*z}%PkbnDss8t(Hmbo|IDL@Cgs9TgG9byhFL$SQ7<1=m4vT>I%5up#Ck0y1XzSvn@nlQC_8Jsfqt*=)a$h%+hTlceyRaPFPH?RKo+rMvnp)+J(_#VL<y}j-O)cWg-r)AS<;xHD{7wYN#-d|?Xu82zRazYqUCdD7~e2A>uHJK|7M&e^&De;Knor{9P5|#7A(L5h4fB>A|MMM-2?cNqhEG9@!$!v$Gd%FC5DffJ&?=-xO6@jp5r4|#)hr_`E7e0b#`s4WFaPS5zF<3;%Lp<RoRQv7ms}m2`{w(zPDLN)mFe3R6FS*oEW4bM`(9s6}T27{e!E!dr=>}x+awj?#dg^Q<68TZV8?>0?$B((y<3u*MAEDKni16YPnx4azI=IA}8cz|8>7m7(`2_+7pu>1<kYh15m&L5VxD>mN1F*VBK*UTYqP6>yZg;~4GDm>(jmz!uC7~8e-!bl(lGh0d9AUity^_h9^_)<Drd5DY2StKb93X?iiyRq5OXI+wqrPG7IY;OY=hImVXAdsE&(jHP0^H+5=<HoMjHc%~T-5mcmC${6JHC%jekU<ocg6XO_2_~xVZ}ez^I$ojBfK)Y2Yo7~uf&I3sM2LzxrFc)`k-lt#73*HO7U?6J|av!S5t77lEb9)r|1M|%-w>>7SV#}YKiVVh2cT*2MVi9?NFE`VgwK7`PU`fQP{S>P#y9&Q|OJkPs8pp*PLkzmu^JR-+%fcf{q&5FhNI6CcYGMLi*qJ+=imq_?0JpR(#=K^t3swl5WNiyD_-~R#GE!wS0wB1qoZD24BAqB#Z@^xO$lQSJHdg(0DUAcRUq&cg<(PZvY0YeX%hDmc8g}r9Xd1D*SG@v_{?D(t3Ln>3_}<kf^*sMR-GIVr&1sgM0%ZncnnR&dQ>?HAql_ZY(v{?3_*jk@Pg+cXfvWibI-Um`2+`KB$%N;;n%z4BZ=M2W%~PS4eAUK99Re@QZDp&<1$<>+#v&$*@-1wJ>6ChIfp$_MA37s!j(bwD&&V7!jrCO4I@}Xo1%^yoNUXr|6VXnLaE;&j3v8)p88SC$iQa^)KK&MP9E-s86^RNbJXci4DCO0gGw(^%9Lwz%mc$is?t^%g;OIo`f>5Idm3z1mYHcN0bq$*BQETrjeivvg@NOV4Z=}jHLVM8v*u@8jk!&t$QCmUe#q?Hh?xoeQIqA_K;`>h8n>?@MvYP9+0DN1`L1Hyq7$}%I<8)t2tWFu!r~%-PTc$!_vkjDe^0NjW^?ec)%=lv_0iTkXkg|My@PqpOAz{3UF(iQSDXNCfklwt9%T-YWYKCkMuvcY9%J|r!tQ&2u#XhA^?jjlBr!D{-f79YkVAe%yzyA$Yo`@>Ue<0Q_|S|Z#V$|JE8w<x73gJ|BmXr5nX`fwpKvvc*kcCDlRsb7|bGr&Di@RVq<rE17N>?^NR;&#XoMqJN^`%<)~0E*wkMn5eD19fAXu@0!{wY*;BOZk#^zr6xzK%{RF+>H$Yq2PAvv(SEgLC&1<mSpW|ldrwTkj|MuqX>mK<^r9Z-DSIy#^_Sm2??l+y0a%1{8Pf87Qzsad}1*Q!StiO{RGu;;(Gu#yiTRCX0J=3M}O3Zo{JYJo?eS7liQ_rFy|90qLmmUqV-7UHsE;jTj8V`-#xqJcmv~mLg9iN^2rx3<=PT33!&KD~-o$Tf0W6d&bA?7@ENK(_Y%%qv+3nB+kK!uma9WHF%pM1{iVZ|9Xx?R_((*6er{I(n9MJixEotH1wM^(Z?wFT*FY}k&n*H(E(*!~Fka2Ot2z;Af6@o1<Zrq3GqSseWZgvLi<Jyy7b5c}h*G927;8&+uH0vq8F+TW3Sl6zAvQ4t66Ru2LF{^Sh94>&SRzv5T}bo_5UZ#$tAY4q`OI%~dYzKF(f_cjWQe`v6?%jI1Gt;W${JS|YPK$M-3xtm<^B^qLMHJi@yHWv?kj3|S@#P|MUbc+`{JUDy*3;X0>O+g|XpdKH3`gAE);bJ+%e)6I6>KAa+Eb*GcD40WF3%^XJH<Jigw4_)4bHGJFNafN`_SwO7b|*87hIu~2zvb0pbfqR8&GPw0k9-aD`5%*ypN`KyCDC8sygz=6N{j>avp#4&hcm}6`vqXO?GzD5{}X-tgQAK4s3p-K6qm%m*iI<F(Q=`GzW>BONe9J0MULY?G-;GB-~7Tq40B=sEqAW^^yD@FLUJj;N;k_NH9(y=pHAN4lGsC4`6H8wl%5ESPEI3(F&gB4o9Hd7{~dMJCm5`wND12PG=<}^v41Afm;4T>Lw?a;ju#T&=J*joBO(`eyV&FCs3WE@q^5!P{l#Kl9w4@z<It0|sB@g08kC)mNLQE;w&Nz#WVs!52c5BhF8)H&*{T0gBf|L+KK_GZIMF%yj~aFJA)miS(KU(Md;B<`jf&|o{|F?dz~=|;X52M?KK0TZh(^}Y&A+|J&22q$e*`PGA^%5BYpG}GJ3WL6-L{%f+j~#p+eigei7xGrnhZVY&%}b_toM(aZK&bpX)_18TNk45!U+aG2Lk<$j_WBk26$lTl)x!A>&Eeqq81VW=@$N4v(S;aaw&7iQ~EL*kFG`wy)Z!}7q0L{BI9VJP}HnP;k_8uXgQX>nqJNNb21+zanY2I+8!#3#PM(JA=ICIA?x;tLW@RX63cHTyhyv26?uPQLmtLOxN#gM!byW0i9Ra0tycI5uqf_eFRlk({o9u`x>l%q9f{Mb^7BBLe?iQGErMeQTYH~4U5gSPIyc8=Ful4$lZf_yn|Q~yBVKmc&mR&eiEHR68rX$H@Wg3EE}Yb+8+SYD1ht_Z)1GrX7#>B;ToZDIZCJLEKzHyz<8=vl)FhX^b=(it5-?PNZbf{nKn@e3TNY=M!Mvye@ro<NN}@u<GF=GEL-U{B@M_3cRXlx7_gyohf}3`qm8SZ+H9J`^%}%@8(l8Q3E8nVi)-XCP!`EehI7eo)JY=wq3_qJIh$%>_=N%>+&@_?1P>$!`>-HC>VVp$%3;8CtyS3bEWzqB%Jmy8CpQGh#;(y>Bkqew&?4!s4X0*iQ1M<V*cmbCPj@dOlHX|R7shV3vi!8PZNMgcs#0j&f*h9&ERQC}=3bF8&(LzlPu0Fb;tV8jK#PCAnvipFv56;5O1X>wn6N137mp@D&)YfsCLI5h)Y3j};(%eAr2T^xCuL`tq?lsv?H{642*Ig}?AlIf0tPSMX4B-mG?%o&q0(X;QLuGEoB{04EPFgju038!A;|)foucE6Za4zSl62tGDQN4ZoQF@)O6aQo^&re#-z)Yso>?*r-*($vT#!o?phjm?%8|G+XJW!pLD?Y=FhXGgX`0)EQXsy9(N8|nxNQBl(T6!hKLRT1ZR^be-0`KY40z$*Tgy9jG^)To=?JhB9ar}UpOr{g$TRsCfMUw&Q4?GMqfYyYB=~2E0f*<OT5Q3~icW!852Cj-9;-dJNZ-DbM4d(smoIM5hu7vgc9(IM>uf<$y@6C>$5kJ1G8CAq2Vxv&}1UinPd;G#6f{~u^ZD=5~OI8PMmjB}oj{_#x8l%815Fs*vLNW#{vHSEj^P{-0I9(E~2~tdPt(;PI6<K+i&oLoFPi+@!TzGG?mK}QW8_Wj~VTlfd(_+yO4;tVe4w1Z5&=h%510PMX_UL9u9f2U0pwkT{)<F(ij68lgCvmRc^{aQ)pQw8S_!~l9rDeZAok2u>5Zgm<!`L+Ks%#qc%pRia2};P=GM*^+JmmXJe#l<J4jvuT8<iZEUq&jFM96~Nj_^VWWoE<H368RT7bG9j)=|<9bmEoMHJ%~^=w8~~w?ZMz*ktD>^Fw`DM6&RPGfe=4_L+v|!EdIphx4cCQ%JN;FnUfr02$eprJ1JrR1%M9RFSEvdF1uK!Yn9-%{L=jQCPf|q0*>>zq*d>lbVSe%9i4_!6Qlw_;r78<fLN6UIi>lDOuPsF;5a+riN3VN&dDW9+YuDRDaKaBg*I3d5_yG`HNqdfhc!}B<VoFA+j}iD={=h^5^VC@2AmV+#g+`<C>Z-63;pM$$G%7keqw0GAQX%<Res>mc?3pjL6?;S+c)qWl6iHwY1xW&YA}&dGDpD-V_z}IrJ*To-P-~XqZO_t@h5&LF@S;98xlBo?{?+Uwp=uW~9GdOdCV;iZFBRPwwjV`huO4{=(28e@2$^jDSj6^@2-?<8|}2r=$l*tZeENgb}C#O@(QyTjJjMLa)`2e1e{!Et#n&D#i|BsS>H^I-C$^2m7<z-b`onbUC4g0=n^8xBT~AUU`)ifG8-<hB@7E6#K$5VeG38Cb5vpVmiiV-^2`AO!_k*Td1K1(Hzaq*Oa#gWYrF)W9wS13~QBXluB_?X#RCRzd-*|?dm}H)x|}Dx2O?WOpYtnw~Zy_3(UC0pE}o?A{X9GrWd4cvlYq@mqlA3O7j`hEM)}^^0^XOQMUF5u*rcUsd2|p)UL<#DzJ1I0A0;+TCp8tGcVl#ACzkB09TSz46X!ZD0}d9mAsK<My?2YK~qB`M&L^MA$!p37vkh>S&+_IOM?@27?gzbB;YiqCrkauuU<xFm5ak;cq<36P3MUX0E59H1!24~ObF#HDGIulbp>s0SF$x`S3TEpimmlkXGF_!2~506Rp-&Y4!_6vb+^acJ8cg!aqyNCyychfQ%UaA>yP6id;F!*V}+ztV>fb4K|$M%oeUr_oCF9~sNvR(E)sI(80%xbUHnNDcYzMS!eSUsqb$DLYpC{2k)Q%{G^GQPM*S<o_J-Yq%SxLwBW)c$qQJ*KKho@FWtaB>!);&?nyJop92U~`^(Kg<10nnuek24B6VxKmxE15^6{xcu01L&p5AeI{3L|-8N_@2JDYTm5p9X*M+3g)ZyS>BTdbaq8#=j41ID8Vp7+5h`ebsJr;EXa7=shnbLTCuk9ic0@Ip&P&2-yvkO6w2l!!fqBh|qMcd4^Eb@8kae6vXHc$YvSF3*bO{TG>=Vw^p$+6&wt`^8~RVQ`523=A?Z$yp5R}Gm1W2YWZxVWiVUM{j8fnIr%0InMMs1wd@3(qQPL)eiA8_lBX8yDMbWYjq?IoTdltjY4kh*DaD`|fpsTIfOOi3mqah{f90d3I}FEfmwGjUXIe*D4*CXu-voWbA3l*Re;V?^vV-VQP_PbxL4zL8D~)W-fCkfI;kT$Xgjsa5cYfE?r-|hnv2<PY)gkTZK;hDK$M{ElZ)CWDtneLjf=(?}LkQqtEHa^Fk0B2D#FD4}s4>03$NWpk18O1Yp)mJEBZQ1AXZ=(YcBQ7lXFRPc9gPA3{`cN;H>r>F3)qPH==0^mvsnZGUZ>ZdW-Sx*oguUQH_DUPjeQ^RIKwF_5C97G0ftErX--Nj>tv~6dW0YWc~|RgSrV3HHK>V`?L}O=7~P6z0r5WKN{MM}@0>I>dgM+Spxu?pe{h|X9AJ&oF)Z?5qgmbVjW@h&R%D*VfoumTaOXFjz}e{15WzQL4@}=xv@`_oy6#v0+_*E+?Wi96us}Xx_)7tT#|a%vJ!<e}9Iu~6C}k3%T7ABp_!?8;-r-HhIs;upKb$^^8>(|-fr?N2%F1WBEO<<znIUbr<rH$$AAhM+NM3@;w$wy1eIY3?;DscSfk$*xKr7d41ZIaxB+)<*gA4bzz!YM)vABc$e&WB!*9-lgQd4>zVD#9XYCtJ1-3An0c4t<?)2Umj(O~ib8oIa3Nf=v7hsLes3Y1UC^GsdQw|8k&@1GZSPR)ety2Iin$e+v8mOO3Y1t5LwP)@n}?Op7VUxqnlmgJ^tk<2D{jn~U;K80q0Ym7_6kMZ`hX#itl>9IwiPAB=(#m$rw_)yB`^W|hXrnrn{A!#~*(#(N)iR-7ha}F@DtZyh<vAh7nWe5}oNRuH%=~EE5=wcwUN_eVnjBe_=N*ErIv<}$JJ6qiS4sfC3@mH5R64bcEEGGh3{Y{Z?@Hct@#PJTgpQ~5;1S{_o0i<PsD)M|HPM^~166F1VkroV9ro0|SBYh>HQB2BLv6oKI*5t|&mOHkqbYT{cOxg}!?T_g^Y*t$9WFMf}K)e_cks|72v80;nJQ3YfKux8G42fHO5B7x878))BVu1>+qKY8Pbjc{ltqB~Ces75m$<?DY*7k1tnx2nGpZiKPS`Cl-K!iPSiIozn2ADV75ZKh|HJGa-1QO+qW|VlIUz&e^QOrQ1z1qxo&`})rme@Evd9@=wtKbqV!&ss{tm>RDmxZuqSf`+454@mIWk{`<jAY2MzUGM8*y|I0@A0Xl(j?>(S81+mV?N;M6Zj+n^o!-8*uI0FAMA8NLA)7Hi<I=HCl$y-1~N9MDZ2qj)bFWyYCm=t*<B9F=FBvq=-Y}JVT1)wN}CLX4&#dIEZ=K|h$gn^X+@{yz6uLCX`<R|UE9OztZGWe_ij6Ld6_3sFf4c-zJ!99aIbO20r0N?L`WI*0GI^f5$dxFG(K3YaDuZcK$t<Rw38z~+H&t%><RKNVqG-7s3(fyy1k35aZC4ol<DiWAxOOfafIe6DAj|6BhkGF$Yn{<a*he1&(SOAX>{J%1wuoAfs}x}Tx`32&PRQ=Aa=-vC|6?KWeJV%N3JyE5oDgGnID-iWkQ{Fzguz+<&pF<JF?IsHp2q6+1Juj{jfWZ-T!VkK<tYg$nkJ>JJ~DhQ5>T9Gx#H|Ld4S4)#r(HM7(faca*3;;kmOqZrw)DacR-+`im_V2CTRiHM}pKVI3>VV^AQw#1`PPvB%mfay!M&Lta|g8y+@35!@>s3Z%=kB4E~11K-{SCPrP!q2ZQg1sXA)*ck7$4Gv6ImPQ%Qa>YmLS@(MsAABCsn+-Y?0|=E~n3+@I$Oqyx%%1V|aUAqW7I!M-wL|$F;W4r8&@>LI3x#XfSjGs@uo$yME3Rep$!P<3G>(1dzeE#eNE2q9ZkLp;GYj`bPy!&%*dw1!N=lHxR7pTam?<YcH69)G=)#J35|)E$L?@*7roOgU^rb+>O<hF*_tQxoiBF4($P4t8|BLIjq(nJDJ50SzzIsnsaumNIq^H^bUP2LGd6BgeGcPk%wr9zg#|w1D!Tc*&5sD%TRTkz$<Ef~FY=$}JkOV1YR79w`?~{C>h062%^AbU1$LiI!N6dJ9-XDCS+7>k)khQPkVTIvR{Q`Z3Q~1pW8%sZl#zoRAnzfSSn1~BlQ_W2@u)^O7siO9ALL@80mfEo4q7f+uuC;*^EJa?T@U4Tt$ZLe4449Rpe;CG4+Mw1+5)WF>J;z>Wb49Vyezs5hj%7Y@nO-YbP$N`OA3C4$D^IK|p^FMCbB7ExbOjxycgQMy@Iu(EeeO^J=gdk>He!#Hd0dRAeRq$IeZ*$t*_>mZscejv6lW{cEcsJh8)3G&L?C*eMghFB^vq>&^puo@9qK<s$EX@<MWZ#5OH@x3N#r6!-IuYHzQa-sJDN$)Va&UjZ}NO*bRmKszSGDRZ2dJU6+HE=<CbFkdIc_v8>%H!O4i|-!@S#onVz1Ow=I<#g0rMk3*l&5$=%wJeVO?uZxw$FWa~@#^#VAXalk<>oC?w{ycqWD5m1OExnfq1qG?4tJ<ZOj>%}(71ggLF3Qw%++SUZRK3=Q8H)SHMNou!9ETe!rrZtoB)~G(W<TmAuwlSOn+PZZR%kh-=*<&$G6RH1-IoRmAorMzMQLDjpZc!9HWPqgj=ucn1v<VX;pE0Caeffs8U@QyYe&h(JRKK>`SE+pYr&jT%@~6@-v&|!C|Bqr}E2V>~Xue~u9`PDyLB4vfst?b*Fmx_i)22XCp(C#VacQy}psg_On*(rtm&1Z1g?K9V!K4WNZ}%JY5psC(Hg1g@RbW=eld4#3>O5tWJG35?-B2x!(iPB(K-`R-0BnVxMoGJ7o_u2^?!{eas$^~|`-(zc5+16s3Y7b+(z+_XBh)X9%O8io6;>roVRXk;td=tiW;$%=3d|gVHhtt2!tO<wx1-0rb{e^gPiFDij@1(l2DY*nM~(``VszD~`nwn|!O*6L8unE<YH=rSUY{Sm=mVF$Qm3mSk%M|@*Iu_Ofgl4Od$iE)DD&9JusQqNlP5~l@@1;Ybd^Z=85mfWx;zof`V-&s?1nw|k>Gfpu6bcUKwT|mugN7=$J)>yj@RoidcZa>;1Kmp98XjEkFfVDd7&w9njJ(iAXJ4;ghJkPlBj_suO0^x)m<r+-1k5{MH^46*=*WoCJHuLduIGdA;hp1LW=_MntscXndZ(_eN<n$;gHejU-^96(9W3BA>~}PGW9r;rswiVYprrsXxNS?!4_-eDr2!jB*5DU{XW!~n_Kt%2)wdC%!4}#N-&j<7y5Rfv(V!cpw?r2@X48HLMjFG5a31+L*70L6k7k`I{TQ%j?k@?DtJRU6CzH|g*lb{z*$~XW?ajU4ZU$q%@=NnSL-#Bv=uEa)Q5vuR`aOeK2`eXbOe{0p+|(QlR=J5(uG+QDd`Ehh2mlz45n8KGT7fn61t;U3c(sKq1&0%fY;u1`b=osf7)nuHzk<#M$7Rq+&MrNrs}G|0>ri8bNy{#bG2GyD?Q%FVS;OxFHVBxBSNH8#j&Yo>Vex#_ohgT*JH`ejV7f$*PO2uFoU0jXg~g7J>Di)h;?r^ylc~02-9@s2U?{z(cW@sjI}>dtEg(NuG^jlupGl4-?piDER7PY=pD4$@c#z@2SExtM0gU*7>g2iz(p8^|MCH-YD`4VkUdBF3FrI%UIpHfuDEVpI!W}m{El*rCDEt5SuQ_K>aiqxgAP=jh73!^LMLgNU{tX##bREEFO-u!Ht%@%JteeRedD|3A>m`~o0VT&<dor8MaooVRSL#cB`ch?ZI`wI;(8Bw0Q5K=y37EF;`daiI6F&A|FZ;W4*K)Y)2Qf=ulq>VDLeWwza9~-Ig~uwn7ZXEr`tqTYQw~UGH;`-_Z^!mV_8)%WJUEonv&>AIIodLx5((x!#erc3ZZjLsBD@9P0E#(9o_xg$qr5IYKzkO*iC1!GDM(Kx9nuWL6m2^I|q{4Iy}7KHB6IAcP>XHT^(enJZHVT9Aiuj1VZ-JD06%*SDP^Pjs~Lv7>bs#Se6a#Ti-6zM`-;=%@^t@^|>>H*2?fhYNiW%I=$GjNMD)KeJw)>MJz<XbwQ5e+ZnPl1NJtu4TtmR8C0g~R2!?)UHB16CL4vfqclaXj9E^+;W=923aFXmLOVjy%5t4cxXLyfiKui?S1K7LQA15p+-FB|m3N63yrN4G1&bnu&xA@C#2`=|b}KUN$VOF1s;tm$KN#16XC-Ixfu>tFs%y$D-Klg!Qd(REb6d%<bof>w@moTAt>>+@>n6%jnN{n~O%uGb?yYje74%vKb)~lO5eU$-ooY;2dQcpeQ1Nb+oL!ZGgT!u^8PYX<nNN_L_m0$3!Ybe^S-k(+jp?Q;O1$45!{VAWbT1eE(HOJn3N$SwQChJosB$a~kHx_>{M##RtwQdZuTZ`MigS4YSMjQk$)!-{q7thF;)Z4?C|NJ(%LLI>n3hN_(`P=c93>A(H$q7bkoS{pWY(@2$<d^0Vs+w?<Y71x9E{7cG|sxsK3csS;Vba@ckpUn{|pzNJqfO%86iznNhO5ME@uJ>aAkns<=h}wqk;-Dh8a23JjiJ%3AP|S;7qy*<}xSjEmo8m<!0Quk;GN6rg|oNr^+O~lhkoGq@0x9Z6n&{u7Ix8Oi^BJU?f{XVMc8<<fs)GR|O$ZLsW;7xK--9=wIXueZ)gk{f}ahPJ?NPyFkhk3~3j<3_k(#y|Fal4E$$h7#Z0qZMpZd!mNafWi&ESbGOFHYFSIH^y~4~PtnPLeQ30v?Qudj_DjXWx5t;5$na-m+N0Bdq45V(q#+fg<m9ifMvM7q6#X(@{%bhBjxb;Q4)lg;($!@(jIP@UI?r06uf8FdZB^hdF8jmjjg-P#XosPlt=@;TlisJ3vv;^#z2jd_-hawK^qVZUuu>!cGVgI!AG2@w_M@|tzrQ*Cji3JZ;g_@H*C#z7HJ{*#5uho<do67yn(WdaN*P5LBfmZ;d7pmsq%k}fPY=J(QNZV)<lkl_ba=MYyx-d5SSa2}lN78XHQAOFiWHwIfMu-^Z9QC$|0FsVAvbQNuS3u8OPv58+G`*qBJWU_*Q5L<>v0w`NkT?*rUYcQRML+rtxhdf6B8V+rQZaD=@sQ1OPWdV&ASh8cK~QbF)DTj7oX!To4?CC9g-Z_a_tpXMNtuFwL=NaS?fxF11k)Cv$!ae<R{v{Q6g!DjbKS1Xvs3}ZW*=<cAL4!T59u-e2ZAmkZ;th>~!z9_)4D+zGUw5dU^wBB}yHFQ;4l`JF2rTE+2EC)e_>+sZS%bG~;fia8qjfve^v|Q4YvY8w?XG7$1E@oAHw=dWUY6*s3kNG()H6(>GW~33&yl<v2LLn<}UMmP#+4AzK7*ItuH$@u5J}g|EOJ*<S^5{(@J5eevvti=w8Su??{t+te^+J)PZJ{3ttUwY=9PF5{~<&(DGddS%Jkdp||hD~vp(5>QWdon)ex^K0f1K~%g$Rc1b6%7)qRuBPxI5=#n;+S$?<ke&fHhcjD?t0%xv&G2&Dp&L%15g+m!1r=_g??De{vDMekaMb@inHG!Dpvb;`Nm*a=MRe)(mDPd|fW~!i7`zn^75|WRx`)hD!LKdj?QZ+jVVR19)(dOFy6X<H$bew@pF`{Muzx2{pFVBx*>5hV%elPzOM6du$&)bRq&;I7YMiV-j<;MHEt6J#%8DSvDP-QVP3bUl2B`JitjFr5Oy;e*GpX(J3=->wBD&5)b;3J}?sm;QNK)T&BeSq|W{J3}gQR19Xz92l+3ln31s;-bNF+Eu)TF7sf0SWpkdr_bm2Oj@qmMEsZ8|$G^h9}S7k4w@owx>eAztANN>~fej#|y8KT7PbBy=P!g<lz?je02!eK%jiwm1Y%7?N7Iw9|=_$ek_608GVSlBoSi;2a!JoehPIlH1#!52FHZEmc1Ne6GS0-6C{&oR*o@?_mhHxPkuDz6Z1yE*~SN?!;-;EfGRsv}A5k>ZNQcM|A5b!T}k%pQ?inB+R-I%v*gkY+2Tp6Z#K=*(a3&7O{Gwv5+%hlc!6#a!Pz+`F7zmPBokA>O(u>7iYqNMn{=?9j6{o3QWn2dQAvW)OC4m**olOOY!uX-&1LRq(7jT4JAAoCYJdqAi75x&(I-Nq}t&bKce@IR!W^LXGE>EOv)N4mMb&N$PF+psHV|1S4TpNRn1_vg91iga!;P8xPb#$xoZhYH(IzxEXUcxy1MkLMA9r!s5R6Al*Q7(sMf$hi2*J6g%-8Erd<Njp(|l@w7AgS&RXk7SHfG>z%B#oTw&h5%u|%>htF}3^KRL&i^HR=N8Jhv{T(M!59$HVmAn9aNF^AH9%a?o5N`#h#h;n$vXY>LCL#h|X#*W3TeWd`7BqY;^e<^h{=5SaT&0>ri6?PrugjIVWLfhi&f_Z1L?|aR-tF}J5yamQPh{8of?yV2IV6d!8baCgEo;5D<{M+aWxrPg>6@PNp_2wfRhQkW!Y5y8;2_rbgUP$Jg@IK<;<7)^zN1qPp@5GcW!f>M*?rjB4vlmP9hDgbzT@nJ;Y0wgJQRA-fe=6)0T>PdtXzYywdrPP1YFxHF~q=ro-}uwvBN%k_l>@&(=wdQT@xnc8UKCg5p1~-@XN|o>Tx~v`>|V^=k5`t2T^wJA>OK{;MLT71?BdTaErLvodrneS2<u~=6<?mG@@zLhsE6<L}z)QmF3rM%+}t>iRYWn(G5k6NBv=+{PY&n2*gs-dy?*jb(m+Tr|+utm-J5QBo%xp^(^Zghr?X*L%2G|`-igTN-lLLw2lmv8G72h-t@x$vLqLSME)pi@20MLNypq94WvDM&o-Osr%EJ9;4oEb8LfOj8M+^^_W^b{iJgL6N~>6~<7vq)LH9Pv^PzM`-%xro-E3P1YrJ^joNxN*C~H=%o7J*IR7X1{VZ2nAw!EJDjK`F?dX&2}C3oWk7QA0(yQvJ)K{1{w!9=1#MEkdGphLoW?9%|3ofN6{kT5Z8ZuwC#6cx{2n%XJdc9>YCloHfeMUED@q2BtwF77gPfnY!;JW&IGOAFet7bef*k|+TE;clqh0wLuGui3CN0K--@rO<?c6l1AdRs5W-*M;#wvgO$~GWd(3wv9ji!40ywgW<)CAEg#)=y;TY=$iB1rW5d%{`@{lYWEr@qogS=kx}y<($F^`CeCNmIhlKwvtj{jyo#_M>gZg`;bN9`_RxlV5%3~K(u-w(b)nyFQw*b2<G0SmfuZ+yv%TMFzJUKjbC%ekgksN@=i|}9vO)c0SuDTg(XaixesMXf717;g|L%F~DeAf#q!gYvXzc4<G-vS>;hT46T+BoiKs)~|a2YU{+^rpRez!m`XT0qv`3+jNmWxZcwTG(1Vu$TyJgWh`!Ter)+rX6ceRD>mvnI7a##)Y7l-N$PZ$fvX-XRCb#btltK$u<D3Z2@*TE|Xws%AUu-;Jkz?CkBES10d3p6o1c2~@_6r=4)4`thtkfT_QQ{hAbc)ZW2TuD6T$5DvO3@`UOU=x)+S3nJ1G%2OJB#=4E9HymQ!lOo4Tri(m!fBFdqj&Aw|)m9$Eo>E~Lz+v&<gw8mF(dB4H$sGYH$pan}9UJW8O3n=sZ2FuUwUFX6l&@i!?=X72of00g>v(GBfxrULJmvDxwApm4&krM2BZRU@Vlo3Uut9IIzL7GD!ax7YCy2z&rymcVbIBlurkukD^ByRr36}UE%t3@_%N}%S8C<Y_3sB6(iom6(gc$5SO&5b@=qKK!z*UnsIhiDIS)+n&6lzngX>&d5=|%Ln{^!r*yoSEYi+P_+jRjV2AT_I;v5V<=JiS4T)TmE=@NvdrxZoBVg7f(~Op1N@Y#hBD6_iB-NyY*{xX1xT(E$}$sZW>Ft3KL1F>Bl~F9z^gPGj)t;ui{O*!YPBxB;AaHUPvxrBXyT=}Ag{1G;d4nN$Mj`nX&XH@eK{3R4>RKXX#-#nCHhf^dQ@0=f!BDUIG@Y4Ud%wf$=!j!v<NK4?riUPb&A<)UeRDh8LM$roS}M+Mn)nSkKFEDYTc+D3wakS@I-a&asW2-t}H??5se6*xAp-VieWHo{}G6Mdur#nfNdi`|{)e~Eq{O`yv&JS-o0@Q-A9V>(59lAlkfUvRr=s9lT#osVXVzw?7P!)SN!;KlBK>-jE{`^&{*R-{j#-rU^md?Ae68BDL94ggc1E(jJ21&XI%>=`v)>^7b^0QVYrbQ=tch7F4U<qe$aft8m6uRzavG(>A29Mu9r2gi^p9BhY6^O8vVMB;~tVkoJGwUCnN<FCh!_OpFl>iP47e)}LF?zi^(ySuyX7w7r&XBVyW{rz_StTnjUYaTrBH(M_*^8Cf_LH}ZqpTqwy+857S7tf!y+l(NuBsdZd2a$Wpi%PYh5cM&Sm_jmgh(sb@pz9g}<tc7GtPH>dGBI>_1GtWeF&)zJSzxMFAbRMQOh-koI6xIb8v64v>Qk!*Icx|UCK4iUX+8#7o2WT4A%}8sXGFg);X};4N-QhUVWQS}0U;fs`G_o~sLa${RD|{yIcND3!bD_})&+ke-eKh|#BJnHqVxQ+e?3B)Gsh#Q=n>t&g~kECl!**b1S(z{nB2PX?r$<`&6%Q|-$8$P8Z`oG*iYv|KU}h_Z^vQIY|RLbR@A5=-d98x=D>+?<us7+D7Xih4d~5KV!CWJh310;Qt9<*KAl`q&8XJSZhPmzJ;iWwex?K~g4#oxA=cAdh2YUEVoy>TkC6cy2S%maaH3!UOwI@9PEvi&KaUod%X2#8+}+b~Hy;J9fsm#$QqOMk6#gh&is1eQ@HXApr*QDR|7`F1!HfK&*&ZC^{r$c3=jU)9hx`2(2m2RIc=Bwp_hRowYwyLd*}B-<-@Rz>o(~3jzW;1D7o0R(Vb!^_c1xpD-^ZZOz!b?w971Py!S-l)2P_lZuNuY|%kh{!4d56fc~{-SQCRS8fUM0FWBml>jy2OnQIPshbO}RWb2R8C(U`z&6bcUDQqR?4=FmwAkrtjO3Gf)yP<h7!BNC}%d<`@fEs<-5-Y!T1$F~#gB6f+<kI3a<HZ2TYM26^UHpWYu$F@%jU+B=6W<U@=E07njPkpb$X|@<#>;ia9i<d;Fx6=I!2VU?VySifLlp*~Z2^Ym;Qotshs;kkW&+z4b;A{$gCbovCwxT%1sxt9|g%YR@BWrpkii^~$Z^;+=Y*YYIX^erJ9AhT3(FJ7yl1|^eE@bOQli}!kG{hAC!W#=C!FSyP?>Zv6<V2&`D5`6@&^IJ<@<QUC0-c;K7tyj%d=kPC`F!H0WL^=fWl=W<q&Ei&oU3fg47&HynM|LF5fcC5G*?~%>`Vinz|qO68Y|qZG^eXkL5+!Ujqez-LX7_<Ucn+}5g(`)Sitn@hSOIalYeF@``jI6za4QaL}Bv#A?I0Q!A5hl8|J|1>j5VSU0{xOL6#2+bqSWK!i&)az-d@pLQ<3-8!Q*}&jAnQ>Y7aFSA9(VN=nAxKK-@v{OQ}jlJLbl))(JEBsl2Mt4_W-T-Hu1pzJX37X9BJPv3J=N<Pbq#bkEUzD14Pu!C~`Uk&FoXz-yXl5iMZ<$V;?hg!fI->J%KGpg^2#0GA9YGaWU09o(v?dQE$r)MW+b*ZEOvs^zqwQKl>CeTw_i>&!W?hDEwqb9SVDJi5=dC05-HX-A_sGK0R?_$UESX`r)Vb%6C46nE@Mt4AT{@!l4evato0u~C$)?!M>wOFu#tu)&Km*6t<JJ5)2$$rdZpfaI9A6&Ngo^sPeq1+8(qu(9>SMT)f^~u@Ctkv8C4y-65T`ZSW#j-`@Zb&iDy<WY@#}{akj8iz=81=fNU!ezL+262AfsDQCl<(ZFq)by4EZ`Bjqw-@ytQAh6+UU8t+vpbNz3#nx^S<}t4DPAypotsy_jc<=%4Zuli0b2(c8Li9TyhlrKC~iMB$A$vRO?0W!|~b4`%jpn!9Q2@TBl$Z_i=QA{CPPV<NoQiEXef}VYwLhKa1o6MCzRFZ)zmOxRZ~RQ%atkoqRYw!?C~B(BXDOGcP#dN$qkp#~5v@+N7S*5d*Z>`2gs7$=J=`7I3XCF4&WszNWn5)m?H$rHUSJ`lu}PzSX(6{TG)k>AR21otm-L2V&KsPO#4Jx*mAnd#Lg8=WxCpe$E$`J32ibW3?x<()%s(*1g94xEP&ddP24hVTqY6Y9&!i`))cFYg5(a94Nx}x#K$Q^op5Qu5--PUgw_)n#>e=1(_0qZk<EHw{M4Acf>Kb>v*N%cOB^FNqa9v^FSSbZ?|Ijll)1$&J4Z+yF+3;hhA{MG65u`8Bqk~>)?=w?v`a3pulb}KOohGTjU3ZevtfbIpMOPIOK3nQ09i;YVUuqCAIe_@XyEJYxku)g)&~K!MHQ=?u(A1R=e4xk!WsZ7mH5q)9Jh8Pp8z_B^s*-LSq(bgX^^qZ~ptgk9!|~J^cVv{Oj@AJ0QHhcgJUcJNZ;&gBdlX87!tZ6C7197j@+YCm`HiJtDekI&kdVbvw6|mag{p^wr-^Ue~z(1I_ALvd?4R8q|B1w3>0edKBJ6<)PBy^0AVXN5Lv-#u}t{(@V62`z_Zm$}S_3PzebZ+PEha;cIc0sV-?zQsBY?hgGzDdb4U+mUk?Qzr?(@1u{}_Cr~z<A##4EiuI5zZPD@dbOiJ2e<3w3h-cCrEJBWkx=zr0msmfzz~N}-cFm9}hn0MpU}<;!f0bcoN0SRW+oMVFaBIV9>R1o^#G~Tb-rH@baFV+vYbW^_+ZNBCfBQ%8?eTlT8@8O5jHdefH30Pb9AsICi(>%(>*J*J%5<bpVY}k)wme|2t6~hRfR;m%N>!40nYYPZ22T$cx5?rz<5vpDZt;jLZq*nas!LVlH%C~Kib7(j8dXhGtvmYY&U{40sA~M`2<uSsS5$(^_4_4nCVS2xz;bor)NV$^x4xI`#R;5D{C6*TM!%oMapL47H)W<MFL}IeDoiynMR^=M<H>!-G+U|lNT2JGaNDWaI1|H#V1fiV;hqy2u#WH_3Qfpd0wIUx&4rGtLahb}ns+GZO$x0uS&oOSI-H{Rg6a#i@2PcWccizsebUs+F#`#lzDB{8l|Ibr!5HoRot^fJNJ_EbKsRu%r#CyL*cTx?#8w&CU+Cgd)_NXrMsOohbhS*pwvle(`gDD$mmx0><;hTW>EoTh<s?GMv|j@Cy4^~<TUCer*EeS;HA5myGL@BQ-lv}SblHarImLo}IzR><*QarKP|F{~@(U0dkK>2QF$}qcGF>q9mKDcku7aDxp|B^0r&skK0J*9ClF{samR)w7Zc<>3+6TP4-u&Q2PdG=bd<@(Zi%ZHcOh-<fJYqr7Q>+4sXo^5zfJm2OWabsHlTlk#cUN$k6w!9{B0A3(H>l*HI}LLOH2K>?A^dTK>r<6oQIf>ioi@Ouk9|PT@T*v~TgEaf5%$6}i)2AwKFBAFJdm}hFpI3Y^&)9ElXfd<x0Cj6(takpOXzl#wVC+Uck!F{{t+TMJbliO+r08I`fO*t`Lo>~{p{f__L+3f7MKzPN3*xs+$ja_%#>btYD$plN^InEvdHXq+ep|<?`lu8tbM>H6E%u)$0k*@in&@z6H?0y8ouGG8s<XnCj7_86X449r3mbou*DYgsKg-vse`6+D#rJ{uZueyP;vqiRr?9KU9&oZd{eW*`bA3fGY&R5D{(DCZdKO_swIIrbuF;S66s`jeW8Za)H*DG>CA8AQ|c;yX*3jp!%50yK$azHNXR5l`aj@9ccL??vbvizCFDu8YlguRr5iH%(%qHh_&cOKL1dZ%b7Pi5O_y0{wW)W>1t(4fSd^7Lgh@Ok%4Z9I#tDu=BE$(h1d>6acS+!xL$$UUj`0Tg$FtbZ_MSCEyRKdbHuXupQWmn%p}YMo*8C{NM284@>U-pJ;p&o)7z!3tt6KA&3J3K}i49Tpte@01R}z?~DuWF;1p>{Mlb^{OhRd{7jq<Jb>h$+oy4P%iZ29ZS$%k6{41fP}diwFCmhLv2_hzt_<Iq8`9Od_4PmW=!E$I5y@!8oa{08!>$O4QB!+$zE{{82Zv$KC-J9zN!<n8JETKc>+1z*ZNl2Ku{O-oH`>4`cel}5k`O~-q7bTn-@`^HRRivK23K8}=jwN%}%+pVNVXp6Dmn0L8m6XBXex)ZDDS}irdlNu4@T1x-9uf7K?&pNnP-X8yadZyRNmyFish+AJpGd_cT6zIfxySgo&(``%~h>Vz4DGRJzey@u+;Z_;`u1XI2AW%KznvvnmI~6G9jAJVU^-RUYP~nOyZ_Pp5&WK2FIhBT}jVp-Jma>c?vW29qnoq|{z`*;~moG@H2umL1%d9oksEgi`w1FAtt#|VhNZQ@_5afbPc_WM3Syanb!QTw$)v3u}S{^y(vfP{yaX7kAK2w(BS0wfcnui8Dtt80*qYGfaSOF;vnhuDR{i4Wu1eGX8tZ7+6gK}f>A3A(nA_={hl3)SfcU09xn-Kd55muDKE*sBfmGjNg+QbbrA~U>BKoWwIKS9si!eWL+TtbvKjnE~<MQ0R=Gfz6!FHy$NJaHdeIiB(OMllcgd&{JRE-)W|W`+-3g2;=hg><OQCj!(aPs~w0bH>zwviYWF{4$d<Q|?Zd%O}*mHq4}EqrsP@R#TTl-hd0DB%B4MLYU{XoZNrOXBieeS|}Ngl3K6_6FLeZ<j=7Ll*q^?QC(?jhx{y52oqX3nBgwiyK`Nq@=D+Yrc(lfnwZkMe0cNfZ@+!;I92Je>jv&u!5$c`shvkbDtnW<j7Wi*7;2ID;UNlOOjAe{V_mC>%cUcc`2}z-<)x}KTGC(I_$1mw`haJpD&sNrz)E<^=HTh`LM^Cho~XqRtLT7gY@0@LDjF)Gp7ohtH)FJtR7{m$t(c>2VR5D>Bro$57gQhKlJ3)<_oze{(Agh;(G{LFTJaIuMB3u^5kB(Ga)7Furo~6sJg$-C&(?DfoaNb$*hxl5#c!yUf7wjUBz%qRRw<Y1Cg@AhMfClhI9${H>o>pns|&4$gt;+m>5T4;y`8m=j$9Y-fRC73nv%<W!s<M)KlXwm2mV|FKhTRq?l;@b%Ico3_Z=V3sJ3XoFuS2wO|!rV5faj3YuUdae?sY~49vAx;{Qi@XLd)AZ`DaIVU6Y}?1Lt+QzWIj>LwrJL?szj>MnL-5dwbMr>t4iIc8*?qGy+wyj7fst^_<A0~<sMxFphoH!H*^9IB5pOD~RZebqQUYq??6FEMjNLl5hKdWF#{#Eqa@x5MdTEger4L#2f`1fuERg(3xPVFO$Lf1TE?g<fOQMsK0?3@hu(jV4)guc?M{yk;#zjBzzGi!Gnsu>kYE@Gyqf*wIxx5rgY{=d@?GY@)n)wyN^0L~ElsEq9J}eo3p-u&@baMIsi*wqj-Gvi&>=<oreK7#Q@8AjzPP93g|};0cT%z?+vM#Ursv8z8l75&heJVhJkrr+xdURLDHx3tax#D*020Se16_z(vKISfac4p3uVU@9^3sD%+N|t#z8*E^H*p!YMtZD>nlT8c3x_AdH_!%<cvp9yB6|(X2vV*PRfK;+?Y6P9%A_9-*qBB;RW#!l69STJG%I{b5<CRN=46CRa{CVS_*vI}BOMN1}B-R#FlaO!=v!-V0b<T3E<M+IAfEP<LBDX5|pGkhPB|HDkCB`Fyz78%G;sc3V?S)ay5pDa19+e4!HwePo&W)>ct<LkVR{eWXf_Ae-Ro5lXjh#ZbWg{aGEgD?8#;txuk%^55-ep}s+6QXO<UWOYvJ`}FSt{M$`B_2=~03;A~+{(TPr;+tpm*FODwfX`HV*wQ#Y8h0wtg+B*+MGR-^NaCUG5c4}pSBarM@*@dm*C}&mshN#=$f#OpThSh#>C))&eF$fyGnF$DHJWi)ed$ZW*~7q8*^h+;S%+0-v)HOFt~%w(p-#QOxYQWaUDa_3cHOMpo&CD3UC_#xZpOtJoC$|BA|b;>>u^ZVO7Jv*6M7^H!}1i1S^s8YmK>**dS7bUtIAvCsD+5SG^@MP;&-{9Dmh)F!gl79JXh}#&@@{PoK>GI-p6+s(RZF$9a**OU4TBeroN~SB*o6bpj>~1LnMB!yFCUbylyw+&E-}WI_Od>Y=N{g*IE|H(CyT~hw5v^fxD$u5C<*wg<g@aIl2`ph5RyedJgAAP%OQ#F4qud(3*L~q0g0)JF|InomiuPg)AOR`l;`xn~1QQ`aZapHqEBAR)YYhYsoMzJLx9Om#3cR?fni{+@Xo1EK*9_0P}b87oU`lNSW<XKymF;N;R&;*~F->({Ki^j$J9ot>b42J@<ZA(~6`X1H*YC0Zh~b!N|<cmu%-r4!9Bd?ngn3`jIiCYaIqB5>4A=9^h;mu1{XrA;;jk_D2Z&*W1<^ov|kNA!hT9szYCOu@Nk*GKA={JXeJ<RSi0O1DdIiL)n|l(H~Zq98J(}{hYFqlH+%k-#!L5Nge?F=9fd-nY&ztnL|^m=ZjH47>5jpMg<p;LnsGDvEoXF(~1j^GVkp~$2uLkctA*|vY$-smjuaTe~t@kL7Fz54rHDegJm9FEm3=RUZ_tnav!kk%`cP@>*KG-uTOv18L~>b;WkWNjrq!bSvL?_q6%X<wopGUtWp+K&CwoxYwP&KhqwReeR>10e#zL+bgOsrUms4+-n;|BU*!KR9L%8*i7cbG>lZMI9317go9AS+hqJ>7tWwRC8vaS)Rq8e`t!I<8k+3H=a6~5-R&l({T6=q`VIJChbgF<cL_VyXmA51XuL$f40_IPS;BwBe7_-A`;EAco#Htj~oz&3au6%((`B{g!n`vxlt`2h4iKi&zGsh5+TUBBl$hcIEPDEw)1U*V#YIR~jKBjMI4z?R7J<0gFFREGIEGqqUnWGsj=fzY6X=A8GpGSG+lXq<imqED|@aCoLpUTi|vTy}bM!5|cXs$1t<?a)^HuDyAtbAw!UHcPEoCjTZ0p44GJ@QnLRzbGF@p<<HilNO&$9H;j$m~Pj9E`SAgzrzGNA?ICk|Lk-JVwg%_&-N^9y8^69HcytneseNJCBj_JnklsiSj%~%JZ0E{&)k*)BF2w>%l>}#(ri;<*tf{g+YEaikEX_W?|x7z42;&n{4{Smb_gtmj=)D`w-wEF#LDkNIzL|yoX{o9rPEAan3Ok7ns2e6PBou<0N`FouZSv<ZWv;V~iwLWnR$?%Kjn65;(ijV#(>vM9n~%z%brvbOj8N>~(}WO6C}ak4a>#m{)*=Lo4{vtK$#Zp8RH}F5@s&_?wr7f2+F|9!~NbDFx-m!?3A>O%q06f_XYs?lk&Cv>Cng<y|dBM0JJL8fKHkR0eFvP_2Qrq#`@G9atn;tR7dR@s}RwjKj{|o{U!tSM-6qZrD?yCRQ3zbYfAH!(oQyuGGi)T0(U(|0r>jGzSr>r5o#o7ogg7KuH;5FlEBrb`)p5!k|A}8Rb(|M5Y|wLuqiuyDb#hbBOX`ri#x>7~r8LSeti_w7un&1kSi1u#ujhIO?2$0i)yKLf5lhgcYx!DU9CoeKT%L=nBl9Z%SXL@&ntdD8WlI)ar&tv6=7yaoKOvQUK~d4z&}h%3N-}^t%Wnpsn9C(`Z!o;(*L})R2l;0dC?qmebR9J*D)xc%Wd%$z>`Kp=#gm^-X+Em>!YW32sVB8RfR-GpE9>^rXm=7KfdthyZliQrY!BQF|gsxulHZ#Mk;8(LAfKH1*LcDn_-^YvT=y30PZR5gKpH4cYMBkNq7|j&7F9%r=&;N=i!&RE$lBTUM)YlP7h%iL1JmIiL+$2pd(r0rqryV|)y=@cBEO4hG8^Wgo_Gwp@j+O2lMpmC2MlDHXXiHOs8M*KWa?MSx_3v{Ls7zqj5?sWdCUumqVp&c~Hle~4I?5sEnn0H^D??s5uJ3{$k6?-)bPE9GTpfi!q5{@usRJ3;tg|Bw+~vM6k&-;tMMSh}{aWWaW6txi|)!c>ji&MZKY%OTD*_RZP27a&-$#})pe6sOwll%qmCtF%^;nnlvbRGi7~S>ohco@O7@H_fwhUWBJl+rG?O*Y&KqIF5H+yv&^U&hdKbsy9QsOlpHV&j;8p{=w*S7kpLOlTtnFlg9K7GX>}nPC4QNQx#=X%3H0-5Q^>eMfsL&=%u+7rBS`)O8UH-GH<WzP?$eonXI;NHI|91lptB~Ag-R?%ck;Rv!Jdh>C5JKlL+ng3B>->Z)YcABB3aWqE6(t>XR%*&AnZAb~r=_{F_aezm_XEWf(B?k>bDN+N9j8?&A%1f^<&259VR3y8>OqFm*OSI-xCyQw0Ez45=LGn%`9ql(k6RYt3$=ekxm$l%=yTV81LKZ%t2WBd30OI7O(8dU0Bcnv-iBGn8a2vzHulXkJzkuxA2vi%df9Ou$|9Uf=L4)tW;U-82{M89kw<CEM&nE<dW$NK@jlwW5P<I_iDb=-be9n4&#x&rYOlPB8?Faad3BOfA%&$liJY%@3Io)_28{BJC69rNI=~<<vB9UY+z_ot+%NQx87AJ3amM>zfn)TT<1?x73?9_}AO5eO39at>UKvd$^Utpt7uU1X8HJA4+6=tLhdh6Fu!B>i6c8iC~``+-1JB0B@|T_JIM?*DCeov|x{0R9Mr7aC&mR4T{Fp{&9TvzW4FdDW*eEJRCOIDpL@wkf|#=)ZV>$`!`~eUS|HQ-%n59x=}c)6TxIc72pl_oQHDDz|=~10e@wOn!|A-S#5gSI?@;3ku28y4ZGebBSTu&x(Duf9SO7@zRZJi>^CK=r(u~QF|u*0e%*`1`p0}cmV((P->b?EKh^<$`Opqv!0G@Xgxq2>RS=meO1&6Q7dxJdjKj^^CK9fM?Ah3*UbzziX%z#UE?|>Xwh*^83l|^Rgvh(D0fc6D2~^5}fXa=~0uoDaqDv_oRBbpjbjvkS$7SbE9a+v34ptV_7A><1!ky4cjG;^+Z75>G;9tAzqWHFneXU^l9Dp=9RyH7?lza`UJ2n<6zb;C~KeozsM;@xi|MBvn+EHEWc|#RG*SV>eJ=E<UrQ%jePwaW?Yen=r>w>Wsm6WAnKR2z?j+JFjYq56*<GeqwQ_z}uC<Q+4Ng3t7(OsZh6W_>&9$I?s#G-punr5wM)>`I(msUVyRH0^Gp=5T|;;~~V(<{f@8SN6i+BW%Y$tKh9t;)ty>ZxZk3=qFkecgvVa{?w5J<^s=%({PI@~x7|<n`5PF&~YhU&hOS4X4+&BKmYQjb2Sh=nsQQdOlv}`Ak)1K?@9-QdGtf&d_hJ1ub++04{szn20m+$`Nm6Rpw@^cecCV`*{5J#IvQ~<M+p(-u(Tf_x9xYb@oc7IW6FN>H+Evoi-nT`|#oH<l{%v4#cDJT@UjzW0e8~lo`sm!_47@<e9>rb&gh*K-Cn?PRRc#A^$pzf5!ogD-Rv-_VefJsK3niU+kA-r@Xa2bU16$YI$|%)M9g)R0u|a#+;mBDSp?IxeLA==I3|-(EW(2wc<R!-B%_;Sy5qZ#l2<uIe1mEFi{Pg7Ob0HT}^QU6Xx1u!a9|BK-IR9Nng%e_}rut$pW1<)K^<&?8|(l*{a4og6e1hr}J_;tUDWLZ>EqxJ7?AkgO!!KQ4ScV(8qnr2Uy&MN{yiZ93MS$a{Gcu8lO{odrtW#Awy2)H$Y;MZxDmaw&RsiMrA8nSEp#rHc!x+OZ?ydi)VzT+kngmy<bny-u(CJ`$|{2cAV5%<#Eu)Jgx_AWlcb?Yck|z1(vkAt9J5ag_}0dy&`i5?iW?RQkAu+0>gxvZ1R{axQqv~)K}+leRDPp6i1?AKEo^$H1T4`4jLry>-Z(7#u${8I0Mn(uo?q+i?iQep%!b)6PHxVkot(d4F^hT!-9P>a19OGWtS`sZYq*^P13v|yYHl|w)8p`#D)DaSo4y7!vcgW`2CjF=`i0RT(a#lC4SA)qhe(Ss^0;tNW}MEdM-ho73J~Do`A1d68odjUSI$EqGto-v|yOi2V3D_i?y>L8lI`4h=385htZ0B0o11t475+~xBM!Ms{Eou(bdbkWOQ=UQqX}=Ed&(!s9da(3M?MA%T9L2G!_!H%9R;~co|j%m1sHM9*7NGeX%qMlU*tB+i`;;;8LIfOXE&Rgs7Fkh5>Q$<hLc(8^}1Jw0zhtDS5LFw_8bG>?R@d;$@umaKJO#Ae<}|CZ%uKqfqr_ofqlLaV<0*(pj@7>m<!AkFlAsUeV!N>7C}~X@RaN4<cS88Y@I<m$@_u>1i;vY1-Yovi2u;TNb2kT+)dFbhrFf93NT+8h%y^1hKK$gCo{4MfiQ?ke}o$r!h4HyU_-VJ9nB4(CP}KC6xqBzdS~H=?KP*VQG;S{P^2_9Bkd8{i1JdXm5XYBNsfb9Y`2UXDllkw-3Jz>dFs!6~=wQ6R94kO(?r*GV&6uh6sega9dRqQ};t)U0E|JO<Fx;|GTW|iuRs3^`H{Wl-a^+VrwpdJ*hh{X@y^sYgpSg;R*Z(C^%ctsn^b@{rT|Y=wCTzm;%<UM%>GXeEu49e6%smGkHk8>VudmiKwU@9MX4$s;8udg)ctAm(+`IVOMMSm}pO>Q98ucKK;P`Zsu9Ls?==|`?TX}2?-cstTuQQT3f3S#@}pT)3UazuG=lC3MNzl7sYn2P1OK~A?Lzyk~n6}SF2Qy%x`B@pIOTJ?C>a27JVL1&--IwWk_;-OphPOq>qsLjsw{8^U)ZM^~#wujmFEt2!6C)97cZ~=eLZ6I8cEqBdwvpe5k<R!T;}gPNJMg<zhKu4TD|JZbdb_9jjpEMup_4+~HF!BAL&hZdOHZYjlOM0t&pi+#*?qiz49e9lxV`CMq)~{5tO!Is9lpOQPDFCj5eP(Zs?V_<E0io#*4}O%Im}k7~u&Ioy-#DTfhbWsaTa@Q|uK>89i9=bC<KwQS$RQ<ag^%wab?hiBzETml!H2PV<lX_h9T=3AP@0nS3@0yVSPZ-!>kZf-t{=FZ;cv)IR3*jY%;Oj_;WR9X+&l!Nl7U}dv;U@k45izGm`Q6o5w-M}<t!%MSh2jCHy!7k6><d<LU@hgVkZU4}Qti(++YS=pwMv7Luz4=b;KX@m|;An3|vjQKU{rMX~V6dbt8yL-vPl#FJCOc9E4rqLT&y<WR#Pt69_Vn}&YyI%UU*Ej{<)rt^@tgPX?D<Ypsk#m@E^?OFaeyScO`=aWGj$7%i9<)4ajR7~*f`3v=rK>{ajLGj+fs|-E4;NXz}x6au+@A)?Nt2<Aj>=WupSz8`kDK*8^gmoP_}403V+dR;y2zGt!DK%bWR={CgP{)bq@C?savSRBd;C5`9y`srC=diRHl=Ls^q=n3&fwW#i!^fi<-WQ)dC?oD+2Iuqo>g)Uw5;F8T*_1#O%{DY9Nu1qbE<I_LkGN*>wrvBH5q0ed8#Upq8?KRoIyD`0?Yvj;B=AJMyt?2VSxZJj}Sx-{LCE_iD^?25V%n_DmMdI*;+!V}AIi$&cPN<-sTZ`_tpDmHCUO8681$%LPX|(}(M^k<a^AGy5%{n#Wy$8qMOVa}wYT!2cN*m5a7!+;p4zdg6Tm({9-Clds8N+Ml2~(4h(1$NIY_*a%M00IdO&)nDuLa9j5`H6fmSj=?(Y;en#JMKwlOAQ5BkJSW;-Bg@Iv40%Z^6)^zeeu})-7vN@rYg|s|`MB?SEsXnjQ>>9W#;TS5@#ssg(TajSV11z+QXJQMGf5(h;}}CLdeI45i@S-uZ8gI0b!6ntB%A^YYj@B<>oP<MWZ^)H_H+SUF)o7q5a7Q}<2;TP*FdKK*JxJnEbJOmG>uRRsctxlko&dFGx)uoy}OU6wr^*IO;p_#<&hS9E^IJmc3|UL8x-nf)vBOwv5H*%-0k*!b+_m1yG6@hA9uliOE$i9|EWt}8SdLz)bdm?yfN?s0RjJ>yC;KRKb7w(M<V`s3Ad;ewty|R4#wBzbdmQ|Vo5jKAK9}KR=As^a+br{T(bC9aKg}jQ~-d+g45?X2`*bZnQ~qPtglWkiF5u~pA}N_X4->k%k~M!%Us*<BF}&qdCJ_V>DGGCohVJRxQvyFeB~i9M94eK#w}Af!6Zofs*o!Mgt9LwVMzTt7ClFF9iD_rl4CqLzu>h7fRuW7<<TXkfW`t23l)#0OgjJ!3i04pUQ{_f>wM`dod69dgjR@CFb&>HYDB5kWLH^(AJX-UplEWB*y55Z*dXUh2UdwK3uRprC&Y16{FPGi>SPXET=ecWf@seh4$R6TqemvxS_I-M5U1h|NmPZ2#VN=aX)oGP>w4Y5&I-6I7x2SlubUz<2@+TF!mR2_JMThg{5#fpXGO_!fv_l|;gnO*!^ws2P@lrsL9T((3D&jdKou&(45VO?PvEAVCej<%o0lr=SfG0}9Oe^!Ko6s901}n79?kNIHUO)m&vU9mM*5q@oJv+R8-{0sDMK3s4&1`$iQZUQ!vyw=w|Fi0xcyJXKk`=p(->^j6k}C|3EfgQ?Up9GdFMiVVB!rL$EMMe$ADFdE{S%G{XkUzhDLhndZdbGtICPz<T5lb4si-!C}A)Fnv>gzy_km3)S2CJvWkkrQ8<B-mAV5yEhq2r^AYJNyZJ)%8JrD+pB+V39*(q>YNO~-Gfd=5^*aOJAly&sE-jh!usT4-t-ZV)_{N!s@j)p%@RCN)!lUE2&e&woRF|GiKgCXbj~CA{P2b(qiN&|{XPk!eZ<sD|J)~si=1F8FL!PD^umI}%Ply1_i=3(fI6=9zZ@I~JEPD@|QC;JRZa^ykjj^^kzDd?0Svp*xNlC+u?^A_=V&#`j3Fe++C$`qXvlMGNP$>UQDUplI22hsSbW%`ZtaG%i7c(q~fYso}`4k>rPvQR)=*1|I-;Ges4u32$k}JAyxt?yArd@~}Q%V!kvei4=+e2&U&l1~@e~L>PEsSQ-6E-euCj9rsbU7!m^iYRxo*O%k5DVyyy|>A4BvmyLtxU>xe0sHWP_f9YnTFFShn!PYFSci@-IGp8tg_8vh5|0W#0dy$v@i;4VIT}^DP@?tcM}g|f=CTI&A@yo@QAU-Nl2#4i}AECKv6u;sH=-aNvsMnG`yUo)ez`d;4!6>{TmqH@iM#UR?w7sh$SgCewppOurvlBUS-rL<}SAez$gPkG<n-{-?rTJL^l(ObfZt?Hzq)bHw}6gACb?*vz=zL(|q>Cnr=CVBL{(W^k#MKeUb0fj?@C#2(2$onUP|l?V7mRWB}*=p?`V*)W*qqFBaArAO})6kHg9Y4wXm5mMC_2Rou@0p15h+>*jeV4ocK%X!~rax_jq%A5-1qE6kZsrH^yWlFfx_R1uolJkPIYrnIE8>3g)4EUQk1Qk8LPRc=CSmh9ZKGTFE1^AqsxRQim}xI&Pd%dUROIG0*j<hgJF%kX4Hi8{xdLi-gZ6Hz!QnFmN9-6_Yfk3XQxD%=!=PxSEr`R(+RexjV3pQo-<vo;@s^!%$-z<56$AiO|CHw2!nhDz84;qW5wqlvD-+Slr8QGtrAQNpB!I>lh%N<d8(Sz%#{+J_mQy*>U1=A+D_Z=xz!4vf|#K55nix>VIY8N6D70s>nNSmshim?SUOqKs>ESXEa95Yk8|4rRg0kcgzxnr?z6paUBu8|U>$y|s$74<+^Y`c-&OGA3V85!_Hw4JDBN7qnYA_YksT%@9+g=+}?J-_Eo_^C$jPGEg`uVNJa)31*P=N??e~7-(KS=*Qt6BfhW@k`a)-a1Ia#4$sYfvK>(^+*K-AR87G(XIEXdCIdv5_fXBij*isBKo|REx1uWe?1ozkH|p<<&4{k_+0Z?B`)GHd!0XQ{6R$LD)9kxtYvq!%J?nkx^7Yssa|sJy{34C8NkfH1sf{_ro3&4QG)Qnj0v{G9^)-dz3)W`6*iWmrU~kLe=`9ktg1Sf`k#oWOu}2mFV+Z1V1!!yrj6qKw6A(DR3pobi+%}WIt+2l;9C-kqSfpK%5+Hf~#bRE!U<HIpyX3G2Fs}OAjJu+V#+^-kfYi2G4nXA36MbIx=ff1J*_=Jk2cv?#X<$?CqR|jt)(5CansS5>C-*G|aOx(OFA((m_T>0uDYi$RnlK#w=;Z9|^z0+0<@?^mi|lw5d0m#Nq~!kb=D5ybZWZ-#26PLyp%ZGe8c_$@OLc!QAq_=HXIp*!c1@c1O_6v8%%TsAReAfW63t7gYd5Q(1COdB;J`hxPl0s^oIWrf8q2vmestw+_CiC;es4~v{62I2)oS#x7E=mBo556x2GK?inE!D)8b@c)@zpu1OU{7S`fTRk+q=73ZIO^0-DH53yhpc<f<o**o}e0!RI7KyR2H1mnz9D&uE@j;FWnS_yN(`$dhc<{XdTXo?xWUoSeSm|ROQViXCJ?kj2otKgZq&wu%MrNC6?Xp2IB2$zveeO$}`aAMGoB8`SPxEj?=M;=(N6?FJRtdc4O3!Ioy-Jp$oz7%OtkeZ}|l$2j{7nX~ZTA;cidWKO)&xDn+DDRU&hw9Z7Ygydh5IA(biVQPye_y9_LKbuA%i_O13F?JMso&Cc(&dld)83BZ-8w<@>E4juTqoyj3SYDXmV;xs#Kyy+gb+fC;wS&h0c_nrmEL@79sH4Mbzx2LcEcJjKmwu07kQ!lN&8oO2`v12R;OZJdvtOW6q|6Hqy{BE4oje(Ibxb~HqVXK|&+LiQlDibb<A)$ouNu7TtSYA5~g=kpWvL!(c<;cc0H~|}ywnisZvi0|YaE;OuykQ4A#7HGmk)&W8YGw*ESfMC(N)R1^(&@XwFWv3F9qLjs8+kzDzjfNIS}nmEEvr;UcJ;Z#OJJbsufIqsI#@P7ds%O<C`KU(%~)17v9nGU^f;s1X2B9@)wRxb)g?IGZFccmte`M)pnnx^vkJquN{A(RMJd<~mP%h81j%+Q?QVoi)?C)oz72x?IFOZ#)sq719LP7?rGnG80s$58Hsd=0^;VSQri6B-^~PJ(<=*V+Jjzt>p?#i=7Nh<+tKRG_lWe5k>6kO!O@c=`+3$AQFO528EQI-Qcf3$7Wvt)FjtDg7hjX6@oVM{d%_DwkGWn+u3*ZdoYZ*-j=tc55+uz=QBC}>wFtG{QL9^LZ&!I(qg)UKFCGmDkv@6`nHlYOL`o-wHKH=S*7!oY|hx;X^Y`aO|nX|Fm?bwQU52a?2Q#>t)7UJ#W5$OOZ%PrORdDLHExSi>sxkI&b{bV)h&=zUIO&8TG=Q{hIp}(mdX3v_<g!7tVpn|w~q`S@LJ<p(W@Q+kfIn=WRMzosR?<8$Ewd)Ct=y;!`=nJhp=AS#mA}OaF8|ZnY^J;9X6~2To-3%YQCW0r-yOK8MFi71Qxx|QbB<rdJAeyDqi!R&o>XAK`x@ufMbyQCm(F|nDrDRbQ*i=gu1>0^VHJKYGkfI;=TUE1iqz0T*`!Lk3#JpqEedAzDqohHAy8?_Iv{#+I_pl;&kd@FVkU@yAS5uozs*y=ez6Wse!L;C&mfG30$U;^9Xml*!huOm`vW83EPSM4|F)y-ueJ{blx@Yw79@VSFdX}*%Ax#0X)Y`7BJaZ~5&-}8=GpeaPlYYLd>NIB$BsV)Jyj*2*a|6h&n#*E5(r4JDuoIRIwSo0^TWR%?rOUGR3qD-!-B{PSet0cw8VmDIV%=u-hmaF-!KbSSZUH-=cR-x6QZNSImeLV3ITcwaAz4D9qJo180Z_^Obd72w!mv)-$wu!rc?d|{Ouz{_ck&z(ZxEegPj>BcIr%)l`x7B`VJgzEgb@#129fYat3afid=H4^V+0XTp$s9#%BTb>#TT7}fE*KPCDxkUjPv}kI_0)1xv0G_RRz(}SX<ewTK7Ny_U7&Do=S1+-O>+Allwc}<EH1=Z+`iq{?wfG9+nfBmb0IhPJ%U|s#L=pPpx!HgI*P!L)GA^%E#v;pf$6<BIw*K4>T#$9Z(?Cq4g#3MJJ&z`77Hg*8WQ8U9bAks_v#!61d}@zI}W0>Qm1m-CZ}PoLZ)Y-8w6Rb!*%@_r+bT?%N%zu2tv1)KgU{(zxkRzZU0|^t`5Y+fl3dFnoGt2t2=bT~T$<q<_XuA$=%4x?wo>M5$O(VVLX!R)SYul9im|J#GT-!I!PQfQ%t|k5>uTc7`r=TUsgpl{Hb}x^0Ooy_PGK5y$UTcT<^1NOi;~BzDTiu@C8YXc#K^GP#!|7oV7E{_MiNrecF2_hwag6$X3NZv)3G)3n3F;!^&8xo;|@PyxaAYF5*uUHWT=BTM1`1~y33{!RFSN_OCZdJD2b^grhrFRmdFYFO!BT|mfTZ62vFkFx!ea*#|2JX6f_pqjo6Wm+TelWZe>$d;}!3!fads8YKlfTE14M|;AKEvO8j2?m5>JuwuD%Ts%h{ADuE1Qk?jsBY}auokGg*(N}&lv6{N8<ZuT4}UDHtd){=cSC#f>O{1zPh-IF>!`53ytKP^UfN9@t(?rXy`ZmE_RV@JWu~dh0QvwcI!2|$s4zACQuV~XNmx(Qk@C{!{k^C4kmR?@GlmBCtN6(DOevpG@*2lumrNMJ3^tF#8EoqRp6oS2?KsRHz<AVZuejIB4gmro`(g_e_4ZP5(@}av776<aJ~;F*r0m<h{mgmboz13q&8cdm-aQw(<m_6eg8GkWV;e-ntP=C#dnr(G9>)M&H8t8ry%6A4QyUvO*38UCl~C_G<=Q12&Sm+XQflijP*G?!%ve?&qT??ca@SJ{w>e;OJLAFqlHVm7=$goDY9YvIA$(X_QVZG5YwjRU;W_r0PzuGw1kal4bC%hY{2<WFxn^ox1jC+*wZtQ=i;F3xWLFH!D^waUv;9(DKpr}%V+^<*xSsAL5S48}M|y6oL+x(2(n=+1b>Br^P6a<Qc8hkL`X{qf8hF&ru!7sO?NQJ;-rWE>JRc8I=9a1smo_8Sr45uhi^Bj8tChJ6yONxhmE1AsR-`0f<?9}=H>*C7kMV?ob!Fg07^p4<=k$lZf|ZU{0fS?!LFRS&S{1$a3G|;KZ}^IEdoL`7_@O74WVZBNnwx5*+_IT{P$X7q6cURNrV>xJf|yFQ_1f}5dZr+q>))z?kn#oeq;b7!DMVqpq;6WT!V9TftMj(_E%X~+jwM^BwuKo<*ypg>h`1^vrg1Y2UL%RB&FE!VZRn}ExmKw=?e*bLCqA+?2wv8iD$&47MoFDhd|P0qjcia#n;D`uzLVGL8_#$8;_{<v98Wapo$c-S=(_10|6)70(&OXTr@t%5J<NzQ#0)U`oDwN;8{_;+dLXY2>4KPBHCmt&Qy)lwtx}hUNS;;$>qPYaT<u{kSHmAEeZD$3W~1dL$JCh<$TP~?QukW@(zV=awI{E;BR}HGt2ant5;|uBkZs3P-9{D^YG;x-Zs=wkbfh&S`A)Q!q*>)bTl&}rWE?>4vcIhEVy&+%Mk73Lr|9Jh$E+c&W^{9z_ZLr5c|!J=*>a)M)gQvWIQmTaMJSMqF$9jtvsUlT`wzc;@_onkbIbG9JhWZP^+TjXzn`A{4G0nNGuDe&XQv;a2jyH)Zzp@ner!Ap;K6RP7wa++@F7jUr*iDmdEpuj%6PuH!`V80r8W(7_`O>WI__@s<C)|NH_WuM9&3Akw|giTw#^;fC91xhN$Ku-mAbx)w}~n?XWC;A;ds)p4Mi%}If*cIP8WAu$}*jn>%<I#pLY+5{Y_9MfF-^&zZ)x~i(9P=wxYouAtq`!i6iW(9EK$DQzeqW@#rbtRJHTbd~sPz1HC0|c2Vyf8%#Kz5{N?3mOoHhO|ie=m+*^vM#7oYZZOG2KD1lQBc&K4_es@~$%~vx6{{`29}Gq*IZx260XTsv>(A~t@+~;T60=r`;-8}T<YcHaG_X=8N3NHH*f<MGj(j$>QWsw|7*8n$t<Z-ZV+(bYu#;K!pw&*$MRI#De$w8vOAb`mnjjZ}1;~ecl1z!7cMtoVxqabgfM<T!t9;-)u=>Rzf{EL8LFTw31bYxh`CApyj`lI(dF4E3%20;m-QJZl94qs&pFM^icQht+wJkVQZcJ(l{0`LijFs^91=6ns2DIxihs%oL8>;kaJ_=XOWgu`X@|Sg{sxZA9JC8;7J<8~_hI-5g5+BP0DYfPFX86uUykgy2g;auCR!ZBhe52o;!Y6?g>9@)7E~A$nY}!3ETbgy4`9`dMk(lSuY<qYc$4SNO7QDWTV;5;F+;U4;SN8WUPg>O}ouw#-FeZM*!tz|of?Q>%g>?xrp|~$~1g#Xn>Efz%L6QQEvu?t##SA=)W3SdjeX-p_lP5BebWe7GHo4YTIUuDA993CjSWeKSbKM_X{`UIl1kOmL-7>6~N11e@Sna0e(2H=B*GCJ3lcs0f1l0aA7Sq+V%_Su1!Atc8Ukz5uDi5Dyo#PVkWjzpILEUYf2NORY#_yKh5ZCl>Wp89`q%|h0E#4X3hMI!O-lzdT>n|>eYm%K&${pI{jd>@TN`H8lQ0|(nH{Wl|2=u_VrE}L?I%_2%GcKT&tTL<3x^bgz;dLqTnacB~{Ub8&w47GTmhcmAJUjcG-LBhc?Z?|~T{j31UW(ecKJfw(S?QAK^L$zq>JKat7G2D9xK3e{l1P=~p)7Ql2_`ofs*GO5l5@_lBofY?{-V@3Y#%}hO9pkFGOSkUEC_i0NkXr@f0cCrBbte|D@i?0I5d>l0(hw$_|5_GjaL$d%76@ZX>R!E7O?_0wof`^uQ*FF04^dS03?&?4N}E~)=A5x^(9*5pedHc%-g%9<YQIVidi)0mQ2bxLin~OR*4UL29w&h-CXW`*T~yD?HK9D+i({zkrn`Fv)$gU9u8?)wgGM13SYVOeRc25ajWCaJU5o2B)Dhv?>6@EL;}6a{5IusgD-GzGW=(|UME;2w0>6icvp9vl&7okK)Vp>X*60p&z_ib_yid(HDa9TW(peHEf_gb4Z?i5^FQ3~|M_-%ch6Y0UB?C6sw5%|lKmhSq${-&vZ0!l>lUd6dIewfx-kh@-MTA_wB8_nW&^P)0oE4(*|TGS>0qi3I_Y!AbA-;^%dG9iB4ZLhR1{?RF?8>94S0C%+pylypU`B?Uoa{2N8A1bCV5nEU{CZ3{eaf=-?sc0+N4A%^k-Xv8K@f<L>n#0TR{TNd;8iV#zS>(+S8N$uK9`{9CXWqzZ6Zaa@BfZm|1nsqJVNBVC?mcyeq5n$@h*mm{~;fn6WIvN>POKhW}5*UXXpePn6w{WJj*Mn%OQ+;&P*647f0><wQ{Bs=D>Svh|fh@k{2O(vnUHZ(7+wwr)|K#?ip91f&gel|5MBX?39b+FQ7~vRrs8OR06ktgxrX0eKXKW8%EIP58zN8Z2LC)I-^xC}Ifqe7FJdC^o8`3~V*z$O`I%%y=f~$yoNNOJoOzD{OONz^u1)Npj%Tu5O3xBQpk9<WI~#oq^jGX9NQ1v_CA@e6KVl$+A6<w8O7tyWt*n0r#@m#$h*0<;#?l7-y`*;d=eyKLKBX2q6pAYQ|wvURJ7H3GG3?b`RZkiUGI^5xALEyQ@ki@b$=FZ8dfg06ZhR|8?VQ+09wIW4mB01A)_sa+k<Cgtk*$^AqHh5@KdD`vXk#x0VL}h&uIAhW~Zn*eXeAK(lgNtR<(FJd(XQqFYTg3Rm!w3X42HYEr6-5>4+xtr8yCL*}$hJ+iAgMC$l3G{=podpYE+x>mhMvC5<(gL(TwB+}RpH&-e(5G32lZma^oTW!({9A)k2-x=tA_(S2(4+#%n6C6%U@++T)7DZ9GMb_#49amLyW<(=bQsL-`@46!C9;@2pSN%H#Jca^Maq>P_L6%yvRAlv#iX&sDdp2%gEep(=ebv|1T6ZjeJ>I8w(t55FNX)`<{Ig(Q0OGlfhCvn_SNq$~JO-ltZ8@W4IVQd$3LrG@^oGiHLQ6?vk(eM+%ca4dLTab`#T0n>wM<;y$HqQsvP<T}|BFAUa89Qt|HJG{v1Nbv$eCcsZzi}lWfQBi>v=_~jfIa03dU(2QeB4L*Q_bO22&7Mnul=vNd3kYBFW~S=!eQXtyohywPItf!l`Yq401UjLt-^_9{JxmEBs)g%r<gdI1Xbc`XCh-NkhaSBRv%7qMMBdUzRhabK+x%4F}6P9ZANS8Vx{6RMGF^sa|iQ20dTiVU*(n2_$WmB&_o*1XnXbZVdm%_3Eo?%{r>O)Srea@ak_?T=pmojNxuec&P2n@0(>nfW6!RnVYktdngwJZ;Fph1M)616{Xucw~0yIVs*aK0#<70*}?*atNln7d(o2PaynbiWYO+Q8X73Ez7C5-4rgMHW?~LzZ@&M$_s=OTyhp8ViC(FuIccInMw)=*Vo0B+l_B_6jD3hIZI4otSGc;m5?zd$k$R7n6UZu=%4Sr^%1B>UGdbGb)&5Z6wN1-hJ*c9SJ<^Ba#~QLa9OZER*Q_Y>J{K}6l}J{}e}|ga(k^i9P+56=5@(IwF6K+&WGKe_l3>?8^f<I1@`3u%AP<S@@O@uOfgKwKcldD~R?vt)Kp@lmy_X5o+hV0U3gBjjWG0|Q(9bRHfPPj&RF$U972|!GQh)Eug!)B@%@54b))7??UDgQcuDn0pgHwLYCn9UcTc!NGI^5+@*LwDRq}DmS{aH1h|KoL?L+aO{cX+(D7Bfx$Ne_A@|G7YFl~pc-6~xYR81!dEH>1<acNe7bAK)%ncHQ&F+}MFn_)<>q%H_X+i7<>BbMgWk?b6d<E~ZzoEd#VIpz%>W`F;lo|9bjcITNh&&N%NGwL7{4<-HqV6Jh+qj*)(tu~^cA{H4A3C~NO-CRP7LmQY~&mt*E@rF5Xqp8QmxAhP0gn2%G07_dzUtpZCjVEW~quoSIDec>tN?-J1vIaIt+hDp%hzdt$az5m#$A@C--%KTqn;cA$;#)SBRA<*daR2lx<l|ls+S77-c5=W`VC^6m^XGkI}t_x%*?xur@1-lvlL1W?rb9l}<nkf7BMW`0JsYr9t1e130r3oU+^PwtK@CADEqbNKNFdadVt_VvD$P?jZ>12tzLJ*pl%W7qJck$M_qx-JM(1$&?kwIcL20&U-0R%*uXs~iCU1V->1&2}n333uGjB6<2Fr0f!#l^t`>TSv!Rv{lvU^gW2fxfUplC{k8Rg!Z~=t>H2f$>p7C9m$K0dUZ4JPmb}JvhKwQY!et3_>}6nWM^xBHXoII=bv%YZ;lnpr_~K(Pw5a4x=fMj=3T-s%G1c(qavV<ja9Bl7M{EA=X7CahXcCsj^7tcZxzI;2M$veRg?QP{MAOk4vE_QeLV)Guj;CpFdK}m!VY32gZ(BiCqV-#;PJ9GnB;*`g~p7kxhel!JtPe_EA-YT-=pi74lmug7!H>v9k^Y)=L(#R+#57nWR+pQ*~^6P^Ri3psOW;TTcN?VVVIspEFk+wVJ6YPeO4o@M$-8Es>=KGqf5Rv^}IFCP6XyckoU{CoZ``CCWtyxY$#5j)PHc>&GvxK#i5Q=CVN>1uD`LS#cWyuiouea?pC=@qrghtj)5aH?OC)>)H;z&ehZ5ZdcxFb`$Sb3wj5w(P^0M8w3hlgHx$}31#B-GNlBSMlzN;3-($Zmcq4R=uDlqcF50Hv;pZg6DP99+NQ5eUlQduy0CBK*(06r<$K7y^oo-WcWRYYKY@?wfK$qKrn{2bgK46$uSLXu@xasp3HVGjUFkb7MoI~}ijWzYCEg8a-st3$4e6VU6vHLeVA#FP)SzC0l^>>jI(JB!_`!<U+dD)B@f5f;t?Yd8r%+E#1j8g#0nw$JK8^fdnz;5mqrJ!a;>C1)&4JCLq@*cmg&}hak0c3Gy0&vsJ*iTPChO&oq$ts`asjDqo&I1=T%VS{xUyehbxH=WSYy^qf?MJ|4}skM&?&t1{{x+e+L8)!*9$_fh+SNd;*K7ov@;Ss<h3>;)yn;JbSgepRq9nL!wmx?t<hVMi!u=I`C>X91J%2tA~v&pezoL?LST{$R4DBB?1!j;*zLVN{`vIGDV#CxpHJtuT96-bl4z>_M>xk%i&_81Qv2~kyG{l_0ST<V@BDS({Po<b01A`h7s%>-o?OZ4R;n%|QT)llhGNxL%zk@6X&)r*=dscuV|uxEDVq4F=rvcDmZFDIU#ct<<KF=BCU>MDfopU#T3o8C^YC~%F<vOqBJW?>mi8=bAD9GSy06k)IKq9Gaxa?^IR~r~d`GO!{l2J5lbdr?jtcuBHHnT7<N5j81GoA@Ai^jZUBrELM&<Y-{iZ6y$TV<bjMm2i;(5?nOvO<5kd*yZy05K3W)!>oZ7xM*I}ZXeomA&KUKP+Jn@|oxGEjA*nRqN$U>&{8?D?e(8TFZb!8NNuTA3kG35H31pZ-0df1e{%lR9h){`CR_l;PPvJ$puf?bE*p_$&@#zAO@x!vs>pU`nb)x7e>}Au7SkDFx7UP@F@j800<MyA8W+2I<T+-J;N5pYGOqy-wZ{B(#2j2L+L!-xl`ruhFc|5KK(>9@pvQq^8d)ku~%H|JBhK{Gw%E_(2$|8_s2jpY*YxbKOH9zdpzBc)7{Bnju9LP`|CT8&`EpD2Z_(*s#&++F4N5BT9v8J5@6(#$MqtfSbZ7-G4ny0AmV_pr8meKEr)s-6Rg?A}OG*<D*P(bSk^Az3WV?1mPq<t<7<4<??ljWMjTo^9_MUnmQ3@tgM8&7ny0D2(b~oFgU4yY3Fq!4%XF~S!>_283C`{)`Pff*L&Um#5l2sGhw({FiZvvpFv4WPfRWh!`r)SS;yB~AG|t!_w$?gC$9sXo!e)bt6R}aY0jav6M^s1To7On{bm~(wO;I|Ak}l2ISQMD2(O)pFh?AdL^F#Tb!FH>$v>Zz9GZ57QCk;uxLLt~KbbM92wDF>i#gfyLW70ds0fC3-mYO-LV?j66-M}XuqwR@R)x@GyY=s8R<=qIu4Y=~H2Px1c01u_TWgM)oQUuAs@kx$hG9T7h|%ntkeaSXuI~=o*Ez|BAGzzK^H+BU-7q?{**YIj{)0?PmP&gg&(8oK8_rhaaW};h==dW0c6*mDbo44`0+Th^nok|BnPmn#_jE<UUBjj+12xWsq5APEg<?#tc+Hgqxp`Ka?qQUQW;3=*;T*e4b69nBw%gmImr~6zkdj#A&9-vo5@)WLFP^1kZ+y<4{AWdE%aDz)4a*hKX2r7l*X=sJ({NVNo+{mPJKi=Ws)z?G7odu_M{eJYW>h)jNmY<JS&vpMg)@a3M?3y|qJ!M&d-6c!FmOSo(jL-WjxgdDSG@hKS<W+U4$3zDg9VQQn?pxT5u|S%ppg=`nTVbAYodaRQc7lXAFC6&oqe^AB&KRB7WT2+&geho#G2uSraox0NjY3RxdU#M-2`EtsX2H^3n|>hQ>0UndRDil)v)|gt;dP|ykVR;lLivF*uEzmCY+<-*L%FGd|(?iCH1w8W8Xs8>9tOSE4Rh^&emNUy|aU+lgkVKEI9(fGpWf&+4;V5`FUM0oZXz0>BR+5BCQic_j+-0Snp$R`8}YH_CdZ6_(6OWh|r(T#HK6QoKe$Gy4yX}kBkvSJ@O(WX4;#Amc+=JRi$dwop?3YX6yX@eTPspRB4cHWKz^X!iAx+DnPIQ-`=imw{0AVzV|EKz9jM`u#)Cd+X~Rl2JK=~bdeU=2O}_C+u0ONsvu4g6h;1hIWy#Nc#|b1yDvp5OB6|wLvm)$IWj3-r}LSlboyGooOmnse<o(&1NMk)r%oBN*+`<W%}MB+@{;tf3_>-{^WmAtwM{y^GXiqx!JeqhpdP3b+5V}V<LVC~Qx$oN0V)osl}@jWO56GQi1{#hbv~?9Jc`Qu;jB83P%OxzMfGb3`B2&-b6FS-N=w8UJ~GIF1mHgILpO=hY@?c!>Bo8WMUjy#>pnvVFvat(guvtoSyUcly=D&DsqQX@{pxP@bDN#JMIgi-E+WX`{NUtUr_1G-N%bH!p}a5?nr*fTDG2HT1{S(-G`7Y4@_Je5=j1Iv=GPKfQ>FUs=%IMibD)aP>P$IFw(Zzes!N?RT*$Ldn=o@k78AfyrVd#a2`$*!gI<vo)kS(HLAb`BNSudj9TYIAHOXrV-5G)gqv8SJU4+u;cCeo7nf2~%MdSfsQv>uClp;y;ifuo&tcfa@De<&Q0#4l++=72YdOY={=PX{W)sa9A(J}yO-_3Y?tjs;87L-6QiE?S+r4Ws$MQcCU#oJ=_?mFY}n*E8KT{A|l>YG&0Omy|aEU3D}R2MK%g$3;6;i3&2FGf;h8Wj9JDg2Y9#4?j8vY$h1mTIjcW)|#(UXla7^G=q*2F?^B_YLMX@qDgyXGvfdFKm%9#B<pStGQ+lDyI4vHVm}6cKq8-9o)2dvF{!-LiBp(F1Jj3p2}88wOgbq*wu{&W8^lQpTNFvZei1G7GA|USIISk6=d-LL;E6&z4gs=o8uhNUdB-9QRkv-x%dv>%n;>K3L^_r0ryu(kP$*OV1J*|u96dMQtWXSHt>hka|x!FoAH<O`;WEIr6IUJgiJkRV=va}&huJq9nD!fSh+e%mO;cJ5klcH(sL9wm^<9^perbviFZwwX!VXJ#j=+V@NJVKC+^3k?G9KWf&XLAjpSiz=0l-VMr*ulXH${(LRl8ZG+CnVnLa-@HJ2`W)DZQN2AUbMDf^-*)FoRSGTGvD#{p=YjR`$P4cDI&C)jglFt(nE_Ku6)t~xae*1Mi~4@kTZU(|_<?dAug7zMjJk#|;VY?e1)7ah<{^RNs9$jpPZIxu<b9aY!~{SD9q5^ETBKHaY%(e`Y$>bwU(QRmfJdAIC*C~wNnIW}qis^9sfo?6{}S>5Hko%Q$mt=cCbZtQ4(27-p2f{cN_aQ@l3Kz2`EGKeyDjrNl28v2ve1u|}QK^rl36<USHs?!osh6u7DBgmXAsXWfolGrlWCT=5I>k5_^J>nWLy+Dy!!qhV7oDEL8amju+5`=tK;FzXtO)MY=0$x}%BtoE6j}&stGMV_%OuFTzqo^rW$T;E_T5nt~X7KN9e(&<>2QkV3-BM%yFtV5yk@N23$4{pyZT2?{8pjmTE*-untCXo_WW^GtYPeebz|jRsc}qOdzAT2(LYtkEErMVYEstc|s7a%((|7N5!$w;l{y6)S+o$#19;dWK>P`_+Wm^05p52@aIxN~U1J-&q!y7%8G+>xSOS^rk4o3))V`kH=x1V+Mo!$N8!ugRC>7$(lA4}PsgBkiH+3*AdB7E2vSTZw*2R?6<XqU*=4uqS*<NfXHtxL8?xj6W6TTRclB1=KyI@)24Z6H&;nN9N<tSDlG3J)nXMyCLYdBD4L#@l>xwbHbV>(#xg^jy_`drMniU3YR$aH2nyVJ0XRFcR34-{y;N4|BB)-mQoQX92)sT~6l`$hrq~kQzv;9ZZ*ntp;*C??_D<gjK07?;h?Z!BvO(k=hl)#?ZsqHxfzk!BpfRXh9^?MnIvJmz`^><Hnm9l<JTtZ6xW*e7P(5LaOfy>Mw1T_<9dg5q14UpbrMll=|J^e>9YO31ky2RgL*T#Bc@+LQA$Z?@;9W=&1NQ?zs4B2Ibh*Y?HqZ?g{-<@sm<xHyPc8uClC#tIVi^KiNCzd^zK#HBK=7mMq|dl^t(c-&FtndVCRzGWs2$I!(ON#BGp1tCCHIjaf*n`m5X3{9D0b=y$u@@;9%(*<;dq3x#_hJLoEytv;J)7HAUWulsV|Oi3i>O`ou(*+q}b&76~f=`nd8sIwTQjs*u*yb@$xio~Juh450+<?OhyQhFi5-%A}IwpL6(7A2-?B!7OT79-LBo1&3xh^bT?SiPhrjcSBlO`M_&sL%jjGYGa0!XJm=Lv?!#RR1PzLZ{*Qk_(eW>ukNWhM<UcjuUrqSQAUD5Ao}4#`y91xREdPiELFtpp#GJ&(BxIMb}`e;?6hzQ@cMdhJEjgIaf@C)c8X@k-uuTrb-lID}nO=6pAPZo~Su{V=P)xcu=AcMfb_`&)$>L;#RoUUGAQu*xh>`ykz{yl=#BvuD9a{4Pfe4>9P-ooORjr)NJXp=c4O5`3^^hp_g*m0p<rirOvJ}4k6kE8xh0fwEVhWs44*6$lXtJ|1FSnQ@!{A$HDC6?aBL7*QMqkZ#Bbe;{}xPen&DCV91!D*Fh}l0az2SugeFfp898g!zyy5A^BV`xALY`pY{Qnu=6pU9qD|ee>nzlVvKx39x2Vuyh4dsvDd4kFm1b-V(R+En0VuWMtma>55B`%DDs&N-4&6j8g22{^-hH50H~V*qBKy@g_i@%p1&3cydd4OmK!e(Oz`#f=#jM44Uql&E}3z8YrGo16bdKkRY<=1*oX2~!ZNi_XWQ3yOKHAoOhZH(=i0h0Q>-JzLtU;dlq_Cd9AUa@y(dMtMfRaOX`w3>D<R)-JltH1L4tf;Gp4Iuc0R(vj+qpoMpsdd-_Jk(wH4_L`9hJ<U)jTFZ%<C2*1@`eoz^z&@+_D(HiLD81fUtgct38#F+a6R4zLbX#ez)`bhmw5g-nU!fIN&WSv_-cD08A5!#P#;xG1dT7!C?&K_vu@UY8=K7faX>IQth}!kR_'
SUBMISSION_SHA256 = '8f03b16618586e1d9a55c38d5232ea61729b82cdc5abcf383cca73227d4875b5'

main_bytes = zlib.decompress(base64.b85decode(SUBMISSION_B85.encode("ascii")))
assert hashlib.sha256(main_bytes).hexdigest() == SUBMISSION_SHA256
(WORK / "main.py").write_bytes(main_bytes)

archive = WORK / "submission.tar.gz"
with tarfile.open(archive, "w:gz") as tf:
    info = tarfile.TarInfo("main.py")
    info.size = len(main_bytes)
    info.mode = 0o644
    tf.addfile(info, io.BytesIO(main_bytes))

with tarfile.open(archive, "r:gz") as tf:
    assert tf.getnames() == ["main.py"]
    assert hashlib.sha256(tf.extractfile("main.py").read()).hexdigest() == SUBMISSION_SHA256

print(f"wrote {WORK / 'main.py'}  ({len(main_bytes):,} bytes, sha256={SUBMISSION_SHA256})")
print(f"wrote {archive}  ({archive.stat().st_size:,} bytes)")

_check = load_submission_agent(WORK / "main.py")
_env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 8000}, debug=True)
_env.run([_check, "starter"])
_final = _env.steps[-1][0]
print(f"packaged v38_low_pressure_opening_20260913: status={_final.status}  bank={_final.reward:,.0f}")
assert _final.status == "DONE", "packaged v38_low_pressure_opening_20260913 did not survive a full season"


---
## 6. Plug in your own agent

Three ways, pick whichever suits you. **Option A** is the one to use if you are just
forking this notebook to try an idea.


### Option A — write it in a cell

Edit the cell below. The template is a deliberately mediocre wheat loop so you can see
the machinery work end to end; replace the body with your own policy.


In [ ]:
%%writefile my_agent.py
"""My Kaggriculture agent.

Contract: `agent(obs) -> {"farmer": [op, *args], "hands": [...], "market": [...]}`
Handy reminders:
  - tiles are indexed tiles[y][x]; the farmer is at [x, y]
  - the only shed-access tile while just NW is unlocked is (4, 4)
  - plants die after 2 unwatered days; animals die after 2 unfed days (permanently)
  - end of day dumps every inventory into the shed, which caps at 100 items
"""

def agent(obs):
    me = obs["farms"][obs["player"]]
    priv = obs["private"]
    fx, fy = me["farmer"]
    tile = me["tiles"][fy][fx]

    market = []
    if priv["seeds"].get("WHEAT", 0) < 4 and me["money"] >= 40:
        market.append(["BUY_SEED", "WHEAT", 4])
    if obs["hour"] == 0:
        market += [["HIRE"]] * 4          # four hands cost 7 coins for the whole day
    wheat = priv["shed"].get("WHEAT", 0)
    if wheat:
        market.append(["SELL", "WHEAT", wheat])

    # One naive farmer loop; the hands just mirror it.
    if tile is None and priv["seeds"].get("WHEAT", 0) > 0:
        op = ["PLANT", "WHEAT"]
    elif isinstance(tile, dict) and tile.get("kind") == "PLANT":
        age = obs["day"] - tile["planted_day"]
        if age >= 4:                       # wait for the full watering bonus window
            op = ["HARVEST"]
        elif not tile["watered_today"]:
            op = ["WATER"]
        else:
            op = ["EAST" if fx < 4 else "WEST"]
    else:
        op = ["EAST" if fx < 4 else "WEST"]

    return {"farmer": op, "hands": [["PASS"]] * len(me["hands"]), "market": market}


### Option B — from your own Kaggle dataset

If your agent already lives in a dataset (handy for anything with weights or several
modules), attach it and point at the file.

### Option C — from a submission archive

If you submit a `submission.tar.gz`, evaluate *that exact artifact* rather than a copy
of the source. This is the option I trust most, because it catches packaging mistakes —
a missing module or a wrong path shows up here instead of on the leaderboard.

Handles `.tar.gz` / `.tgz` / `.tar.bz2` / `.tar.xz` / plain `.tar`, and `.zip`.
**`.7z` is not supported** — `py7zr` is not installed on Kaggle images, and the
competition wants a `.tar.gz` anyway, so repack rather than fight it.


In [ ]:
# ---------------------------------------------------------------------------
# Choose how to load your challenger: "cell", "dataset" or "archive".
#
# Default is "archive" pointing at the submission.tar.gz built above, so the
# ranking below measures the exact artifact this notebook submits. Switch to
# "cell" to rank the template agent instead, or "dataset" for your own file.
# ---------------------------------------------------------------------------
CHALLENGER_MODE = "archive"

CHALLENGER_NAME = "v38_low_pressure_opening_20260913"   # label used in the results table
DATASET_AGENT_PATH = "/kaggle/input/my-agent-dataset/main.py"
ARCHIVE_PATH = str(WORK / "submission.tar.gz")   # what we just packaged


def _unsafe(name):
    """Reject absolute paths and anything escaping the extraction directory."""
    parts = Path(name).parts
    return Path(name).is_absolute() or ".." in parts or name.startswith("/")


def load_from_archive(archive_path, workdir=None):
    """Extract a submission archive and import the main.py inside it.

    Supports the .tar.gz the competition expects, its siblings (.tgz, .tar.bz2,
    .tar.xz, plain .tar) and .zip. 7-Zip is deliberately not supported: py7zr is
    not installed on Kaggle images.
    """
    archive_path = Path(archive_path)
    if not archive_path.exists():
        raise FileNotFoundError(f"no archive at {archive_path}")
    # Derive from WORK rather than hard-coding /kaggle/working, so this also runs
    # locally or in a fork with a different working directory.
    workdir = Path(workdir) if workdir else WORK / "_unpacked"
    if workdir.exists():
        shutil.rmtree(workdir)               # never mix two runs' extractions
    workdir.mkdir(parents=True, exist_ok=True)

    # Check zip first: a .zip is not a tarfile, and vice versa.
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as zf:
            bad = [n for n in zf.namelist() if _unsafe(n)]
            if bad:
                raise ValueError(f"unsafe path in archive: {bad[0]}")
            zf.extractall(workdir)
    elif tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path) as tf:
            bad = [n for n in tf.getnames() if _unsafe(n)]
            if bad:
                raise ValueError(f"unsafe path in archive: {bad[0]}")
            try:
                tf.extractall(workdir, filter="data")   # Python 3.12+
            except TypeError:
                tf.extractall(workdir)
    else:
        raise ValueError(
            f"{archive_path.name} is not a readable tar or zip archive. "
            "7-Zip (.7z) is not supported here -- py7zr is not installed on Kaggle "
            "images, and the competition expects submission.tar.gz. Repack with: "
            "tar -czf submission.tar.gz main.py ..."
        )

    main = workdir / "main.py"
    if not main.exists():
        found = sorted(workdir.rglob("main.py"))
        if not found:
            raise FileNotFoundError(
                "no main.py in the archive. The competition requires main.py at the "
                f"root; archive contains: {sorted(p.name for p in workdir.rglob('*'))[:12]}"
            )
        # Not fatal here, but it *is* fatal on submission -- so say so loudly.
        main = found[0]
        print(f"WARNING: main.py is not at the archive root (found at "
              f"{main.relative_to(workdir)}). Kaggle will reject this submission.")
    sys.path.insert(0, str(main.parent))       # so sibling modules import cleanly
    return load_submission_agent(main)


if CHALLENGER_MODE == "cell":
    challenger = load_agent("my_agent.py", "challenger")
elif CHALLENGER_MODE == "dataset":
    challenger = load_agent(DATASET_AGENT_PATH, "challenger")
elif CHALLENGER_MODE == "archive":
    challenger = load_from_archive(ARCHIVE_PATH)
else:
    raise ValueError(f"unknown CHALLENGER_MODE: {CHALLENGER_MODE!r}")

print("challenger loaded:", CHALLENGER_NAME, "via", CHALLENGER_MODE)


### Sanity check first

Before spending minutes on a round robin, play one short game and confirm the agent
does not crash. A Kaggriculture agent that raises gets status `ERROR` and forfeits, and
because invalid actions are *silent no-ops* you can otherwise burn a full evaluation on
an agent that quietly did nothing at all.


In [ ]:
def play(agent_a, agent_b, seed=0, steps=720, debug=False):
    """One episode. Returns (rewards, statuses)."""
    env = make("kaggriculture",
               configuration={"episodeSteps": steps, "seed": seed},
               debug=debug)
    env.run([agent_a, agent_b])
    final = env.steps[-1]
    return [s.reward for s in final], [s.status for s in final]


rewards, statuses = play(challenger, reference["fallow_finn"], seed=1, steps=120, debug=True)
print("120-turn smoke test — rewards:", rewards, "statuses:", statuses)
assert statuses[0] == "DONE", f"challenger did not survive: {statuses[0]}"
print("OK")


---
## 7. The evaluation

Two details make the difference between a number you can trust and one you cannot:

**Swap seats.** Player 0 and player 1 are not symmetric — market orders are processed
in player order, so seat 0 gets first call on a contested price. Every pairing is
played from both seats.

**Fix the seeds.** Weeds, shop unlock order and shop selection are all seeded. Reusing
the same seed list keeps runs comparable when you tweak your agent.


In [ ]:
# ---- evaluation budget ----------------------------------------------------
SEEDS = [9001, 9002, 9003]     # add more for tighter error bars
FULL_ROUND_ROBIN = False       # True also replays reference-vs-reference (much slower)
# ---------------------------------------------------------------------------

def duel(name_a, agent_a, name_b, agent_b, seeds=SEEDS):
    """Seat-swapped series. Returns a result row from name_a's perspective."""
    wins_a = wins_b = ties = errors = 0
    margins = []
    for seed in seeds:
        for seat_a in (0, 1):
            pair = [agent_a, agent_b] if seat_a == 0 else [agent_b, agent_a]
            rewards, statuses = play(*pair, seed=seed)
            mine, theirs = rewards[seat_a], rewards[1 - seat_a]
            if mine is None or theirs is None or any(
                    s in {"ERROR", "INVALID"} for s in statuses):
                errors += 1
                continue
            margins.append(mine - theirs)
            if mine > theirs:
                wins_a += 1
            elif mine < theirs:
                wins_b += 1
            else:
                ties += 1
    return {
        "agent_a": name_a, "agent_b": name_b,
        "wins_a": wins_a, "wins_b": wins_b, "ties": ties, "errors": errors,
        "games": wins_a + wins_b + ties,
        "mean_margin_a": round(sum(margins) / len(margins), 1) if margins else 0.0,
    }


In [ ]:
started = time.time()
rows = []

# Challenger against every rung. This is the part that must be measured fresh.
for slug, ref in reference.items():
    row = duel(CHALLENGER_NAME, challenger, slug, ref)
    rows.append(row)
    print(f"  {CHALLENGER_NAME} {row['wins_a']}-{row['wins_b']} {slug}"
          f"  (tier {TIER_OF[slug]}, margin {row['mean_margin_a']:+,.0f})")

# The rung above the ladder, plus one pairing that ties it to the ladder -- otherwise
# Bradley-Terry sees the host only through the challenger and cannot place either one.
if top_meta is not None:
    row = duel(CHALLENGER_NAME, challenger, TOP_META_SLUG, top_meta)
    rows.append(row)
    print(f"  {CHALLENGER_NAME} {row['wins_a']}-{row['wins_b']} {TOP_META_SLUG}"
          f"  (tier 10, margin {row['mean_margin_a']:+,.0f})")
    row = duel(TOP_META_SLUG, top_meta, "closer_cleo", reference["closer_cleo"])
    rows.append(row)
    print(f"  {TOP_META_SLUG} {row['wins_a']}-{row['wins_b']} closer_cleo"
          f"  (margin {row['mean_margin_a']:+,.0f})")

# Reference-vs-reference comes precomputed in the dataset, so the default run stays
# fast. Flip FULL_ROUND_ROBIN to replay it on your own seeds instead.
if FULL_ROUND_ROBIN:
    print("\nreplaying reference-vs-reference...")
    for a, b in itertools.combinations(reference, 2):
        row = duel(a, reference[a], b, reference[b])
        rows.append(row)
        print(f"  {a} {row['wins_a']}-{row['wins_b']} {b}")
else:
    baseline = pd.read_csv(DATASET_DIR / "baseline_league.csv")
    rows += baseline.to_dict("records")
    print(f"\nreused {len(baseline)} precomputed reference pairings from the dataset")

results = pd.DataFrame(rows)
print(f"\n{len(results)} pairings in {time.time() - started:.0f}s"
      f"   errors: {int(results.errors.sum())}")


---
## 8. Ranking with Bradley-Terry

Win rate alone is misleading in a ladder: beating tier 0 four times is not the same
achievement as beating tier 5 twice, but a raw win rate treats them identically.

Bradley-Terry fits each agent a latent strength from *who* it beat, so wins against
strong opponents count for more. The competition uses the same family of model for
final standings, which is the main reason I rank this way locally.

I report it on an Elo-like scale (400 points per 10x strength, mean anchored at 1500)
because those numbers are easier to hold in your head than raw strengths.


In [ ]:
def bradley_terry(rows, iterations=10_000, tol=1e-10, prior=0.5):
    """Fit BT strengths by MM iteration.

    `prior` adds half a phantom win each way against an average opponent, which keeps
    an undefeated (or winless) agent from running off to infinity.
    """
    pairs = {}
    for r in rows:
        key = (r["agent_a"], r["agent_b"])
        wa, wb = pairs.get(key, (0.0, 0.0))
        # A tie counts as half a win to each side.
        pairs[key] = (wa + r["wins_a"] + 0.5 * r["ties"],
                      wb + r["wins_b"] + 0.5 * r["ties"])

    names = sorted({n for pair in pairs for n in pair})
    strength = {n: 1.0 for n in names}
    wins = {n: 0.0 for n in names}
    games = {n: [] for n in names}
    for (a, b), (wa, wb) in pairs.items():
        wins[a] += wa
        wins[b] += wb
        games[a].append((b, wa + wb))
        games[b].append((a, wa + wb))

    for _ in range(iterations):
        delta = 0.0
        for n in names:
            numerator = wins[n] + prior
            denominator = prior / (prior + 1.0) * 2.0
            for other, total in games[n]:
                denominator += total / (strength[n] + strength[other])
            if denominator <= 0:
                continue
            updated = numerator / denominator
            delta = max(delta, abs(updated - strength[n]) / max(updated, 1e-12))
            strength[n] = updated
        geo = math.exp(sum(math.log(max(s, 1e-12)) for s in strength.values()) / len(strength))
        for n in names:
            strength[n] /= geo
        if delta < tol:
            break

    return {n: 1500 + 400 * math.log10(max(s, 1e-12)) for n, s in strength.items()}


def rank_table(rows):
    ratings = bradley_terry(rows)
    stats = {n: {"wins": 0, "losses": 0, "ties": 0, "margins": []} for n in ratings}
    for r in rows:
        a, b = r["agent_a"], r["agent_b"]
        stats[a]["wins"] += r["wins_a"]; stats[a]["losses"] += r["wins_b"]
        stats[b]["wins"] += r["wins_b"]; stats[b]["losses"] += r["wins_a"]
        stats[a]["ties"] += r["ties"];   stats[b]["ties"] += r["ties"]
        stats[a]["margins"].append(r["mean_margin_a"])
        stats[b]["margins"].append(-r["mean_margin_a"])

    out = []
    for n, s in stats.items():
        played = s["wins"] + s["losses"] + s["ties"]
        out.append({
            "agent": NAME_OF.get(n, n),
            "slug": n,
            "tier": TIER_OF.get(n, "you"),
            "bt_rating": round(ratings[n]),
            "record": f"{s['wins']}-{s['losses']}-{s['ties']}",
            "win_pct": round(100 * s["wins"] / played, 1) if played else 0.0,
            "mean_margin": round(sum(s["margins"]) / len(s["margins"])) if s["margins"] else 0,
        })
    return (pd.DataFrame(out)
            .sort_values("bt_rating", ascending=False)
            .reset_index(drop=True))


table = rank_table(results.to_dict("records"))
table.index += 1
table


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.4))

colors = ["#d1495b" if s == CHALLENGER_NAME else "#4c6ef5" for s in table.slug]
ax1.barh(table.agent[::-1], table.bt_rating[::-1], color=colors[::-1])
ax1.set_xlabel("Bradley-Terry rating (1500 = field average)")
ax1.set_title("Ranking — your agent in red")
ax1.grid(axis="x", alpha=.3)

ref_rows = table[table.slug != CHALLENGER_NAME].sort_values("tier")
ax2.plot(ref_rows.tier, ref_rows.bt_rating, "-o", color="#4c6ef5", label="reference ladder")
you = table[table.slug == CHALLENGER_NAME]
if len(you):
    ax2.axhline(you.bt_rating.iloc[0], color="#d1495b", ls="--",
                label=f"you ({int(you.bt_rating.iloc[0])})")
ax2.set_xlabel("reference tier"); ax2.set_ylabel("BT rating")
ax2.set_title("Which rung did you land on?")
ax2.legend(); ax2.grid(alpha=.3)

plt.tight_layout(); plt.show()


---
## 9. Reading the result

Find the highest tier you beat consistently, then look up what that tier does in the
manifest above. The gaps are where your next improvement is.


In [ ]:
you_rows = results[(results.agent_a == CHALLENGER_NAME) | (results.agent_b == CHALLENGER_NAME)]
print(f"{CHALLENGER_NAME} vs the ladder\n" + "=" * 58)
beaten = []
for _, r in you_rows.iterrows():
    if r.agent_a == CHALLENGER_NAME:
        opp, my_w, their_w, margin = r.agent_b, r.wins_a, r.wins_b, r.mean_margin_a
    else:
        opp, my_w, their_w, margin = r.agent_a, r.wins_b, r.wins_a, -r.mean_margin_a
    verdict = "WIN " if my_w > their_w else ("tie " if my_w == their_w else "LOSS")
    if my_w > their_w:
        beaten.append(TIER_OF[opp])
    print(f"  {verdict} vs tier {TIER_OF[opp]} {NAME_OF[opp]:<18} "
          f"{my_w}-{their_w}   margin {margin:+,.0f}")

print()
if beaten:
    top = max(beaten)
    top_slug = next(k for k, v in TIER_OF.items() if v == top)
    nxt = manifest[manifest.tier == top + 1]
    print(f"Highest tier beaten: {top} ({NAME_OF[top_slug]})")
    if len(nxt):
        print(f"\nNext rung — tier {top + 1}, {nxt.agent_name.iloc[0]}:")
        print("  " + nxt.headline.iloc[0])
        print("  " + nxt.lesson.iloc[0])
    elif top_meta is not None and top >= TIER_OF[TOP_META_SLUG]:
        print("\nYou beat the top-meta host as well. There is no rung left here --")
        print("raise SEEDS for tighter error bars, then submit.")
    else:
        print("\nYou beat the whole ladder. Time to raise the seed count and submit.")
else:
    print("No wins yet. Start with tier 0 (Fallow Finn) — it literally passes every turn,")
    print("so losing to it means the agent is destroying value: buying seeds it never")
    print("waters, or livestock it never feeds.")


### The checklist I actually use

Most of my own broken agents failed one of these, and every one of them is cheap to
check. In rough order of how much money they cost me:

1. **Hire hands.** Four hands cost `1+1+2+3 = 7` coins for a whole day and take you
   from 24 actions to 120. Not hiring is the single most expensive mistake available.
2. **Sell before you buy, in the same turn.** The market queue is processed in list
   order, so a `SELL` placed ahead of a `BUY` funds it immediately. Budget against
   post-sale cash or you will sit at zero coins all season with a full shed.
3. **Feed before you expand.** An animal dies *permanently* after two unfed days.
   Wheat has to be bought before land or livestock, never after.
4. **Do not hoard seeds.** Twenty-five melon seeds is 2,000 coins earning nothing.
   Hold only what you can plant in the next few turns.
5. **Spread your carriers.** One hand with a full sack cannot walk a whole quadrant in
   24 turns. Send several part-loaded hands instead.
6. **Meter premium sales.** Melon, milk, wool and strawberry all floor fast. Check
   `price_curves.csv` before dumping a harvest.
7. **Stop investing near the end.** Coins spent on day 28 never come back, and produce
   still in the shed at the final bell scores exactly nothing — liquidate.
8. **Count what your hands are carrying.** Wheat in a hand's inventory is still yours;
   forget it and you will sell your feed each morning and buy it back at double by
   afternoon.

Two engine details that cost me real time, and that the written rules get wrong:

- The rules say `CARE` banks **+2** per day. The engine adds **+1**
  (`kaggriculture.py`, `_daily_refresh_animals`). Trust the source.
- While only NW is unlocked, `(4, 4)` is the **only** usable shed tile — the other
  three access tiles sit in locked quadrants, and `PICKUP`/`DROP` silently no-op on
  `LOCKED`. Hired hands spawn on those locked tiles and lose a turn walking in.


---
## 10. Turning up the rigour

The default budget (3 seeds, challenger-only) is tuned to be fast enough that you
actually run it. Before trusting a close result, raise it:

```python
SEEDS = list(range(9001, 9021))   # 20 seeds
FULL_ROUND_ROBIN = True           # replay the reference pairings on your seeds too
```

That is 20 seeds x 2 seats x 21 pairings = 840 games, roughly 20 minutes. Worth it when
two candidates are within ~50 BT points, because a 6-game sample cannot separate them.

A few other things worth trying from here:

- **Beat the ladder, then beat yourself.** Add your previous submission as a seventh
  agent — the rung that matters most is your own last version.
- **Check seat bias.** If your agent wins from seat 0 and loses from seat 1, you have a
  market-ordering dependency worth understanding.
- **Watch a game.** `env.render(mode="ipython", width=900, height=700)` after a `play()`
  call is the fastest way to spot a farmer walking in circles.

---

*Reference agents: [kaggriculture-reference-agents](https://www.kaggle.com/datasets/raykkretzschmar/kaggriculture-reference-agents)
(agent code MIT; data and analysis CC BY-SA 4.0; see the dataset's provenance notes).
Measured on `kaggle-environments` 1.32.7. If you find a rung mis-ranked, tell me
in the comments and I will re-measure.*
